# Test tất cả hostname — URL mới và chẩn đoán IP

Upload notebook lên Kaggle, bật **Internet**, chạy **Run All**; không cần GPU hay clone repo.
Notebook nhúng **1,764 URL mới thuộc 94 hostname**, tối đa 20 URL duy nhất/hostname,
seed 20261014, giữ ID/URL từ corpus 1.220.598 dòng đã ghim revision/hash.
Mẫu cũ của VOV/Long Châu và các URL PubMed đã thử được loại khỏi lượt chọn này.
Các hostname không đủ 20 URL có bảng số thiếu; không lặp URL để đủ quota.

Phần IP đo IP đầu ra của Requests và Chrome, DNS, một lượt homepage đối chứng/hostname,
và phân tích HTTP/raw/screenshot của các trang mẫu. **403 không tự chứng minh chặn IP**.
`IP_BLOCK_MESSAGE` chỉ ghi khi trang thông báo rõ chặn IP; `ACCESS_BLOCK_SUSPECTED`
là nghi ngờ chặn truy cập/anti-bot, chưa tách riêng nguyên nhân IP.
Giữ cùng notebook/mẫu khi chạy local và Kaggle để so sánh hai môi trường.

PubMed mặc định chạy Selenium trước; các hostname khác chạy Requests rồi Selenium khi lỗi/QC lỗi.
IP check dùng Requests tới **homepage**, không thay đổi việc bỏ Requests tới **bài viết PubMed**.
Chạy tất cả hostname có thể mất nhiều giờ. Có thể dừng cell và resume từ output đã lưu.


`TARGET_DOMAINS = None` chọn toàn bộ hostname trong corpus. Muốn giới hạn lượt thử,
đặt tuple hostname trong ô cấu hình. Mỗi hostname có tối đa 20 URL mới không trùng nhau.
Homepage, URL lỗi, challenge và lỗi parser/QC được báo riêng.


## 1. Cấu hình

Mặc định thử toàn bộ mẫu mới với `WORKERS = 2`, `MAX_NEW = None`.
`MAX_NEW` giới hạn URL bài viết mới, gồm PubMed chạy Selenium trực tiếp;
homepage/IP check là phép đo riêng. `IP_CHECK_ENABLED = False` tắt chẩn đoán IP.
`IP_HOMEPAGE_CHECK` và `IP_BROWSER_EGRESS_CHECK` có thể tắt riêng.

Dùng `RESUME_RUN_DIR` để tiếp tục cùng mẫu/code/cấu hình; checkpoint đã có, kể cả lỗi,
không tự thử lại. Để thử lại lỗi hoặc nâng cấp từ notebook 40 URL cũ, tạo run mới
và dùng `RECOVERY_RUN_DIR` nếu cần reuse raw cho ID/URL trùng mẫu.


In [ ]:
TARGET_DOMAINS = None  # None: toàn bộ hostname; hoặc tuple hostname cần thử
SAMPLES_PER_DOMAIN = 20
SAMPLE_SEED = 20261014

USER_AGENT = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/130.0.0.0 Safari/537.36'

WORKERS = 2
DELAY = 1.0
LOG_EVERY = 5            # Tiến độ mỗi 5 URL; URL lỗi luôn in ngay
ATTEMPTS = 3
ROBOTS_ATTEMPTS = 3
ROBOTS_CONNECT_TIMEOUT = 15.0
ROBOTS_READ_TIMEOUT = 30.0
ROBOTS_RETRY_BUDGET = 90.0
ROBOTS_FAILURE_TTL = 30.0
NETWORK_CHECK_DOMAINS = TARGET_DOMAINS
IP_CHECK_ENABLED = True
IP_RECHECK_ON_RESUME = True  # Đo lại IP/homepage khi resume session mới
IP_HOMEPAGE_CHECK = True  # 1 lượt Requests tới homepage/hostname làm đối chứng
IP_BROWSER_EGRESS_CHECK = True  # Đo IP của Chrome, đối chiếu với Requests
IP_CHECK_TIMEOUT = (5.0, 10.0)
IP_CHECK_BUDGET = 20.0
BROWSER_TEST_MODE = 'failed_only'  # off / first_each_domain_and_failures / all
PUBMED_SELENIUM_FIRST = True  # PubMed chạy Selenium trước, bỏ Requests bài viết
BROWSER_USER_AGENT = None  # None: dùng User-Agent thực của Chrome đang chạy
BROWSER_LOAD_TIMEOUT = 45.0
BROWSER_ATTEMPTS = 2
CHROME_BINARY = None       # None: Selenium Manager tìm hoặc tải Chrome phù hợp
PAGE_CONNECT_TIMEOUT = 20.0
PAGE_READ_TIMEOUT = 45.0
PAGE_REQUEST_BUDGET = 180.0
PAGE_ATTEMPTS = 3
MAX_NEW = None           # None: toàn bộ; 10: thêm tối đa 10 URL
RESUME_RUN_DIR = None    # Hoặc đường dẫn thư mục run đã có
RECOVERY_RUN_DIR = None  # Run cũ: tạo run mới, reuse raw 2xx và thử lại các URL lỗi

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import uuid

if sys.version_info < (3, 11):
    raise RuntimeError('Cần Python 3.11 trở lên.')
if type(WORKERS) is not int or WORKERS < 1:
    raise ValueError('WORKERS phải là số nguyên dương.')
if type(ATTEMPTS) is not int or ATTEMPTS < 1:
    raise ValueError('ATTEMPTS phải là số nguyên dương.')
if isinstance(DELAY, bool) or not isinstance(DELAY, (int, float)) or not 0 <= DELAY < float('inf'):
    raise ValueError('DELAY phải hữu hạn và không âm.')
if type(LOG_EVERY) is not int or LOG_EVERY < 1:
    raise ValueError('LOG_EVERY phải là số nguyên dương.')
if MAX_NEW is not None and (type(MAX_NEW) is not int or MAX_NEW < 1):
    raise ValueError('MAX_NEW phải là None hoặc số nguyên dương.')
if RECOVERY_RUN_DIR and RESUME_RUN_DIR:
    raise ValueError('Recovery tạo run mới; không đặt đồng thời RESUME_RUN_DIR.')
OUTPUT_BASE = (Path('/kaggle/working') if sys.platform != 'win32' and Path('/kaggle/working').is_dir()
               else Path('/content') if sys.platform != 'win32' and Path('/content').is_dir()
               else Path.cwd() / 'runs')
RUN_DIR = (Path(RESUME_RUN_DIR).expanduser().resolve() if RESUME_RUN_DIR else
           OUTPUT_BASE / ('domain_probe_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                          + '_' + uuid.uuid4().hex[:6]))
if RESUME_RUN_DIR and not (RUN_DIR / 'manifest.json').is_file():
    raise ValueError('RESUME_RUN_DIR phải chứa manifest.json của notebook này.')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Output:', RUN_DIR)

DIRECT_PUBLIC_PROBE = True


## 2. Cài dependencies

Cài các package crawler cần trong môi trường hiện tại. Phiên bản thực tế
và log pip được lưu trong output. Khi resume, dùng phiên bản đã lưu.


In [ ]:
import subprocess
import importlib.metadata

packages = ['pyarrow', 'requests', 'trafilatura', 'pymupdf', 'tld', 'ipython']
if DIRECT_PUBLIC_PROBE and BROWSER_TEST_MODE != 'off':
    packages.append('selenium')
requirement_file = RUN_DIR / 'notebook_requirements.txt'
install_args = ['-r', str(requirement_file)] if requirement_file.exists() else packages
installation = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet', *install_args],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    encoding='utf-8', errors='replace',
)
(RUN_DIR / 'pip_install.log').write_text(installation.stdout or '', encoding='utf-8')
if installation.returncode:
    print(installation.stdout)
    raise RuntimeError('Cài dependencies thất bại; xem pip_install.log.')
versions = {name: importlib.metadata.version(name) for name in packages}
if not requirement_file.exists():
    requirement_file.write_text(''.join(f'{p}=={v}\n' for p, v in versions.items()), encoding='utf-8')
print(json.dumps(versions, indent=2))


## 3. Mẫu mới và snapshot

Mẫu được nhúng từ `links_corpus_vi_en_zh.parquet`, không cần tải Parquet trên Kaggle.
Danh sách hostname, số URL nguồn, URL duy nhất có sẵn và số mẫu thực tế được lưu
trong `source_profile.json`. ID/URL và hash mẫu được kiểm tra khi resume.
Code chẩn đoán IP được nhúng cùng notebook và có SHA-256 riêng.


In [ ]:
EXCLUDED_SAMPLE_IDS = [213425,
 214251,
 217630,
 219040,
 219815,
 220312,
 225132,
 229499,
 230885,
 238312,
 246897,
 249470,
 250706,
 254687,
 264708,
 265237,
 266033,
 268911,
 281452,
 282916,
 704563,
 706114,
 706377,
 706678,
 706928,
 707097,
 707299,
 707479,
 708399,
 708669,
 708775,
 709340,
 709402,
 709448,
 709673,
 709963,
 710876,
 711104,
 712207,
 713042,
 5880584,
 5880586,
 5880597,
 5880598,
 5880636,
 5880645,
 5880663,
 5880664,
 5880681,
 5880684,
 6034570,
 6264823,
 6272144,
 6331460,
 6494525,
 6584929,
 6689567,
 6713129,
 6743359,
 6806142,
 6814865,
 7000989,
 7009785,
 7025795,
 7117564,
 7181339,
 7226872,
 7279666,
 7280508,
 7410454]
EXCLUDED_SAMPLE_URLS = ['https://nhathuoclongchau.com.vn/bai-viet/bi-di-ung-thoi-tiet-nen-boi-thuoc-gi-nhanh-khoi-53300.html',
 'https://nhathuoclongchau.com.vn/bai-viet/bi-trat-khop-lung-phai-lam-sao-cach-nhan-biet-trat-khop-lung.html',
 'https://nhathuoclongchau.com.vn/bai-viet/cac-trieu-chung-khi-mac-lao-phoi-la-gi-va-benh-lao-phoi-co-nguy-hiem-khong-55060.html',
 'https://nhathuoclongchau.com.vn/bai-viet/cach-dung-chi-nha-khoa-co-can-chuan-nhat-59725.html',
 'https://nhathuoclongchau.com.vn/bai-viet/cach-lam-toi-den-bang-noi-com-dien-don-gian-68993.html',
 'https://nhathuoclongchau.com.vn/bai-viet/cach-phan-biet-tri-va-ung-thu-hau-mon.html',
 'https://nhathuoclongchau.com.vn/bai-viet/chup-mach-mau-so-hoa-xoa-nen-dsa-la-gi-dung-trong-chan-doan-benh-nao.html',
 'https://nhathuoclongchau.com.vn/bai-viet/dau-tuc-tinh-hoan-nhung-khong-sung-co-nguy-hiem-khong-67806.html',
 'https://nhathuoclongchau.com.vn/bai-viet/dieu-tri-viem-va-cap-tinh-o-tre-va-bien-chung-neu-khong-chua-tri-kip-thoi.html',
 'https://nhathuoclongchau.com.vn/bai-viet/hop-thuoc-chong-soc-gom-nhung-gi.html',
 'https://nhathuoclongchau.com.vn/bai-viet/moi-mo-xong-nen-an-gi-kieng-an-gi-dam-bao-suc-khoe-70824.html',
 'https://nhathuoclongchau.com.vn/bai-viet/nghien-ruou-nguyen-nhan-trieu-chung-va-cach-xu-tri-51141.html',
 'https://nhathuoclongchau.com.vn/bai-viet/nguoi-tre-van-co-nguy-co-cao-bi-gai-dot-song-co-42025.html',
 'https://nhathuoclongchau.com.vn/bai-viet/nhung-hau-qua-cua-benh-suy-dinh-duong-o-tre-duoi-5-tuoi-41829.html',
 'https://nhathuoclongchau.com.vn/bai-viet/tac-hai-cua-thuoc-nhuom-toc-la-gi-nhung-dieu-ban-can-luu-y-de-nhuom-toc-an-toan-66186.html',
 'https://nhathuoclongchau.com.vn/bai-viet/tai-sao-nhan-vien-van-phong-thuong-gap-van-de-dau-co-vai-gay.html',
 'https://nhathuoclongchau.com.vn/bai-viet/tap-yoga-nen-tap-luc-nao-la-tot-nhat-57374.html',
 'https://nhathuoclongchau.com.vn/bai-viet/thuoc-kalimate-la-thuoc-gi-nhung-dieu-can-biet-khi-su-dung-thuoc-kalimate-69001.html',
 'https://nhathuoclongchau.com.vn/bai-viet/xet-nghiem-hpv-o-nam-gioi-va-nhung-thong-tin-can-biet-56801.html',
 'https://nhathuoclongchau.com.vn/benh/mnemophobia.html',
 'https://pubmed.ncbi.nlm.nih.gov/23227385/',
 'https://pubmed.ncbi.nlm.nih.gov/23227395/',
 'https://pubmed.ncbi.nlm.nih.gov/23227411/',
 'https://pubmed.ncbi.nlm.nih.gov/23227412/',
 'https://pubmed.ncbi.nlm.nih.gov/23227839/',
 'https://pubmed.ncbi.nlm.nih.gov/23227941/',
 'https://pubmed.ncbi.nlm.nih.gov/23228149/',
 'https://pubmed.ncbi.nlm.nih.gov/23228162/',
 'https://pubmed.ncbi.nlm.nih.gov/23228314/',
 'https://pubmed.ncbi.nlm.nih.gov/23228320/',
 'https://pubmed.ncbi.nlm.nih.gov/24991081/',
 'https://pubmed.ncbi.nlm.nih.gov/27550230/',
 'https://pubmed.ncbi.nlm.nih.gov/27634614/',
 'https://pubmed.ncbi.nlm.nih.gov/28382722/',
 'https://pubmed.ncbi.nlm.nih.gov/30191729/',
 'https://pubmed.ncbi.nlm.nih.gov/31177164/',
 'https://pubmed.ncbi.nlm.nih.gov/32269084/',
 'https://pubmed.ncbi.nlm.nih.gov/32503703/',
 'https://pubmed.ncbi.nlm.nih.gov/32790965/',
 'https://pubmed.ncbi.nlm.nih.gov/33372449/',
 'https://pubmed.ncbi.nlm.nih.gov/33452770/',
 'https://pubmed.ncbi.nlm.nih.gov/35135462/',
 'https://pubmed.ncbi.nlm.nih.gov/35211476/',
 'https://pubmed.ncbi.nlm.nih.gov/35347653/',
 'https://pubmed.ncbi.nlm.nih.gov/36187066/',
 'https://pubmed.ncbi.nlm.nih.gov/36772621/',
 'https://pubmed.ncbi.nlm.nih.gov/37323816/',
 'https://pubmed.ncbi.nlm.nih.gov/38192654/',
 'https://pubmed.ncbi.nlm.nih.gov/38206743/',
 'https://pubmed.ncbi.nlm.nih.gov/40386372/',
 'https://vov.vn/suc-khoe/4-loai-rau-ban-nen-tranh-trong-mua-mua-post973112.vov',
 'https://vov.vn/suc-khoe/bo-gan-tam-binh-to-chuc-gay-quy-tang-benh-nhan-gan-ngay-viem-gan-the-gioi-post1035616.vov',
 'https://vov.vn/suc-khoe/cach-cho-con-bu-khi-me-duong-tinh-voi-covid-19-879521.vov',
 'https://vov.vn/suc-khoe/cay-thuoc-quy-khong-trong-cung-moc-tua-tua-khap-vuon-it-nguoi-biet-tan-dung-post1244137.vov',
 'https://vov.vn/suc-khoe/co-nen-an-bi-lon-thuong-xuyen-post1102200.vov',
 'https://vov.vn/suc-khoe/cuu-song-be-trai-nguoi-nhat-bi-chan-thuong-so-nao-khi-dang-trong-thoi-gian-cach-ly-880653.vov',
 'https://vov.vn/suc-khoe/dau-mua-khi-lay-lan-the-nao-post946723.vov',
 'https://vov.vn/suc-khoe/dieu-gi-xay-ra-khi-ban-ngung-uong-nuoc-ngot-post1010466.vov',
 'https://vov.vn/suc-khoe/loai-cay-kho-ngot-nhu-duong-la-vi-thuoc-quy-viet-nam-co-nhieu-lai-it-duoc-chu-y-post1263715.vov',
 'https://vov.vn/suc-khoe/loai-qua-sieu-giau-tinh-bot-an-vao-lam-cham-lao-hoa-lai-ngua-du-benh-post1213223.vov',
 'https://vov.vn/suc-khoe/loai-rau-viet-ra-cho-la-thay-la-than-duoc-bo-xuong-ngua-ung-thu-post1129703.vov',
 'https://vov.vn/suc-khoe/nam-dau-bep-suy-da-tang-sau-3-ngay-che-bien-thit-lon-post1036522.vov',
 'https://vov.vn/suc-khoe/nau-nuong-thuc-pham-khong-dung-cach-lam-tang-muc-cholesterol-khi-ban-su-dung-post978352.vov',
 'https://vov.vn/suc-khoe/neu-ban-bi-nhip-tim-nhanh-hay-thu-nhung-loai-do-uong-nay-post972732.vov',
 'https://vov.vn/suc-khoe/nguoi-viet-chi-tien-ty-moi-nam-mua-tao-do-an-kieu-nay-thanh-dau-doc-co-the-post1155173.vov',
 'https://vov.vn/suc-khoe/nhung-lam-tuong-ve-vaccine-covid-19-khien-nhieu-nguoi-khong-tiem-chung-875456.vov',
 'https://vov.vn/suc-khoe/su-dung-hieu-qua-thuoc-nham-trung-dich-egfr-tki-the-he-3-cho-benh-ung-thu-phoi-post1012057.vov',
 'https://vov.vn/suc-khoe/tam-dem-bang-nuoc-lanh-lien-tuc-tai-trai-nguoi-dan-ong-mat-thinh-luc-hoan-toan-post1206514.vov',
 'https://vov.vn/suc-khoe/viet-nam-co-nu-hoang-cua-cac-loai-trai-cay-moi-ngay-mot-qua-bo-ngang-to-yen-post1191038.vov',
 'https://vov.vn/xa-hoi/lan-dau-tien-cac-tram-y-te-tai-tphcm-co-300-loai-thuoc-post1126708.vov']
IP_DIAGNOSTICS_PAYLOAD = 'c-pmGTW=)Cb$;isC{qZ!TRUuCc4b+^on5TBw3pCoNt2Z1fSR;=W_ma+_H>WBdw4N9K$HXsf;=Pw3<OSMz-z;C9LKPn*f6k*AV4VifAGPv^Dpw9Q&oK*lIzPt;>>hcpE^}_F5fwIsuvd*w|j{fAA7<(jJ+fgTRXxH2g2?5>DRyueK$zfg)8o@+!byRg`OBhULwLM6<(6My}(aKc!VD8xuLzdxVSKkqp|2mLE!aMKME7!k0()_ij8O*re3V?hy7_Bdtqu1r|A@(>t_3-*mDOvQ50;Rdi`k{#Vz3`PCpt?0x$Ij3#wf|Io4kfzv)l(_mP{70>792OveEp=TFB=`tHQjxBQ7a7|^Kog^!ac)Ze2-@;Px+BBz<#!B5>J&L|~O|Hw<TUuPM`)E{S!rc-|)X-?w+B<+bCC!QvZYad5`*rG4VB=FOPh4(ji_qMj5iH<M^(b)CFC4aIs@ZG~ON>aa{EFG^J3tKzR&hGa6TaVEgElp#Trjw+-yzEYV+n@NuGdqe89}c5<?4}(Wq0th>Mqj=+r<t{|x3Rsm$waZbUwrnf&#zv7zb`(SV!4yc=P7*?gR7T++aHPJtCv3z{gKx{f+Vy=|MEXYI=XuKm*Qx2`Mi(sUL+=Qbb7Y@#b=j4!2@YLJrn(_7eATU!dMXcpYddWe}{yvCGM@gC5|t@3#u+eDLnLu*3ZL3kzT$08LnQv{1JYP{cyy>2R`WHi(iIig6~|t`0rE%Q*;&bQ-32apHGPIAK>x+C=%nV7eAi1nBVXC7^e@u$K3V!^5;G{rGbBk8ehHqp)2BPD8kF{p4ph8g@u7P6ugj*KQUA9G_~5?c^aR!GcJzZVCrEaX<qFhatDcQT5G_c_Mw-WhCeYZ;SYtM_#qT6?0Y8PZHa;3Pc7jE5FWn5RJ_x^H%Z0&bYU}&qqyEXb!jJQ3`VUwk4hKNF4GGLhNHfkcuw|jX%f~={l*^jIuy@fG!_cKiISqq1I#*0oWmgMxq%!SKPKn*7<14IJckBg_M*X=nqyX`;1mj8Q1<%B(%7Tn(@mQmA^yWdNhogw+N5d1ut^V1ddNN{?Kh3(W$xKJfYmc;B@!5=c01`~9<UN*Fe`V`v%s^uMtP3Re3Dv{8;^V0A@Pa%y49{SL|obfuOAIO)0n2irLU0w#@-}w`<`JHCpPx%WZE-h<HO_*H4^y4VwNqsoX&pWCJ8j&j6pUYw78E>&oET)u;IIsA{0Gh|KO4#7kxMN4x{+2B|`7mi-jK!@i#O6ay=lc=`@kPg53d;#4?k|kR6IBR^Ozg%O^<MLztS07Y?vIG=SvDpG>@glX%d7e>!%Cew?I+RTHsuL$hYIh4E~A-`U^X+uwXlL?FYx0QT7#L&)4i4=uh5pXL$$km#rMj4@j%OnP15Xaw{{%BERlpBjPr%Ah=2PI=i-%ar<QK!R9$n6A~I?#0mw=G`gWHl$Hgqabx=a(8H*@t3&Z2Cu<^df?HG;i`QQg+n3_AL~bx7O#aL_I>#G6#6#Chji4iY~%spF=Kd`z)Nhod=m$n0N)*wH-Lck%!lo>Xc|M(@C~x{u#K-;s~=v3UV0M6N7jWu$*<$v!OGHG?$WSrckfsQBc1(Y*#$8onp~W?$pzdwF7-SwwD4&?#2sEvZ*T#T3+V%za0D0vXmCM*#T!^3_L_-bFyr%|r!Lg6YGay9^NY{mdR*mN{E5JG(oyuTUOW#4vka&AeSwxPyomLXeHArhcXa_T_=EAB>A3jczxdbR{NrDJn7qEVgFkra*Z=(aufOx>;79)Y<$wJ4dq2Lw|Jjw#KmE<We0uTgpZ(o$zx7`i`1OC^`qWYqsC2U|f`kPet%G{Gfdb&9J8V-tH|~$jTn1WNl3H|oOWQbrmcB1dRuq_;hBS|+>X|fLJ9n#Lz_ou6d=ttq-KDrMR^M1zVTzhQY#U_Nq<d>CE6U)G03f}X26}!3e|zG^rqve50`Nv0weW@bD9!<vcmf6R9QCABx$~K2`>8ihOiP-FLs%KsfWo60JaGm<r+r9~d{|LZjfP8OKlSPP0U0I3(rk$ioK8lgECis!bD1wfY<xUTQo!+XL`r)BwT2-VkXhFE(RA!Wo}%c;2c8~N(;lFG2$&8*2lmB;xfcx6U}EVp^dL<Q`3f>9YE7+$>sI0Jiduyld#}Wh{1hL%$FB6HOjEU1&P(%S&6^pP0`&Z6kp1%HkhHR41Bp+jFolDQLR^Di6w^Q?dj0?xnx}<KNfRS*2U_BDZ={_N;RxXJx1VlryyLvPxwrS|YnyUqy#NoA6f{4y@#N9dr<>2dw&`r|I-9$@_`6enR%>vFG^m-#?xX!p=jqnFTjT(1Pf873Z4t({nh8AG*Z|MY<IQJVB?{7~IruM^*P13FCP46V<_rV(P^r;D*ZMt&E1A9rrdL<y(0Xt8sY8zctK08AYZ|r;SqW6KkdYa=-_dvPi!ZONG}GPN+(jHFxx1O{T7~T0d1NWfanwX_zhjWw{`v-!fT+bT2hOZPiHGQd%uWG_DX+|oGyo}SU)U}tCq4n#IO;`d;uv%0Zg>0b?fpIHYY-eg#AJ?yKHJ>?V0-r+TClfww?Eh`PtB`%jiL$_5WjL$F*%a4dupz<MA^<SiB;HndRq3zkFG4tLxVYHZ9;;Yff5*0w-30-wq)jxcync?J5va{hfKx*z%+JG%s&1E9uR#rWg9`uq2{@SdX=MbPiR6$4Is9s5FXtoCqt<Q6GSbstVkT>k9RE_J}U~2J<}@oMK>vuioF1$atsI?<%IG_ioLr&CNK|8Ok>wQ5#BL@!@ftOO7NAUUWE=wq&VYZN5JYP^+*v(56X@_3qlRg9p~g3G<Vcxn=(7*>WX+Bi@0`Af7}tP*&xWxNW}d>)EdH=sOh!`K9H;i=U?j7COH|oYj1qnxcxvWZ?v5F-}I1DNLaMW(G~TimVfBsE)r5<8p8)=0>GCuNkAT_C0=iF3J4cE<{K>x!a}q)9e~Tu8cX)6Kc0>|g)!za*dBaQ;4P8%jNuxU%*kHXkxcPq;tp<&QGUYK17kT`noxkC2bRTPEXW|xVV3N11bK-<zpn%c{*LgH0luLF%)J2<?}wx(26<4-Hyam4$Z<(o^m<HF5aP&Gv4eXDT?w01q7URjwjQ4oDs#xj$AHJ4d9Sc$IcRNZK-JPBrKa0ftT_>-NK{wZ)aLWAK3lgzosItn=s&=8bo4~tQA=5dCOP)hQ={jO!tiOIbDj2vn|NCQoMGaVnm$HK$5c<2$!U!z<Ex@7EqTc@ZZ1>s*QgYT4DG{jmJy|H=%&%w?>oJ-)Jw{SMs?Sc8cs=9PX7=|Oe#%cXQcwhAUXDQGJv5q3$a8;uiMZM-N3=hce2G%Eez??2old*70pj4XDrw5PSd#Old53+&iPR_CeWB&fqt?_S^U5t<T~LoWSC7FLJNd}E~RG^Po`@i@JsG6RkthA0Di~r8GSqUUHX8RVn=USv+`7zA<<pJad%hN=+9mH^CtcIiVV3*@yFAQV7oR1NHf|<<>b^?woHn|LH>BR9<*Wg4+>h}wcQD&aLjXq(m`Y)G&v)U@Kbgt%?P2Dwi?l_PCB<6@}YVI%(EY=3}}Nm=9#YOMB0tI=pGnSKu1Wbhwj)jx&_sxi+VmPPpX`oCcMzly%uvlJ5JzZ-Lb$7q&voj)Z@&A3%<|kLT^BEWHEcV$sY_06S~&Eu194d9GQjE9S)6il1LLzTejoSXdP$P7U#GyGY;AS#of#qG|eFlJaEM%;hY@$7$lvT+>%-KC_rh-N<t#7a+Z>33|L3{(y`|zu!t0}IY-`^$}zW#PssdrV=5dLRZ2S`<!~AVIv7-WrtbOOwHbyliAtzQ+b}l$xo2AT$UD_ZQMq5`C>xu5b?jR7Jmu5nHcY~PeM<Q`I9ifyovzETbqLHho1*>*df_=eNL_p;+hqj&NoPN%fsN3F+_?HlERo4tyB<%#mi6T$IgEImZ5WB6>j%8kMuCL{OmJfi8zPAwW8`(AaM-TV89=ED8Hhm7P>;-GER+lg-CQoXDGI1<-b><kYW4^XMkE?$0@I`}W|{H{r%)J)z?ddvblO7ggthcW7+H;MEet(Zj>;?GroL2P0VAL4Li_yUOaoClD%(3nW)Qw0SGUAHW(Kv5p#2<&Mrq1Nayy|gIj;&&Ge25MdUO6q?QI%z^EC*}yyX!$Df=lsP1;dath(N4AsP!+*MAb@A<1uLy+%ImUYr@YR@KS^<HcJ(Y?L);$y>62IUDLeW{ozlOwE5wZW%I`>{O6sW;~~1$om0J%2Z~iQ`POpXT0#(kE4*D=@>`uAwaglj`b|Dv8$5O9w<c=#)%)^U6V_)Blm%1>tz}Wv1C!C1=DRx)6&!#mNsyEx23OeLzS=u1VQr@@UE&;7?i`I0R~6-hO=Z*VypW~qAV#*5T$v5#*`TfY%PD|PYIFGuf6;y@;3R8XWQz#HEZM$npR9+bPl?(r0O&2BZ%zqMzfTZ1~Ju$L2oL^y+4Wm3wrofdBv2L8h+}m19+sur=+u5dJg%Cgp3Qzi3Cc@-?6>7pKLy&9d%gx*Uydj!2Hr9GV2r>Wj0<R7Me;xC(_H}W_-y%mxGpDg^ty@McaA#k;jf-S$%8GS||Y;Y#kGb$+o3$*h6HTOCG<z9YCgsa$4p5OhKcmv(oD30Fdy)b=IeGS!tUsv&H!o4r2e97rVqmDT*vLlZ)6#x&gf@f~8DN$!O_tO7~n)CgWA&#e(~<BZ5}N@t1zRN-ykhD$;SVT0!PF3S~iHt_l0~&a2!=#TA<WSv`cPE<|}jSg=64dB;a-hHC0@&5!5E4FTGjVP7_g`cqmDn_E9D|F(T><i-26ciRT!F`D}+U|PVel5$wZC{;0{P`}g2_l=mm7Gy{yPHK4@6kenAit6^NYIo}~WqcH2)bBPlp7Ed}IM9Ek#D&y)eOlETVOJ+LO9<m2%T$50gqc!c^E_6DkyIn*t?s#V$_BAehQkrYf!2#IW40_KHmfj@RV)0F8OU_A%78581AZ8yJcCkzPL^ls>h0Ni)5Wzc%_Di$#2Eu#j9^YzoV6|8FL!6%9uLUq0zn*w>zQ+v0YVYDt8InG^e2<x;N;xMQUN;3m|f#?LgYtOk$_iS?|a>YmBM<<GC^8adH~fkkzx`XTlel3b3IBIxA!<x-KdG&ajtq)RYhud(7sz#GTEvevvd3F<93SLi(!n8r^+m-yyP5rN^yj#sxfVuuv5bY8f7eEF-TX*JDIRlm4{`3i&S<rO&UWkVuA4sZ+z%rP2!dFS=F>?Qq7SJ31lXWm7CqV4Jy!Uw)?+V#C1KZ6m;>3NlaTAA9V-tnkuJV;s`=0)C<#!3=@a>2?%J&mF2D>Z^abA1Q{T^M*0trq$^Idg<XMLBIc~1m9pVd$Vi-)jhk@voP1^}87S#BH>`?@xJHC-<DFHCyXea`0t%A_Rp9lG=b?ag8gumo49_&c?$3nQ4{WcCjWiCHHn8&<Qn3ZyA7im;+m6H!$;ODP9I|?9p%qpx@dWMxqo`M5n}|3ZorqGODgsgk+#PX1Pg4~l@|b4P8tdGdj6Br}`Jv9@bengDBw=b-EC4rEf7TpmGAP0#+18e`WLDC@uL6$>`;+CKEb%9_Qwf*6LK~D#4RDVx>s(jeTuk5%Qj^Pr2h;H+LG~P&fRiJfm@bSSiqp_VOX2}as>laaGu`EkAZ{>CMlw5C4dd$hW0jC<<#JNX(Wu3*!npFnskJH(SuIMRQn^9F0ZZu_P18Yi5}GHjpQ`+)Z0RUsl`-6Q6_p~XYr3}Bi#a7##9ysUB|2Gd4I5=LFL@B>4!!sgL$-sw^q|W*o<c2Zx42I)$<=BADz&Q{qhEdQaxwi8SFwMBD)#R~O{k2JtJc>u&#dYX15~X~MwdUDh$C6Xj>dv26hHHI$-Hf7M%)lt{eya4L;<o$KI_E__4+2TyPq=#|G~BTz02p3h%bMkO7jtnF01)p7pLBsHuJdBK4a0xK&OJw8(h8k;S>#O{1&FT(@;utGw*xb?`}F&FeIPadiyD}zkzvH%*Qm(d=z2r^mH9;>2*L&=g;`qfINMmMwLdF-wo*q!ncL?9Jcrd4bQ*)H#GbD#&T9NrG_Jq)-c>G*ITvb*G07|#a*+E5=$oRd}Vsbz{-BAlcJ|oJ*-Lxn2xknb0<%6D;Iz7y}g~y4K1y7(j(wgBB|htS2|Mh6JFa-r}Bgb&TjmeXYVsVpOR%RaN^A`J=ZB{jq)=mkO!PF`6nNYrv4I&nU)pm$c00)U+4GTt#52T*7CuVxu+L9xO({yevzfZsT7t<*?^C*XvSzTKdg$uQSpok_ENTX6knQ&?foaZQi^+j%7h<3+glz}k?^5TJuO2jm*qb4%~Hp`NGm2yA5KK&0D^bky<ni-jRNbOX=c={%o?4XaJjgP0VTl%Cz%e1{wX)J<#z_>>3CAC0m`10g+Rs$Z8Q?H2fXOWJmXU0Qzd`sop9xZ(Oe_M+;b{7Iku@3;sd^5%64iTq*O^q4XN^|H2o5On$0(9Mkq&;RK{(qG^m;rRo^xB15|f;bf~0K;03%4cbM?L;Ls2BMqAUqSib2|k6Y$%m2ot4-Tme$N0<Aw*a9&YLwfz7o295F;~a*{A(TC~OfAqBVld#N9pb<&RsG2QLga*7*c>S41$8DUYtX$WN>t?WW$|3{0sjMRJmbMFR<^PDw8-xt_9;SKh+eD{qe!Y%svM&%sKtSxS>#SH`vEuTHUPMKaT!dO6d<|BH38^O#nxle@M?wGwQ1Di0h~_L*_|6o5zAfZqev>V&?>NO%}rvKStuX2*?shZvq{P2XB(T&*0c8?J>9zfu^XMVD6d#nxuzg|6`!>-C)?4>B4I8q6m|j2D@Q5a((2tJ*DjTzqjjQK)in3y#4X6`+UN#S<y$P4S>;n>`RFaWLkb<oW|C2q)(KVa8M>~$9G&S*R+7)Kv?T>IGAn392@jW{a#m+4VRA9R))96dAnWp#KYoHq(P}g@MK0Lb6fev-I<$mb<m@J_FHI3fDY>S|>U3{KZyKs0i_~jC+}VESJORMfi1CRI5Q>OALl6puOPC6iB~LC9B*Jeg|BQs@#2<EEUYM1|dyLBGz(i?;OC!##iPZ$>e}{5Sd5EOYL#4pW%0=#`$$hlDzqRpnQ-kwtVu`cj-0mtzSp<+7cnyK$=-6^0I;vvf_1cN)Pe3QVdhye~AV?@zLWACxf`)Tp=N~d)m0E(6`=rt82!AEaXbdiRK2=IW1t&!_k}W~apyZD;lF91{YiW7%B{-{t4Gw}JxU4XlG%t?i+UOoA^D@t5=9v>Zdrxjj{U-V&n7|?86co}sYb$g*Kpsk1TcKRJYB<*#lgCLM#?%O29aW5=ax=UkG^o$tY(BYc3q?`lbdZYsns=vai*xNP@b=41v@`H`d)n+NapEYaSoEDtRvpHA6?$uLT}KbE54bhGs?{Yakm7St&veWeq^uIH$W6O!jBW)Qa!SRcER9yPRnpm8P;5YE*L$FrL~gerT`{!n>&@_7ddg#}(qNB+53<O(AivzKAGC!Cbh@sO^XojPI_+6#MM08v3$3`8VyTs+IMYc-jsfP?Gh+9{qwL3=(-d4Dh3i}(Rpl`QM46zKdCE?MA>ruo`5i8nl+&*h#B5_r>0@OC>87^GwVu@qX%pL)Wqk$jpqH><++<IhhlHy%@mZr?S;HGvu2oy28y04^aQ)s~CjN5ebTOm+Ro;`PTS4JD%$$=2FMN|SBI<%loMefbIo;uwp<?)!pi}8`g|Lro$&FY-84NiKAi9<7Os*{5`8}2xl|AP91MXD+)ct1gy`gJg9%9JwfZi9NOby|WQsc6iwdQ%qR>5bq3iiI$WUFf;)M$rWO;)ve=43`C>4W>g9rp&VtRQa}-y&qokNm|bG)?MD;Z&%s-z3%hWE!PjVe2C0?K}9>`~2;}{ge_|_=q3SE~cUE3~rD30{SzD9K_sT=1aWM!h0+-DW-1459q!mMa4}Ij6!#JIz_^@o(5S@w7WU%YMO)Vx>Gm+g=CqQS9(M<<^-QSRDIBq-(!J(zl3rwDjeeZeQ=-O*%SaRcNUNR#RvDtD43wv0HI<l?JRO)QD5aO09tt`q&<s5RnT@8Py7Kw-T~yc?=A6<mLKBKjT<as-2$((y7-_RTwwTg0RF)SVxu30fbY@ebAGi6#XkjH&+jtQB8?I8L-+ZWFAr#x=R@$&UcEOtU0$`{pt4_x^wQK{A7fL>58G?FvyQT0yeQ_i)i>|1+(T(pnGv3(^B$Gw$4hiDfyy`9`pYaGw9?Tzy`%?;*7jSVCY!cbC#MM61Aic18@S$@*ITE9>r2Qi9ER<{8>Z`!b!_Ql4&=2on$-Fpo%irQj6<#uX}{*V!<FHi>ywDisYPL%&Ji4)t>eM=%FGWZ)3n40Pjvh4+REf~W+TS{md_(}I>aHWwu@cl+5KfXQ`Na2t==^8_m|P;{v?x7Su`xAa6&h-AZ%-=EiyS~vsy1cx>N37{Le4Np7Irq&sU0*3f)#)R(xaLwmM{<K1x_E%EQu&f-})0PLXyN_b>l_oXHu!AY$*ALcT>5X!!0D>dU>L(j4kB@Q-D-5af9w7MW9}loeK%(Hb($HJ1;v+DdrP&4B9{-agwJn3_$?#)@y!L6N8Q))W93bCJig*z!6LApS80_;?dTETUmvsfqcI&-U2-NAV+^aad%dfE8Y}DsMkAaPo>@%Fy{zY?BOcNVT@p=qa7Xqtlb9H^D1xg6RM!7(H8^){yVdDzyR-gUrd1L4?S_7-ctJIt)3-Ez@)G5M;7+a!FNEtHc@YZ%A+Vnz%ub`G9^)VLJL+c3;EAk|3WqOYm&HEU)Lyf;zJo8@ezq#1k!87vfz`NnCIQcENGeg($r>?SeSI5KBu-g8#OgKG3S0&jiwADVJX?UWoHjzHcVvm1e)+oh{SNGD^sv$sj@H3V=a&B}1OLuvkT(g{8VfYb7_?vG3>ecvapj&$zV3M-6c4qxH&;T${4BtwO?;SNx+|f%!*v9jkot_muRh?0;8YJjW>^9LdF&{!c}CI6XtUjv$7ITF#?onNpIk%ENBr8MZB7Jg3L0)jRUY3YUM`Sn8MGPE~;b#FJ!Gy;QICK0SWLA1?ega#xq%3uWHVX1VR&s5)7^yt}!xy}NIZ86JKZDwM&qZ#65Ceu2YLTjA}4qi%Lo1wf$^@RY)tCTv+T#Vt_{Aj;Ze%@+P2X~V8J'
IP_DIAGNOSTICS_SHA256 = 'f01441ca077a528b7424d3b5d9557191e24199ffd1c456e60652663d60c9121b'
EMBEDDED_SAMPLE_SEED = 20261014

SAMPLES = [{'id': 2132063,
  'url': 'https://article.iiyi.com/detail/201060.html',
  'domain': 'article.iiyi.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2132091,
  'url': 'https://article.iiyi.com/detail/201489.html',
  'domain': 'article.iiyi.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2132561,
  'url': 'https://article.iiyi.com/detail/25211.html',
  'domain': 'article.iiyi.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2133111,
  'url': 'https://article.iiyi.com/detail/30124.html',
  'domain': 'article.iiyi.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2133130,
  'url': 'https://article.iiyi.com/detail/30306.html',
  'domain': 'article.iiyi.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2133639,
  'url': 'https://article.iiyi.com/detail/34242.html',
  'domain': 'article.iiyi.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2133919,
  'url': 'https://article.iiyi.com/detail/38956.html',
  'domain': 'article.iiyi.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2133937,
  'url': 'https://article.iiyi.com/detail/39065.html',
  'domain': 'article.iiyi.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2133964,
  'url': 'https://article.iiyi.com/detail/39482.html',
  'domain': 'article.iiyi.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2134438,
  'url': 'https://article.iiyi.com/detail/410178.html',
  'domain': 'article.iiyi.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2134828,
  'url': 'https://article.iiyi.com/detail/410848.html',
  'domain': 'article.iiyi.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2134900,
  'url': 'https://article.iiyi.com/detail/410971.html',
  'domain': 'article.iiyi.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2135069,
  'url': 'https://article.iiyi.com/detail/411214.html',
  'domain': 'article.iiyi.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2135803,
  'url': 'https://article.iiyi.com/detail/412415.html',
  'domain': 'article.iiyi.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2135865,
  'url': 'https://article.iiyi.com/detail/412690.html',
  'domain': 'article.iiyi.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2135975,
  'url': 'https://article.iiyi.com/detail/413961.html',
  'domain': 'article.iiyi.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2136146,
  'url': 'https://article.iiyi.com/detail/417168.html',
  'domain': 'article.iiyi.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2136147,
  'url': 'https://article.iiyi.com/detail/417186.html',
  'domain': 'article.iiyi.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2136616,
  'url': 'https://article.iiyi.com/detail/8408.html',
  'domain': 'article.iiyi.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2136714,
  'url': 'https://article.iiyi.com/detail/9785.html',
  'domain': 'article.iiyi.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 2138677,
  'url': 'https://ask.39.net/question/100292322.html',
  'domain': 'ask.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2161647,
  'url': 'https://ask.39.net/question/105031200.html',
  'domain': 'ask.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2162144,
  'url': 'https://ask.39.net/question/105081468.html',
  'domain': 'ask.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2177627,
  'url': 'https://ask.39.net/question/107372665.html',
  'domain': 'ask.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2214821,
  'url': 'https://ask.39.net/question/23086559.html',
  'domain': 'ask.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2222078,
  'url': 'https://ask.39.net/question/27751146.html',
  'domain': 'ask.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2222680,
  'url': 'https://ask.39.net/question/28102871.html',
  'domain': 'ask.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2231481,
  'url': 'https://ask.39.net/question/32111438.html',
  'domain': 'ask.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2241332,
  'url': 'https://ask.39.net/question/35417969.html',
  'domain': 'ask.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2249304,
  'url': 'https://ask.39.net/question/38984842.html',
  'domain': 'ask.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2251380,
  'url': 'https://ask.39.net/question/40062469.html',
  'domain': 'ask.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2261225,
  'url': 'https://ask.39.net/question/46741436.html',
  'domain': 'ask.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2287238,
  'url': 'https://ask.39.net/question/5966114.html',
  'domain': 'ask.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2304872,
  'url': 'https://ask.39.net/question/64327776.html',
  'domain': 'ask.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2309313,
  'url': 'https://ask.39.net/question/65508099.html',
  'domain': 'ask.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2368344,
  'url': 'https://ask.39.net/question/92098409.html',
  'domain': 'ask.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2379227,
  'url': 'https://ask.39.net/question/94080632.html',
  'domain': 'ask.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2398016,
  'url': 'https://ask.39.net/question/_0kyu05.html',
  'domain': 'ask.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2426542,
  'url': 'https://ask.39.net/question/_cd9u05.html',
  'domain': 'ask.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2427334,
  'url': 'https://ask.39.net/question/_csbhc5.html',
  'domain': 'ask.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 766444,
  'url': 'http://ask.familydoctor.com.cn/zt/mn.html',
  'domain': 'ask.familydoctor.com.cn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 772464,
  'url': 'http://baby.39.net/a/141024/4501347.html',
  'domain': 'baby.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 773007,
  'url': 'http://baby.39.net/a/150105/4550080.html',
  'domain': 'baby.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 773028,
  'url': 'http://baby.39.net/a/150107/4552879.html',
  'domain': 'baby.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 773357,
  'url': 'http://baby.39.net/a/150302/4582331.html',
  'domain': 'baby.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 773779,
  'url': 'http://baby.39.net/a/150430/4616128.html',
  'domain': 'baby.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 774470,
  'url': 'http://baby.39.net/a/151010/4707082.html',
  'domain': 'baby.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 775124,
  'url': 'http://baby.39.net/a/160513/4851730.html',
  'domain': 'baby.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 775598,
  'url': 'http://baby.39.net/a/161116/5046162.html',
  'domain': 'baby.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 779450,
  'url': 'http://baby.39.net/a/200825/8152077.html',
  'domain': 'baby.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 781910,
  'url': 'http://baby.39.net/a/2010419/1234368.html',
  'domain': 'baby.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 783013,
  'url': 'http://baby.39.net/a/2010521/1285918.html',
  'domain': 'baby.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 787019,
  'url': 'http://baby.39.net/a/201077/1379001.html',
  'domain': 'baby.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 787932,
  'url': 'http://baby.39.net/a/201086/1426029.html',
  'domain': 'baby.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 789155,
  'url': 'http://baby.39.net/a/2011214/1609137.html',
  'domain': 'baby.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 790564,
  'url': 'http://baby.39.net/a/2011528/1703886.html',
  'domain': 'baby.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 790678,
  'url': 'http://baby.39.net/a/201159/1683912.html',
  'domain': 'baby.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 791143,
  'url': 'http://baby.39.net/a/201171/1734161.html',
  'domain': 'baby.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 791777,
  'url': 'http://baby.39.net/a/2011825/1786532.html',
  'domain': 'baby.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 792119,
  'url': 'http://baby.39.net/a/201192/1795161.html',
  'domain': 'baby.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 793812,
  'url': 'http://baby.39.net/a/230906/v8edqm6.html',
  'domain': 'baby.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 974,
  'url': 'https://bachmai.gov.vn/bai-viet/bo-y-te-khang-dinh-khong-dung-vac-xin-combe-five?id=9260c49b-a80b-e54c-f39b-fbf56579d661',
  'domain': 'bachmai.gov.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 992,
  'url': 'https://bachmai.gov.vn/bai-viet/bs-viet-cuu-song-benh-nhan-mac-sot-xuat-huyet-o-bi-nho-hoi-chan-dac-biet?id=adeadda3-d563-4016-5bb7-b5a8bc768365',
  'domain': 'bachmai.gov.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 1026,
  'url': 'https://bachmai.gov.vn/bai-viet/ca-si-mia-cap-cuu-vi-lam-dung-paracetamol-chuyen-gia-canh-bao-co-benh-sau-can-than-trong?id=318a4876-5857-776b-a988-44d52245f790',
  'domain': 'bachmai.gov.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 1070,
  'url': 'https://bachmai.gov.vn/bai-viet/cach-xu-ly-khi-phat-hien-tre-tham-gia-tro-choi-tu-sat-kinh-di-momo?id=196f9481-217b-0174-29e7-a4b8f894a688',
  'domain': 'bachmai.gov.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 1127,
  'url': 'https://bachmai.gov.vn/bai-viet/canh-bao-hut-thuoc-la-thu-dong-co-the-gay-dot-tu-o-tre-em?id=2734b66b-8009-2472-6acd-730c80673a74',
  'domain': 'bachmai.gov.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 1180,
  'url': 'https://bachmai.gov.vn/bai-viet/canh-bao-tinh-trang-ngo-doc-ruou-trong-dip-tet?id=27f6bd55-d067-28eb-df9b-87065af9f2fc',
  'domain': 'bachmai.gov.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 1264,
  'url': 'https://bachmai.gov.vn/bai-viet/chen-ruou-va-cai-chet-duoc-bao-truoc?id=1295c855-2c8c-e6bc-2389-0653baba470b',
  'domain': 'bachmai.gov.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 1307,
  'url': 'https://bachmai.gov.vn/bai-viet/chung-nghien-va-trung-tam-lac-thu-o-nao?id=92655fa3-38e5-98c2-b7cf-86d1a65b587e',
  'domain': 'bachmai.gov.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 1340,
  'url': 'https://bachmai.gov.vn/bai-viet/chuyen-gia-khuyen-cao-phong-ngua-benh-roi-loan-tam-than?id=ddd95200-7f5d-2697-5321-17786b1888a1',
  'domain': 'bachmai.gov.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 1459,
  'url': 'https://bachmai.gov.vn/bai-viet/dau-hieu-nhan-biet-tram-cam-o-tre?id=36251470-7036-5551-0c92-29ec8fc2026f',
  'domain': 'bachmai.gov.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 1494,
  'url': 'https://bachmai.gov.vn/bai-viet/dia-an-lanh-manh-huong-dan-don-gian-de-co-mot-bua-an-can-bang?id=5f77c70a-e53e-462e-9463-5ecf5961a4cd',
  'domain': 'bachmai.gov.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 1580,
  'url': 'https://bachmai.gov.vn/bai-viet/dung-cao-dan-chong-buon-ngu-co-thuc-su-an-toan?id=43a89b46-e346-58f5-6507-ec375d44079a',
  'domain': 'bachmai.gov.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2305,
  'url': 'https://bachmai.gov.vn/bai-viet/nhieu-loi-ich-khi-mua-thuoc-tai-nha-thuoc-benh-vien?id=edc40081-8099-46ce-8c8d-a880b9d5725f',
  'domain': 'bachmai.gov.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2501,
  'url': 'https://bachmai.gov.vn/bai-viet/phat-hien-va-phong-tranh-dich-benh-mua-dong-xuan?id=5f8efd9a-af70-4781-944c-fc486cd8fb8f',
  'domain': 'bachmai.gov.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2548,
  'url': 'https://bachmai.gov.vn/bai-viet/phong-ngua-benh-van-phong?id=7b6f0776-620f-ca36-7b92-884ffa07e8b8',
  'domain': 'bachmai.gov.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2696,
  'url': 'https://bachmai.gov.vn/bai-viet/so-cuu-khi-bi-ong-dot?id=4a0a5fcf-d3d6-f883-0c39-52c8ca7940d1',
  'domain': 'bachmai.gov.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2782,
  'url': 'https://bachmai.gov.vn/bai-viet/tai-sao-co-nguoi-sau-khi-boi-len-bo-van-chet-duoi?id=1521b419-58db-bef5-4b31-22b58ed32a15',
  'domain': 'bachmai.gov.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2898,
  'url': 'https://bachmai.gov.vn/bai-viet/thong-bao-lich-sinh-hoat-clb-benh-nhan-dai-thao-duong-thang-3-2018?id=be8dbb39-4a09-1ab9-4b47-3875afb37f8f',
  'domain': 'bachmai.gov.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 3054,
  'url': 'https://bachmai.gov.vn/bai-viet/trai-nghiem-kham-benh-buoi-chieu-hay-la-nguoi-benh-thong-thai?id=7032447a-ffdc-4d82-9a53-bfc498458870',
  'domain': 'bachmai.gov.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 3332,
  'url': 'https://bachmai.gov.vn/bai-viet/xem-clip-be-so-sinh-tap-yoga-bac-si-cung-dung-tim?id=a37dcce1-c0f9-613c-69a9-dd1903f81c29',
  'domain': 'bachmai.gov.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 2474133,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201211/9861281133554.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2474623,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201412/730305.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2474707,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201501/739229.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2474862,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201504/756173.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2475421,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201603/964390.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2475660,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201605/1003170.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2475697,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201605/1009871.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2475722,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201605/1025221.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2475801,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201605/1044531.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2476026,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201607/1148871.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2476173,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201608/1203780.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2476538,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201611/1450883.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2477246,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/201804/2474593.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2477651,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/202006/2612437.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2477683,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/202501/3146430.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2477708,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/202502/3170542.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2477780,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/202511/3738672.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2477785,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/202601/3880846.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2477788,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/202601/3880849.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2477796,
  'url': 'https://baidianfeng.familydoctor.com.cn/a/202603/3947399.html',
  'domain': 'baidianfeng.familydoctor.com.cn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 3414,
  'url': 'https://baoangiang.com.vn/10-dia-phuong-co-ca-mac-moi-nhieu-nhat-trong-sang-27-7-a309010.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 3947,
  'url': 'https://baoangiang.com.vn/5-meo-don-gian-giup-mo-thua-dot-chay-am-am-ngay-ca-khi-ngu-a395341.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 4138,
  'url': 'https://baoangiang.com.vn/6-tu-the-giup-ban-chim-vao-giac-ngu-ngay-tuc-khac-a221069.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 4452,
  'url': 'https://baoangiang.com.vn/an-gi-vao-bua-sang-de-tranh-met-moi-sau-khi-say-ruou-a384198.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 4790,
  'url': 'https://baoangiang.com.vn/bai-thuoc-tu-gung-nau-voi-la-tia-to-don-gian-a421772.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 5789,
  'url': 'https://baoangiang.com.vn/ca-nuoc-ghi-nhan-gan-23-000-truong-hop-mac-sot-xuat-huyet-voi-5-ca-tu-vong-a422598.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 6075,
  'url': 'https://baoangiang.com.vn/cach-ngam-hoa-du-du-duc-voi-mat-ong-de-tri-ho-viem-hong-a465215.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 6974,
  'url': 'https://baoangiang.com.vn/cuu-song-benh-nhan-viem-ruot-thua-cap-co-phu-tang-dao-nguoc-hoan-toan-a236962.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 7215,
  'url': 'https://baoangiang.com.vn/day-tre-mam-non-lam-banh-pizza-a217889.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 7615,
  'url': 'https://baoangiang.com.vn/dieu-tri-hau-covid-19-a333862.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 8192,
  'url': 'https://baoangiang.com.vn/hieu-qua-cua-lieu-vac-xin-pfizer-tang-cuong-a317651.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 9348,
  'url': 'https://baoangiang.com.vn/mui-vaccine-ngua-covid-19-thu-4-co-hieu-qua-rat-cao-voi-nguoi-lon-tuoi-a337697.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 10283,
  'url': 'https://baoangiang.com.vn/nguy-co-dot-quy-no-luc-trong-dieu-tri-a400382.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 10600,
  'url': 'https://baoangiang.com.vn/nhung-loai-nuoc-ep-giup-tang-cuong-mien-dich-a265209.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 10691,
  'url': 'https://baoangiang.com.vn/nhung-nguoi-co-nguy-co-cao-mac-ung-thu-khoang-mieng-a358195.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 10768,
  'url': 'https://baoangiang.com.vn/nhung-phat-hien-khoa-hoc-moi-giup-dieu-tri-ung-thu-thuc-quan-a357283.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 10827,
  'url': 'https://baoangiang.com.vn/nhung-thuc-pham-ban-co-the-an-thoa-thich-ma-khong-lo-tang-can-a250426.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 11204,
  'url': 'https://baoangiang.com.vn/phong-chong-tai-nan-duoi-nuoc-cho-tre-em-trong-dip-he-a365933.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 12545,
  'url': 'https://baoangiang.com.vn/thuc-day-nghien-cuu-tiep-nhan-chuyen-giao-cong-nghe-san-xuat-thuoc-a381297.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 12869,
  'url': 'https://baoangiang.com.vn/tp-ho-chi-minh-17-000-tre-em-duoc-tiem-vaccine-soi-trong-ky-nghi-le-quoc-khanh-a404458.html',
  'domain': 'baoangiang.com.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 14401,
  'url': 'https://baocantho.com.vn/10-dieu-nen-lam-truoc-khi-ngu-de-duoc-khoe-manh-a82901.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 14684,
  'url': 'https://baocantho.com.vn/5-nhom-nguoi-xet-nghiem-covid-19-tang-cuong-duoc-bao-hiem-y-te-chi-tra-a134930.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 14919,
  'url': 'https://baocantho.com.vn/amway-viet-nam-tu-hao-lan-thu-2-don-nhan-bang-khen-cua-bo-y-te-a131807.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 15017,
  'url': 'https://baocantho.com.vn/an-hanh-nhan-giup-loai-bo-cholesterol-xau-a89566.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 15304,
  'url': 'https://baocantho.com.vn/ban-chai-danh-rang-sac-pin-qua-cong-usb-a71438.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 16822,
  'url': 'https://baocantho.com.vn/cham-soc-nguoi-cao-tuoi-giua-dai-dich-a120273.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 17786,
  'url': 'https://baocantho.com.vn/da-banh-de-sac-pin-a62406.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 18261,
  'url': 'https://baocantho.com.vn/dieu-tri-thanh-cong-cho-cap-vo-chong-hiem-muon-ma-nguoi-chong-vo-tinh-a149236.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 18480,
  'url': 'https://baocantho.com.vn/dung-cam-va-buoi-giup-phu-nu-giam-nguy-co-dot-qui-a75804.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 18876,
  'url': 'https://baocantho.com.vn/gioi-thieu-cong-nghe-in-3d-moi-nhanh-gap-tram-lan-a81730.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 20128,
  'url': 'https://baocantho.com.vn/lieu-phap-testosterone-day-lui-tram-cam-o-nam-gioi-a103872.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 20563,
  'url': 'https://baocantho.com.vn/mien-phi-xet-nghiem-chan-doan-benh-tat-bam-sinh-o-tre-so-sinh-a151665.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 21484,
  'url': 'https://baocantho.com.vn/nhieu-bien-chung-nguy-hiem-o-benh-nhan-bi-soi-duong-mat-trong-gan-a176450.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 21584,
  'url': 'https://baocantho.com.vn/nhung-bien-phap-ren-luyen-giup-ich-cho-benh-nhan-viem-khop-a66678.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 22734,
  'url': 'https://baocantho.com.vn/phu-nu-ngay-cang-noi-troi-trong-linh-vuc-cong-nghe-a78755.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 22825,
  'url': 'https://baocantho.com.vn/prebiotic-va-loi-ich-suc-khoe-khac-tu-chat-xo-a84177.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 23977,
  'url': 'https://baocantho.com.vn/thanh-lap-chi-hoi-van-dong-hien-mo-bo-phan-co-the-nguoi-tai-tp-can-tho-a181830.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 24539,
  'url': 'https://baocantho.com.vn/thuoc-tru-sau-100-tu-nhien-a60258.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 25136,
  'url': 'https://baocantho.com.vn/trien-vong-trong-dieu-tri-benh-parkinson-va-dong-kinh-a105341.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 25139,
  'url': 'https://baocantho.com.vn/trien-vong-vac-xin-ngua-benh-alzheimer-a67204.html',
  'domain': 'baocantho.com.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 26236,
  'url': 'https://baochinhphu.vn/benh-soi-chua-co-thuoc-dieu-tri-dac-hieu-102240829151448771.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 26628,
  'url': 'https://baochinhphu.vn/chan-chinh-hoat-dong-giam-dinh-phap-y-phap-y-tam-than-102250619184933967.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 26635,
  'url': 'https://baochinhphu.vn/chat-luong-dan-so-duoc-cai-thien-nhieu-mat-102221226164000305.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 26804,
  'url': 'https://baochinhphu.vn/da-nang-tich-cuc-van-dong-nguoi-dan-tiem-vaccine-phong-covid-19-102220713181607764.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 26890,
  'url': 'https://baochinhphu.vn/de-xuat-danh-muc-thuoc-co-it-nhat-3-hang-trong-nuoc-san-xuat-tren-day-chuyen-dap-ung-tieu-chuan-eu-gmp-102231218170106926.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 27268,
  'url': 'https://baochinhphu.vn/huong-dan-phan-nhom-thiet-bi-y-te-theo-tieu-chuan-chat-luong-102260107101658252.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 27360,
  'url': 'https://baochinhphu.vn/khong-duoc-bo-sung-pham-vi-hanh-nghe-cua-chuc-danh-khac-102240611132107099.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 27519,
  'url': 'https://baochinhphu.vn/nam-dinh-co-nen-tang-phat-trien-nganh-duoc-voi-nhieu-doanh-nghiep-lon-102220407172239381.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 27525,
  'url': 'https://baochinhphu.vn/nang-cao-hieu-biet-bao-ve-tre-em-truoc-benh-do-phe-cau-khuan-gay-ra-102251117115416562.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 27528,
  'url': 'https://baochinhphu.vn/nang-cao-nhan-thuc-ve-du-phong-kiem-soat-benh-tim-mach-than-chuyen-hoa-phoi-ke-102230617183246933.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 27614,
  'url': 'https://baochinhphu.vn/nhieu-dich-benh-dang-dien-bien-phuc-tap-102220730083039956.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 27638,
  'url': 'https://baochinhphu.vn/nhung-luu-y-nguoi-dan-can-biet-de-tranh-mua-thuoc-gia-102250418131317472.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 27752,
  'url': 'https://baochinhphu.vn/quang-nam-phat-hien-xu-ly-94-o-dich-sot-xuat-huyet-102220715014749203.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 27826,
  'url': 'https://baochinhphu.vn/sang-kien-bac-si-cho-moi-nha-thuc-day-chuyen-doi-so-cua-nganh-y-te-102230720110854126.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 27849,
  'url': 'https://baochinhphu.vn/so-ca-mac-covid-19-ca-nuoc-giam-manh-con-34140-ca-102220409183826962.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 27980,
  'url': 'https://baochinhphu.vn/than-toc-phoi-hop-thuc-hien-lay-ghep-da-tang-tu-nguoi-cho-chet-nao-102241129085704803.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 27987,
  'url': 'https://baochinhphu.vn/thanh-lap-truong-dai-hoc-tam-anh-dao-tao-nhan-luc-chuan-quoc-te-102251231080013731.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 28131,
  'url': 'https://baochinhphu.vn/tim-giai-phap-kiem-soat-va-quan-ly-benh-khong-lay-nhiem-102230627163449244.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 28215,
  'url': 'https://baochinhphu.vn/truc-tiep-toa-dam-nganh-y-vuot-kho-102230223074006505.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 28311,
  'url': 'https://baochinhphu.vn/viet-nam-da-coi-covid-19-la-benh-luu-hanh-chua-102220429184610671.htm',
  'domain': 'baochinhphu.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 28760,
  'url': 'https://baodanang.vn/6-nguoi-dan-quang-ngai-nhap-vien-vi-ngo-doc-ca-noc-3080760.html',
  'domain': 'baodanang.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 28931,
  'url': 'https://baodanang.vn/an-ninh-duoc-lieu-3239103.html',
  'domain': 'baodanang.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 29097,
  'url': 'https://baodanang.vn/ban-hanh-he-thong-chi-tieu-theo-doi-danh-gia-cong-tac-phong-chong-hiv-aids-3110606.html',
  'domain': 'baodanang.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 30754,
  'url': 'https://baodanang.vn/can-nhac-khi-thuc-hien-nhin-an-gian-doan-3153199.html',
  'domain': 'baodanang.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 31017,
  'url': 'https://baodanang.vn/cat-buou-giap-khong-lo-cho-nu-benh-nhan-3246406.html',
  'domain': 'baodanang.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 31861,
  'url': 'https://baodanang.vn/cong-dung-cua-hoa-van-tho-3296689.html',
  'domain': 'baodanang.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 32280,
  'url': 'https://baodanang.vn/da-nang-rut-giay-phep-co-so-ban-thuoc-dong-y-tron-tan-duoc-3238819.html',
  'domain': 'baodanang.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 32409,
  'url': 'https://baodanang.vn/dam-bao-an-toan-phong-chong-dich-trong-bau-cu-3068933.html',
  'domain': 'baodanang.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 34096,
  'url': 'https://baodanang.vn/hon-1-600-nguoi-dan-nui-thanh-duoc-kham-tam-soat-tim-mien-phi-3128382.html',
  'domain': 'baodanang.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 34902,
  'url': 'https://baodanang.vn/khong-nen-cho-vac-xin-dich-vu-3240127.html',
  'domain': 'baodanang.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 35047,
  'url': 'https://baodanang.vn/kiem-tra-cong-tac-cach-ly-y-te-tai-cac-co-so-do-quan-doi-quan-ly-3076831.html',
  'domain': 'baodanang.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 35064,
  'url': 'https://baodanang.vn/kiem-tra-hien-trang-su-dung-cac-tram-y-te-xuong-cap-3050009.html',
  'domain': 'baodanang.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 36484,
  'url': 'https://baodanang.vn/ngay-22-10-viet-nam-ghi-nhan-475-ca-mac-moi-covid-19-3245887.html',
  'domain': 'baodanang.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 37090,
  'url': 'https://baodanang.vn/ngo-doc-ruou-3112495.html',
  'domain': 'baodanang.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 37188,
  'url': 'https://baodanang.vn/nguy-co-cum-a-h7n9-xam-nhap-vao-viet-nam-la-rat-cao-3238995.html',
  'domain': 'baodanang.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 38907,
  'url': 'https://baodanang.vn/sang-15-2-viet-nam-ghi-nhan-1-ca-mac-moi-covid-19-la-chuyen-gia-nguoi-nhat-ban-3243447.html',
  'domain': 'baodanang.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 39790,
  'url': 'https://baodanang.vn/tang-cuong-tiem-vac-xin-phong-covid-19-mui-3-4-kip-thoi-day-du-3059857.html',
  'domain': 'baodanang.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 39934,
  'url': 'https://baodanang.vn/tap-trung-phong-chong-tranh-nguy-co-bung-phat-dich-soi-3150720.html',
  'domain': 'baodanang.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 40309,
  'url': 'https://baodanang.vn/them-1-truong-hop-nghi-nhiem-covid-19-huyen-bac-tra-my-thanh-lap-2-khu-cach-ly-tap-trung-3064654.html',
  'domain': 'baodanang.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 41675,
  'url': 'https://baodanang.vn/trien-khai-tiem-vac-xin-cho-hon-40-000-nguoi-3244512.html',
  'domain': 'baodanang.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 43130,
  'url': 'https://baogialai.com.vn/an-cu-cai-trang-co-tot-khong-post261991.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 43173,
  'url': 'https://baogialai.com.vn/an-yen-mach-thay-bua-sang-loi-va-hai-the-nao-post577157.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 43189,
  'url': 'https://baogialai.com.vn/ba-cach-phong-ngua-dot-quy-nho-an-uong-post315755.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 43264,
  'url': 'https://baogialai.com.vn/bao-dong-gia-tang-nguoi-tre-tai-bien-mach-mau-nao-nhoi-mau-co-tim-post308597.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 43366,
  'url': 'https://baogialai.com.vn/benh-vien-binh-dinh-kham-va-dieu-tri-benh-ly-dong-mach-vanh-cap-cung-chuyen-gia-post343965.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 43479,
  'url': 'https://baogialai.com.vn/benh-vien-trung-uong-hue-kham-sang-loc-benh-tim-mien-phi-cho-nguoi-dan-gia-lai-post312700.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 43560,
  'url': 'https://baogialai.com.vn/bo-y-te-cong-bo-tinh-hinh-cap-cuu-va-tu-vong-trong-dip-nghi-le-post321694.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 43583,
  'url': 'https://baogialai.com.vn/bo-y-te-giam-sat-dich-cum-cung-ung-kip-thoi-thuoc-dieu-tri-post310925.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 43992,
  'url': 'https://baogialai.com.vn/dak-po-ghi-nhan-14-truong-hop-tre-em-mac-benh-tay-chan-mieng-chi-trong-3-ngay-post273132.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 44082,
  'url': 'https://baogialai.com.vn/dieu-chinh-chu-truong-dau-tu-du-an-khu-xa-tri-va-trang-thiet-bi-benh-vien-da-khoa-tinh-gia-lai-post284897.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 44182,
  'url': 'https://baogialai.com.vn/gan-300-vien-chuc-dan-so-gia-lai-tap-huan-nang-cao-nang-luc-ky-nang-cham-soc-suc-khoe-nguoi-cao-tuoi-post570614.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 44465,
  'url': 'https://baogialai.com.vn/gia-lai-tren-100-y-bac-si-tap-huan-ung-dung-chatgpt-trong-y-khoa-post565818.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 44808,
  'url': 'https://baogialai.com.vn/lien-tiep-nhieu-nguoi-nguy-kich-vi-ngo-doc-ruou-bo-y-te-chi-dao-khan-post286639.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 44921,
  'url': 'https://baogialai.com.vn/mo-dot-cao-diem-dau-tranh-voi-thuoc-gia-thuc-pham-gia-post333571.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 45016,
  'url': 'https://baogialai.com.vn/nap-khoang-chat-nao-de-giam-huyet-ap-mot-cach-tu-nhien-post310484.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 45055,
  'url': 'https://baogialai.com.vn/nghe-nhan-binh-dinh-dang-hat-boi-bat-ngo-nga-guc-tren-san-khau-roi-tu-vong-post288109.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 45477,
  'url': 'https://baogialai.com.vn/quan-ly-nguy-co-va-chu-dong-phong-ngua-sot-xuat-huyet-dengue-post567092.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 45479,
  'url': 'https://baogialai.com.vn/quan-y-viet-nam-campuchia-kham-benh-cap-thuoc-mien-phi-tai-huyen-duc-co-post257756.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 45921,
  'url': 'https://baogialai.com.vn/trang-bi-kien-thuc-ve-gioi-tinh-va-suc-khoe-sinh-san-cho-sinh-vien-post341136.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 46206,
  'url': 'https://baogialai.com.vn/xuat-khau-duoc-pham-dat-312-trieu-usd-viet-nam-dung-thu-4-dong-nam-a-post576649.html',
  'domain': 'baogialai.com.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 46627,
  'url': 'https://baohaiphong.vn/3-benh-nhan-o-o-dich-36-ngo-quyen-duoc-xuat-vien-211538.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 46735,
  'url': 'https://baohaiphong.vn/34-sinh-con-gai-duoc-thuong-34-la-tin-don-that-thiet-338511.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 47049,
  'url': 'https://baohaiphong.vn/6-dieu-can-biet-ve-vaccine-phong-covid-19-cho-tre-tu-5-11-tuoi-148412.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 48282,
  'url': 'https://baohaiphong.vn/benh-vien-dau-tien-cua-viet-nam-co-ten-tren-ban-do-ghep-tim-the-gioi-304205.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 48912,
  'url': 'https://baohaiphong.vn/bo-y-te-hop-khan-ve-dich-bach-hau-dang-lan-rong-o-tay-nguyen-216301.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 49599,
  'url': 'https://baohaiphong.vn/cach-giu-am-co-the-trong-mua-dong-285943.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 51267,
  'url': 'https://baohaiphong.vn/den-19-gio-30-ngay-25-2-tren-dia-ban-thanh-pho-khong-ghi-nhan-ca-mac-covid-19-moi-454988.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 53214,
  'url': 'https://baohaiphong.vn/hoai-tu-nguc-sau-khi-bo-ra-hang-chuc-trieu-dong-nang-vong-1-tai-spa-144843.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 53279,
  'url': 'https://baohaiphong.vn/hoi-dong-dao-duc-vaccine-nano-covax-co-tac-dung-voi-chung-delta-alpha-178473.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 53503,
  'url': 'https://baohaiphong.vn/huong-dan-moi-nhat-ve-tiem-vaccine-cho-nguoi-da-mac-covid-19-159461.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 53706,
  'url': 'https://baohaiphong.vn/infographics-ngay-suc-khoe-the-gioi-10-van-de-suc-khoe-toan-cau-trong-nam-2021-194226.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 53874,
  'url': 'https://baohaiphong.vn/kham-benh-tinh-nguyen-tai-xa-le-hong-244655.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 54159,
  'url': 'https://baohaiphong.vn/khong-de-nguoi-benh-phai-tu-mua-thuoc-vat-tu-y-te-ngoai-benh-vien-130060.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 57769,
  'url': 'https://baohaiphong.vn/ra-soat-doi-tuong-tang-cuong-tiem-chung-vaccine-ngua-covid-19-128630.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 58404,
  'url': 'https://baohaiphong.vn/so-y-te-thanh-pho-hai-phong-tiep-nhan-tai-tro-15-xe-cuu-thuong-va-trang-thiet-bi-tri-gia-hon-26-ty-dong-529287.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 59648,
  'url': 'https://baohaiphong.vn/thiet-thuc-va-hieu-qua-324239.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 60688,
  'url': 'https://baohaiphong.vn/trien-khai-cac-phuong-an-van-chuyen-dieu-tri-nguoi-mac-covid-19-469567.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 62003,
  'url': 'https://baohaiphong.vn/viet-nam-lot-top-11-quoc-qua-khoe-manh-nhat-the-gioi-250719.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 62225,
  'url': 'https://baohaiphong.vn/who-dat-ten-moi-cho-ncov-la-covid-19-441170.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 62450,
  'url': 'https://baohaiphong.vn/xuat-huyet-da-day-do-uong-giam-giam-can-344620.html',
  'domain': 'baohaiphong.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 62514,
  'url': 'https://baolangson.vn/2024-la-nam-thay-thuoc-tre-xung-kich-chuyen-doi-so-trong-hoat-dong-tinh-nguyen-1637114.html',
  'domain': 'baolangson.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 62553,
  'url': 'https://baolangson.vn/ai-con-mat-thu-ba-giup-bac-si-phat-hien-som-ung-thu-tu-dau-hieu-nho-5068950.html',
  'domain': 'baolangson.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 62826,
  'url': 'https://baolangson.vn/bo-y-te-yeu-cau-ra-soat-viec-su-dung-sua-trong-benh-vien-5044528.html',
  'domain': 'baolangson.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 62864,
  'url': 'https://baolangson.vn/cac-dieu-kien-thanh-toan-bao-hiem-y-te-khi-mua-thuoc-ngoai-benh-vien-5026755.html',
  'domain': 'baolangson.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 62949,
  'url': 'https://baolangson.vn/cham-cuu-cau-noi-y-hoc-va-van-hoa-giua-viet-nam-va-phap-5059255.html',
  'domain': 'baolangson.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 62956,
  'url': 'https://baolangson.vn/cham-soc-suc-khoe-giao-vien-can-chu-y-phong-benh-trao-nguoc-da-day-thuc-quan-5029107.html',
  'domain': 'baolangson.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 63092,
  'url': 'https://baolangson.vn/cuu-nguoi-dan-ong-ngung-tim-khi-mot-minh-toi-khoa-cap-cuu-5058885.html',
  'domain': 'baolangson.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 63113,
  'url': 'https://baolangson.vn/dak-lak-nguoi-giu-la-chan-vi-sinh-bao-ve-suc-khoe-cong-dong-5062244.html',
  'domain': 'baolangson.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 63253,
  'url': 'https://baolangson.vn/gia-tang-so-ca-mac-benh-thuy-dau-nguoi-dan-khong-nen-chu-quan-5009111.html',
  'domain': 'baolangson.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 63332,
  'url': 'https://baolangson.vn/hanh-trinh-do-nam-2024-du-kien-tiep-nhan-120-000-don-vi-mau-5010475.html',
  'domain': 'baolangson.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 63335,
  'url': 'https://baolangson.vn/hau-het-nguoi-benh-bi-sa-sut-tri-tue-den-vien-o-giai-doan-muon-1571957.html',
  'domain': 'baolangson.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 63411,
  'url': 'https://baolangson.vn/human-act-prize-2023-vinh-danh-du-an-thien-nguyen-thuoc-dung-cho-em-cua-genestory-1630421.html',
  'domain': 'baolangson.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 63556,
  'url': 'https://baolangson.vn/lan-toa-kien-thuc-nuoi-con-khoe-5062400.html',
  'domain': 'baolangson.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 63749,
  'url': 'https://baolangson.vn/nhan-dien-nhung-yeu-to-nguy-co-de-phong-chong-benh-dai-thao-duong-1623579.html',
  'domain': 'baolangson.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 63939,
  'url': 'https://baolangson.vn/siet-lai-he-thong-thong-tin-nhan-dien-dia-chi-an-toan-thuc-pham-1516644.html',
  'domain': 'baolangson.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 63957,
  'url': 'https://baolangson.vn/so-ket-cong-tac-y-te-quy-i-mot-so-chi-tieu-chuyen-mon-dat-va-vuot-so-voi-ke-hoach-de-ra-5044348.html',
  'domain': 'baolangson.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 64029,
  'url': 'https://baolangson.vn/tang-cuong-quan-ly-cac-benh-khong-lay-nhiem-5019137.html',
  'domain': 'baolangson.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 64057,
  'url': 'https://baolangson.vn/tat-tan-tat-quyen-loi-bao-hiem-y-te-nguoi-dan-duoc-huong-tu-nam-2026-5071586.html',
  'domain': 'baolangson.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 64156,
  'url': 'https://baolangson.vn/thuc-day-hop-tac-chuyen-doi-so-khoa-hoc-cong-nghe-trong-nganh-y-te-duoc-pham-5044451.html',
  'domain': 'baolangson.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 64223,
  'url': 'https://baolangson.vn/tp-ho-chi-minh-keu-goi-nguoi-dan-hien-mau-nhan-dao-truoc-va-sau-ky-nghi-tet-1643478.html',
  'domain': 'baolangson.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 66073,
  'url': 'https://baonghean.vn/6-meo-nho-nhung-tac-dung-lon-giup-eo-thon-10142847.html',
  'domain': 'baonghean.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 66525,
  'url': 'https://baonghean.vn/8-bai-thuoc-dieu-tri-mat-ngu-tu-cay-thuoc-dan-gian-10285796.html',
  'domain': 'baonghean.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 67067,
  'url': 'https://baonghean.vn/an-hoa-ung-thu-tu-do-nuong-10043841.html',
  'domain': 'baonghean.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 67396,
  'url': 'https://baonghean.vn/bai-2-can-co-giai-phap-dong-bo-va-quyet-liet-10029329.html',
  'domain': 'baonghean.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 67397,
  'url': 'https://baonghean.vn/bai-2-can-giai-phap-dong-bo-quyet-liet-10036692.html',
  'domain': 'baonghean.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 67783,
  'url': 'https://baonghean.vn/be-mac-va-trao-giai-giai-bong-chuyen-hoi-nganh-y-te-nghe-an-nam-2025-10291779.html',
  'domain': 'baonghean.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 68230,
  'url': 'https://baonghean.vn/benh-vien-phuc-hoi-chuc-nang-nghe-an-nhan-len-niem-tin-yeu-10153697.html',
  'domain': 'baonghean.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 69278,
  'url': 'https://baonghean.vn/cach-lam-nuoc-detox-giup-tieu-hao-mo-tich-tu-trong-ngay-tet-10164369.html',
  'domain': 'baonghean.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 69362,
  'url': 'https://baonghean.vn/cach-nhanh-nhat-de-nhan-biet-nuoc-mam-tuong-ot-co-doc-to-10083915.html',
  'domain': 'baonghean.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 70276,
  'url': 'https://baonghean.vn/chua-viem-hong-do-lanh-hieu-qua-voi-rau-bo-binh-10056566.html',
  'domain': 'baonghean.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 71291,
  'url': 'https://baonghean.vn/dich-tay-chan-mieng-trong-tam-kiem-soat-10031593.html',
  'domain': 'baonghean.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 72567,
  'url': 'https://baonghean.vn/hon-93-000-truong-hop-mac-benh-chan-tay-mieng-10039895.html',
  'domain': 'baonghean.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 73132,
  'url': 'https://baonghean.vn/ky-ket-hop-dong-kham-chua-benh-bhyt-giua-bhxh-va-benh-vien-da-khoa-quoc-te-vinh-10081088.html',
  'domain': 'baonghean.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 73852,
  'url': 'https://baonghean.vn/meo-bu-nuoc-nhanh-chong-trong-nhung-ngay-nang-nong-10274425.html',
  'domain': 'baonghean.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 76420,
  'url': 'https://baonghean.vn/nhung-truong-hop-khong-duoc-tiem-vac-xin-soi-10058750.html',
  'domain': 'baonghean.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 77425,
  'url': 'https://baonghean.vn/sang-18-8-them-7-ca-mac-moi-covid-19-trong-do-quang-nam-hai-duong-va-ha-noi-co-6-ca-10216764.html',
  'domain': 'baonghean.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 78208,
  'url': 'https://baonghean.vn/thanh-chuong-ki-cam-ket-khong-sinh-con-thu-3-trong-nganh-giao-duc-10045610.html',
  'domain': 'baonghean.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 78228,
  'url': 'https://baonghean.vn/thanh-lap-trung-tam-benh-nhiet-doi-tai-benh-vien-hndk-nghe-an-10208498.html',
  'domain': 'baonghean.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 78369,
  'url': 'https://baonghean.vn/them-4-truong-hop-mac-covid-19-viet-nam-co-590-ca-benh-10216185.html',
  'domain': 'baonghean.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 78498,
  'url': 'https://baonghean.vn/thoi-quen-an-uong-dang-tan-pha-co-the-nguoi-viet-10026851.html',
  'domain': 'baonghean.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 80427,
  'url': 'https://baophutho.vn/10-loai-thuc-pham-tot-cho-gan-ho-tro-phuc-hoi-234971.htm',
  'domain': 'baophutho.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 80465,
  'url': 'https://baophutho.vn/3-ke-thu-khien-dau-an-de-bi-hong-vay-bao-quan-the-nao-la-dung-cach-237078.htm',
  'domain': 'baophutho.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 80481,
  'url': 'https://baophutho.vn/3-nhom-thuc-pham-can-tranh-xa-khi-dang-giam-can-243105.htm',
  'domain': 'baophutho.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 80674,
  'url': 'https://baophutho.vn/6-dau-hieu-canh-bao-tre-dang-gap-van-de-ve-dinh-duong-241933.htm',
  'domain': 'baophutho.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 80733,
  'url': 'https://baophutho.vn/7-dieu-co-the-xay-ra-voi-co-the-nguoi-khong-an-rau-va-trai-cay-248922.htm',
  'domain': 'baophutho.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 80771,
  'url': 'https://baophutho.vn/7-sai-lam-khien-bo-sung-sat-khong-dat-hieu-qua-246782.htm',
  'domain': 'baophutho.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 80782,
  'url': 'https://baophutho.vn/7-thuc-pham-nen-an-khi-bi-cang-co-qua-muc-230964.htm',
  'domain': 'baophutho.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 80788,
  'url': 'https://baophutho.vn/8-loai-hat-giau-magie-nen-an-de-giup-ngu-ngon-242608.htm',
  'domain': 'baophutho.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 80847,
  'url': 'https://baophutho.vn/an-nhieu-thuc-pham-mau-den-giup-bao-ve-than-233056.htm',
  'domain': 'baophutho.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 80886,
  'url': 'https://baophutho.vn/benh-vien-da-khoa-hoa-binh-phau-thuat-thanh-cong-ca-mo-hiem-u-tuy-song-vung-co-239664.htm',
  'domain': 'baophutho.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 81053,
  'url': 'https://baophutho.vn/che-do-dinh-duong-bao-ve-suc-khoe-mat-cho-nguoi-bi-loan-thi-230932.htm',
  'domain': 'baophutho.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 81069,
  'url': 'https://baophutho.vn/chu-dong-ung-pho-voi-covid-19-233261.htm',
  'domain': 'baophutho.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 81085,
  'url': 'https://baophutho.vn/co-nen-cho-con-an-chay-tu-be-234220.htm',
  'domain': 'baophutho.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 81114,
  'url': 'https://baophutho.vn/de-vet-mo-sau-sinh-mau-lanh-va-khong-de-lai-seo-240108.htm',
  'domain': 'baophutho.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 81129,
  'url': 'https://baophutho.vn/diem-sang-phong-trao-thi-dua-yeu-nuoc-nganh-y-te-248500.htm',
  'domain': 'baophutho.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 81154,
  'url': 'https://baophutho.vn/dinh-duong-ho-tro-dieu-tri-hoai-tu-chom-xuong-dui-231759.htm',
  'domain': 'baophutho.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 81561,
  'url': 'https://baophutho.vn/thoi-diem-an-sang-tot-nhat-de-giam-can-241407.htm',
  'domain': 'baophutho.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 81566,
  'url': 'https://baophutho.vn/thoi-diem-tot-nhat-de-an-bua-sang-neu-ban-bi-cholesterol-cao-247648.htm',
  'domain': 'baophutho.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 81638,
  'url': 'https://baophutho.vn/trung-tam-nbsp-tim-mach-chuyen-sau-hang-dau-khu-vuc-243269.htm',
  'domain': 'baophutho.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 81691,
  'url': 'https://baophutho.vn/vi-suc-khoe-nhan-dan-242586.htm',
  'domain': 'baophutho.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 81715,
  'url': 'https://baoquangninh.vn/113-truong-hop-duoc-kham-chua-benh-tu-chuyen-gia-y-te-cua-cong-hoa-phap-tai-benh-vien-viet-nam-thuy-3299037.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 82089,
  'url': 'https://baoquangninh.vn/bo-y-te-bo-tren-45-tuoi-me-ngoai-35-co-nguy-co-sinh-con-bi-khuyet-tat-3224614.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 82108,
  'url': 'https://baoquangninh.vn/bo-y-te-chi-dao-tang-cuong-phong-chong-dich-covid-19-tai-co-so-kham-chua-benh-2511711.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 82191,
  'url': 'https://baoquangninh.vn/bo-y-te-tang-cuong-kiem-tra-niem-yet-gia-thuoc-dieu-tri-covid-19-3176871.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 82365,
  'url': 'https://baoquangninh.vn/cap-nhat-cac-xet-nghiem-trong-chan-doan-va-dieu-tri-3208460.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 82385,
  'url': 'https://baoquangninh.vn/cham-lo-tot-cho-tre-em-la-chuan-bi-chu-dao-nhan-luc-cho-tuong-lai-3167459.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 82521,
  'url': 'https://baoquangninh.vn/chung-tay-vi-suc-khoe-cong-dong-3248134.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 82595,
  'url': 'https://baoquangninh.vn/cuc-pin-duong-kinh-15mm-da-chay-khet-80-nam-trong-mui-be-gai-4-tuoi-3261444.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 82733,
  'url': 'https://baoquangninh.vn/dau-hieu-canh-bao-chuc-nang-than-suy-giam-2463041.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 82921,
  'url': 'https://baoquangninh.vn/dua-mot-thuy-thu-tau-nuoc-ngoai-bi-thuong-vao-dieu-tri-tai-nha-trang-3216532.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 83159,
  'url': 'https://baoquangninh.vn/huyen-van-don-to-chuc-ngay-hoi-hien-mau-tinh-nguyen-dot-3-nam-2025-3361582.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 83196,
  'url': 'https://baoquangninh.vn/kham-suc-khoe-hoc-duong-than-toc-30-giay-hoc-sinh-3330202.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 83329,
  'url': 'https://baoquangninh.vn/lua-chon-gioi-tinh-thai-nhi-la-hanh-dong-vi-pham-phap-luat-2404167.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 83830,
  'url': 'https://baoquangninh.vn/phong-20-chung-vi-khuan-phe-cau-bang-vaccine-pcv20-cong-nghe-moi-3359868.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 84041,
  'url': 'https://baoquangninh.vn/superpath-dot-pha-moi-voi-ky-thuat-thay-khop-hang-nhan-tao-it-xam-lan-3328095.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 84161,
  'url': 'https://baoquangninh.vn/thap-sang-tuong-lai-cho-mam-xanh-vung-phen-dau-to-quoc-3383779.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 84169,
  'url': 'https://baoquangninh.vn/them-co-hoi-bao-ve-suc-khoe-nguoi-dan-3191283.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 84215,
  'url': 'https://baoquangninh.vn/thu-tuong-phe-duyet-nhiem-vu-lap-quy-hoach-mang-luoi-co-so-y-te-2484640.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 84401,
  'url': 'https://baoquangninh.vn/tu-y-dap-la-tri-tac-tuyen-sua-san-phu-bi-nhiem-trung-chay-mu-2488176.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 84485,
  'url': 'https://baoquangninh.vn/viem-loet-giac-mac-de-lai-nhieu-di-chung-cho-mat-2435286.html',
  'domain': 'baoquangninh.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 84894,
  'url': 'https://baoquangtri.vn/multimedia/infographic/202601/muc-tieu-tu-nam-2030-100-nguoi-dan-co-so-suc-khoe-dien-tu-2c67ad2/',
  'domain': 'baoquangtri.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 85117,
  'url': 'https://baoquangtri.vn/suc-khoe/201603/cuoi-thang-3-se-tiem-ngua-soi-rubella-cho-hoc-sinh-thpt-2133317/',
  'domain': 'baoquangtri.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 85534,
  'url': 'https://baoquangtri.vn/suc-khoe/201703/cac-bieu-hien-cua-nguoi-mac-benh-ho-ga-nhu-the-nao-2143417/',
  'domain': 'baoquangtri.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 85641,
  'url': 'https://baoquangtri.vn/suc-khoe/201706/infographics-hien-mau-tinh-nguyen-nhung-dieu-can-biet-2146354/',
  'domain': 'baoquangtri.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 85884,
  'url': 'https://baoquangtri.vn/suc-khoe/201801/bo-y-te-khuyen-cao-cach-tranh-soc-nhiet-trong-ret-dam-ai-cung-can-biet-2152995/',
  'domain': 'baoquangtri.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 86337,
  'url': 'https://baoquangtri.vn/suc-khoe/201901/san-pham-co-nguy-co-nhiem-salmonella-poona-da-nhap-khau-ve-viet-nam-2163925/',
  'domain': 'baoquangtri.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 86409,
  'url': 'https://baoquangtri.vn/suc-khoe/201903/xac-dinh-ham-luong-sam-bo-chinh-tren-dinh-nui-chop-chai-2164973/',
  'domain': 'baoquangtri.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 86443,
  'url': 'https://baoquangtri.vn/suc-khoe/201904/viet-nam-da-xuat-hien-vi-khuan-da-khang-toan-khang-2166165/',
  'domain': 'baoquangtri.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 86862,
  'url': 'https://baoquangtri.vn/suc-khoe/202002/phong-chong-dich-corona-san-sang-ung-pho-voi-moi-cap-do-lay-nhiem-2174108/',
  'domain': 'baoquangtri.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 87168,
  'url': 'https://baoquangtri.vn/suc-khoe/202006/khong-ghi-nhan-ca-mac-moi-benh-nhan-91-rut-duoc-ong-tho-qua-khi-quan-2178381/',
  'domain': 'baoquangtri.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 87199,
  'url': 'https://baoquangtri.vn/suc-khoe/202007/81-ngay-viet-nam-khong-co-ca-lay-nhiem-covid-19-trong-cong-dong-2178941/',
  'domain': 'baoquangtri.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 87946,
  'url': 'https://baoquangtri.vn/suc-khoe/202106/tiem-vac-xin-phong-covid-19-dot-2-cong-khai-minh-bach-dung-lo-trinh-va-doi-tuong-2190146/',
  'domain': 'baoquangtri.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 88284,
  'url': 'https://baoquangtri.vn/suc-khoe/202111/mot-so-luu-y-sau-tiem-vaccine-phong-covid-19-cho-tre-em-2195062/',
  'domain': 'baoquangtri.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 88314,
  'url': 'https://baoquangtri.vn/suc-khoe/202111/tuyen-hoa-kich-hoat-khu-dieu-tri-da-chien-tiep-nhan-benh-nhan-covid-19-2195621/',
  'domain': 'baoquangtri.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 88556,
  'url': 'https://baoquangtri.vn/suc-khoe/202205/6-hieu-nham-pho-bien-ve-benh-dau-mua-khi-2200680/',
  'domain': 'baoquangtri.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 88816,
  'url': 'https://baoquangtri.vn/suc-khoe/202211/thuoc-la-dien-tu-len-loi-vao-hoc-duong-can-theo-doi-dau-hieu-la-o-tre-2204937/',
  'domain': 'baoquangtri.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 88977,
  'url': 'https://baoquangtri.vn/suc-khoe/202306/dieu-chinh-covid-19-tu-nhom-a-sang-nhom-b-xem-xet-cong-bo-het-dich-2210117/',
  'domain': 'baoquangtri.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 89117,
  'url': 'https://baoquangtri.vn/suc-khoe/202312/moi-truong-lao-dong-hiem-hoa-cua-benh-nghe-nghiep-2214643/',
  'domain': 'baoquangtri.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 89180,
  'url': 'https://baoquangtri.vn/suc-khoe/202403/nguy-co-thieu-vac-xin-phong-benh-dai-tren-nguoi-2216906/',
  'domain': 'baoquangtri.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 89588,
  'url': 'https://baoquangtri.vn/thoi-su/202002/can-giai-phap-moi-ung-pho-voi-dich-covid-19-2174797/',
  'domain': 'baoquangtri.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 90190,
  'url': 'https://baotayninh.vn/ben-luc-co-648-nguoi-dang-dieu-tri-bang-thuoc-arv-129768.html',
  'domain': 'baotayninh.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 90255,
  'url': 'https://baotayninh.vn/benh-vien-108-thuc-hien-thanh-cong-3-ca-ghep-tang-trong-dip-nghi-le-131299.html',
  'domain': 'baotayninh.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 90257,
  'url': 'https://baotayninh.vn/benh-vien-cho-ray-trien-khai-30-tong-dai-dac-biet-tu-van-kham-benh-online-31535.html',
  'domain': 'baotayninh.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 90284,
  'url': 'https://baotayninh.vn/binh-duong-long-an-day-nhanh-tien-do-tiem-chung-31551.html',
  'domain': 'baotayninh.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 90378,
  'url': 'https://baotayninh.vn/can-duoc-ra-quan-chien-dich-ve-sinh-moi-truong-dot-2-2024-122027.html',
  'domain': 'baotayninh.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 90524,
  'url': 'https://baotayninh.vn/duc-hoa-nang-cao-nang-luc-dieu-tri-sot-xuat-huyet-cho-cac-co-so-y-te-tu-nhan-124809.html',
  'domain': 'baotayninh.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 90579,
  'url': 'https://baotayninh.vn/hieu-qua-tu-chuong-trinh-nuoc-sach-hoc-duong-tai-huyen-tan-hung-105739.html',
  'domain': 'baotayninh.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 90601,
  'url': 'https://baotayninh.vn/hoi-thao-chuyen-de-tuan-le-dinh-duong-va-phat-trien-nam-2024-126066.html',
  'domain': 'baotayninh.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 90624,
  'url': 'https://baotayninh.vn/huong-ung-ngay-asean-phong-chong-benh-sot-xuat-huyet-dengue-108458.html',
  'domain': 'baotayninh.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 90661,
  'url': 'https://baotayninh.vn/kiem-tra-danh-gia-tinh-trang-dinh-duong-cua-tre-tai-can-duoc-102031.html',
  'domain': 'baotayninh.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 90729,
  'url': 'https://baotayninh.vn/long-an-trien-khai-thuc-hien-cong-dien-so-72-cua-thu-tuong-chinh-phu-de-bao-dam-du-thuoc-trang-thiet-106282.html',
  'domain': 'baotayninh.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 90832,
  'url': 'https://baotayninh.vn/nhung-nguoi-me-thu-hai-cua-tre-so-sinh-140038.html',
  'domain': 'baotayninh.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 90858,
  'url': 'https://baotayninh.vn/pho-chu-tich-ubnd-huyen-can-duoc-kiem-tra-cong-tac-phong-chong-dich-covid-19-108267.html',
  'domain': 'baotayninh.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 90862,
  'url': 'https://baotayninh.vn/phong-benh-sot-xuat-huyet-ngay-tu-trong-gia-dinh-96949.html',
  'domain': 'baotayninh.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 90883,
  'url': 'https://baotayninh.vn/sang-17-7-viet-nam-co-2-106-ca-mac-moi-covid-19-da-tiem-4-233-896-lieu-vaccine-31591.html',
  'domain': 'baotayninh.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 90953,
  'url': 'https://baotayninh.vn/tan-hung-bac-si-huyen-bien-gioi-tan-tam-voi-nghe-129262.html',
  'domain': 'baotayninh.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 90977,
  'url': 'https://baotayninh.vn/tan-hung-hop-mat-ky-niem-70-nam-ngay-thay-thuoc-viet-nam-129489.html',
  'domain': 'baotayninh.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 91174,
  'url': 'https://baotayninh.vn/tp-ho-chi-minh-co-gan-8-500-ca-mac-moi-trong-24-gio-qua-chua-phat-hien-bien-chung-moi-31492.html',
  'domain': 'baotayninh.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 91200,
  'url': 'https://baotayninh.vn/trang-bi-kien-thuc-bao-ve-suc-khoe-cho-hoc-sinh-truong-thpt-dan-toc-noi-tru-tinh-tay-ninh-137179.html',
  'domain': 'baotayninh.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 91244,
  'url': 'https://baotayninh.vn/uong-nuoc-chanh-chuyen-gia-chi-cach-pha-phong-ung-thu-benh-tim-134318.html',
  'domain': 'baotayninh.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 91324,
  'url': 'https://baothanhhoa.vn/bao-dam-cong-tac-kham-chua-benh-va-khong-de-dich-benh-bung-phat-sau-mua-bao-263801.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 91357,
  'url': 'https://baothanhhoa.vn/benh-vien-da-khoa-tinh-thanh-hoa-thuc-hien-thanh-cong-ky-thuat-thay-van-dong-mach-chu-qua-da-269030.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 91402,
  'url': 'https://baothanhhoa.vn/bo-y-te-de-nghi-siet-chat-kinh-doanh-thuc-pham-tren-thuong-mai-dien-tu-268744.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 91492,
  'url': 'https://baothanhhoa.vn/cai-thien-tinh-trang-suy-dinh-duong-o-tre-em-muong-lat-250922.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 91499,
  'url': 'https://baothanhhoa.vn/canh-bao-he-luy-cua-trao-luu-su-dung-thuoc-tieu-duong-de-giam-can-273051.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 91595,
  'url': 'https://baothanhhoa.vn/de-xuat-90-benh-can-sang-loc-chan-doan-dieu-tri-truoc-sinh-va-so-sinh-281410.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 91604,
  'url': 'https://baothanhhoa.vn/doan-cong-tac-benh-vien-e-khao-sat-chuyen-giao-ky-thuat-tai-benh-vien-da-khoa-tinh-thanh-hoa-269453.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 91651,
  'url': 'https://baothanhhoa.vn/ho-tro-dia-phuong-xu-ly-moi-truong-va-phong-chong-dich-benh-sau-bao-wipha-255963.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 91675,
  'url': 'https://baothanhhoa.vn/huong-dan-phong-chong-cac-dich-benh-trong-va-sau-lut-bao-mua-lu-252031.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 91698,
  'url': 'https://baothanhhoa.vn/khi-niem-vui-sum-vay-nhuong-cho-cho-trach-nhiem-278401.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 91699,
  'url': 'https://baothanhhoa.vn/kho-khan-trong-viec-xu-phat-nbsp-nguoi-hut-thuoc-la-tai-benh-vien-262422.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 91718,
  'url': 'https://baothanhhoa.vn/kiem-tra-cong-tac-khac-phuc-hau-qua-do-mua-lu-tai-mot-so-co-so-y-te-263619.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 91722,
  'url': 'https://baothanhhoa.vn/ky-ket-bien-ban-ghi-nho-hop-tac-giua-benh-vien-ung-buou-tinh-thanh-hoa-va-tap-doan-y-te-parkway-singapore-256130.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 91830,
  'url': 'https://baothanhhoa.vn/phat-hien-manh-moi-giup-lam-cham-tien-trien-benh-tieu-duong-tuyp-1-271865.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 91857,
  'url': 'https://baothanhhoa.vn/ra-soat-xu-ly-nghiem-nhan-vien-y-te-vi-pham-ke-don-thuc-pham-chuc-nang-va-sua-246535.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 91919,
  'url': 'https://baothanhhoa.vn/tap-huan-su-dung-he-thong-quan-ly-giam-sat-benh-truyen-nhiem-phien-ban-nang-cap-258052.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 91987,
  'url': 'https://baothanhhoa.vn/trien-khai-chien-dich-bo-sung-vitamin-a-va-tay-giun-cho-tre-dot-2-nam-2025-270506.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 92005,
  'url': 'https://baothanhhoa.vn/tuan-le-glocom-the-gioi-2026-chung-tay-vi-mot-the-gioi-khong-con-benh-glocom-280861.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 92021,
  'url': 'https://baothanhhoa.vn/ve-vung-dat-quy-huong-281632.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 92055,
  'url': 'https://baothanhhoa.vn/who-tai-khang-dinh-khong-co-moi-lien-he-giua-vaccine-va-chung-tu-ky-271527.htm',
  'domain': 'baothanhhoa.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 92100,
  'url': 'https://baovinhlong.com.vn/phong-su-anh/201702/nam-gioi-bi-than-hu-nen-an-gi-2785468/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 92353,
  'url': 'https://baovinhlong.com.vn/xa-hoi/202002/huong-dan-dieu-tri-va-phong-chong-benh-do-virus-corona-moi-gay-ra-2987153/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 92740,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201205/kip-thoi-boi-bo-tri-nao-cho-con-tao-thanh-tich-mua-thi-2573580/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 92826,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201211/can-than-khi-dung-paracetamol-2572249/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 93114,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201402/khuyen-cao-phong-chong-benh-soi-cua-bo-y-te-2570915/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 93719,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201512/cao-rau-khong-dung-cach-hai-suc-khoe-2652344/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 93810,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201602/the-tri-luc-nguoi-dan-tren-da-cai-thien-2666719/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 93831,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201603/dau-hieu-som-cua-ung-thu-luoi-2672318/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 94046,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201609/malaysia-benh-nhan-nhiem-virus-zika-tu-vong-do-bien-chung-tim-2731397/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 94295,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201708/tu-xu-tri-ca-vo-gan-noi-ve-chuyen-tuyen-2833761/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 94448,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201808/dat-stent-mach-vanh-cuu-thanh-cong-benh-nhan-nhoi-mau-co-tim-cap-2907994/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 94450,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201808/hai-than-vi-tan-trang-vung-kin-sai-cach-2905740/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 94532,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201811/oi-kho-dinh-duong-khong-phai-ai-cung-biet-2920610/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 94825,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/201910/be-gai-7-tuoi-day-thi-som-vi-bat-den-luc-ngu-suot-3-nam-2968773/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 95458,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202009/sang-109-viet-nam-da-8-ngay-khong-co-ca-mac-moi-covid-19-trong-cong-dong-3021344/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 95711,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202102/vinh-long-truy-vet-cach-ly-5-truong-hop-f1-cua-benh-nhan-covid-19-3043482/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 96367,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202203/hieu-ro-viec-tiem-vac-xin-phong-covid-19-cho-tre-em-3106398/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 96648,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202310/thang-loi-dai-dich-covid-19-la-thang-loi-cua-nhan-dan-3176918/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 96820,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202502/mua-trai-mua-phong-benh-sot-xuat-huyet-1752ef1/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 96970,
  'url': 'https://baovinhlong.com.vn/xa-hoi/y-te/202509/viet-nam-lan-dau-tien-co-vaccine-virus-hop-bao-ho-hap-rsv-e1e21f2/',
  'domain': 'baovinhlong.com.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 97237,
  'url': 'https://benhviennhitrunguong.gov.vn/bac-si-benh-vien-nhi-trung-uong-huong-dan-chuom-am-ha-sot-tai-nha-cho-tre.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 97306,
  'url': 'https://benhviennhitrunguong.gov.vn/be-trai-36-ngay-tuoi-suyt-tu-vong-do-bien-chung-tieu-duong-so-sinh.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 97403,
  'url': 'https://benhviennhitrunguong.gov.vn/cac-hoat-dong-ho-tro-giao-tiep-cho-tre-tu-ky.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 97442,
  'url': 'https://benhviennhitrunguong.gov.vn/canh-bao-gia-tang-truong-hop-bi-cho-meo-vat-nuoi-can-gay-chan-thuong-trong-cac-dip-le-tet.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 97449,
  'url': 'https://benhviennhitrunguong.gov.vn/canh-bao-nguyen-nhan-dan-den-suy-than-man-o-tre.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 97531,
  'url': 'https://benhviennhitrunguong.gov.vn/chan-thuong-than-o-tre-em.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 97550,
  'url': 'https://benhviennhitrunguong.gov.vn/chu-dong-phong-covid-19-cho-tre-em-bang-vaccine-la-can-thiet.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 97583,
  'url': 'https://benhviennhitrunguong.gov.vn/cuu-song-2-chau-be-bi-bo-ep-uong-thuoc-doc.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 97660,
  'url': 'https://benhviennhitrunguong.gov.vn/dieu-tri-nang-thung-tinh-tran-dich-mang-tinh-hoan-va-thoat-vi-ben-nen-mo-mo-hay-noi-soi.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 97775,
  'url': 'https://benhviennhitrunguong.gov.vn/hep-khi-quan-phuc-tap-tren-nen-tim-bam-sinh-be-gai-8-thang-tuoi-nguoi-lao-duoc-cuu-khoi-cua-tu.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 97811,
  'url': 'https://benhviennhitrunguong.gov.vn/hoi-suc-luu-dong-cuu-song-be-gai-soc-thuoc-tai-ninh-binh.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 97814,
  'url': 'https://benhviennhitrunguong.gov.vn/huong-dan-an-bo-sung-cho-be-2.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 97884,
  'url': 'https://benhviennhitrunguong.gov.vn/huong-ung-thang-hanh-dong-vi-binh-dang-gioi-va-phong-ngua-ung-pho-voi-bao-luc-tren-co-so-gioi-nam-2025.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 97920,
  'url': 'https://benhviennhitrunguong.gov.vn/khong-duoc-tiem-vac-xin-nhieu-tre-mac-ho-ga-dan-den-suy-ho-hap.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 97932,
  'url': 'https://benhviennhitrunguong.gov.vn/la-thu-cam-on-xuc-dong-tu-benh-nhi-va-gia-dinh-gui-toi-y-bac-si-da-cuu-con-thoat-khoi-tu-than.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 97986,
  'url': 'https://benhviennhitrunguong.gov.vn/mot-so-sai-lam-trong-cham-soc-tre-tieu-chay-khien-benh-nang-them.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 98131,
  'url': 'https://benhviennhitrunguong.gov.vn/phau-thuat-lay-khoi-u-toc-khong-lo-trong-co-the-be-gai-11-tuoi.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 98167,
  'url': 'https://benhviennhitrunguong.gov.vn/phong-ngua-va-xu-tri-thoat-mach-hoa-chat.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 98333,
  'url': 'https://benhviennhitrunguong.gov.vn/tu-lam-thit-coc-de-an-chi-em-sinh-doi-bi-ngo-doc-nang.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 98352,
  'url': 'https://benhviennhitrunguong.gov.vn/uong-nham-axit-be-trai-7-tuoi-phai-cat-bo-23-da-day-2.html',
  'domain': 'benhviennhitrunguong.gov.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 98432,
  'url': 'https://benhvienvietduc.org/benh-ly-van-tim-va-cac-loai-van-tim-nhan-tao.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 98442,
  'url': 'https://benhvienvietduc.org/benh-tri-va-cac-phuong-phap-dieu-tri-2.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 98445,
  'url': 'https://benhvienvietduc.org/benh-vien-huu-nghi-viet-duc-cham-soc-nguoi-benh-sau-phau-thuat-2.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 98511,
  'url': 'https://benhvienvietduc.org/chan-thuong-nao-tre-em.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 98519,
  'url': 'https://benhvienvietduc.org/che-dinh-duong-cho-nguoi-benh-chan-thuong-nao.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 98526,
  'url': 'https://benhvienvietduc.org/che-dinh-duong-sau-ghep-tim.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 98553,
  'url': 'https://benhvienvietduc.org/de-phong-benh-ly-chan-thuong.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 98590,
  'url': 'https://benhvienvietduc.org/gia-tri-cua-ngan-hang-mo-trong-chan-thuong-chinh-hinh-2.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 98660,
  'url': 'https://benhvienvietduc.org/nhieu-ban-tre-dang-mat-kiem-soat-can-nang.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 98696,
  'url': 'https://benhvienvietduc.org/nhung-yeu-nguy-co-cua-benh-dong-mach-chi-duoi-man-tinh-2.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 98713,
  'url': 'https://benhvienvietduc.org/phinh-dong-mach-nao-nguy-hiem-nhu-nao.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 98718,
  'url': 'https://benhvienvietduc.org/phuc-hoi-chuc-nang-chan-thuong-thao-rach-co.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 98720,
  'url': 'https://benhvienvietduc.org/phuc-hoi-chuc-nang-cho-nguoi-benh-sau-phau-thuat-noi-soi-hep-khoang-duoi-mom-cung-vai.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 98721,
  'url': 'https://benhvienvietduc.org/phuc-hoi-chuc-nang-dau-xo-co.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 98744,
  'url': 'https://benhvienvietduc.org/tap-luyen-va-nhung-luu-y-doi-voi-nguoi-benh-sau-mo-thay-khop-hang.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 98754,
  'url': 'https://benhvienvietduc.org/thu-pham-gay-ra-benh-tri-la-gi.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 98763,
  'url': 'https://benhvienvietduc.org/tim-hieu-ve-hoi-suc-nguoi-benh-chan-thuong-nao.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 98790,
  'url': 'https://benhvienvietduc.org/ung-thu-da-day-va-nhung-dieu-can-biet.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 98791,
  'url': 'https://benhvienvietduc.org/ung-thu-dai-trang-benh-ly-ung-thu-pho-bien-dung-thu-5-tai-viet-nam.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 98804,
  'url': 'https://benhvienvietduc.org/vai-tro-cua-phuc-hoi-chuc-nang-sau-phau-thuat-dut-day-chang-cheo-truoc.html',
  'domain': 'benhvienvietduc.org',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 2502455,
  'url': 'https://bingli.iiyi.com/show/66406-1.html',
  'domain': 'bingli.iiyi.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2502469,
  'url': 'https://bingli.iiyi.com/show/66420-1.html',
  'domain': 'bingli.iiyi.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2502471,
  'url': 'https://bingli.iiyi.com/show/66423-1.html',
  'domain': 'bingli.iiyi.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2502476,
  'url': 'https://bingli.iiyi.com/show/66428-1.html',
  'domain': 'bingli.iiyi.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2502681,
  'url': 'https://bingli.iiyi.com/show/66749-1.html',
  'domain': 'bingli.iiyi.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 794140,
  'url': 'http://byby.39.net/a/20101111/1540015.html',
  'domain': 'byby.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 794145,
  'url': 'http://byby.39.net/a/20101114/1539875.html',
  'domain': 'byby.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 794173,
  'url': 'http://byby.39.net/a/20101125/1548981.html',
  'domain': 'byby.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 794175,
  'url': 'http://byby.39.net/a/20101126/1548838.html',
  'domain': 'byby.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 794182,
  'url': 'http://byby.39.net/a/2010113/1527671.html',
  'domain': 'byby.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 794184,
  'url': 'http://byby.39.net/a/20101130/1540908.html',
  'domain': 'byby.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 794185,
  'url': 'http://byby.39.net/a/20101130/1548848.html',
  'domain': 'byby.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 794241,
  'url': 'http://byby.39.net/a/2010823/1455528.html',
  'domain': 'byby.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 794252,
  'url': 'http://byby.39.net/a/2010827/1455003.html',
  'domain': 'byby.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 794254,
  'url': 'http://byby.39.net/a/2010827/1461536.html',
  'domain': 'byby.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 794262,
  'url': 'http://byby.39.net/a/2010916/1481861.html',
  'domain': 'byby.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 794366,
  'url': 'http://byby.39.net/a/230529/13823767.html',
  'domain': 'byby.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 794393,
  'url': 'http://byby.39.net/a/230601/13824087.html',
  'domain': 'byby.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 794410,
  'url': 'http://byby.39.net/a/230602/13824203.html',
  'domain': 'byby.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 794424,
  'url': 'http://byby.39.net/a/230830/a4dcu2y.html',
  'domain': 'byby.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 794451,
  'url': 'http://byby.39.net/a/231111/a7j4alv.html',
  'domain': 'byby.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 794467,
  'url': 'http://byby.39.net/a/231129/k3jxnc8.html',
  'domain': 'byby.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 794473,
  'url': 'http://byby.39.net/a/231203/o8k4my0.html',
  'domain': 'byby.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 794475,
  'url': 'http://byby.39.net/a/231209/k6kekhs.html',
  'domain': 'byby.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 794480,
  'url': 'http://byby.39.net/a/231209/s2kekuh.html',
  'domain': 'byby.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 795051,
  'url': 'http://cancer.39.net/a/20111020/1828097.html',
  'domain': 'cancer.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 795433,
  'url': 'http://cancer.39.net/a/2011512/1688598.html',
  'domain': 'cancer.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 795526,
  'url': 'http://cancer.39.net/a/2011621/1721385.html',
  'domain': 'cancer.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 795737,
  'url': 'http://cancer.39.net/a/2011819/1782436.html',
  'domain': 'cancer.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2503979,
  'url': 'https://cancer.39.net/a/130401/4148472.html',
  'domain': 'cancer.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2504067,
  'url': 'https://cancer.39.net/a/130428/4164262.html',
  'domain': 'cancer.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2504278,
  'url': 'https://cancer.39.net/a/130830/4245985.html',
  'domain': 'cancer.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2504579,
  'url': 'https://cancer.39.net/a/140326/4360085.html',
  'domain': 'cancer.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2505651,
  'url': 'https://cancer.39.net/a/150228/4582356.html',
  'domain': 'cancer.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2506009,
  'url': 'https://cancer.39.net/a/150720/4658435.html',
  'domain': 'cancer.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2506355,
  'url': 'https://cancer.39.net/a/160328/4802076.html',
  'domain': 'cancer.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2506866,
  'url': 'https://cancer.39.net/a/180503/6206652.html',
  'domain': 'cancer.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2506907,
  'url': 'https://cancer.39.net/a/180607/6310358.html',
  'domain': 'cancer.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2507364,
  'url': 'https://cancer.39.net/a/190929/7504371.html',
  'domain': 'cancer.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2507948,
  'url': 'https://cancer.39.net/a/210630/9104060.html',
  'domain': 'cancer.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2507973,
  'url': 'https://cancer.39.net/a/210716/9158388.html',
  'domain': 'cancer.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2508024,
  'url': 'https://cancer.39.net/a/210923/9462030.html',
  'domain': 'cancer.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2508038,
  'url': 'https://cancer.39.net/a/211020/9562563.html',
  'domain': 'cancer.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2508156,
  'url': 'https://cancer.39.net/a/220210/10252047.html',
  'domain': 'cancer.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2509080,
  'url': 'https://cancer.39.net/a/260702/q6wnh7j.html',
  'domain': 'cancer.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 796448,
  'url': 'http://care.39.net/a/111217/1865282.html',
  'domain': 'care.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 796780,
  'url': 'http://care.39.net/a/120820/4037295.html',
  'domain': 'care.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 797505,
  'url': 'http://care.39.net/a/140101/4365045.html',
  'domain': 'care.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 798096,
  'url': 'http://care.39.net/a/141210/4532111.html',
  'domain': 'care.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 803452,
  'url': 'http://care.39.net/a/200609/7956269.html',
  'domain': 'care.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 803942,
  'url': 'http://care.39.net/a/2010128/1563606.html',
  'domain': 'care.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 804145,
  'url': 'http://care.39.net/a/201042/1210084.html',
  'domain': 'care.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 807482,
  'url': 'http://care.39.net/a/2012223/1979288.html',
  'domain': 'care.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 807506,
  'url': 'http://care.39.net/a/201225/8524105.html',
  'domain': 'care.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 807524,
  'url': 'http://care.39.net/a/201229/8542328.html',
  'domain': 'care.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 808208,
  'url': 'http://care.39.net/a/210323/8770438.html',
  'domain': 'care.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 808249,
  'url': 'http://care.39.net/a/210402/8806241.html',
  'domain': 'care.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 808314,
  'url': 'http://care.39.net/a/210429/8902106.html',
  'domain': 'care.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 808399,
  'url': 'http://care.39.net/a/210518/8962935.html',
  'domain': 'care.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 809537,
  'url': 'http://care.39.net/a/220523/10876926.html',
  'domain': 'care.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 809992,
  'url': 'http://care.39.net/a/221111/12298613.html',
  'domain': 'care.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 810029,
  'url': 'http://care.39.net/a/221127/8439830.html',
  'domain': 'care.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 810478,
  'url': 'http://care.39.net/a/230927/j9gmj30.html',
  'domain': 'care.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 810570,
  'url': 'http://care.39.net/a/231018/v5i6yha.html',
  'domain': 'care.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 810976,
  'url': 'http://care.39.net/a/240228/r5n2e5f.html',
  'domain': 'care.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 99130,
  'url': 'https://dantri.com.vn/suc-khoe/benh-vien-bach-mai-khoi-cong-khoa-kham-benh-dap-ung-7000-luot-khamngay-20251030184505740.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 99325,
  'url': 'https://dantri.com.vn/suc-khoe/canh-bao-nhung-benh-ly-de-bung-phat-va-gay-bien-chung-nang-trong-mua-he-20250721153745089.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 99349,
  'url': 'https://dantri.com.vn/suc-khoe/cham-do-viet-nam-tren-ban-do-ghep-phoi-the-gioi-20260224191630060.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 99374,
  'url': 'https://dantri.com.vn/suc-khoe/chau-be-bi-cho-can-vao-mat-phai-khau-20-mui-20260121151352781.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 99508,
  'url': 'https://dantri.com.vn/suc-khoe/dan-ong-dang-cuong-dieu-hoa-kich-thuoc-cau-nho-20170411204915449.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 99591,
  'url': 'https://dantri.com.vn/suc-khoe/dien-quang-can-thiep-loi-thoat-cho-nguoi-song-chung-voi-dau-dau-man-tinh-20250822003959692.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 99653,
  'url': 'https://dantri.com.vn/suc-khoe/gap-thanh-cong-xuong-ca-dam-thung-ruot-non-20200929181208673.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 99710,
  'url': 'https://dantri.com.vn/suc-khoe/ha-noi-se-co-mo-to-cap-cuu-vao-duoc-tung-ngo-ngach-cuu-nguoi-20260225194007301.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 99902,
  'url': 'https://dantri.com.vn/suc-khoe/loi-tu-biet-nghen-ngao-cua-nguoi-me-truoc-ca-hien-tang-cuu-8-cuoc-doi-20260223183040140.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 100176,
  'url': 'https://dantri.com.vn/suc-khoe/nhung-dieu-can-biet-khi-bo-sung-thuc-pham-chuc-nang-cho-co-the-20240805163321903.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 100194,
  'url': 'https://dantri.com.vn/suc-khoe/nhung-nguoi-linh-50-nam-cuoi-ngua-sat-khien-bom-dan-cui-dau-20260216134047870.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 100263,
  'url': 'https://dantri.com.vn/suc-khoe/pharmacity-va-viatris-hop-tac-huong-toi-muc-tieu-nang-cao-suc-khoe-cong-dong-20250415155356820.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 100282,
  'url': 'https://dantri.com.vn/suc-khoe/phau-thuat-cat-ban-phan-da-day-cuu-cu-ba-70-tuoi-mac-ung-thu-tai-vinmec-nha-trang-20251210173209047.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 100403,
  'url': 'https://dantri.com.vn/suc-khoe/song-khoe/ung-thu.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 100404,
  'url': 'https://dantri.com.vn/suc-khoe/sot-xuat-huyet-dengue-va-nhung-du-dinh-do-dang-ben-giuong-benh-20260320192813385.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 100424,
  'url': 'https://dantri.com.vn/suc-khoe/tai-nan-lien-hoan-tren-pho-nguyen-chanh-ha-noi-mot-phu-nu-tu-vong-20260307114133341.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 100444,
  'url': 'https://dantri.com.vn/suc-khoe/tap-di-sau-6-tieng-mo-thay-khop-goi-nho-ung-dung-cong-nghe-3d-20251117134635980.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 100514,
  'url': 'https://dantri.com.vn/suc-khoe/thung-ruot-non-sau-an-ca-ro-phi-20170303140352382.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 100587,
  'url': 'https://dantri.com.vn/suc-khoe/tphcm-khan-cap-chuyen-cac-tui-thuoc-giup-cuu-chua-ba-con-anh-huong-bao-20251007144651824.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 100748,
  'url': 'https://dantri.com.vn/suc-khoe/viet-nam-co-loai-ngu-coc-duoc-vi-la-vang-den-rat-tot-cho-than-20240905075452771.htm',
  'domain': 'dantri.com.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 812100,
  'url': 'http://ek.39.net/a/150728/4662600.html',
  'domain': 'ek.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 812360,
  'url': 'http://ek.39.net/a/170804/5536372.html',
  'domain': 'ek.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 812719,
  'url': 'http://ek.39.net/a/2010815/1434737.html',
  'domain': 'ek.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 812725,
  'url': 'http://ek.39.net/a/2010820/1445248.html',
  'domain': 'ek.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 812776,
  'url': 'http://ek.39.net/a/2010924/1486183.html',
  'domain': 'ek.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 813207,
  'url': 'http://ek.39.net/a/2011512/1688621.html',
  'domain': 'ek.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 813267,
  'url': 'http://ek.39.net/a/2011527/1703091.html',
  'domain': 'ek.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 813298,
  'url': 'http://ek.39.net/a/201157/1683476.html',
  'domain': 'ek.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 813335,
  'url': 'http://ek.39.net/a/2011616/1722173.html',
  'domain': 'ek.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 813362,
  'url': 'http://ek.39.net/a/2011621/1727080.html',
  'domain': 'ek.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 813532,
  'url': 'http://ek.39.net/a/2011815/1778515.html',
  'domain': 'ek.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 813542,
  'url': 'http://ek.39.net/a/2011818/1782052.html',
  'domain': 'ek.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 813574,
  'url': 'http://ek.39.net/a/2011829/1792038.html',
  'domain': 'ek.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 813645,
  'url': 'http://ek.39.net/a/2011924/1814663.html',
  'domain': 'ek.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 813650,
  'url': 'http://ek.39.net/a/2011926/1814435.html',
  'domain': 'ek.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 813659,
  'url': 'http://ek.39.net/a/2011928/1816868.html',
  'domain': 'ek.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 813960,
  'url': 'http://ek.39.net/a/241031/t6ucrpp.html',
  'domain': 'ek.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 814241,
  'url': 'http://ek.39.net/a/260121/i3wd5p9.html',
  'domain': 'ek.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 814493,
  'url': 'http://ek.39.net/a/260408/d8wh9cx.html',
  'domain': 'ek.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 814594,
  'url': 'http://ek.39.net/a/260604/b4wlp0v.html',
  'domain': 'ek.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 817250,
  'url': 'http://fitness.39.net/a/170220/5216037.html',
  'domain': 'fitness.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 817928,
  'url': 'http://fitness.39.net/a/170925/5715342.html',
  'domain': 'fitness.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 818077,
  'url': 'http://fitness.39.net/a/171114/5838746.html',
  'domain': 'fitness.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 818263,
  'url': 'http://fitness.39.net/a/180118/6005429.html',
  'domain': 'fitness.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 818319,
  'url': 'http://fitness.39.net/a/180205/6035020.html',
  'domain': 'fitness.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 818931,
  'url': 'http://fitness.39.net/a/180807/6442869.html',
  'domain': 'fitness.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 821823,
  'url': 'http://fitness.39.net/a/200807/8099911.html',
  'domain': 'fitness.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 822964,
  'url': 'http://fitness.39.net/a/2010811/1452200.html',
  'domain': 'fitness.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 823098,
  'url': 'http://fitness.39.net/a/2010822/1452805.html',
  'domain': 'fitness.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 823227,
  'url': 'http://fitness.39.net/a/201085/1451642.html',
  'domain': 'fitness.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 823414,
  'url': 'http://fitness.39.net/a/20111130/1850983.html',
  'domain': 'fitness.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 823727,
  'url': 'http://fitness.39.net/a/201132/1623539.html',
  'domain': 'fitness.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 824163,
  'url': 'http://fitness.39.net/a/201169/1715969.html',
  'domain': 'fitness.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 824201,
  'url': 'http://fitness.39.net/a/2011715/1748987.html',
  'domain': 'fitness.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 824451,
  'url': 'http://fitness.39.net/a/2011910/1802133.html',
  'domain': 'fitness.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 824665,
  'url': 'http://fitness.39.net/a/2012221/1977585.html',
  'domain': 'fitness.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 825753,
  'url': 'http://fitness.39.net/a/220311/10417286.html',
  'domain': 'fitness.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 825773,
  'url': 'http://fitness.39.net/a/220324/10492256.html',
  'domain': 'fitness.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 826520,
  'url': 'http://fitness.39.net/a/240113/b2lrnu7.html',
  'domain': 'fitness.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 826637,
  'url': 'http://fitness.39.net/a/240402/y5o4eke.html',
  'domain': 'fitness.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 827254,
  'url': 'http://fk.39.net/a/121116/4084215.html',
  'domain': 'fk.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 827374,
  'url': 'http://fk.39.net/a/121223/4100478.html',
  'domain': 'fk.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 827410,
  'url': 'http://fk.39.net/a/121231/4099489.html',
  'domain': 'fk.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 828060,
  'url': 'http://fk.39.net/a/130827/4243023.html',
  'domain': 'fk.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 828336,
  'url': 'http://fk.39.net/a/131106/4285110.html',
  'domain': 'fk.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 828431,
  'url': 'http://fk.39.net/a/131201/4298928.html',
  'domain': 'fk.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 828736,
  'url': 'http://fk.39.net/a/140324/4358314.html',
  'domain': 'fk.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 829196,
  'url': 'http://fk.39.net/a/140807/4441908.html',
  'domain': 'fk.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 829326,
  'url': 'http://fk.39.net/a/140903/4460950.html',
  'domain': 'fk.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 829448,
  'url': 'http://fk.39.net/a/141012/4491641.html',
  'domain': 'fk.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 829562,
  'url': 'http://fk.39.net/a/141030/4504264.html',
  'domain': 'fk.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 832094,
  'url': 'http://fk.39.net/a/191220/7701010.html',
  'domain': 'fk.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 832370,
  'url': 'http://fk.39.net/a/2010111/1526427.html',
  'domain': 'fk.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 832545,
  'url': 'http://fk.39.net/a/20101217/1568275.html',
  'domain': 'fk.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 832970,
  'url': 'http://fk.39.net/a/20111110/1839926.html',
  'domain': 'fk.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 833068,
  'url': 'http://fk.39.net/a/20111217/1864151.html',
  'domain': 'fk.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 833975,
  'url': 'http://fk.39.net/a/2011728/1761355.html',
  'domain': 'fk.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 834594,
  'url': 'http://fk.39.net/a/2012413/2005996.html',
  'domain': 'fk.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 834888,
  'url': 'http://fk.39.net/a/210226/8692327.html',
  'domain': 'fk.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 835392,
  'url': 'http://fk.39.net/a/230728/t4acicv.html',
  'domain': 'fk.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 836748,
  'url': 'http://food.39.net/a/121130/4090247.html',
  'domain': 'food.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 837626,
  'url': 'http://food.39.net/a/141109/4513288.html',
  'domain': 'food.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 837721,
  'url': 'http://food.39.net/a/141213/4534840.html',
  'domain': 'food.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 837769,
  'url': 'http://food.39.net/a/141226/4543939.html',
  'domain': 'food.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 838333,
  'url': 'http://food.39.net/a/150716/4656973.html',
  'domain': 'food.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 838551,
  'url': 'http://food.39.net/a/151116/4726117.html',
  'domain': 'food.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 838563,
  'url': 'http://food.39.net/a/151128/4733877.html',
  'domain': 'food.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 842635,
  'url': 'http://food.39.net/a/200424/7858631.html',
  'domain': 'food.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 843012,
  'url': 'http://food.39.net/a/201020/8310145.html',
  'domain': 'food.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 843018,
  'url': 'http://food.39.net/a/201020/8315135.html',
  'domain': 'food.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 844604,
  'url': 'http://food.39.net/a/2011224/1619647.html',
  'domain': 'food.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 845473,
  'url': 'http://food.39.net/a/2011812/1774742.html',
  'domain': 'food.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 845570,
  'url': 'http://food.39.net/a/201183/1766001.html',
  'domain': 'food.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 846540,
  'url': 'http://food.39.net/a/231208/a3kcv3u.html',
  'domain': 'food.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 846651,
  'url': 'http://food.39.net/a/231208/d9kcunn.html',
  'domain': 'food.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 846655,
  'url': 'http://food.39.net/a/231208/e1kcv58.html',
  'domain': 'food.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 846675,
  'url': 'http://food.39.net/a/231208/e9kcuvr.html',
  'domain': 'food.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 846808,
  'url': 'http://food.39.net/a/231208/j9kcur3.html',
  'domain': 'food.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 846810,
  'url': 'http://food.39.net/a/231208/j9kcut9.html',
  'domain': 'food.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 847146,
  'url': 'http://food.39.net/a/231208/w4kcufj.html',
  'domain': 'food.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 847738,
  'url': 'http://gan.39.net/a/140121/4335976.html',
  'domain': 'gan.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 847837,
  'url': 'http://gan.39.net/a/140402/4418753.html',
  'domain': 'gan.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 847862,
  'url': 'http://gan.39.net/a/140417/4375022.html',
  'domain': 'gan.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 848075,
  'url': 'http://gan.39.net/a/140822/4454767.html',
  'domain': 'gan.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 848215,
  'url': 'http://gan.39.net/a/150320/4593760.html',
  'domain': 'gan.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 848675,
  'url': 'http://gan.39.net/a/151030/4718914.html',
  'domain': 'gan.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 848765,
  'url': 'http://gan.39.net/a/151111/4723813.html',
  'domain': 'gan.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 848938,
  'url': 'http://gan.39.net/a/151127/4733487.html',
  'domain': 'gan.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 849272,
  'url': 'http://gan.39.net/a/160116/4757351.html',
  'domain': 'gan.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 849277,
  'url': 'http://gan.39.net/a/160118/4760748.html',
  'domain': 'gan.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 850153,
  'url': 'http://gan.39.net/a/2010112/1529719.html',
  'domain': 'gan.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 850215,
  'url': 'http://gan.39.net/a/20101228/1581152.html',
  'domain': 'gan.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 850578,
  'url': 'http://gan.39.net/a/2011312/1632064.html',
  'domain': 'gan.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 850692,
  'url': 'http://gan.39.net/a/201141/1652225.html',
  'domain': 'gan.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 850838,
  'url': 'http://gan.39.net/a/2011521/1697084.html',
  'domain': 'gan.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 850866,
  'url': 'http://gan.39.net/a/2011531/1707118.html',
  'domain': 'gan.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 850971,
  'url': 'http://gan.39.net/a/201168/1712590.html',
  'domain': 'gan.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 851074,
  'url': 'http://gan.39.net/a/2011810/1774801.html',
  'domain': 'gan.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 851202,
  'url': 'http://gan.39.net/a/2012112/1878790.html',
  'domain': 'gan.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 852134,
  'url': 'http://gan.39.net/a/260121/q4wd5nb.html',
  'domain': 'gan.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 852819,
  'url': 'http://gc.39.net/a/131218/4310054.html',
  'domain': 'gc.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 852885,
  'url': 'http://gc.39.net/a/140304/4346159.html',
  'domain': 'gc.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 852909,
  'url': 'http://gc.39.net/a/140312/4364058.html',
  'domain': 'gc.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 852996,
  'url': 'http://gc.39.net/a/140430/4417380.html',
  'domain': 'gc.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 853088,
  'url': 'http://gc.39.net/a/140618/4415929.html',
  'domain': 'gc.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 853102,
  'url': 'http://gc.39.net/a/140629/4416254.html',
  'domain': 'gc.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 853151,
  'url': 'http://gc.39.net/a/140809/4443419.html',
  'domain': 'gc.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 853200,
  'url': 'http://gc.39.net/a/141010/4486836.html',
  'domain': 'gc.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 853317,
  'url': 'http://gc.39.net/a/160705/4890958.html',
  'domain': 'gc.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 853362,
  'url': 'http://gc.39.net/a/190131/6847887.html',
  'domain': 'gc.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 853433,
  'url': 'http://gc.39.net/a/2010816/1445649.html',
  'domain': 'gc.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 853442,
  'url': 'http://gc.39.net/a/2010819/1450815.html',
  'domain': 'gc.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 853447,
  'url': 'http://gc.39.net/a/2010821/1452523.html',
  'domain': 'gc.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 853510,
  'url': 'http://gc.39.net/a/2011112/1591857.html',
  'domain': 'gc.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 853567,
  'url': 'http://gc.39.net/a/2011325/1640345.html',
  'domain': 'gc.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 853721,
  'url': 'http://gc.39.net/a/2011819/1762769.html',
  'domain': 'gc.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 853808,
  'url': 'http://gc.39.net/a/2012119/1964165.html',
  'domain': 'gc.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 854190,
  'url': 'http://gc.39.net/a/240809/e4rdufq.html',
  'domain': 'gc.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 854193,
  'url': 'http://gc.39.net/a/240822/k9rxh31.html',
  'domain': 'gc.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 854287,
  'url': 'http://gc.39.net/a/241022/q1u6jfw.html',
  'domain': 'gc.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 101297,
  'url': 'https://giadinhonline.vn/3-dau-hieu-o-mat-canh-bao-ung-thu-gan-d204646.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 101536,
  'url': 'https://giadinhonline.vn/4-meo-hay-dieu-tri-viem-hong-don-gian-nhung-cuc-hieu-qua-d142585.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 101926,
  'url': 'https://giadinhonline.vn/5-tac-hai-cua-viec-an-nhieu-dau-phu-doi-voi-nam-gioi-d163047.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 102365,
  'url': 'https://giadinhonline.vn/7-can-benh-nguy-hiem-cua-nguoi-gia-nhung-de-bat-gap-o-nguoi-tre-d165318.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 103062,
  'url': 'https://giadinhonline.vn/an-rau-muong-bi-chuot-rut-dung-khong-vi-sao-nhieu-nguoi-gap-hien-tuong-nay-vao-ban-dem-d187175.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 104302,
  'url': 'https://giadinhonline.vn/bi-quyet-hoi-phuc-ky-dieu-cua-nguoi-benh-mach-vanh-d203741.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 105426,
  'url': 'https://giadinhonline.vn/cach-tri-nghet-mui-hieu-qua-d19823.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 105663,
  'url': 'https://giadinhonline.vn/canh-bao-benh-viem-nao-bung-phat-trong-mua-he-d93542.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 105767,
  'url': 'https://giadinhonline.vn/canh-giac-voi-can-benh-nguy-hiem-de-doa-tinh-mang-dip-tep-d166439.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 106013,
  'url': 'https://giadinhonline.vn/che-do-dinh-duong-tang-suc-khoe-de-khang-cho-f0-f1-cach-ly-d172604.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 107721,
  'url': 'https://giadinhonline.vn/duc-thuy-tinh-the-nhung-dieu-ban-nen-biet-d36777.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 108559,
  'url': 'https://giadinhonline.vn/huda-cong-bo-thong-tin-ve-le-hoi-bong-da-bien-2021-tai-7-tinh-mien-trung-d168773.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 108668,
  'url': 'https://giadinhonline.vn/khai-mac-trien-lam-y-te-quoc-te-viet-nam-lan-thu-12-d118247.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 108820,
  'url': 'https://giadinhonline.vn/khoi-nghi-he-cau-cuu-vi-thu-vui-hat-karaoke-cua-phu-huynh-d206776.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 108876,
  'url': 'https://giadinhonline.vn/khong-phai-dien-thoai-day-moi-la-nguyen-nhan-lon-nhat-gay-can-thi-o-tre-d192171.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 110366,
  'url': 'https://giadinhonline.vn/ngu-tu-the-nao-de-khong-bi-dau-lung-d51053.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 112024,
  'url': 'https://giadinhonline.vn/nhung-tac-dung-huu-hieu-cua-duong-doi-voi-suc-khoe-d89689.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 112036,
  'url': 'https://giadinhonline.vn/nhung-tac-hai-cua-song-di-dong-len-thai-nhi-d116238.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 112860,
  'url': 'https://giadinhonline.vn/phu-nu-sau-sinh-nen-an-gi-de-mau-ve-sua-d118818.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 116038,
  'url': 'https://giadinhonline.vn/tu-vi-thang-1-2015-cua-12-cung-hoang-dao-1-d38633.html',
  'domain': 'giadinhonline.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 854997,
  'url': 'http://gk.39.net/a/100224/1162932.html',
  'domain': 'gk.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 855074,
  'url': 'http://gk.39.net/a/100516/1270663.html',
  'domain': 'gk.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 855091,
  'url': 'http://gk.39.net/a/100531/1312202.html',
  'domain': 'gk.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 855196,
  'url': 'http://gk.39.net/a/100909/1468479.html',
  'domain': 'gk.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 855202,
  'url': 'http://gk.39.net/a/100912/1474164.html',
  'domain': 'gk.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 855225,
  'url': 'http://gk.39.net/a/100922/1480858.html',
  'domain': 'gk.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 855355,
  'url': 'http://gk.39.net/a/110225/1619859.html',
  'domain': 'gk.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 855358,
  'url': 'http://gk.39.net/a/110226/1619775.html',
  'domain': 'gk.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 855411,
  'url': 'http://gk.39.net/a/110425/1672634.html',
  'domain': 'gk.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 855461,
  'url': 'http://gk.39.net/a/110611/1716370.html',
  'domain': 'gk.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 855546,
  'url': 'http://gk.39.net/a/110909/1793715.html',
  'domain': 'gk.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 855664,
  'url': 'http://gk.39.net/a/120925/4058519.html',
  'domain': 'gk.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 855708,
  'url': 'http://gk.39.net/a/130101/4103485.html',
  'domain': 'gk.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 855727,
  'url': 'http://gk.39.net/a/130216/4125091.html',
  'domain': 'gk.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 855790,
  'url': 'http://gk.39.net/a/130625/4195451.html',
  'domain': 'gk.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 855876,
  'url': 'http://gk.39.net/a/140224/4341299.html',
  'domain': 'gk.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 855918,
  'url': 'http://gk.39.net/a/230907/14237461.html',
  'domain': 'gk.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 856024,
  'url': 'http://gk.39.net/a/231002/14262556.html',
  'domain': 'gk.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 856301,
  'url': 'http://gk.39.net/a/241010/s2tr3sl.html',
  'domain': 'gk.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 856375,
  'url': 'http://gk.39.net/a/250902/x3vs43s.html',
  'domain': 'gk.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 117812,
  'url': 'https://hanoimoi.vn/bat-trang-phat-dong-chien-dich-tong-ve-sinh-moi-truong-phong-chong-sot-xuat-huyet-718402.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 117879,
  'url': 'https://hanoimoi.vn/bo-truong-bo-y-te-khong-co-thuoc-gia-trong-benh-vien-705937.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 118042,
  'url': 'https://hanoimoi.vn/co-nen-ap-dung-che-do-an-kieng-cho-tre-beo-phi-696500.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 118154,
  'url': 'https://hanoimoi.vn/ghep-da-tang-domino-ky-tich-moi-cua-y-hoc-viet-nam-739654.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 118171,
  'url': 'https://hanoimoi.vn/giam-ty-le-mac-hoac-tu-vong-do-cac-benh-co-the-phong-duoc-bang-vac-xin-tren-dia-ban-thanh-pho-719102.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 118295,
  'url': 'https://hanoimoi.vn/ha-noi-xay-dung-quy-chuan-ky-thuat-dia-phuong-ve-nuoc-sach-sinh-hoat-722184.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 118386,
  'url': 'https://hanoimoi.vn/khoanh-khac-thieng-lieng-noi-canh-cua-phong-sinh-733921.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 118419,
  'url': 'https://hanoimoi.vn/kiem-tra-dot-xuat-an-toan-thuc-pham-doi-voi-banh-trung-thu-tai-2-khach-san-5-sao-716496.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 118455,
  'url': 'https://hanoimoi.vn/lien-tiep-ghi-nhan-2-truong-hop-mac-lien-cau-khuan-lon-710489.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 118480,
  'url': 'https://hanoimoi.vn/mit-tinh-huong-ung-ngay-an-toan-nguoi-benh-the-gioi-716365.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 118521,
  'url': 'https://hanoimoi.vn/ngan-chan-su-lay-lan-cua-dai-dich-thuoc-gia-707355.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 118522,
  'url': 'https://hanoimoi.vn/ngan-chan-tac-hai-nghiem-trong-cua-thuoc-la-704079.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 118611,
  'url': 'https://hanoimoi.vn/nuoc-chanh-co-phai-than-duoc-697338.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 118758,
  'url': 'https://hanoimoi.vn/tang-cuong-cong-tac-kham-phat-hien-dieu-tri-ca-benh-covid-19-702768.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 118801,
  'url': 'https://hanoimoi.vn/thang-hanh-dong-vi-tre-em-nam-2025-co-chu-de-uu-tien-nguon-luc-hoan-thanh-cac-muc-tieu-vi-tre-em-704140.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 118832,
  'url': 'https://hanoimoi.vn/thu-hoi-thuoc-diclofen-tren-toan-quoc-do-vi-pham-muc-do-2-ve-chat-luong-726359.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 118858,
  'url': 'https://hanoimoi.vn/tien-phong-ung-dung-cong-nghe-moi-trong-dieu-tri-benh-khong-lay-nhiem-738300.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 118864,
  'url': 'https://hanoimoi.vn/ton-vinh-20-co-so-y-te-xuat-sac-trong-cuoc-chien-chong-tac-hai-thuoc-la-728121.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 118882,
  'url': 'https://hanoimoi.vn/tray-hoi-vui-xuan-dung-quen-chu-dong-phong-benh-736059.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 118915,
  'url': 'https://hanoimoi.vn/vach-tran-su-hap-dan-gia-tao-cua-quang-cao-sai-su-that-702912.html',
  'domain': 'hanoimoi.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 856491,
  'url': 'http://health.people.com.cn/n1/2022/0524/c14739-32428613.html',
  'domain': 'health.people.com.cn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 856591,
  'url': 'http://health.people.com.cn/n1/2023/0629/c14739-40023558.html',
  'domain': 'health.people.com.cn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 856823,
  'url': 'http://health.people.com.cn/n1/2025/1219/c14739-40628249.html',
  'domain': 'health.people.com.cn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 856928,
  'url': 'http://health.people.com.cn/n1/2025/1231/c14739-40636751.html',
  'domain': 'health.people.com.cn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 856943,
  'url': 'http://health.people.com.cn/n1/2026/0113/c14739-40644517.html',
  'domain': 'health.people.com.cn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 856945,
  'url': 'http://health.people.com.cn/n1/2026/0113/c14739-40644536.html',
  'domain': 'health.people.com.cn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 857018,
  'url': 'http://health.people.com.cn/n1/2026/0212/c14739-40664914.html',
  'domain': 'health.people.com.cn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 857053,
  'url': 'http://health.people.com.cn/n1/2026/0304/c14739-40674290.html',
  'domain': 'health.people.com.cn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 857237,
  'url': 'http://health.people.com.cn/n1/2026/0520/c14739-40723845.html',
  'domain': 'health.people.com.cn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 857243,
  'url': 'http://health.people.com.cn/n1/2026/0522/c14739-40725306.html',
  'domain': 'health.people.com.cn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 857266,
  'url': 'http://health.people.com.cn/n1/2026/0529/c14739-40730381.html',
  'domain': 'health.people.com.cn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 857417,
  'url': 'http://health.people.com.cn/n1/2026/0610/c14739-40737531.html',
  'domain': 'health.people.com.cn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 857431,
  'url': 'http://health.people.com.cn/n1/2026/0611/c14739-40738262.html',
  'domain': 'health.people.com.cn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 857432,
  'url': 'http://health.people.com.cn/n1/2026/0611/c14739-40738272.html',
  'domain': 'health.people.com.cn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 857447,
  'url': 'http://health.people.com.cn/n1/2026/0612/c14739-40739060.html',
  'domain': 'health.people.com.cn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 857452,
  'url': 'http://health.people.com.cn/n1/2026/0613/c14739-40739618.html',
  'domain': 'health.people.com.cn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 857498,
  'url': 'http://health.people.com.cn/n1/2026/0617/c14739-40742109.html',
  'domain': 'health.people.com.cn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 857592,
  'url': 'http://health.people.com.cn/n1/2026/0626/c14739-40748408.html',
  'domain': 'health.people.com.cn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 857634,
  'url': 'http://health.people.com.cn/n1/2026/0701/c14739-40751400.html',
  'domain': 'health.people.com.cn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 857676,
  'url': 'http://health.people.com.cn/n1/2026/0704/c14739-40753316.html',
  'domain': 'health.people.com.cn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 858828,
  'url': 'http://heart.39.net/a/140331/4363922.html',
  'domain': 'heart.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 858918,
  'url': 'http://heart.39.net/a/140504/4412145.html',
  'domain': 'heart.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 859642,
  'url': 'http://heart.39.net/a/150604/4633860.html',
  'domain': 'heart.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 859647,
  'url': 'http://heart.39.net/a/150611/4637532.html',
  'domain': 'heart.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 859724,
  'url': 'http://heart.39.net/a/150811/4676355.html',
  'domain': 'heart.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 860356,
  'url': 'http://heart.39.net/a/180518/6263606.html',
  'domain': 'heart.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 860588,
  'url': 'http://heart.39.net/a/180926/6546864.html',
  'domain': 'heart.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 860832,
  'url': 'http://heart.39.net/a/190608/7203442.html',
  'domain': 'heart.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 861437,
  'url': 'http://heart.39.net/a/20101111/1539892.html',
  'domain': 'heart.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 861486,
  'url': 'http://heart.39.net/a/20101122/1548705.html',
  'domain': 'heart.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 861547,
  'url': 'http://heart.39.net/a/2010117/1533805.html',
  'domain': 'heart.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 861621,
  'url': 'http://heart.39.net/a/20101230/1582298.html',
  'domain': 'heart.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 861807,
  'url': 'http://heart.39.net/a/201086/1429794.html',
  'domain': 'heart.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 861981,
  'url': 'http://heart.39.net/a/20111027/1831945.html',
  'domain': 'heart.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 862009,
  'url': 'http://heart.39.net/a/201111/1583281.html',
  'domain': 'heart.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 862137,
  'url': 'http://heart.39.net/a/2011117/1838400.html',
  'domain': 'heart.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 862521,
  'url': 'http://heart.39.net/a/2011513/1689857.html',
  'domain': 'heart.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 862664,
  'url': 'http://heart.39.net/a/2011727/1757083.html',
  'domain': 'heart.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 863239,
  'url': 'http://heart.39.net/a/2012525/2031505.html',
  'domain': 'heart.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 863516,
  'url': 'http://heart.39.net/a/210715/9158271.html',
  'domain': 'heart.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 120075,
  'url': 'https://hellobacsi.com/an-uong-lanh-manh/thong-tin-dinh-duong/kiem-tra-ngay-8-dau-hieu-cho-thay-ban-thieu-kali/',
  'domain': 'hellobacsi.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 121198,
  'url': 'https://hellobacsi.com/da-lieu/nhiem-trung-da/nam-da-dui/',
  'domain': 'hellobacsi.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 121329,
  'url': 'https://hellobacsi.com/da-lieu/tri-mun/mun-duoi-cam/',
  'domain': 'hellobacsi.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 121797,
  'url': 'https://hellobacsi.com/di-ung/di-ung-khac/hoi-chung-lyell/',
  'domain': 'hellobacsi.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 122028,
  'url': 'https://hellobacsi.com/duoc-lieu/thao-duoc/cay-sam-bo-chinh-mot-loai-hoa-dep-cung-la-mot-vi-thuoc/',
  'domain': 'hellobacsi.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 122239,
  'url': 'https://hellobacsi.com/duoc-lieu/thao-duoc/que/',
  'domain': 'hellobacsi.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 122358,
  'url': 'https://hellobacsi.com/duoc-lieu/thao-duoc/tra-hoa-dau-biec-co-tac-dung-gi/',
  'domain': 'hellobacsi.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 122362,
  'url': 'https://hellobacsi.com/duoc-lieu/thao-duoc/tra-hoa-vang/',
  'domain': 'hellobacsi.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 122495,
  'url': 'https://hellobacsi.com/mang-thai/cham-soc-me-sau-sinh/chu-de-sau-sinh-khac/co-be-bi-rong-sau-sinh-phai-lam-sao/',
  'domain': 'hellobacsi.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 124200,
  'url': 'https://hellobacsi.com/mang-thai/thai-ky/tam-ca-nguyet-3/thai-nhi-39-tuan/',
  'domain': 'hellobacsi.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 124731,
  'url': 'https://hellobacsi.com/nuoi-day-con/be-0-1-tuoi/dinh-duong-cho-tre/chon-trai-cay-an-dam-cho-be-moi-lan-dau-an/',
  'domain': 'hellobacsi.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 125193,
  'url': 'https://hellobacsi.com/nuoi-day-con/nhi-khoa/benh-truyen-nhiem-tre-em/4-bi-quyet-giup-dieu-tri-va-phong-dau-mat-do-cho-tre/',
  'domain': 'hellobacsi.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 126451,
  'url': 'https://hellobacsi.com/nuoi-day-con/tuoi-day-thi/suc-khoe-tuoi-day-thi/che-do-an-uong-cho-tuoi-day-thi-vua-cao-vua-khoe-manh/',
  'domain': 'hellobacsi.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 126641,
  'url': 'https://hellobacsi.com/suc-khoe-nam-gioi/suc-khoe-duong-vat/ngua-dau-duong-vat/',
  'domain': 'hellobacsi.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 127159,
  'url': 'https://hellobacsi.com/suc-khoe-phu-nu/van-de-phu-khoa-khac/am-dao-co-mui/',
  'domain': 'hellobacsi.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 128169,
  'url': 'https://hellobacsi.com/suc-khoe-tinh-duc/tinh-duc-an-toan/quan-he-tinh-duc-deu-dan-co-tac-dung-gi/',
  'domain': 'hellobacsi.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 129061,
  'url': 'https://hellobacsi.com/tam-ly-tam-than/moi-quan-he/nguoi-huong-ngoai-la-gi/',
  'domain': 'hellobacsi.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 129223,
  'url': 'https://hellobacsi.com/tam-ly-tam-than/quan-ly-cang-thang/tac-hai-cua-ngoi-thien/',
  'domain': 'hellobacsi.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 129799,
  'url': 'https://hellobacsi.com/the-duc-the-thao/hoat-dong-the-chat-khac/bai-tap-tho-giam-mo-bung/',
  'domain': 'hellobacsi.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 130446,
  'url': 'https://hellobacsi.com/ung-thu-ung-buou/ung-thu-gan/benh-nhan-ung-thu-gan/',
  'domain': 'hellobacsi.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 2509852,
  'url': 'https://jb39.com/jibing/ChangBingXingGuanJieYan263402.htm',
  'domain': 'jb39.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2510137,
  'url': 'https://jb39.com/jibing/DanShenJingBingYuShenJingCongShenJing256420.htm',
  'domain': 'jb39.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2511415,
  'url': 'https://jb39.com/jibing/JiFaXingTieLiYouXiBaoXingPin259049.htm',
  'domain': 'jb39.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2511893,
  'url': 'https://jb39.com/jibing/JingHangFaRe251363.htm',
  'domain': 'jb39.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2512030,
  'url': 'https://jb39.com/jibing/JunTuanFei268570.htm',
  'domain': 'jb39.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2512387,
  'url': 'https://jb39.com/jibing/LaoNianXingManXingYingMoXiaXueZhong260474.htm',
  'domain': 'jb39.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2512730,
  'url': 'https://jb39.com/jibing/ManXingKeSou251066.htm',
  'domain': 'jb39.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2513332,
  'url': 'https://jb39.com/jibing/QiNiaoGuanNangZhongJiQiNiaoGuanLou268470.htm',
  'domain': 'jb39.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2513345,
  'url': 'https://jb39.com/jibing/QiTangNiaoBing261823.htm',
  'domain': 'jb39.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2513743,
  'url': 'https://jb39.com/jibing/SeSuXingJingMaiPangShiWangMoMai252469.htm',
  'domain': 'jb39.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2513746,
  'url': 'https://jb39.com/jibing/SeSuXingRongMaoHuaMoYan263400.htm',
  'domain': 'jb39.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2513773,
  'url': 'https://jb39.com/jibing/ShangHan265681.htm',
  'domain': 'jb39.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2513860,
  'url': 'https://jb39.com/jibing/ShenJingXiTongDeFangSheXingSunShang251078.htm',
  'domain': 'jb39.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2513913,
  'url': 'https://jb39.com/jibing/ShenXingGaoXueYa268318.htm',
  'domain': 'jb39.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2514554,
  'url': 'https://jb39.com/jibing/WeiShengSuYiLaiXingGouLouBing257494.htm',
  'domain': 'jb39.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2514670,
  'url': 'https://jb39.com/jibing/XiaZhiDongMaiYingHuaShuanSe256238.htm',
  'domain': 'jb39.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2515033,
  'url': 'https://jb39.com/jibing/XiaoErGaoXueYa268224.htm',
  'domain': 'jb39.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2515716,
  'url': 'https://jb39.com/jibing/XingYouErNaoLiuZhiBing263451.htm',
  'domain': 'jb39.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2516288,
  'url': 'https://jb39.com/jibing/YingHuaXingZhiMoYan263362.htm',
  'domain': 'jb39.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2516586,
  'url': 'https://jb39.com/jibing/ZhiYiXingPiYan265717.htm',
  'domain': 'jb39.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 2517142,
  'url': 'https://jbk.39.net/cdbd71xszkb/',
  'domain': 'jbk.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2517539,
  'url': 'https://jbk.39.net/dnjs/',
  'domain': 'jbk.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2520202,
  'url': 'https://jbk.39.net/jiancha/lgtzsy/',
  'domain': 'jbk.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2520450,
  'url': 'https://jbk.39.net/jiancha/ngtjyfdwbaep/',
  'domain': 'jbk.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2520546,
  'url': 'https://jbk.39.net/jiancha/njyxns/',
  'domain': 'jbk.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2520609,
  'url': 'https://jbk.39.net/jiancha/nsjdph/',
  'domain': 'jbk.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2520745,
  'url': 'https://jbk.39.net/jiancha/plycsz/',
  'domain': 'jbk.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2521158,
  'url': 'https://jbk.39.net/jiancha/tjhjjc/',
  'domain': 'jbk.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2521401,
  'url': 'https://jbk.39.net/jiancha/xbmyjc/',
  'domain': 'jbk.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2521765,
  'url': 'https://jbk.39.net/jiancha/xserxzsc/',
  'domain': 'jbk.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2521893,
  'url': 'https://jbk.39.net/jiancha/yckfpttnlsy/',
  'domain': 'jbk.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2521949,
  'url': 'https://jbk.39.net/jiancha/ygbjadbfitcspajcf/',
  'domain': 'jbk.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2523157,
  'url': 'https://jbk.39.net/lnrfhqjexlbzl/',
  'domain': 'jbk.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2525239,
  'url': 'https://jbk.39.net/shoushu/kcy/',
  'domain': 'jbk.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2525467,
  'url': 'https://jbk.39.net/shoushu/rxxwlssmammotome/',
  'domain': 'jbk.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2529413,
  'url': 'https://jbk.39.net/zhengzhuang/chdbcx/',
  'domain': 'jbk.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2530638,
  'url': 'https://jbk.39.net/zhengzhuang/hjhot/',
  'domain': 'jbk.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2530759,
  'url': 'https://jbk.39.net/zhengzhuang/hxsljmqgz/',
  'domain': 'jbk.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2533244,
  'url': 'https://jbk.39.net/zhengzhuang/ssyjdszxzhsb/',
  'domain': 'jbk.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2533531,
  'url': 'https://jbk.39.net/zhengzhuang/tbjyhbttbntdlxbddlls/',
  'domain': 'jbk.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 135148,
  'url': 'https://khoahocphothong.vn/astrazeneca-dong-hanh-cung-cong-dong-benh-hiem-tai-viet-nam-264302.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 135745,
  'url': 'https://khoahocphothong.vn/benh-vien-nhi-dong-1-dieu-tri-di-tat-tim-cho-benh-nhi-campuchia-mien-phi-257758.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 136180,
  'url': 'https://khoahocphothong.vn/buoc-thoi-viec-y-si-tac-trach-lam-tre-tu-vong-238831.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 136583,
  'url': 'https://khoahocphothong.vn/cao-tra-xanh-ho-tro-dieu-tri-benh-ung-thu-238918.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 136758,
  'url': 'https://khoahocphothong.vn/chan-doan-tram-cam-va-su-dong-thuan-cua-benh-nhan-240629.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 137246,
  'url': 'https://khoahocphothong.vn/cuu-song-benh-nhan-2-lan-ngung-tim-do-bi-dien-giat-202602.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 137518,
  'url': 'https://khoahocphothong.vn/de-tieu-dam-khong-con-la-noi-lo-238333.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 137882,
  'url': 'https://khoahocphothong.vn/dua-noi-tre-thoi-quen-can-loai-bo-239175.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 138066,
  'url': 'https://khoahocphothong.vn/gan-nhu-khong-phan-biet-duoc-covid-19-voi-cac-benh-nhiem-trung-ho-hap-202592.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 138475,
  'url': 'https://khoahocphothong.vn/hoi-an-xay-dung-moi-truong-khong-khoi-thuoc-la-tai-nha-hang-khach-san-237547.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 138528,
  'url': 'https://khoahocphothong.vn/hoi-nghi-khoa-hoc-sot-xuat-huyet-cap-nhat-va-xu-tri-241014.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 139026,
  'url': 'https://khoahocphothong.vn/lam-the-nao-de-tang-suc-de-khang-241461.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 139209,
  'url': 'https://khoahocphothong.vn/mac-tay-chan-mieng-do-4-benh-nhi-17-thang-tuoi-phai-loc-mau-248592.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 140116,
  'url': 'https://khoahocphothong.vn/noi-niem-buc-xuc-ve-khoi-thuoc-la-903.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 140490,
  'url': 'https://khoahocphothong.vn/phong-hoc-sac-o-tre-238158.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 141003,
  'url': 'https://khoahocphothong.vn/so-y-te-tp-hcm-yeu-cau-benh-vien-da-khoa-gia-dinh-tam-ngung-thuc-hien-phau-thuat-hut-mo-bung-259841.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 141005,
  'url': 'https://khoahocphothong.vn/so-y-te-tp-hcm-yeu-cau-tuan-thu-nguyen-tac-6-gio-tai-cac-khoa-cap-cuu-257977.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 141151,
  'url': 'https://khoahocphothong.vn/tac-dung-chong-ung-thu-cua-nhan-sam-ginsenoside-rh2-241900.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 141626,
  'url': 'https://khoahocphothong.vn/ths-bs-ckii-nguyen-duc-minh-giam-doc-benh-vien-rang-ham-mat-tp-hcm-tu-hao-voi-hanh-trinh-vuon-minh-gan-50-nam-cua-benh-vien-258603.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 142695,
  'url': 'https://khoahocphothong.vn/viet-cho-nguoi-benh-tieu-duong-1-dau-la-dap-an-241859.html',
  'domain': 'khoahocphothong.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 145347,
  'url': 'https://laodong.vn/ban-tin/nhung-luu-y-dac-biet-khi-su-dung-nam-dui-ga-cho-nguoi-huyet-ap-cao-1596744.ldo',
  'domain': 'laodong.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 147285,
  'url': 'https://laodong.vn/dinh-duong-am-thuc/4-loai-thuc-pham-nen-an-de-ho-tro-giai-doc-than-ngua-soi-than-1550990.ldo',
  'domain': 'laodong.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 148857,
  'url': 'https://laodong.vn/dinh-duong-am-thuc/cach-su-dung-loai-rau-quen-thuoc-de-giam-beo-va-kiem-soat-tieu-duong-1307601.ldo',
  'domain': 'laodong.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 149256,
  'url': 'https://laodong.vn/dinh-duong-am-thuc/giam-mo-bung-bang-mot-loai-qua-trong-mam-ngu-qua-1451787.ldo',
  'domain': 'laodong.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 152841,
  'url': 'https://laodong.vn/photo/thoat-khoi-stress-voi-10-thu-thuat-tam-ly-don-gian-821341.ldo',
  'domain': 'laodong.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 153118,
  'url': 'https://laodong.vn/suc-khoe/2-mon-an-nhe-tu-ca-rot-co-chi-so-duong-huyet-thap-1205185.ldo',
  'domain': 'laodong.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 155084,
  'url': 'https://laodong.vn/suc-khoe/5-thuc-pham-giup-xuong-chac-khoe-cho-phu-nu-man-kinh-1408588.ldo',
  'domain': 'laodong.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 157829,
  'url': 'https://laodong.vn/suc-khoe/buoc-tien-dot-pha-trong-nghien-cuu-dieu-tri-ung-thu-tai-viet-nam-1434540.ldo',
  'domain': 'laodong.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 158320,
  'url': 'https://laodong.vn/suc-khoe/cach-giup-de-dang-cat-giam-calo-va-giam-can-hieu-qua-1520440.ldo',
  'domain': 'laodong.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 158407,
  'url': 'https://laodong.vn/suc-khoe/cach-lam-sua-dong-chanh-day-thuan-chay-it-calo-1484088.ldo',
  'domain': 'laodong.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 159564,
  'url': 'https://laodong.vn/suc-khoe/con-20-benh-nhan-mac-covid-19-dang-dieu-tri-tai-cac-co-so-y-te-814704.ldo',
  'domain': 'laodong.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 160726,
  'url': 'https://laodong.vn/suc-khoe/dot-bien-gene-khien-nhung-bao-thai-de-mat-di-co-hoi-song-1365857.ldo',
  'domain': 'laodong.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 161416,
  'url': 'https://laodong.vn/suc-khoe/ha-noi-quan-chat-cac-bep-an-truong-hoc-646510.ldo',
  'domain': 'laodong.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 163643,
  'url': 'https://laodong.vn/suc-khoe/mot-chiec-banh-trung-thu-co-nang-luong-gan-bang-ba-bat-pho-629303.ldo',
  'domain': 'laodong.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 164287,
  'url': 'https://laodong.vn/suc-khoe/nguoi-dan-can-tiem-ngua-cho-cho-meo-de-phong-chong-benh-dai-mua-nang-nong-1173847.ldo',
  'domain': 'laodong.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 166932,
  'url': 'https://laodong.vn/suc-khoe/soi-tui-mat-co-the-bat-dau-tu-chung-kho-tieu-1536976.ldo',
  'domain': 'laodong.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 167465,
  'url': 'https://laodong.vn/suc-khoe/tao-ham-mon-an-don-gian-giup-cai-thien-tieu-hoa-va-bao-ve-tim-mach-1551386.ldo',
  'domain': 'laodong.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 170584,
  'url': 'https://laodong.vn/video-xa-hoi/nguyen-tac-vang-giup-nguoi-co-axit-uric-cao-ngu-ngon-1500072.ldo',
  'domain': 'laodong.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 172251,
  'url': 'https://laodong.vn/video/phuc-hoi-than-hieu-qua-voi-cac-loai-qua-nay-1500490.ldo',
  'domain': 'laodong.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 173714,
  'url': 'https://laodong.vn/y-te/nam-dieu-duong-bi-da-vao-bung-van-co-gang-cuu-nguoi-nhan-mua-loi-khen-1498451.ldo',
  'domain': 'laodong.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 864173,
  'url': 'http://man.39.net/a/140704/4419241.html',
  'domain': 'man.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 864419,
  'url': 'http://man.39.net/a/150320/4594035.html',
  'domain': 'man.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 865025,
  'url': 'http://man.39.net/a/20101015/1504654.html',
  'domain': 'man.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 865541,
  'url': 'http://man.39.net/a/2010421/1240947.html',
  'domain': 'man.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 865678,
  'url': 'http://man.39.net/a/2010515/1301579.html',
  'domain': 'man.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 865982,
  'url': 'http://man.39.net/a/201053/1259307.html',
  'domain': 'man.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 866968,
  'url': 'http://man.39.net/a/201078/1381529.html',
  'domain': 'man.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 867126,
  'url': 'http://man.39.net/a/2010831/1465600.html',
  'domain': 'man.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 867210,
  'url': 'http://man.39.net/a/2010912/1482211.html',
  'domain': 'man.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 867544,
  'url': 'http://man.39.net/a/2011331/1649188.html',
  'domain': 'man.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 867874,
  'url': 'http://man.39.net/a/201175/1771796.html',
  'domain': 'man.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 867916,
  'url': 'http://man.39.net/a/201181/1762875.html',
  'domain': 'man.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 868073,
  'url': 'http://man.39.net/a/2011917/1808827.html',
  'domain': 'man.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 868246,
  'url': 'http://man.39.net/a/220330/10525286.html',
  'domain': 'man.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 868334,
  'url': 'http://man.39.net/a/220510/10797983.html',
  'domain': 'man.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 868615,
  'url': 'http://man.39.net/a/220921/8235010.html',
  'domain': 'man.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 868695,
  'url': 'http://man.39.net/a/230121/10226782.html',
  'domain': 'man.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 868704,
  'url': 'http://man.39.net/a/230209/13150465.html',
  'domain': 'man.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 868722,
  'url': 'http://man.39.net/a/230303/13317157.html',
  'domain': 'man.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 868749,
  'url': 'http://man.39.net/a/230329/8793094.html',
  'domain': 'man.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 174516,
  'url': 'https://medlatec.vn/tin-tuc/-test-covid19-va-nhung-sai-lam-nghiem-trong-s194-n31815',
  'domain': 'medlatec.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 174996,
  'url': 'https://medlatec.vn/tin-tuc/7-van-de-khong-nen-bo-lo-ve-kinh-nguyet-o-tuoi-moi-lon-s74-n22006',
  'domain': 'medlatec.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 175527,
  'url': 'https://medlatec.vn/tin-tuc/bac-si-medlatec-chia-se-kinh-nghiem-ung-dung-noi-soi-sieu-am-miniprobe-tai-hoi-nghi-noi-soi-tieu-hoa-can-thiep-toan-quoc-viges-2025',
  'domain': 'medlatec.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 175865,
  'url': 'https://medlatec.vn/tin-tuc/bao-lanh-vien-phi-pti-tai-medlatec-mang-den-nhung-quyen-loi-gi-cho-khach-hang-s191-n28249',
  'domain': 'medlatec.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 178726,
  'url': 'https://medlatec.vn/tin-tuc/camphor-la-gi-cong-dung-cach-dung-va-canh-bao-an-toan-theo-y-khoa',
  'domain': 'medlatec.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 183321,
  'url': 'https://medlatec.vn/tin-tuc/giam-kho-mat-voi-thuoc-nho-mat-syseye-nhung-dieu-can-luu-y',
  'domain': 'medlatec.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 183426,
  'url': 'https://medlatec.vn/tin-tuc/giup-me-giai-dap-thac-mac-tre-bi-sot-phat-ban-uong-thuoc-gi-s75-n25863',
  'domain': 'medlatec.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 185460,
  'url': 'https://medlatec.vn/tin-tuc/kham-than-kinh-gom-nhung-gi-nen-kham-khi-nao-s65-n31097',
  'domain': 'medlatec.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 185736,
  'url': 'https://medlatec.vn/tin-tuc/khong-can-den-benh-vien-cung-co-the-xet-nghiem-kiem-tra-suc-khoe-voi-medlatec-thai-binh-s171-n16076',
  'domain': 'medlatec.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 187787,
  'url': 'https://medlatec.vn/tin-tuc/nen-kham-benh-ly-truyen-nhiem-lay-truyen-qua-duong-tinh-duc-tai-go-vap-o-dau',
  'domain': 'medlatec.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 188190,
  'url': 'https://medlatec.vn/tin-tuc/nguy-co-tiem-an-tu-hoi-chung-ngung-tho-khi-ngu-va-giai-phap-khac-phuc',
  'domain': 'medlatec.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 188206,
  'url': 'https://medlatec.vn/tin-tuc/nguyen-nhan-bi-cang-co-bap-chan-la-gi-va-cach-chua-tri-hieu-qua',
  'domain': 'medlatec.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 190214,
  'url': 'https://medlatec.vn/tin-tuc/nut-got-chan-la-tai-sao-mot-so-bien-phap-khac-phuc-don-gian',
  'domain': 'medlatec.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 190568,
  'url': 'https://medlatec.vn/tin-tuc/phong-tranh-tac-nhan-gay-viem-mang-nao-mu-nhu-the-nao-la-hieu-qua-s195-n21350',
  'domain': 'medlatec.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 190591,
  'url': 'https://medlatec.vn/tin-tuc/phu-nu-bi-nhan-tuyen-giap-co-nen-mang-thai-khong-an-toan-hay-rui-ro',
  'domain': 'medlatec.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 191027,
  'url': 'https://medlatec.vn/tin-tuc/rang-nao-khong-nen-nho-nhung-dieu-can-biet-khi-nho-rang',
  'domain': 'medlatec.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 192568,
  'url': 'https://medlatec.vn/tin-tuc/test-nhanh-va-test-pcr-co-uu-nhuoc-diem-gi-s58-n31750',
  'domain': 'medlatec.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 193913,
  'url': 'https://medlatec.vn/tin-tuc/tim-hieu-cac-trieu-chung-cua-benh-roi-loan-lo-au-s65-n25520',
  'domain': 'medlatec.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 193922,
  'url': 'https://medlatec.vn/tin-tuc/tim-hieu-cach-sinh-con-theo-y-muon-cho-cap-doi-hiem-muon-s74-n30982',
  'domain': 'medlatec.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 196119,
  'url': 'https://medlatec.vn/tin-tuc/ung-dung-y-te-my-medlatec--buoc-chuyen-minh-trong-hanh-trinh-nang-cao-trai-nghiem-khach-hang--s177-n31268',
  'domain': 'medlatec.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 199047,
  'url': 'https://mega.vietnamplus.vn/thai-huong-nu-anh-hung-dam-me-lam-cho-dat-no-hoa-5415.html',
  'domain': 'mega.vietnamplus.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 199048,
  'url': 'https://mega.vietnamplus.vn/theo-chan-nguoi-dan-bien-vien-mien-tay-xu-nghe-xuyen-rung-hai-loc-troi-5412.html',
  'domain': 'mega.vietnamplus.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 199105,
  'url': 'https://nhandan.vn/46-tan-bac-si-rang-ham-mat-dau-tien-cua-truong-dai-hoc-khoa-hoc-suc-khoe-tot-nghiep-post919968.html',
  'domain': 'nhandan.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 199153,
  'url': 'https://nhandan.vn/an-toan-nguoi-benh-la-thuoc-do-chat-luong-chuyen-mon-cua-cac-benh-vien-post908643.html',
  'domain': 'nhandan.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 199271,
  'url': 'https://nhandan.vn/bao-dam-cong-tac-y-te-phuc-vu-le-tang-tong-bi-thu-nguyen-phu-trong-post820891.html',
  'domain': 'nhandan.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 199406,
  'url': 'https://nhandan.vn/benh-nhi-soc-phan-ve-do-bi-ong-dot-bac-si-huong-dan-cach-xu-tri-post823257.html',
  'domain': 'nhandan.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 199559,
  'url': 'https://nhandan.vn/benh-vien-quan-y175-tri-an-gia-dinh-hien-da-mo-tang-post875280.html',
  'domain': 'nhandan.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 199950,
  'url': 'https://nhandan.vn/can-nao-de-giu-thai-them-10-tuan-trong-buong-tu-cung-cho-san-phu-bi-ri-oi-post865630.html',
  'domain': 'nhandan.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 200051,
  'url': 'https://nhandan.vn/cap-cuu-va-van-chuyen-khan-cap-ngu-dan-nguy-kich-tu-dao-song-tu-tay-ve-dat-lien-dieu-tri-post906366.html',
  'domain': 'nhandan.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 200142,
  'url': 'https://nhandan.vn/chi-dao-quyet-liet-thuc-hien-hieu-qua-chuyen-doi-so-va-de-an-06-post818334.html',
  'domain': 'nhandan.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 200299,
  'url': 'https://nhandan.vn/chuyen-gia-nhat-ban-hoi-chan-cho-be-gai-trong-vu-sat-lo-o-lang-nu-post831077.html',
  'domain': 'nhandan.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 200337,
  'url': 'https://nhandan.vn/cong-bo-chinh-thuc-thanh-toan-benh-mat-hot-tai-viet-nam-post872330.html',
  'domain': 'nhandan.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 200444,
  'url': 'https://nhandan.vn/cuu-song-benh-nhi-bi-da-chan-thuong-nghiem-trong-do-va-cham-xe-cong-nong-post809568.html',
  'domain': 'nhandan.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 200793,
  'url': 'https://nhandan.vn/dua-vao-su-dung-may-chup-cong-huong-tu-15-tesla-post847223.html',
  'domain': 'nhandan.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 200894,
  'url': 'https://nhandan.vn/giai-phap-ai-giup-nang-cao-do-chinh-xac-trong-chan-doan-hinh-anh-post845083.html',
  'domain': 'nhandan.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 200989,
  'url': 'https://nhandan.vn/ha-noi-ghi-nhan-mot-truong-hop-tu-vong-do-soi-post866923.html',
  'domain': 'nhandan.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 201371,
  'url': 'https://nhandan.vn/kham-suc-khoe-cho-quan-va-dan-huyen-dao-truong-sa-va-nha-gian-dk1-post825808.html',
  'domain': 'nhandan.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 201923,
  'url': 'https://nhandan.vn/nganh-y-te-keu-goi-cac-dia-phuong-nguoi-dan-chu-dong-ngan-chan-benh-sot-xuat-huyet-post908797.html',
  'domain': 'nhandan.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 203184,
  'url': 'https://nhandan.vn/thanh-pho-hue-xay-dung-do-thi-di-san-trung-tam-van-hoa-du-lich-dac-sac-tam-co-quoc-te-post938099.html',
  'domain': 'nhandan.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 203577,
  'url': 'https://nhandan.vn/trung-tam-dot-quy-benh-vien-bach-mai-dat-trung-tam-dot-quy-chuyen-sau-post920189.html',
  'domain': 'nhandan.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 203602,
  'url': 'https://nhandan.vn/truy-tim-nguon-goc-lo-san-pham-thuoc-tri-hen-suyen-gia-chi-dat-63-ham-luong-post883088.html',
  'domain': 'nhandan.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 203791,
  'url': 'https://nhandan.vn/unicef-hon-178000-ca-mac-benh-ta-2900-ca-tu-vong-o-khu-vuc-dong-va-nam-chau-phi-post867450.html',
  'domain': 'nhandan.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 209342,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/bai-tap-yoga-tot-cho-tu-cung-ma-chi-em-phu-nu-khong-nen-bo-qua-71971.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 210092,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/banh-gao-lut-tim-than-bao-nhieu-calo-an-banh-gao-lut-co-giam-can-duoc-khong-68241.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 212703,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/benh-tri-do-2-la-gi-cac-bieu-hieu-chuyen-bien-thanh-benh-tri-do-2-49449.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 219379,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-khac-phuc-chay-bo-bi-dau-chan-58796.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 221467,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/cach-xu-ly-khi-tre-bi-quai-bi-de-tranh-bien-chung-ve-sau-43413.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 225269,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/chuyen-gia-giai-dap-viem-cot-song-dinh-khop-co-di-truyen-khong.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 229609,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/day-ron-la-gi-nguyen-nhan-day-ron-quan-co-thai-nhi-63830.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 231345,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/dot-quy-thuong-khong-den-bat-ngo-6-nguyen-nhan-am-tham-nhieu-nguoi-bo-qua.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 235438,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/goc-giai-dap-nguoi-bi-tieu-duong-co-an-duoc-banh-mi-khong-70896.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 236281,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/ha-natri-mau-nguyen-nhan-trieu-chung-chan-doan-dieu-tri-va-cach-phong-ngua.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 243488,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/loi-ich-cua-tap-zumba-cho-be-va-cac-bai-tap-zumba-don-gian-danh-cho-be.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 249751,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/ngu-nhieu-bi-gi-va-cach-han-che-tinh-trang-ngu-qua-muc.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 253330,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/nhung-bien-phap-nao-giup-phong-ngua-tang-bach-cau-ai-toan.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 261424,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/sandwich-bao-nhieu-calo-an-sandwich-nao-giup-giam-can-71618.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 265009,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tai-sao-chao-dinh-duong-co-mui-thom-loi-ich-cua-mui-thom-trong-chao-dinh-duong.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 267893,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/thoi-gian-u-benh-lau-la-bao-lau-phai-lam-gi-khi-bi-benh-lau-69685.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 271620,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/tinh-dau-palmarosa-va-nhung-loi-ich-khi-su-dung.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 272626,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/top-12-thuc-pham-chuc-nang-bo-sung-kem-hieu-qua-co-the-ban-chua-biet-49003.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 278938,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/vi-sao-cang-lon-tuoi-thi-tinh-trang-non-nao-sau-khi-uong-ruou-cua-ban-cang-toi-te-hon-lam-gi-de-cai-thien-tinh-trang-tren-49657.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 280161,
  'url': 'https://nhathuoclongchau.com.vn/bai-viet/viem-tai-ngoai-la-gi-nguyen-nhan-va-cach-dieu-tri-56159.html',
  'domain': 'nhathuoclongchau.com.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 869151,
  'url': 'http://nk.39.net/a/120704/4002825.html',
  'domain': 'nk.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 869580,
  'url': 'http://nk.39.net/a/121214/4096477.html',
  'domain': 'nk.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 869612,
  'url': 'http://nk.39.net/a/130101/4103432.html',
  'domain': 'nk.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 869947,
  'url': 'http://nk.39.net/a/131010/4269179.html',
  'domain': 'nk.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 869984,
  'url': 'http://nk.39.net/a/131221/4311344.html',
  'domain': 'nk.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 871014,
  'url': 'http://nk.39.net/a/141231/4543801.html',
  'domain': 'nk.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 871145,
  'url': 'http://nk.39.net/a/150311/4589121.html',
  'domain': 'nk.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 871882,
  'url': 'http://nk.39.net/a/171128/5880607.html',
  'domain': 'nk.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 872413,
  'url': 'http://nk.39.net/a/2010115/1532431.html',
  'domain': 'nk.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 872555,
  'url': 'http://nk.39.net/a/2010713/1385412.html',
  'domain': 'nk.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 872723,
  'url': 'http://nk.39.net/a/2010920/1484943.html',
  'domain': 'nk.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 872834,
  'url': 'http://nk.39.net/a/20111012/1820996.html',
  'domain': 'nk.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 873427,
  'url': 'http://nk.39.net/a/201138/1629366.html',
  'domain': 'nk.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 873455,
  'url': 'http://nk.39.net/a/2011416/1663766.html',
  'domain': 'nk.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 873769,
  'url': 'http://nk.39.net/a/2011719/1749852.html',
  'domain': 'nk.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 873772,
  'url': 'http://nk.39.net/a/2011719/1751912.html',
  'domain': 'nk.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 874189,
  'url': 'http://nk.39.net/a/201222/1966719.html',
  'domain': 'nk.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 874547,
  'url': 'http://nk.39.net/a/211227/10114001.html',
  'domain': 'nk.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 874559,
  'url': 'http://nk.39.net/a/220302/10363390.html',
  'domain': 'nk.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 875653,
  'url': 'http://nk.39.net/a/260707/s1wnj52.html',
  'domain': 'nk.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 875749,
  'url': 'http://pf.39.net/a/120829/4042716.html',
  'domain': 'pf.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 875784,
  'url': 'http://pf.39.net/a/120928/4060574.html',
  'domain': 'pf.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 875830,
  'url': 'http://pf.39.net/a/121115/4083059.html',
  'domain': 'pf.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 875833,
  'url': 'http://pf.39.net/a/121120/4084796.html',
  'domain': 'pf.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 875845,
  'url': 'http://pf.39.net/a/121206/4092675.html',
  'domain': 'pf.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 875872,
  'url': 'http://pf.39.net/a/130123/4116724.html',
  'domain': 'pf.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 875934,
  'url': 'http://pf.39.net/a/130505/4166915.html',
  'domain': 'pf.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 876093,
  'url': 'http://pf.39.net/a/131109/4286653.html',
  'domain': 'pf.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 876107,
  'url': 'http://pf.39.net/a/131124/4295980.html',
  'domain': 'pf.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 876159,
  'url': 'http://pf.39.net/a/20101028/1525803.html',
  'domain': 'pf.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 876174,
  'url': 'http://pf.39.net/a/20101116/1544349.html',
  'domain': 'pf.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 876176,
  'url': 'http://pf.39.net/a/2010112/1529933.html',
  'domain': 'pf.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 876189,
  'url': 'http://pf.39.net/a/20101210/1567514.html',
  'domain': 'pf.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 876258,
  'url': 'http://pf.39.net/a/20111014/1824861.html',
  'domain': 'pf.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 876415,
  'url': 'http://pf.39.net/a/201127/1603583.html',
  'domain': 'pf.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 876523,
  'url': 'http://pf.39.net/a/2011513/1687172.html',
  'domain': 'pf.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 876706,
  'url': 'http://pf.39.net/a/201179/1742112.html',
  'domain': 'pf.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 876822,
  'url': 'http://pf.39.net/a/2012116/1878842.html',
  'domain': 'pf.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 876830,
  'url': 'http://pf.39.net/a/2012127/1879135.html',
  'domain': 'pf.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 877052,
  'url': 'http://pf.39.net/a/231108/v2j0lhx.html',
  'domain': 'pf.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 284809,
  'url': 'https://phunusuckhoe.giadinhonline.vn/3-hanh-dong-nho-tuong-chung-nhu-vo-hai-lai-la-tac-nhan-lon-gay-anh-huong-den-tu-cung-cac-ban-gai-can-biet-de-bo-ngay-c12a383874.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 286444,
  'url': 'https://phunusuckhoe.giadinhonline.vn/5-nguyen-nhan-pho-bien-gay-dau-nguc-o-phu-nu-c12a406620.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 286842,
  'url': 'https://phunusuckhoe.giadinhonline.vn/6-cach-on-gian-e-phong-benh-cam-cum-ma-uoc-cac-chuyen-gia-tin-dung-c12a489904.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 289021,
  'url': 'https://phunusuckhoe.giadinhonline.vn/bac-si-tiet-nieu-chi-ra-6-thoi-quen-am-tham-pha-hong-than-c12a446675.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 289172,
  'url': 'https://phunusuckhoe.giadinhonline.vn/ban-gai-chia-se-cach-lam-trang-da-giam-tham-mun-chi-trong-1-thang-c12a5117.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 291074,
  'url': 'https://phunusuckhoe.giadinhonline.vn/cac-kieu-toc-ngan-dep-han-quoc-danh-cho-cac-nang-ca-tinh-c12a59561.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 292675,
  'url': 'https://phunusuckhoe.giadinhonline.vn/cdc-my-canh-bao-nguy-co-benh-au-mua-khi-bung-phat-tro-lai-c12a448201.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 295061,
  'url': 'https://phunusuckhoe.giadinhonline.vn/dang-mua-nang-nong-dinh-diem-tia-buc-xa-dat-muc-bao-dong-hay-lam-4-dieu-nay-de-bao-ve-lan-da-c12a309953.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 295310,
  'url': 'https://phunusuckhoe.giadinhonline.vn/dau-hieu-bat-thuong-canh-bao-benh-ung-thu-vu-c12a447319.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 296787,
  'url': 'https://phunusuckhoe.giadinhonline.vn/e-phong-tranh-lay-lan-benh-nhan-au-mat-o-can-tuan-thu-2-hanh-ong-nay-c12a461442.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 297542,
  'url': 'https://phunusuckhoe.giadinhonline.vn/hai-cach-lam-trang-da-mat-tai-nha-cho-lan-da-trang-co-nang-them-hong-c12a111079.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 298353,
  'url': 'https://phunusuckhoe.giadinhonline.vn/i-cho-mua-ca-hay-tranh-xa-3-bo-phan-nay-an-vao-gan-than-tan-ta-suc-khoe-xuong-cap-thay-ro-c12a473815.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 298601,
  'url': 'https://phunusuckhoe.giadinhonline.vn/kham-pha-6-tuyet-chieu-giam-beo-bung-sau-sinh-mo-nhanh-chong-c12a76833.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 299810,
  'url': 'https://phunusuckhoe.giadinhonline.vn/loai-nuoc-tuong-chung-bo-di-nay-lai-co-tac-dung-ngan-ngua-lao-hoa-cai-thien-do-dan-hoi-cua-lan-da-tuong-tu-collagen-c12a305238.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 302580,
  'url': 'https://phunusuckhoe.giadinhonline.vn/nghien-cuu-moi-nhat-chao-chong-dinh-dang-khien-cau-nho-teo-di-moi-ngay-c12a304726.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 303906,
  'url': 'https://phunusuckhoe.giadinhonline.vn/nhung-cach-cham-soc-da-mat-sau-khi-sinh-tot-nhat-c12a102706.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 303994,
  'url': 'https://phunusuckhoe.giadinhonline.vn/nhung-can-benh-ve-da-do-do-mo-hoi-c12a415751.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 307361,
  'url': 'https://phunusuckhoe.giadinhonline.vn/than-tuong-san-pham-lam-dep-sieu-toc-coi-chung-ung-thu-ghe-tham-c12a144918.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 308213,
  'url': 'https://phunusuckhoe.giadinhonline.vn/thuc-pham-ngon-mieng-giup-toc-moc-nhanh-hon-c12a13662.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 309230,
  'url': 'https://phunusuckhoe.giadinhonline.vn/tri-mun-tham-o-mat-tu-thien-nhien-giai-phap-an-toan-va-hoan-hao-nhat-cho-chi-em-c12a124896.html',
  'domain': 'phunusuckhoe.giadinhonline.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 312209,
  'url': 'https://phuyen.baodaklak.vn/95/13295/phat-hien-som-trieu-chung-ung-thu-buong-trung/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 313991,
  'url': 'https://phuyen.baodaklak.vn/95/228583/chuong-trinh-phau-thuat-nhan-dao-danh-cho-benh-nhan-bi-di-tat-khe-moi-ham-ech/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 314153,
  'url': 'https://phuyen.baodaklak.vn/95/23802/nhung-cach-lam-tang-chieu-cao/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 314298,
  'url': 'https://phuyen.baodaklak.vn/95/249008/tac-hai-khong-ngo-cua-thuoc-la-the-he-moi/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 314563,
  'url': 'https://phuyen.baodaklak.vn/95/27753/kho-vi-mui-hoi-co-the/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 314609,
  'url': 'https://phuyen.baodaklak.vn/95/281172/bao-ve-da-truoc-tia-tu-ngoai/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 314737,
  'url': 'https://phuyen.baodaklak.vn/95/2909/an-chay-mot-vai-luu-y-nen-biet/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 314955,
  'url': 'https://phuyen.baodaklak.vn/95/307573/ra-soat-chan-chinh-thai-do-y-duc-cua-nhan-vien-y-te/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 315083,
  'url': 'https://phuyen.baodaklak.vn/95/314845/benh-tim-mach-o-nguoi-tre-dung-chu-quan/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 315427,
  'url': 'https://phuyen.baodaklak.vn/95/36109/dap-ung-tuong-doi-ve-dieu-kien-an-toan-ve-sinh-lao-dong/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 316109,
  'url': 'https://phuyen.baodaklak.vn/95/55318/phong-ngua-benh-tat-bang-rau-qua/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 316233,
  'url': 'https://phuyen.baodaklak.vn/95/58947/tang-cuong-phat-hien-benh-nhan-lao/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 316396,
  'url': 'https://phuyen.baodaklak.vn/95/63480/hoa-hong-chua-bach-benh/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 316563,
  'url': 'https://phuyen.baodaklak.vn/95/67737/cach-bo-sung-vi-chat-dinh-duong/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 316565,
  'url': 'https://phuyen.baodaklak.vn/95/678/nhung-tin-hieu-kha-quan-trong-viec-chong-ung-thu/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 316748,
  'url': 'https://phuyen.baodaklak.vn/95/71701/can-quan-tam-dac-biet-den-van-de-tang-huyet-ap/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 317176,
  'url': 'https://phuyen.baodaklak.vn/95/8185/phuc-hoi-ton-thuong-tuy-song-hieu-qua-cao-benh-nhan-phan-khoi/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 317451,
  'url': 'https://phuyen.baodaklak.vn/95/90567/phong-tranh-benh-o-nguoi-cao-tuoi-trong-mua-lanh/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 317485,
  'url': 'https://phuyen.baodaklak.vn/95/92508/gop-phan-phong-chong-benh-tat/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 317567,
  'url': 'https://phuyen.baodaklak.vn/95/9651/kiem-tra-an-toan-ve-sinh-thuc-pham/',
  'domain': 'phuyen.baodaklak.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 317725,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/10-loi-ich-vang-cua-omega-3-cho-suc-khoe-phu-nu-ma-ban-khong-the-bo-qua/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 318113,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/be-tre-so-sinh-cho-bu-dung-tu-the-me-va-be-cung-thoai-mai/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 318545,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/cac-giai-doan-cua-su-khoai-cam-trong-quan-he-tinh-duc-o-nam-gioi-va-nu-gioi/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 318798,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/cay-bau-dat-la-gi-kham-pha-10-bai-thuoc-dan-gian-tu-cay-bau-dat/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 318856,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/chao-ca-dieu-hong-cho-be/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 319081,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/daily-digestion-regularity-probiotic-tot-cho-tieu-hoa-tang-cuong-suc-khoe-khong-lo-tao-bon/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 319760,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/hoat-chat-chloramphenicol/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 319855,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/hoat-chat-eprazinon/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 319892,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/hoat-chat-gabapentin/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 320179,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/hoat-chat-terbutalin-sulfat/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 320440,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/kham-pha-giai-doan-dau-cua-ung-thu-phoi-khong-te-bao-nho-hieu-biet-de-phong-ngua/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 320792,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/me-bau-co-nen-an-cua-trong-3-thang-dau-thai-ky/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 321251,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/phat-hien-de-chien-thang-ung-thu-ruot-gia-va-nhung-dieu-can-biet/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 321493,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/roi-loan-lo-au-xa-hoi/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 321829,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/tai-sao-ba-bau-nen-an-hat-macca-va-khi-nao-la-thoi-diem-tot-nhat/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 322482,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/tri-viem-nieu-dao-man-tinh/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 322582,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/u-xo-tu-cung-va-u-nang-buong-trung-cong-dung-va-trieu-chung/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 322800,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/viem-duong-tiet-nieu-o-nu-gioi-dau-hieu-nguyen-nhan-va-phuong-phap-dieu-tri-hieu-qua/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 322859,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/viem-nang-long-sau-khi-wax-nguyen-nhan-va-cach-phong-tranh-hieu-qua/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 322908,
  'url': 'https://pmc-ecm-healthblog.beta.pharmacity.io/viem-tuy-nguyen-nhan-dieutri/',
  'domain': 'pmc-ecm-healthblog.beta.pharmacity.io',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 5118789,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/10925184/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 5142877,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/11233899/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 5201233,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/12068320/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 5283417,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/14964758/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 5766791,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/21865450/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 5839204,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/22737488/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 5875884,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/23173919/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 5890996,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/23344413/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 6219834,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/27028499/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 6497259,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/30220045/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 6747339,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/32830604/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 6769689,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/33034863/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 6911900,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/34347954/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 6934367,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/34543425/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 7029928,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/35386638/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 7089044,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/35928869/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 7156683,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/36544945/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 7209175,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/37040150/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 7360221,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/39529701/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 7453065,
  'url': 'https://pubmed.ncbi.nlm.nih.gov/41005560/',
  'domain': 'pubmed.ncbi.nlm.nih.gov',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 877312,
  'url': 'http://qihuangzhishu.com/1002/192.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 877367,
  'url': 'http://qihuangzhishu.com/1002/248.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 878074,
  'url': 'http://qihuangzhishu.com/1014/220.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 878088,
  'url': 'http://qihuangzhishu.com/1014/236.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 878093,
  'url': 'http://qihuangzhishu.com/1014/241.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 878112,
  'url': 'http://qihuangzhishu.com/1014/260.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 878383,
  'url': 'http://qihuangzhishu.com/1017/101.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 878538,
  'url': 'http://qihuangzhishu.com/1017/65.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 878539,
  'url': 'http://qihuangzhishu.com/1017/66.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 879697,
  'url': 'http://qihuangzhishu.com/130/97.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 879698,
  'url': 'http://qihuangzhishu.com/130/98.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 879909,
  'url': 'http://qihuangzhishu.com/140/169.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 885342,
  'url': 'http://qihuangzhishu.com/336/1163.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 886289,
  'url': 'http://qihuangzhishu.com/336/337.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 888050,
  'url': 'http://qihuangzhishu.com/616/160.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 892025,
  'url': 'http://qihuangzhishu.com/671/167.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 899604,
  'url': 'http://qihuangzhishu.com/969/391.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 899675,
  'url': 'http://qihuangzhishu.com/969/459.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 900617,
  'url': 'http://qihuangzhishu.com/985/185.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 900679,
  'url': 'http://qihuangzhishu.com/985/47.htm',
  'domain': 'qihuangzhishu.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 900874,
  'url': 'http://qy.familydoctor.com.cn/a/201404/603902.html',
  'domain': 'qy.familydoctor.com.cn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 900884,
  'url': 'http://qy.familydoctor.com.cn/a/201404/606757.html',
  'domain': 'qy.familydoctor.com.cn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 900935,
  'url': 'http://shen.39.net/a/120827/4042138.html',
  'domain': 'shen.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 901024,
  'url': 'http://shen.39.net/a/130929/4265818.html',
  'domain': 'shen.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 901197,
  'url': 'http://shen.39.net/a/140318/4356426.html',
  'domain': 'shen.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 901486,
  'url': 'http://shen.39.net/a/140616/4406109.html',
  'domain': 'shen.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 901574,
  'url': 'http://shen.39.net/a/140630/4417192.html',
  'domain': 'shen.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 901681,
  'url': 'http://shen.39.net/a/141227/4545837.html',
  'domain': 'shen.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 902170,
  'url': 'http://shen.39.net/a/190510/7129570.html',
  'domain': 'shen.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 902206,
  'url': 'http://shen.39.net/a/20101021/1507178.html',
  'domain': 'shen.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 902389,
  'url': 'http://shen.39.net/a/2010129/1562015.html',
  'domain': 'shen.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 902404,
  'url': 'http://shen.39.net/a/2010815/1444366.html',
  'domain': 'shen.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 902441,
  'url': 'http://shen.39.net/a/201089/1435125.html',
  'domain': 'shen.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 902493,
  'url': 'http://shen.39.net/a/201099/1476310.html',
  'domain': 'shen.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 902604,
  'url': 'http://shen.39.net/a/201112/1580846.html',
  'domain': 'shen.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 902678,
  'url': 'http://shen.39.net/a/201121/1599872.html',
  'domain': 'shen.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 902694,
  'url': 'http://shen.39.net/a/2011214/1611231.html',
  'domain': 'shen.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 902816,
  'url': 'http://shen.39.net/a/2011328/1645261.html',
  'domain': 'shen.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 902933,
  'url': 'http://shen.39.net/a/2011521/1691365.html',
  'domain': 'shen.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 902968,
  'url': 'http://shen.39.net/a/2011528/1693904.html',
  'domain': 'shen.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 903026,
  'url': 'http://shen.39.net/a/2011622/1726789.html',
  'domain': 'shen.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 903372,
  'url': 'http://shen.39.net/a/2011925/1813634.html',
  'domain': 'shen.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 904139,
  'url': 'http://sj.39.net/a/100704/1423173.html',
  'domain': 'sj.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 904279,
  'url': 'http://sj.39.net/a/140227/4366391.html',
  'domain': 'sj.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 904322,
  'url': 'http://sj.39.net/a/140327/4366286.html',
  'domain': 'sj.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 904408,
  'url': 'http://sj.39.net/a/140627/4415807.html',
  'domain': 'sj.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 904454,
  'url': 'http://sj.39.net/a/140730/4436269.html',
  'domain': 'sj.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 904490,
  'url': 'http://sj.39.net/a/140904/4462284.html',
  'domain': 'sj.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 904579,
  'url': 'http://sj.39.net/a/150410/4606423.html',
  'domain': 'sj.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 904600,
  'url': 'http://sj.39.net/a/150624/4645104.html',
  'domain': 'sj.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 904626,
  'url': 'http://sj.39.net/a/150717/4657663.html',
  'domain': 'sj.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 904647,
  'url': 'http://sj.39.net/a/150817/4678954.html',
  'domain': 'sj.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 904684,
  'url': 'http://sj.39.net/a/160129/4765685.html',
  'domain': 'sj.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 904708,
  'url': 'http://sj.39.net/a/20101019/1507260.html',
  'domain': 'sj.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 904719,
  'url': 'http://sj.39.net/a/20101022/1510279.html',
  'domain': 'sj.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 904768,
  'url': 'http://sj.39.net/a/20101123/1548915.html',
  'domain': 'sj.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 904813,
  'url': 'http://sj.39.net/a/2010819/1451187.html',
  'domain': 'sj.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 904843,
  'url': 'http://sj.39.net/a/2010929/1498984.html',
  'domain': 'sj.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 904882,
  'url': 'http://sj.39.net/a/2011126/1603621.html',
  'domain': 'sj.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 904891,
  'url': 'http://sj.39.net/a/2011330/1648688.html',
  'domain': 'sj.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 904906,
  'url': 'http://sj.39.net/a/2011428/1676382.html',
  'domain': 'sj.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 904966,
  'url': 'http://sj.39.net/a/201196/1794737.html',
  'domain': 'sj.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 583,
  'url': 'http://special.vietnamplus.vn/benh_vien_nhan_ai',
  'domain': 'special.vietnamplus.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 327054,
  'url': 'https://suckhoecongdongonline.vn/7-dieu-dan-ong-so-nhat-o-nguoi-phu-nu-ben-canh-minh',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 332886,
  'url': 'https://suckhoecongdongonline.vn/bang-gia-gas-hom-nay-11-12-gia-xang-dau-giam-keo-gia-gas-di-xuong',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 340593,
  'url': 'https://suckhoecongdongonline.vn/bo-giao-duc-va-dao-tao-yeu-cau-ubnd-tinh-lao-cai-xu-ly-nghiem-vu-viec-bua-an-cho-hoc-sinh-ban-tru',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 344689,
  'url': 'https://suckhoecongdongonline.vn/cach-bao-quan-thuc-pham-trong-tu-lanh-ngay-tet-dung-cach',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 363749,
  'url': 'https://suckhoecongdongonline.vn/di-lam-osin-xu-a-rap',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 369911,
  'url': 'https://suckhoecongdongonline.vn/dung-atiso-do-sao-cho-hieu-qua-va-an-toan',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 383449,
  'url': 'https://suckhoecongdongonline.vn/hanh-trinh-cuc-nhoc-cua-nguoi-lai-tau-hoa',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 397750,
  'url': 'https://suckhoecongdongonline.vn/lich-thi-dau-bong-da-hom-nay-ngay-4-3-831755307619',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 410098,
  'url': 'https://suckhoecongdongonline.vn/nguoi-cha-dap-xe-2000km-de-nghe-nhip-tim-con-gai-da-mat',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 414515,
  'url': 'https://suckhoecongdongonline.vn/nhieu-hoc-sinh-o-ha-noi-nghi-bi-ngo-doc-sau-bua-an-trua',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 426783,
  'url': 'https://suckhoecongdongonline.vn/quang-ninh-phau-thuat-loai-bo-khoi-buou-giap-da-nhan-nang-hon-hop-cho-mot-benh-nhan-mang-tren-minh-20-nam',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 427832,
  'url': 'https://suckhoecongdongonline.vn/rau-ma-co-tac-dung-gi',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 428820,
  'url': 'https://suckhoecongdongonline.vn/sang-13-2-hon-221-trieu-benh-nhan-covid-19-khoi-benh-con-2649-ca-nang-dang-dieu-tri',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 439984,
  'url': 'https://suckhoecongdongonline.vn/thien-giup-tang-cuong-gien-kiem-soat-phan-ung-mien-dich',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 447090,
  'url': 'https://suckhoecongdongonline.vn/tin-tuc-moi-nhat-ngay-18-2-2016-nam-thanh-nien-di-phuot-bi-roi-xuong-cong-tu-vong-o-cam-thinh-dong-cam-ranh-khanh-hoa',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 449575,
  'url': 'https://suckhoecongdongonline.vn/toi-14-10-ghi-nhan-11-ca-mac-covid-19-moi-co-1-benh-nhan-khoi-benh',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 455559,
  'url': 'https://suckhoecongdongonline.vn/trua-30-6-ghi-nhan-them-116-ca-mac-covid-19',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 461986,
  'url': 'https://suckhoecongdongonline.vn/van-phat-hung-tu-de-che-bds-phia-nam-den-hoat-dong-lay-lat-bi-beu-ten-no-thue',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 463216,
  'url': 'https://suckhoecongdongonline.vn/video-hai-co-be-don-tim-cu-dan-mang-vi-bieu-cam-sieu-dang-yeu-khi-thay-bo-bi-thuong',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 465373,
  'url': 'https://suckhoecongdongonline.vn/vinmec-ra-mat-chuyen-khoa-y-hoc-bao-thai-buoc-ngoat-dieu-tri-truoc-sinh-tai-viet-nam',
  'domain': 'suckhoecongdongonline.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 479192,
  'url': 'https://suckhoedoisong.vn/3-bai-thuoc-hay-chua-phu-do-than-hu-169100691.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 479583,
  'url': 'https://suckhoedoisong.vn/3-loai-thuoc-dieu-tri-benh-man-tinh-khong-nen-dung-chung-voi-giam-tao-169250811104838975.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 482263,
  'url': 'https://suckhoedoisong.vn/5-loai-thuc-pham-giup-lao-hoa-khoe-manh-va-minh-man-hon-169250507152003709.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 483084,
  'url': 'https://suckhoedoisong.vn/5-tac-dung-bat-ngo-khi-an-khoai-lang-vao-mua-dong-169251023180244664.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 486094,
  'url': 'https://suckhoedoisong.vn/7-tu-the-yoga-giup-giam-dau-do-lac-noi-mac-tu-cung-169221229163543429.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 486419,
  'url': 'https://suckhoedoisong.vn/8-loai-do-uong-tu-lam-ho-tro-than-va-duong-tiet-nieu-khoe-manh-169251023121638261.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 487647,
  'url': 'https://suckhoedoisong.vn/ai-nen-nieng-rang-16927288.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 500496,
  'url': 'https://suckhoedoisong.vn/cach-tri-het-tham-mun-seo-tham-trong-3-tuan-16982434.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 509588,
  'url': 'https://suckhoedoisong.vn/dai-thao-duong-thai-ky-ai-de-mac-16936834.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 517213,
  'url': 'https://suckhoedoisong.vn/giam-nguy-co-ung-thu-phoi-voi-5-thoi-quen-don-gian-sau-169231018114251523.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 521204,
  'url': 'https://suckhoedoisong.vn/ich-nhi-kien-dinh-phat-trien-nguon-duoc-lieu-sach-chuan-gacp-who-169251210112452927.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 531027,
  'url': 'https://suckhoedoisong.vn/neu-khong-quyet-liet-ngan-chan-thuoc-la-dien-tu-se-anh-huong-xau-den-gioi-tre-169230531183905076.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 532468,
  'url': 'https://suckhoedoisong.vn/ngu-vi-tu-tri-hen-suyen-ra-nhieu-mo-hoi-169210812105826617.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 533609,
  'url': 'https://suckhoedoisong.vn/nguoi-mac-benh-cum-nen-an-nhung-trai-cay-gi-giup-tang-suc-de-khang-169220728220525343.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 536704,
  'url': 'https://suckhoedoisong.vn/nhung-cuoc-chia-xa-vi-covid-169185365.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 541501,
  'url': 'https://suckhoedoisong.vn/phong-tranh-trat-khop-hang-sau-mo-thay-khop-hang-1692711.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 543292,
  'url': 'https://suckhoedoisong.vn/roi-loan-tien-dinh-sat-thu-tham-lang-169155177.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 544471,
  'url': 'https://suckhoedoisong.vn/sau-ung-thu-dai-truc-trang-benh-nhan-chien-thang-ung-thu-thuc-quan-giai-doan-cuoi-16925021712060652.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 554832,
  'url': 'https://suckhoedoisong.vn/tra-thanh-nhiet-cho-mua-he-16928250.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 562931,
  'url': 'https://suckhoedoisong.vn/xua-tan-stress-chong-mat-bang-5-loai-tra-thao-moc-169146750.htm',
  'domain': 'suckhoedoisong.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 563755,
  'url': 'https://suckhoeviet.org.vn/bac-si-chi-cach-khong-bi-pha-suc-khi-choi-pickleball-toi-muon-23559.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 563874,
  'url': 'https://suckhoeviet.org.vn/bai-thuoc-dan-gian-ho-tro-hen-suyen-hen-phe-quan-phuc-vu-cho-mua-dong-2224.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 563987,
  'url': 'https://suckhoeviet.org.vn/bao-dam-an-toan-thuc-pham-hieu-ro-va-tuan-thu-nhung-quy-chuan-quan-trong-11274.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 565711,
  'url': 'https://suckhoeviet.org.vn/dong-da-ha-noi-to-chuc-cham-soc-suc-khoe-toan-dien-va-cap-thuoc-mien-phi-cho-nguoi-dan-phuong-tho-quan-11903.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 565811,
  'url': 'https://suckhoeviet.org.vn/e-magazine-ky-tu-kim-cuong-do-trong-dong-y-12767-12767.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 565940,
  'url': 'https://suckhoeviet.org.vn/gioi-thieu-tai-lieu-ho-tro-hanh-vi-tich-cuc-cho-tre-em-roi-loan-pho-tu-ky-11615.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 566540,
  'url': 'https://suckhoeviet.org.vn/lam-the-nao-de-thuong-thuc-thuc-pham-chien-ran-mot-cach-an-toan-22313.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 566573,
  'url': 'https://suckhoeviet.org.vn/lich-an-uong-giam-can-khoa-hoc-cho-nguoi-ban-ron-19996.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 566607,
  'url': 'https://suckhoeviet.org.vn/loi-ich-bat-ngo-cua-viec-uong-tra-xanh-moi-ngay-21470.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 566636,
  'url': 'https://suckhoeviet.org.vn/loi-ich-suc-khoe-cua-phan-xa-lieu-phap-11666.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 566860,
  'url': 'https://suckhoeviet.org.vn/mon-an-bai-thuoc-ho-tro-dieu-tri-benh-tay-chan-mieng-6367.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 567002,
  'url': 'https://suckhoeviet.org.vn/nam-lim-xanh-nhung-cong-dung-noi-bat-it-nguoi-biet-35.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 567071,
  'url': 'https://suckhoeviet.org.vn/nganh-y-te-nghe-an-day-manh-cong-tac-phong-chong-hivaids-trong-gioi-tre-11498.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 567626,
  'url': 'https://suckhoeviet.org.vn/phat-huy-hieu-qua-quy-trinh-bao-dong-do-noi-vien-bao-dong-do-lien-vien-cuu-song-nguoi-benh-trong-tinh-trang-nguy-kich-12168.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 568232,
  'url': 'https://suckhoeviet.org.vn/tang-cuong-van-dong-o-tuoi-trung-nien-chia-khoa-phong-tranh-benh-alzheimer-22054.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 568309,
  'url': 'https://suckhoeviet.org.vn/thao-duoc-trong-y-hoc-co-truyen-giup-cai-thien-chuc-nang-than-20427.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 568860,
  'url': 'https://suckhoeviet.org.vn/uong-nuoc-cot-chanh-co-thuc-su-giup-giam-can-va-cai-thien-suc-khoe-22414.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 568881,
  'url': 'https://suckhoeviet.org.vn/uu-tien-suc-khoe-tam-than-tai-noi-lam-viec-14928.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 568897,
  'url': 'https://suckhoeviet.org.vn/vai-tro-quan-trong-cua-vitamin-nhom-b-doi-voi-nao-bo-21934.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 569047,
  'url': 'https://suckhoeviet.org.vn/xit-xoang-trieu-minh-bai-thuoc-tu-tinh-thuong-va-tri-tue-24824.html',
  'domain': 'suckhoeviet.org.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 569385,
  'url': 'https://tamanhhospital.vn/tin-tuc/be-trai-chao-doi-voi-day-ron-that-nut/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 569470,
  'url': 'https://tamanhhospital.vn/tin-tuc/bi-quyet-giup-vuot-qua-tien-man-kinh/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 569753,
  'url': 'https://tamanhhospital.vn/tin-tuc/cham-soc-va-dieu-tri-benh-tim-mach-cho-benh-nhan-ung-thu/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 570230,
  'url': 'https://tamanhhospital.vn/tin-tuc/di-kham-suc-khoe-phat-hien-khoi-u-to-bang-qua-trung-ga/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 570441,
  'url': 'https://tamanhhospital.vn/tin-tuc/gan-mot-thap-ky-cung-vo-chua-vo-sinh/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 570576,
  'url': 'https://tamanhhospital.vn/tin-tuc/hach-bach-huyet-khong-lo-gay-dau-lung-kinh-nien/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 570614,
  'url': 'https://tamanhhospital.vn/tin-tuc/hanh-phuc-lam-me-sau-5-lan-mat-con/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 570636,
  'url': 'https://tamanhhospital.vn/tin-tuc/het-benh-ngung-tho-khi-ngu-nho-do-da-ky-ho-hap/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 571016,
  'url': 'https://tamanhhospital.vn/tin-tuc/ky-thuat-tao-nhip-bo-nhanh-trai-giam-bien-chung-suy-tim-cho-benh-nhan-nhip-cham/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 571034,
  'url': 'https://tamanhhospital.vn/tin-tuc/lam-dep-toan-dien-khong-dau-voi-song-va-laser/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 571212,
  'url': 'https://tamanhhospital.vn/tin-tuc/mat-co-bap-vi-giam-can-sai-cach/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 571348,
  'url': 'https://tamanhhospital.vn/tin-tuc/mot-minh-chua-vo-sinh/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 571598,
  'url': 'https://tamanhhospital.vn/tin-tuc/nhau-3-lan-tuan-nguoi-dan-ong-cap-cuu-do-viem-tuy-cap/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 571917,
  'url': 'https://tamanhhospital.vn/tin-tuc/phat-hien-ung-thu-vu-sieu-nho-bang-ky-thuat-moi/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 572188,
  'url': 'https://tamanhhospital.vn/tin-tuc/sau-nam-kien-tri-de-co-them-con/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 572260,
  'url': 'https://tamanhhospital.vn/tin-tuc/son-tieu-benh-kho-noi-cua-nu-gioi/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 572422,
  'url': 'https://tamanhhospital.vn/tin-tuc/tam-soat-ung-thu-vu-phat-hien-vo-tui-nguc/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 572696,
  'url': 'https://tamanhhospital.vn/tin-tuc/tien-bo-trong-dieu-tri-vo-sinh-nam/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 573020,
  'url': 'https://tamanhhospital.vn/tin-tuc/u-to-bang-hat-dau-o-day-luoi-be-trai/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 573169,
  'url': 'https://tamanhhospital.vn/tin-tuc/viem-tai-do-ghien-lay-ray-tai/',
  'domain': 'tamanhhospital.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 2537636,
  'url': 'https://test.pmphai.com/jeesitede/appcase/toPcDetail?sessionId=&knowledgeLibPrefix=case&id=15333',
  'domain': 'test.pmphai.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2538067,
  'url': 'https://test.pmphai.com/jeesitede/appcase/toPcDetail?sessionId=&knowledgeLibPrefix=case&id=15768',
  'domain': 'test.pmphai.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2538891,
  'url': 'https://test.pmphai.com/jeesitede/appcase/toPcDetail?sessionId=&knowledgeLibPrefix=case&id=1782299726612267009',
  'domain': 'test.pmphai.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2539106,
  'url': 'https://test.pmphai.com/jeesitede/appcase/toPcDetail?sessionId=&knowledgeLibPrefix=case&id=1782304019746652161',
  'domain': 'test.pmphai.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2539705,
  'url': 'https://test.pmphai.com/jeesitede/appcase/toPcDetail?sessionId=&knowledgeLibPrefix=case&id=31566',
  'domain': 'test.pmphai.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2539842,
  'url': 'https://test.pmphai.com/jeesitede/appcase/toPcDetail?sessionId=&knowledgeLibPrefix=case&id=7189',
  'domain': 'test.pmphai.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2540890,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=0001AA100000000ELZTV',
  'domain': 'test.pmphai.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2541360,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=0001AA100000000EOC1Y',
  'domain': 'test.pmphai.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2542571,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=11007',
  'domain': 'test.pmphai.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2543274,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=13409',
  'domain': 'test.pmphai.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2544443,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=1656113918809997314',
  'domain': 'test.pmphai.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2545272,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=18030',
  'domain': 'test.pmphai.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2545329,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=18091',
  'domain': 'test.pmphai.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2546095,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=19642',
  'domain': 'test.pmphai.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2546129,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=19688',
  'domain': 'test.pmphai.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2546507,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=20152',
  'domain': 'test.pmphai.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2546815,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=21576',
  'domain': 'test.pmphai.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2547042,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=21837',
  'domain': 'test.pmphai.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2547832,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=22972',
  'domain': 'test.pmphai.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2549554,
  'url': 'https://test.pmphai.com/jeesitede/appdisease/toPcDetail?sessionId=&knowledgeLibPrefix=disease&id=26744',
  'domain': 'test.pmphai.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 594363,
  'url': 'https://thanhnien.vn/an-tu-hinh-cho-ke-bo-muoi-cong-nghiep-vao-sua-185312511.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 597073,
  'url': 'https://thanhnien.vn/bao-ve-da-truoc-tia-cuc-tim-uv-nang-nong-gay-gat-nhu-the-nao-185955632.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 598860,
  'url': 'https://thanhnien.vn/benh-tri-de-mac-khi-ngoi-may-tinh-nhieu-18591795.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 599843,
  'url': 'https://thanhnien.vn/bi-dau-bung-duoi-sau-khi-hanh-kinh-185312986.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 605737,
  'url': 'https://thanhnien.vn/chau-be-3-tuoi-tu-vong-co-the-do-ngat-tho-vi-dom-185109352.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 611739,
  'url': 'https://thanhnien.vn/dai-tuong-phung-quang-thanh-den-bo-quoc-phong-chi-dao-cong-viec-185488526.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 617627,
  'url': 'https://thanhnien.vn/giai-dap-nhung-nghi-ngo-ve-trung-song-va-trung-nau-chin-1851110729.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 618940,
  'url': 'https://thanhnien.vn/giup-nao-khoe-18585162.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 623311,
  'url': 'https://thanhnien.vn/khong-phai-thuoc-nao-cung-be-nho-ngam-cho-tan-18542107.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 624783,
  'url': 'https://thanhnien.vn/lam-the-nao-phong-ngua-nhiem-giun-san-tu-thu-cung-18525021818441013.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 628948,
  'url': 'https://thanhnien.vn/mot-y-si-hien-tang-tuy-cho-cau-be-9-tuoi-bi-ung-thu-mau-185792018.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 629905,
  'url': 'https://thanhnien.vn/nang-ta-co-lam-phu-nu-to-va-tho-kech-185971883.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 635337,
  'url': 'https://thanhnien.vn/nhan-biet-dau-hieu-nghien-mua-sam-185230724161824552.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 635805,
  'url': 'https://thanhnien.vn/nhieu-chuyen-gia-hang-dau-tham-du-hoi-nghi-ve-tai-mui-hong-o-nha-trang-1851010098.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 641520,
  'url': 'https://thanhnien.vn/phau-thuat-nam-khoa-trong-dieu-tri-vo-sinh-nam-185164530.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 646094,
  'url': 'https://thanhnien.vn/tai-nan-ky-la-rang-sua-chui-vao-khi-quan-khi-co-be-dang-ngu-185857988.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 648031,
  'url': 'https://thanhnien.vn/thanh-cong-viec-thay-khop-goi-toan-phan-185140162.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 653873,
  'url': 'https://thanhnien.vn/tri-thua-can-tu-rong-bien-185317383.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 656696,
  'url': 'https://thanhnien.vn/vi-sao-an-thit-rua-bien-co-the-gay-ngo-doc-185240311165510549.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 659078,
  'url': 'https://thanhnien.vn/xet-nghiem-omicron-tu-mau-nuoc-bot-chinh-xac-hon-1851417393.htm',
  'domain': 'thanhnien.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 659601,
  'url': 'https://thaythuocvietnam.vn/3-cong-dung-tuyet-voi-cua-hat-sen/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 660083,
  'url': 'https://thaythuocvietnam.vn/cach-chua-benh-tri-ngoai-va-noi-khac-nhau-nhu-the-nao/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 660383,
  'url': 'https://thaythuocvietnam.vn/chua-viem-xoang-bang-hoa-ngu-sac/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 660411,
  'url': 'https://thaythuocvietnam.vn/co-dung-cam-xuyen-huong-cho-tre-duoc-khong/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 660440,
  'url': 'https://thaythuocvietnam.vn/cong-dung-cua-la-sen/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 660712,
  'url': 'https://thaythuocvietnam.vn/giai-dap-su-khac-biet-giua-2-san-pham-bo-doi-hon-dich-anvitra-su-dung-cho-tung-muc-do-trao-nguoc/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 660808,
  'url': 'https://thaythuocvietnam.vn/hon-200-tre-mien-tay-nhap-vien-theo-doi-sau-tiem-vac-xin-combe-five/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 660869,
  'url': 'https://thaythuocvietnam.vn/khoang-cach-giua-2-lan-pha-thai-bang-thuoc/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 660967,
  'url': 'https://thaythuocvietnam.vn/mat-na-cho-da-trong-nhung-ngay/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 661026,
  'url': 'https://thaythuocvietnam.vn/mot-so-bai-thuoc-chua-mat-ngu-bang-dong-y/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 661034,
  'url': 'https://thaythuocvietnam.vn/mot-so-dau-hieu-cua-tuoi-gia/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 661048,
  'url': 'https://thaythuocvietnam.vn/mua-he-van-de-bi-viem-hong-tai-sao/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 661348,
  'url': 'https://thaythuocvietnam.vn/nhung-hau-qua-sau-mac-covid-19/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 661457,
  'url': 'https://thaythuocvietnam.vn/phan-loai-cac-loai-mun-trung-ca/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 661715,
  'url': 'https://thaythuocvietnam.vn/tay-cao-rang-tai-nha-bi-quyet-toan-hieu-qua/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 661745,
  'url': 'https://thaythuocvietnam.vn/thieu-mau-co-tim-hau-covid-nguy-co-va-cach-khac-phuc/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 661780,
  'url': 'https://thaythuocvietnam.vn/thuc-don-cho-nguoi-tang-huyet-ap-co-soi/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 661893,
  'url': 'https://thaythuocvietnam.vn/thuvien/hoi-thao/hoi-thao-15-06-2025-toi-uu-hoa-dieu-tri-dinh-duong-trong-quan-ly-dai-thao-duong/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 662327,
  'url': 'https://thaythuocvietnam.vn/tuc-nang-moi-chi-ne-tim-la-dau-hieu-cua-benh-gi/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 662481,
  'url': 'https://thaythuocvietnam.vn/xuat-huyet-tieu-hoa-cao-tren-va-nhung-dieu-can-biet/',
  'domain': 'thaythuocvietnam.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 663247,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/benh-a-z-dau-dau-van-mach',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 663642,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/benh-crohn-o-tre-em-nguyen-nhan-trieu-chung-va-huong-dieu-tri-hien-nay',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 664349,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/bi-sot-xuat-huyet-co-quan-he-duoc-khong',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 664572,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/boi-bo-suc-khoe-bang-nhan-sam-ngam-mat-ong-va-cach-dung',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 664695,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/cac-benh-ve-mat-thuong-gap-hien-nay-nguyen-nhan-va-cach-phong-ngua',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 665007,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/cach-giam-ure-trong-mau-an-toan-hieu-qua-va-de-ap-dung',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 665103,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/cach-phong-tranh-benh-loang-xuong-nguyen-tac-vang-de-giu-xuong-chac-khoe',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 665309,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/canh-bao-dau-hieu-ung-thu-mau-giai-doan-cuoi',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 666183,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/dai-trang-va-truc-trang-cau-truc-chuc-nang-va-cac-benh-nguy-hiem-lien-quan',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 666429,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/dau-hieu-roi-loan-kinh-nguyet-sau-sot-xuat-huyet',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 666874,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/doi-moi-la-gi-nguyen-nhan-dau-hieu-va-cach-cai-thien-doi-moi-tren-da',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 667005,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/gan-nhiem-mo-co-nguy-hiem-khong-trieu-chung-bien-chung-va-cach-dieu-tri',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 667931,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/khi-hu-co-mui-trung-thoi-nguyen-nhan-phuong-phap-dieu-tri-va-cach-phong-ngua',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 669995,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/nhung-dia-diem-dang-tin-cay-de-tiem-viem-mang-nao-mo-cau-ac-o-ha-noi',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 670031,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/nhung-luu-y-khi-tiem-vac-xin-phong-viem-mang-nao-mo-cau',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 670505,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/phu-chan',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 671314,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/sot-xuat-huyet-co-duoc-nam-may-lanh-khong',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 671465,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/sung-nuou-rang-co-tu-het-khong-khi-nao-can-gap-bac-si',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 671670,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/tai-sao-phu-nu-thuong-bi-tri-sau-sinh-dau-hieu-nhan-biet',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 674984,
  'url': 'https://tiemchunglongchau.com.vn/kien-thuc-tiem-chung/viem-ruot-thua',
  'domain': 'tiemchunglongchau.com.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 675859,
  'url': 'https://tienphong.vn/an-nham-loai-hai-san-chua-chat-kich-doc-nguoi-dan-ong-liet-toan-than-nguy-kich-post1750427.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 675886,
  'url': 'https://tienphong.vn/bac-ninh-thong-tin-ve-benh-nhan-tu-vong-do-soc-phan-ve-post1767050.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 675904,
  'url': 'https://tienphong.vn/bac-si-chuyen-khoa-giai-dap-nhieu-cau-kho-cho-benh-nhan-suy-tuyen-thuong-than-post1769539.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 675911,
  'url': 'https://tienphong.vn/bac-si-doi-mat-rui-ro-phap-ly-khi-ung-dung-ai-trong-dieu-tri-post1797883.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 675987,
  'url': 'https://tienphong.vn/be-gai-16-thang-tuoi-nguy-kich-vi-me-nau-an-bang-nuoc-gieng-post1753953.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 676057,
  'url': 'https://tienphong.vn/benh-nhan-nghien-thuoc-la-bang-hoang-khi-bac-si-thong-bao-bi-hoai-tu-xuong-post1806422.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 676148,
  'url': 'https://tienphong.vn/benh-vien-thanh-nhan-tiep-nhan-nhieu-ca-mac-covid-19-sot-xuat-huyet-tre-em-chiem-ti-le-cao-post1744286.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 676433,
  'url': 'https://tienphong.vn/cap-cuu-ngu-dan-bi-ca-bien-co-doc-dam-vao-chan-o-vung-bien-truong-sa-post1723873.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 676481,
  'url': 'https://tienphong.vn/che-do-dinh-duong-voi-benh-nhan-viem-loet-da-day-ta-trang-post1761466.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 676625,
  'url': 'https://tienphong.vn/cu-ong-dot-quy-tren-dinh-nui-cao-nhat-khu-vuc-nam-bo-post1765313.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 677042,
  'url': 'https://tienphong.vn/hai-xe-may-lao-am-am-vao-nhau-5-nguoi-thuong-vong-post1823920.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 677246,
  'url': 'https://tienphong.vn/khong-cap-nhat-nguyen-lieu-lam-thuoc-cong-ty-co-phan-duoc-thiet-bi-y-te-da-nang-linh-phat-post1723229.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 677436,
  'url': 'https://tienphong.vn/mao-danh-benh-vien-de-lua-dao-post1762542.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 677451,
  'url': 'https://tienphong.vn/me-chon-gio-dep-de-sinh-mo-con-tu-vong-sau-hai-ngay-chao-doi-post1768946.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 677540,
  'url': 'https://tienphong.vn/nam-thanh-nien-thung-ruot-vi-thoi-quen-pho-bien-o-gioi-tre-post1792573.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 677595,
  'url': 'https://tienphong.vn/ngoai-khoa-benh-vien-da-khoa-viet-duc-phu-tho-tien-phong-ve-ky-thuat-hoi-sinh-su-song-post1779812.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 677736,
  'url': 'https://tienphong.vn/nhieu-benh-nhan-soc-nguy-kich-do-sot-xuat-huyet-post1785834.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 677769,
  'url': 'https://tienphong.vn/nhieu-tre-nho-nhap-vien-vi-cum-a-canh-bao-nguy-co-bien-chung-nang-post1794422.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 677909,
  'url': 'https://tienphong.vn/phat-hien-qua-trung-trong-o-bung-cu-ong-gan-100-tuoi-sau-nhieu-ngay-bi-tieu-post1751732.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 678322,
  'url': 'https://tienphong.vn/thuc-hu-con-so-hon-11000-nguoi-o-nghe-an-nhiem-hiv-post1796685.tpo',
  'domain': 'tienphong.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 905094,
  'url': 'http://tnb.39.net/a/100727/1407887.html',
  'domain': 'tnb.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 905187,
  'url': 'http://tnb.39.net/a/121030/4073892.html',
  'domain': 'tnb.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 905195,
  'url': 'http://tnb.39.net/a/121106/4108025.html',
  'domain': 'tnb.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 905361,
  'url': 'http://tnb.39.net/a/130622/4193664.html',
  'domain': 'tnb.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 905379,
  'url': 'http://tnb.39.net/a/130723/4223740.html',
  'domain': 'tnb.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 905447,
  'url': 'http://tnb.39.net/a/130923/4260590.html',
  'domain': 'tnb.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 905659,
  'url': 'http://tnb.39.net/a/170216/5222522.html',
  'domain': 'tnb.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 905941,
  'url': 'http://tnb.39.net/a/171201/5886008.html',
  'domain': 'tnb.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 906871,
  'url': 'http://tnb.39.net/a/190620/7231323.html',
  'domain': 'tnb.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 906971,
  'url': 'http://tnb.39.net/a/190816/7390975.html',
  'domain': 'tnb.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 907343,
  'url': 'http://tnb.39.net/a/200508/7881665.html',
  'domain': 'tnb.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 907735,
  'url': 'http://tnb.39.net/a/2011312/1633081.html',
  'domain': 'tnb.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 907852,
  'url': 'http://tnb.39.net/a/2011530/1705474.html',
  'domain': 'tnb.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 907874,
  'url': 'http://tnb.39.net/a/2011620/1726123.html',
  'domain': 'tnb.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 907912,
  'url': 'http://tnb.39.net/a/2011813/1772960.html',
  'domain': 'tnb.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 908023,
  'url': 'http://tnb.39.net/a/201222/1967286.html',
  'domain': 'tnb.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 908548,
  'url': 'http://tnb.39.net/a/230822/n8ca36v.html',
  'domain': 'tnb.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 908590,
  'url': 'http://tnb.39.net/a/230827/x5d314w.html',
  'domain': 'tnb.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 908809,
  'url': 'http://tnb.39.net/a/230929/x8gtkjf.html',
  'domain': 'tnb.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 909030,
  'url': 'http://tnb.39.net/a/231202/t2k3rqx.html',
  'domain': 'tnb.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 679256,
  'url': 'https://tuoitre.vn/5-moi-truong-de-lay-benh-nao-mo-cau-cho-thanh-thieu-nien-2025061717183559.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 679532,
  'url': 'https://tuoitre.vn/bat-nhao-tri-lieu-tam-ly-online-ky-2-ai-kiem-soat-nha-tri-lieu-online-20250514224153525.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 679568,
  'url': 'https://tuoitre.vn/be-trai-10-tuoi-bi-yeu-liet-nua-nguoi-vi-nhoi-mau-cap-hiem-gap-20260206080334316.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 679673,
  'url': 'https://tuoitre.vn/benh-vien-giao-thong-van-tai-tp-hcm-duoc-ban-giao-ve-tp-hcm-quan-ly-20251028170104082.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 679817,
  'url': 'https://tuoitre.vn/bo-y-te-ban-hanh-huong-dan-che-do-dinh-duong-sinh-ceton-cho-nguoi-benh-dong-kinh-khang-thuoc-20251223155905631.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 680132,
  'url': 'https://tuoitre.vn/chinh-quyen-hai-cap-tram-y-te-xa-benh-vien-tuyen-huyen-sap-xep-the-nao-20250707162534821.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 680236,
  'url': 'https://tuoitre.vn/co-gai-tre-do-rat-mat-vi-o-nhiem-khong-khi-bac-si-huong-dan-cach-bao-ve-da-20251206092734757.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 680243,
  'url': 'https://tuoitre.vn/co-khoi-sui-nho-10-nam-khong-chua-nay-phat-hien-ung-thu-duong-vat-20250530234826677.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 680420,
  'url': 'https://tuoitre.vn/dang-di-sieu-thi-nu-dieu-duong-benh-vien-bach-mai-kip-thoi-cuu-song-bac-bao-ve-bi-ngung-tuan-hoan-20260227195155361.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 680596,
  'url': 'https://tuoitre.vn/dung-bao-cao-su-co-tranh-duoc-sui-mao-ga-sinh-duc-20251113063430153.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 680897,
  'url': 'https://tuoitre.vn/kham-suc-khoe-mien-phi-phat-hien-som-benh-tat-20251005231117219.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 681457,
  'url': 'https://tuoitre.vn/nguoi-dan-tp-hcm-tren-65-tuoi-nhan-the-bao-hiem-y-te-mien-phi-o-dau-20251116092631846.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 681567,
  'url': 'https://tuoitre.vn/nhieu-ky-thuat-co-the-chua-khoi-hoan-toan-ung-thu-duong-tieu-hoa-neu-phat-hien-som-20251116100506236.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 681633,
  'url': 'https://tuoitre.vn/nhung-sai-lam-khi-tap-the-duc-the-thao-de-gay-ngung-tim-dot-quy-20250905154937886.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 682277,
  'url': 'https://tuoitre.vn/thuc-hu-cac-phuong-phap-tang-kich-thuoc-cau-nho-20250905215840225.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 682334,
  'url': 'https://tuoitre.vn/ti-le-sinh-han-quoc-tang-ngoan-muc-trong-quy-i-20250529162637648.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 682572,
  'url': 'https://tuoitre.vn/trai-xoai-thuc-pham-rat-tot-cho-mat-va-suc-khoe-tim-mach-20251203081920775.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 682762,
  'url': 'https://tuoitre.vn/unicef-lan-dau-tien-so-tre-beo-phi-vuot-qua-so-tre-thieu-can-20250910142943769.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 682792,
  'url': 'https://tuoitre.vn/uy-quyen-nhan-luong-huu-khi-van-hanh-chinh-quyen-dia-phuong-2-cap-can-luu-y-gi-20250725182614444.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 682949,
  'url': 'https://tuoitre.vn/vu-ngo-doc-sau-an-banh-mi-o-tp-hcm-tang-len-235-ca-cac-nan-nhan-nam-o-13-benh-vien-20251110111418367.htm',
  'domain': 'tuoitre.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 683168,
  'url': 'https://vietnamnet.vn/14-nguoi-bi-ngo-doc-ruou-trong-dam-tang-3-nguoi-dang-nguy-kich-2078496.html#comment',
  'domain': 'vietnamnet.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 684135,
  'url': 'https://vietnamnet.vn/astrazeneca-thu-hoi-vac-xin-covid-19-tren-toan-cau-2278398.html#comment',
  'domain': 'vietnamnet.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 684476,
  'url': 'https://vietnamnet.vn/bac-si-tai-tphcm-khong-con-phai-lo-tim-thuoc-hiem-trong-ca-truc-cap-cuu-2318403.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 684841,
  'url': 'https://vietnamnet.vn/be-trai-2-tuoi-da-co-soi-than-2135255.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 684990,
  'url': 'https://vietnamnet.vn/benh-nhan-416-dong-dac-phoi-co-the-phai-ghep-phoi-669507.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 685677,
  'url': 'https://vietnamnet.vn/bo-sung-glucosamine-dung-cach-765505.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 686292,
  'url': 'https://vietnamnet.vn/ca-nghi-nhiem-covid-19-o-quan-3-tphcm-ban-quan-an-738374.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 686569,
  'url': 'https://vietnamnet.vn/cac-nguyen-nhan-hang-dau-gay-ra-hang-trieu-ca-benh-lien-quan-thuc-pham-2335081.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 686598,
  'url': 'https://vietnamnet.vn/cac-tac-dong-it-duoc-quan-tam-cua-covid-19-len-nao-809499.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 686625,
  'url': 'https://vietnamnet.vn/cac-trieu-chung-gay-bat-ngo-cua-ung-thu-phoi-677882.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 687965,
  'url': 'https://vietnamnet.vn/cuoc-gap-dac-biet-cua-nguoi-phu-nu-va-bac-si-cuu-minh-30-nam-truoc-2366622.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 689192,
  'url': 'https://vietnamnet.vn/giam-doc-benh-vien-khen-gia-dinh-so-cuu-tot-cho-nu-benh-nhan-bi-set-danh-2288937.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 690166,
  'url': 'https://vietnamnet.vn/hanh-trinh-thien-nguyen-mat-sang-cho-em-den-voi-hoc-sinh-reo-cao-bac-kan-2232759.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 693640,
  'url': 'https://vietnamnet.vn/nhung-nguoi-khong-nhiem-covid-19-du-song-giua-the-gioi-f0-820411.html#comment',
  'domain': 'vietnamnet.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 695795,
  'url': 'https://vietnamnet.vn/the-gioi-co-the-can-cac-loai-vac-xin-moi-chong-covid-19-780428.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 696408,
  'url': 'https://vietnamnet.vn/thuoc-remdesivir-dieu-tri-covid-19-hieu-qua-the-nao-voi-benh-nhan-nang-765061.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 697258,
  'url': 'https://vietnamnet.vn/trua-149-ha-noi-them-8-ca-covid-19-co-1-ca-cong-dong-o-thanh-xuan-774717.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 697438,
  'url': 'https://vietnamnet.vn/tu-vu-thit-thoi-len-loi-vao-cho-chuyen-gia-mach-ba-noi-tro-4-khong-2420248.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 698066,
  'url': 'https://vietnamnet.vn/viet-nam-them-4411-ca-covid-19-da-tiem-gan-76-trieu-lieu-vac-xin-787409.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 698534,
  'url': 'https://vietnamnet.vn/xuat-hien-chum-ca-benh-chua-ro-nguon-lay-4-nguoi-cung-gia-dinh-o-hai-duong-790228.html',
  'domain': 'vietnamnet.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 701721,
  'url': 'https://vnexpress.net/4-hieu-lam-thuong-gap-ve-benh-glocom-5051276.html',
  'domain': 'vnexpress.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 701757,
  'url': 'https://vnexpress.net/5-cach-giup-me-bau-ngu-ngon-trong-thai-ky-5048270.html',
  'domain': 'vnexpress.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 701760,
  'url': 'https://vnexpress.net/5-cach-phong-dau-mat-do-dip-tet-5041229.html',
  'domain': 'vnexpress.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 701850,
  'url': 'https://vnexpress.net/5-tu-the-yoga-tot-cho-mat-5005904.html',
  'domain': 'vnexpress.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 701878,
  'url': 'https://vnexpress.net/6-mon-quen-thuoc-dot-mo-gan-5043457.html',
  'domain': 'vnexpress.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 701928,
  'url': 'https://vnexpress.net/8-cach-don-gian-lam-sach-bui-trong-nha-4992155.html',
  'domain': 'vnexpress.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 702055,
  'url': 'https://vnexpress.net/bac-si-an-do-chi-ra-5-dau-hieu-canh-bao-than-suy-yeu-5045595.html',
  'domain': 'vnexpress.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 702286,
  'url': 'https://vnexpress.net/cach-giup-xuong-tre-chac-khoe-tu-nho-5046934.html',
  'domain': 'vnexpress.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 702302,
  'url': 'https://vnexpress.net/cach-tranh-tang-can-sau-tet-5013198.html',
  'domain': 'vnexpress.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 702476,
  'url': 'https://vnexpress.net/cum-a-h3n2-co-the-bien-chung-viem-nao-4994799.html',
  'domain': 'vnexpress.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 702499,
  'url': 'https://vnexpress.net/dang-nieng-rang-co-tiem-filler-duoc-khong-5047567.html',
  'domain': 'vnexpress.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 702551,
  'url': 'https://vnexpress.net/dieu-gi-xay-ra-khi-ban-an-nhieu-do-ngot-5041162.html',
  'domain': 'vnexpress.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 702584,
  'url': 'https://vnexpress.net/dua-tre-benh-hiem-doi-doi-nho-lieu-thuoc-tri-gia-hang-tram-trieu-dong-5044325.html',
  'domain': 'vnexpress.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 702664,
  'url': 'https://vnexpress.net/hari-won-can-pha-vo-nhieu-dinh-kien-ve-hpv-4934944.html',
  'domain': 'vnexpress.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 702919,
  'url': 'https://vnexpress.net/mo-mat-do-bien-chung-tieu-duong-5049435.html',
  'domain': 'vnexpress.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 703002,
  'url': 'https://vnexpress.net/nguoi-benh-ung-thu-dang-dieu-tri-co-nen-phun-xam-tham-my-5041297.html',
  'domain': 'vnexpress.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 703243,
  'url': 'https://vnexpress.net/ruou-bia-la-chat-xuc-tac-hay-ke-pha-binh-phong-the-5041995.html',
  'domain': 'vnexpress.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 703262,
  'url': 'https://vnexpress.net/sau-tiem-vaccine-hpv-co-can-tam-soat-ung-thu-co-tu-cung-khong-5044954.html',
  'domain': 'vnexpress.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 703447,
  'url': 'https://vnexpress.net/trao-nguoc-hong-thanh-quan-co-chua-khoi-khong-5044665.html',
  'domain': 'vnexpress.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 703580,
  'url': 'https://vnexpress.net/ve-viet-nam-vua-an-tet-vua-chay-than-5046350.html',
  'domain': 'vnexpress.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 704088,
  'url': 'https://vov.vn/an-sach-song-khoe/phat-hien-hang-chuc-tan-long-lon-ngam-tam-phu-gia-khong-ro-nguon-goc-845027.vov',
  'domain': 'vov.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 705213,
  'url': 'https://vov.vn/suc-khoe/8-cach-giam-dau-dau-khong-can-dung-thuoc-post1081582.vov',
  'domain': 'vov.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 705761,
  'url': 'https://vov.vn/suc-khoe/bao-dong-tinh-trang-thieu-sat-kem-o-tre-em-viet-nam-post964949.vov',
  'domain': 'vov.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 706179,
  'url': 'https://vov.vn/suc-khoe/bot-san-day-that-hay-gia-5-cach-nhan-biet-don-gian-it-nguoi-biet-post1270498.vov',
  'domain': 'vov.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 706189,
  'url': 'https://vov.vn/suc-khoe/buoi-co-bo-phan-la-than-duoc-dai-bo-chong-lao-hoa-cuc-tot-lai-it-nguoi-biet-post1153488.vov',
  'domain': 'vov.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 706530,
  'url': 'https://vov.vn/suc-khoe/can-thiep-dieu-tri-benh-ly-thuong-gap-o-nam-gioi-post1055078.vov',
  'domain': 'vov.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 706874,
  'url': 'https://vov.vn/suc-khoe/chuyen-gia-dinh-duong-chi-nguy-co-tiem-an-trong-dia-sashimi-hap-dan-post984312.vov',
  'domain': 'vov.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 707211,
  'url': 'https://vov.vn/suc-khoe/dau-hieu-bat-thuong-khi-mang-thai-me-bau-can-chu-y-post1009308.vov',
  'domain': 'vov.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 707298,
  'url': 'https://vov.vn/suc-khoe/dau-mo-chien-ran-nhieu-lan-co-gay-ung-thu-post1112520.vov',
  'domain': 'vov.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 707554,
  'url': 'https://vov.vn/suc-khoe/dinh-chinh-cac-tin-don-lien-quan-den-xa-tri-ung-thu-post1038363.vov',
  'domain': 'vov.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 708494,
  'url': 'https://vov.vn/suc-khoe/loai-hat-khi-ngam-nuoc-se-hoa-than-duoc-ha-mo-mau-ngua-ung-thu-gia-lai-cuc-re-post1131558.vov',
  'domain': 'vov.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 709007,
  'url': 'https://vov.vn/suc-khoe/mach-ban-10-thuc-pham-giup-tri-tao-bon-post956470.vov',
  'domain': 'vov.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 709288,
  'url': 'https://vov.vn/suc-khoe/mua-thuoc-tren-mang-mua-lo-vao-nguoi-post1065437.vov',
  'domain': 'vov.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 709624,
  'url': 'https://vov.vn/suc-khoe/nguoi-dan-ong-tu-vong-sau-3-thang-bi-meo-can-post1044125.vov',
  'domain': 'vov.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 710375,
  'url': 'https://vov.vn/suc-khoe/nuoc-cam-tuong-sieu-lanh-manh-uong-kieu-nay-lai-chang-khac-gi-tu-dau-doc-co-the-post1207396.vov',
  'domain': 'vov.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 710786,
  'url': 'https://vov.vn/suc-khoe/sau-bua-an-nang-bung-neu-muon-giam-day-hoi-cho-bo-qua-5-loai-do-uong-tu-che-nay-post1061652.vov',
  'domain': 'vov.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 710798,
  'url': 'https://vov.vn/suc-khoe/sau-tet-bao-quan-thuc-an-thua-kieu-nay-coi-chung-mang-hoa-vao-than-post1152309.vov',
  'domain': 'vov.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 711576,
  'url': 'https://vov.vn/suc-khoe/tra-tam-sen-thuc-uong-quen-thuoc-nhung-khong-phai-ai-cung-dung-dung-post1257348.vov',
  'domain': 'vov.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 713331,
  'url': 'https://vov.vn/xa-hoi/thung-co-hong-vi-hoc-xuong-ca-post1001565.vov',
  'domain': 'vov.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 713342,
  'url': 'https://vov.vn/xa-hoi/tieng-khoc-oe-oe-chao-doi-nam-moi-la-niem-vui-cua-bac-si-san-khoa-post1150166.vov',
  'domain': 'vov.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 713702,
  'url': 'https://vov2.vov.vn/suc-khoe/37-trieu-lieu-vaccine-covid-19-vien-tro-cho-chinh-phu-dong-y-33413.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 714316,
  'url': 'https://vov2.vov.vn/suc-khoe/binh-duong-tim-nguoi-lien-quan-den-bn10436-26749.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 714973,
  'url': 'https://vov2.vov.vn/suc-khoe/cham-soc-rang-mieng-me-bau-khong-the-bo-qua-38715.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 715150,
  'url': 'https://vov2.vov.vn/suc-khoe/chua-bao-gio-di-chong-dich-ma-toi-lai-hen-chinh-xac-ngay-ve-26495.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 715611,
  'url': 'https://vov2.vov.vn/suc-khoe/de-xuat-7-loai-giay-to-de-f0-duoc-huong-bhxh-32924.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 716152,
  'url': 'https://vov2.vov.vn/suc-khoe/ha-noi-ghi-nhan-hon-1300-ca-mac-sot-xuat-huyet-37363.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 716793,
  'url': 'https://vov2.vov.vn/suc-khoe/lam-dep-khong-bien-chung-45503.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 717169,
  'url': 'https://vov2.vov.vn/suc-khoe/ngan-chan-dich-covid-19-ngay-tu-cua-ngo-cua-tp-ho-chi-minh-27856.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 717207,
  'url': 'https://vov2.vov.vn/suc-khoe/ngay-157-nuoc-ta-ghi-nhan-con-so-ky-luc-voi-3416-ca-mac-covid-19-27535.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 717387,
  'url': 'https://vov2.vov.vn/suc-khoe/nguoi-dan-vung-lu-can-lam-gi-de-chu-dong-phong-benh-22405.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 718105,
  'url': 'https://vov2.vov.vn/suc-khoe/sai-lam-thuong-gap-khi-chua-cam-cum-tai-nha-36502.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 718144,
  'url': 'https://vov2.vov.vn/suc-khoe/sang-117-ca-nuoc-ghi-nhan-606-ca-mac-covid-19-tphcm-co-443-ca-27406.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 718472,
  'url': 'https://vov2.vov.vn/suc-khoe/suyt-mat-mang-vi-tu-y-dieu-tri-ung-thu-48349.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 719078,
  'url': 'https://vov2.vov.vn/suc-khoe/tiem-an-nguy-co-bung-phat-lan-song-dich-covid-19-39635.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 719102,
  'url': 'https://vov2.vov.vn/suc-khoe/tiem-thu-nghiem-lam-sang-giai-doan-2-vaccine-covivac-28570.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 719163,
  'url': 'https://vov2.vov.vn/suc-khoe/tinh-co-phat-hien-tieu-duong-benh-nhan-da-bi-hoai-tu-mat-chan-37576.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 719310,
  'url': 'https://vov2.vov.vn/suc-khoe/tp-hcm-thanh-lap-to-chuyen-gia-tu-van-dieu-tri-covid-19-28492.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 719323,
  'url': 'https://vov2.vov.vn/suc-khoe/tp-hcm-xin-mua-them-200000-tui-thuoc-dieu-tri-f0-tai-nha-28943.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 719401,
  'url': 'https://vov2.vov.vn/suc-khoe/tphcm-chuan-bi-de-ung-pho-voi-bien-chung-moi-omicron-31059.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 720329,
  'url': 'https://vov2.vov.vn/suc-khoe/who-tuyen-bo-dau-mua-khi-la-tinh-trang-khan-cap-y-te-toan-cau-36536.vov2',
  'domain': 'vov2.vov.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 909511,
  'url': 'http://wei.39.net/a/140123/4331472.html',
  'domain': 'wei.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 909821,
  'url': 'http://wei.39.net/a/150817/4675779.html',
  'domain': 'wei.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 909870,
  'url': 'http://wei.39.net/a/160119/4759788.html',
  'domain': 'wei.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 910115,
  'url': 'http://wei.39.net/a/191025/7563632.html',
  'domain': 'wei.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 910138,
  'url': 'http://wei.39.net/a/20101012/1502100.html',
  'domain': 'wei.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 910139,
  'url': 'http://wei.39.net/a/20101012/1502693.html',
  'domain': 'wei.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 910211,
  'url': 'http://wei.39.net/a/20101111/1541245.html',
  'domain': 'wei.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 910240,
  'url': 'http://wei.39.net/a/20101119/1544738.html',
  'domain': 'wei.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 910448,
  'url': 'http://wei.39.net/a/2010913/1479477.html',
  'domain': 'wei.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 910616,
  'url': 'http://wei.39.net/a/20111229/1872023.html',
  'domain': 'wei.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 910647,
  'url': 'http://wei.39.net/a/201117/1589524.html',
  'domain': 'wei.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 910686,
  'url': 'http://wei.39.net/a/2011224/1618607.html',
  'domain': 'wei.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 910831,
  'url': 'http://wei.39.net/a/2011321/1633967.html',
  'domain': 'wei.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 911303,
  'url': 'http://wei.39.net/a/2011713/1743498.html',
  'domain': 'wei.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 911669,
  'url': 'http://wei.39.net/a/201198/1798234.html',
  'domain': 'wei.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 912199,
  'url': 'http://wei.39.net/a/231210/m6kg6au.html',
  'domain': 'wei.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 912247,
  'url': 'http://wei.39.net/a/240119/w6lz6yw.html',
  'domain': 'wei.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 912543,
  'url': 'http://wei.39.net/a/251231/q2watju.html',
  'domain': 'wei.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 912634,
  'url': 'http://wei.39.net/a/260106/d8wbkpt.html',
  'domain': 'wei.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 913008,
  'url': 'http://wei.39.net/a/260122/s2wda03.html',
  'domain': 'wei.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 913462,
  'url': 'http://woman.39.net/a/130205/4124332.html',
  'domain': 'woman.39.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 914057,
  'url': 'http://woman.39.net/a/131127/4297480.html',
  'domain': 'woman.39.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 914072,
  'url': 'http://woman.39.net/a/131204/4301009.html',
  'domain': 'woman.39.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 915286,
  'url': 'http://woman.39.net/a/150519/4625269.html',
  'domain': 'woman.39.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 915736,
  'url': 'http://woman.39.net/a/160420/4826105.html',
  'domain': 'woman.39.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 916419,
  'url': 'http://woman.39.net/a/161227/5179790.html',
  'domain': 'woman.39.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 916488,
  'url': 'http://woman.39.net/a/161229/5179776.html',
  'domain': 'woman.39.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 917361,
  'url': 'http://woman.39.net/a/170317/5251190.html',
  'domain': 'woman.39.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 918508,
  'url': 'http://woman.39.net/a/180413/6162934.html',
  'domain': 'woman.39.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 918572,
  'url': 'http://woman.39.net/a/180607/6314116.html',
  'domain': 'woman.39.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 918653,
  'url': 'http://woman.39.net/a/180719/6401796.html',
  'domain': 'woman.39.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 920573,
  'url': 'http://woman.39.net/a/200730/8078539.html',
  'domain': 'woman.39.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 921351,
  'url': 'http://woman.39.net/a/2010527/1306888.html',
  'domain': 'woman.39.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 922602,
  'url': 'http://woman.39.net/a/2011514/1683617.html',
  'domain': 'woman.39.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 922930,
  'url': 'http://woman.39.net/a/201167/1713525.html',
  'domain': 'woman.39.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 923378,
  'url': 'http://woman.39.net/a/201188/1760432.html',
  'domain': 'woman.39.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 923521,
  'url': 'http://woman.39.net/a/201196/1796025.html',
  'domain': 'woman.39.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 923797,
  'url': 'http://woman.39.net/a/201254/2017447.html',
  'domain': 'woman.39.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 923946,
  'url': 'http://woman.39.net/a/210226/8694726.html',
  'domain': 'woman.39.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 924032,
  'url': 'http://woman.39.net/a/210402/8810047.html',
  'domain': 'woman.39.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 926610,
  'url': 'http://www.120ask.com/question/1022468.htm',
  'domain': 'www.120ask.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 933851,
  'url': 'http://www.120ask.com/question/103581929.htm',
  'domain': 'www.120ask.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 965639,
  'url': 'http://www.120ask.com/question/107466642.htm',
  'domain': 'www.120ask.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 965752,
  'url': 'http://www.120ask.com/question/1074712.htm',
  'domain': 'www.120ask.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 979186,
  'url': 'http://www.120ask.com/question/108163283.htm',
  'domain': 'www.120ask.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 1006401,
  'url': 'http://www.120ask.com/question/108512186.htm',
  'domain': 'www.120ask.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 1007841,
  'url': 'http://www.120ask.com/question/108518149.htm',
  'domain': 'www.120ask.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 1020565,
  'url': 'http://www.120ask.com/question/1150775.htm',
  'domain': 'www.120ask.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 1045773,
  'url': 'http://www.120ask.com/question/13710875.htm',
  'domain': 'www.120ask.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 1079128,
  'url': 'http://www.120ask.com/question/15275898.htm',
  'domain': 'www.120ask.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 1082950,
  'url': 'http://www.120ask.com/question/15592794.htm',
  'domain': 'www.120ask.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 1102846,
  'url': 'http://www.120ask.com/question/17174057.htm',
  'domain': 'www.120ask.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 1124248,
  'url': 'http://www.120ask.com/question/19667.htm',
  'domain': 'www.120ask.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 1158878,
  'url': 'http://www.120ask.com/question/2339172.htm',
  'domain': 'www.120ask.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 1160946,
  'url': 'http://www.120ask.com/question/236638.htm',
  'domain': 'www.120ask.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 1248039,
  'url': 'http://www.120ask.com/question/37725880.htm',
  'domain': 'www.120ask.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 1252041,
  'url': 'http://www.120ask.com/question/38375536.htm',
  'domain': 'www.120ask.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 1283469,
  'url': 'http://www.120ask.com/question/470020.htm',
  'domain': 'www.120ask.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 1301684,
  'url': 'http://www.120ask.com/question/553657.htm',
  'domain': 'www.120ask.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 1304488,
  'url': 'http://www.120ask.com/question/56496391.htm',
  'domain': 'www.120ask.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 3020657,
  'url': 'https://www.a-hospital.com/w/%E5%8E%A5%E9%80%86',
  'domain': 'www.a-hospital.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 3029482,
  'url': 'https://www.a-hospital.com/w/%E5%9C%B0%E9%BB%84%E5%85%BB%E8%A1%80%E6%B1%A4',
  'domain': 'www.a-hospital.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 3031654,
  'url': 'https://www.a-hospital.com/w/%E5%A4%96%E5%90%B9',
  'domain': 'www.a-hospital.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 3045141,
  'url': 'https://www.a-hospital.com/w/%E5%B0%8F%E5%AE%9A%E5%BF%97%E4%B8%B8',
  'domain': 'www.a-hospital.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 3066995,
  'url': 'https://www.a-hospital.com/w/%E6%9C%AC%E8%8D%89%E7%BA%B2%E7%9B%AE/%E8%92%B4%E7%BF%9F',
  'domain': 'www.a-hospital.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 3070052,
  'url': 'https://www.a-hospital.com/w/%E6%A0%B8%E8%8B%B7%E7%B1%BB%E8%8D%AF%E7%89%A9',
  'domain': 'www.a-hospital.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 3086214,
  'url': 'https://www.a-hospital.com/w/%E6%B9%BF%E7%96%AE',
  'domain': 'www.a-hospital.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 3086432,
  'url': 'https://www.a-hospital.com/w/%E6%BA%B4%E8%8A%90%E8%83%BA%E6%89%98%E8%A5%BF%E9%85%B8%E7%9B%90',
  'domain': 'www.a-hospital.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 3091707,
  'url': 'https://www.a-hospital.com/w/%E7%90%83%E5%90%8E',
  'domain': 'www.a-hospital.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 3094828,
  'url': 'https://www.a-hospital.com/w/%E7%96%BE%E7%97%85%E6%9D%A1%E7%9B%AE%E7%B4%A2%E5%BC%95-J',
  'domain': 'www.a-hospital.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 3095132,
  'url': 'https://www.a-hospital.com/w/%E7%97%85%E7%90%86%E5%AD%A6/%E5%B1%80%E9%83%A8%E8%A1%80%E6%B6%B2%E5%8F%8A%E4%BD%93%E6%B6%B2%E5%BE%AA%E7%8E%AF%E9%9A%9C%E7%A2%8D',
  'domain': 'www.a-hospital.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 3101091,
  'url': 'https://www.a-hospital.com/w/%E7%9D%BE%E4%B8%B8%E7%96%BC%E7%97%9B',
  'domain': 'www.a-hospital.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 3102496,
  'url': 'https://www.a-hospital.com/w/%E7%A1%AC%E5%8C%96%E6%80%A7%E8%84%82%E8%82%AA%E8%82%89%E8%8A%BD%E8%82%BF',
  'domain': 'www.a-hospital.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 3104975,
  'url': 'https://www.a-hospital.com/w/%E7%A7%BD%E7%96%AE',
  'domain': 'www.a-hospital.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 3111293,
  'url': 'https://www.a-hospital.com/w/%E8%80%81%E5%B9%B4%E7%97%85%E9%98%B2%E6%B2%BB/%E8%80%81%E5%B9%B4%E6%80%A7%E7%97%B4%E5%91%86',
  'domain': 'www.a-hospital.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 3114855,
  'url': 'https://www.a-hospital.com/w/%E8%83%81%E7%97%9B',
  'domain': 'www.a-hospital.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 3128966,
  'url': 'https://www.a-hospital.com/w/%E8%A1%A5%E4%B8%AD%E5%AE%89%E8%83%8E%E9%A5%AE',
  'domain': 'www.a-hospital.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 3134824,
  'url': 'https://www.a-hospital.com/w/%E8%BE%A8%E7%97%B0',
  'domain': 'www.a-hospital.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 3144854,
  'url': 'https://www.a-hospital.com/w/%E9%98%B4%E8%8C%8E%E7%A1%AC%E7%BB%93',
  'domain': 'www.a-hospital.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 3158829,
  'url': 'https://www.a-hospital.com/w/SM',
  'domain': 'www.a-hospital.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 3159015,
  'url': 'https://www.baidu.com/bh/dict/ydxx_10564436041452919187?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 3159028,
  'url': 'https://www.baidu.com/bh/dict/ydxx_10583754649732000737?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 3159103,
  'url': 'https://www.baidu.com/bh/dict/ydxx_10657625553175891664?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 3159305,
  'url': 'https://www.baidu.com/bh/dict/ydxx_10780509721602543724?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 3159728,
  'url': 'https://www.baidu.com/bh/dict/ydxx_10970532071317393733?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 3160155,
  'url': 'https://www.baidu.com/bh/dict/ydxx_11180332015317300649?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 3160365,
  'url': 'https://www.baidu.com/bh/dict/ydxx_7709459851145120481?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 3160679,
  'url': 'https://www.baidu.com/bh/dict/ydxx_8195459985128409569?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 3161205,
  'url': 'https://www.baidu.com/bh/dict/ydxx_8709144688059398981?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 3161492,
  'url': 'https://www.baidu.com/bh/dict/ydzz_11444815160868530268?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 3161510,
  'url': 'https://www.baidu.com/bh/dict/ydzz_13221695658602592209?from=dicta&sf_ref=med_pc&sf_ch=ch_med_pc',
  'domain': 'www.baidu.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 1405494,
  'url': 'http://www.cnkang.com/cm/201502/1148790.html',
  'domain': 'www.cnkang.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 1562011,
  'url': 'http://www.cnkang.com/dise/499/342365.html',
  'domain': 'www.cnkang.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 1562027,
  'url': 'http://www.cnkang.com/dise/499/342398.html',
  'domain': 'www.cnkang.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 1646733,
  'url': 'http://www.cnkang.com/dise/706/534327.html',
  'domain': 'www.cnkang.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 1714545,
  'url': 'http://www.cnkang.com/fuke/renliu/579444.html',
  'domain': 'www.cnkang.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 1727426,
  'url': 'http://www.cnkang.com/fuke/ziranliuchan/578408.html',
  'domain': 'www.cnkang.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 1734430,
  'url': 'http://www.cnkang.com/jibing/bizichuxue/676318.html',
  'domain': 'www.cnkang.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 1762904,
  'url': 'http://www.cnkang.com/jibing/maodichong/253467.html',
  'domain': 'www.cnkang.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 1778028,
  'url': 'http://www.cnkang.com/jibing/seruo/679608.html',
  'domain': 'www.cnkang.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 1783761,
  'url': 'http://www.cnkang.com/jibing/szpzzl/656117.html',
  'domain': 'www.cnkang.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 1817844,
  'url': 'http://www.cnkang.com/myjk/201110/435574.html',
  'domain': 'www.cnkang.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 1846162,
  'url': 'http://www.cnkang.com/myjk/201403/455336.html',
  'domain': 'www.cnkang.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 1848692,
  'url': 'http://www.cnkang.com/myjk/bb/xse/jb/200909/129895.html',
  'domain': 'www.cnkang.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 1878034,
  'url': 'http://www.cnkang.com/nkwk/gxyzz/870849.html',
  'domain': 'www.cnkang.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 1891296,
  'url': 'http://www.cnkang.com/nkwk/naoyan/365322.html',
  'domain': 'www.cnkang.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 1895831,
  'url': 'http://www.cnkang.com/nkwk/shangfeng/595133.html',
  'domain': 'www.cnkang.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 1961225,
  'url': 'http://www.cnkang.com/nxjk/201203/127091.html',
  'domain': 'www.cnkang.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 1966610,
  'url': 'http://www.cnkang.com/nxjk/201206/823462.html',
  'domain': 'www.cnkang.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 3426701,
  'url': 'https://www.cnkang.com/fuke/gongwaiyun/580036.html',
  'domain': 'www.cnkang.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 3477493,
  'url': 'https://www.cnkang.com/myjk/201211/429048.html',
  'domain': 'www.cnkang.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 3530727,
  'url': 'https://www.familydoctor.cn/a/201308/493539.html',
  'domain': 'www.familydoctor.cn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 3530755,
  'url': 'https://www.familydoctor.cn/a/201508/803897.html',
  'domain': 'www.familydoctor.cn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 3539320,
  'url': 'https://www.familydoctor.com.cn/baby/a/201512/882317.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 3541920,
  'url': 'https://www.familydoctor.com.cn/baby/a/201612/1582726.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 3545134,
  'url': 'https://www.familydoctor.com.cn/baby/a/201612/1596702.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 3550215,
  'url': 'https://www.familydoctor.com.cn/baby/a/201612/1625266.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 3565485,
  'url': 'https://www.familydoctor.com.cn/baby/a/201705/1994508.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 3592579,
  'url': 'https://www.familydoctor.com.cn/cancer/a/201311/535181.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 3595949,
  'url': 'https://www.familydoctor.com.cn/cancer/a/201410/718148.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 3610055,
  'url': 'https://www.familydoctor.com.cn/cancer/a/201704/1862218.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 3702694,
  'url': 'https://www.familydoctor.com.cn/fuke/a/201707/2154569.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 3712786,
  'url': 'https://www.familydoctor.com.cn/fuke/a/201904/2546190.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 3720046,
  'url': 'https://www.familydoctor.com.cn/ganbing/a/201311/531392.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 3739161,
  'url': 'https://www.familydoctor.com.cn/ganbing/a/201701/1675225.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 3741093,
  'url': 'https://www.familydoctor.com.cn/ganbing/a/201701/1679241.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 3748204,
  'url': 'https://www.familydoctor.com.cn/ganbing/a/201706/2001815.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 3767904,
  'url': 'https://www.familydoctor.com.cn/guke/a/201702/1721220.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 3793348,
  'url': 'https://www.familydoctor.com.cn/jianfei/diet/201202/934113215739.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 3804984,
  'url': 'https://www.familydoctor.com.cn/lady/a/201711/2425160.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 3811973,
  'url': 'https://www.familydoctor.com.cn/lady/health/201112/948626129134.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 3906017,
  'url': 'https://www.familydoctor.com.cn/xnxg/a/201611/1478105.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 3976790,
  'url': 'https://www.familydoctor.com.cn/yinshi/yaoshan/201108/33251581961.html',
  'domain': 'www.familydoctor.com.cn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 3979129,
  'url': 'https://www.msdmanuals.cn/home/digestive-disorders/symptoms-of-digestive-disorders/chronic-abdominal-pain-and-recurring-abdominal-pain',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 3979155,
  'url': 'https://www.msdmanuals.cn/home/disorders-of-nutrition',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 3979357,
  'url': 'https://www.msdmanuals.cn/home/eye-disorders/corneal-disorders/keratoconus',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 3980044,
  'url': 'https://www.msdmanuals.cn/home/injuries-and-poisoning/dislocations',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 3980077,
  'url': 'https://www.msdmanuals.cn/home/injuries-and-poisoning/facial-injuries',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 3980251,
  'url': 'https://www.msdmanuals.cn/home/kidney-and-urinary-tract-disorders/kidney-failure/chronic-kidney-disease',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 3980463,
  'url': 'https://www.msdmanuals.cn/home/men-s-health-issues',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 3980626,
  'url': 'https://www.msdmanuals.cn/home/mouth-and-dental-disorders/urgent-dental-problems/fractured-loosened-or-knocked-out-teeth',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 3980709,
  'url': 'https://www.msdmanuals.cn/home/skin-disorders/fungal-skin-infections/jock-itch-tinea-cruris',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 3981659,
  'url': 'https://www.msdmanuals.cn/professional/ear-nose-and-throat-disorders/laryngeal-disorders/laryngitis',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 3981963,
  'url': 'https://www.msdmanuals.cn/professional/gastrointestinal-disorders/esophageal-and-swallowing-disorders/dysphagia',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 3981991,
  'url': 'https://www.msdmanuals.cn/professional/gastrointestinal-disorders/gastroenteritis/norovirus-gastroenteritis',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 3982182,
  'url': 'https://www.msdmanuals.cn/professional/geriatrics/pharmacologic-therapy-in-older-adults/drug-categories-of-concern-in-older-adults',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 3982673,
  'url': 'https://www.msdmanuals.cn/professional/immunology-allergic-disorders/transplantation/small-bowel-transplantation',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 3982860,
  'url': 'https://www.msdmanuals.cn/professional/infectious-diseases/immunization/human-papillomavirus-hpv-vaccine',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 3983081,
  'url': 'https://www.msdmanuals.cn/professional/injuries-poisoning/how-to-do-anesthesia-procedures/how-to-do-an-ultrasound-guided-peripheral-nerve-block',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 3983224,
  'url': 'https://www.msdmanuals.cn/professional/musculoskeletal-and-connective-tissue-disorders/approach-to-the-patient-with-joint-symptoms/evaluation-of-the-foot',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 3983639,
  'url': 'https://www.msdmanuals.cn/professional/pediatrics/caring-for-sick-children-and-their-families/children-and-youth-with-special-health-care-needs',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 3983763,
  'url': 'https://www.msdmanuals.cn/professional/pediatrics/dehydration-and-fluid-therapy-in-children/oral-rehydration-therapy',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 3983817,
  'url': 'https://www.msdmanuals.cn/professional/pediatrics/hereditary-periodic-fever-syndromes/tnf-receptor-associated-periodic-syndrome-traps',
  'domain': 'www.msdmanuals.cn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 720422,
  'url': 'https://www.pharmacity.vn/bang-can-nang-thai-nhi.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 720426,
  'url': 'https://www.pharmacity.vn/cac-thuoc-va-che-pham-bo-sung-co-kha-nang-lam-tang-huyet-ap.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 720430,
  'url': 'https://www.pharmacity.vn/cat-bao-quy-dau-va-nhung-dieu-can-biet.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 720437,
  'url': 'https://www.pharmacity.vn/dau-hieu-thieu-canxi.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 720448,
  'url': 'https://www.pharmacity.vn/kham-pha-gia-tri-dinh-duong-va-tac-dong-cua-ca-ngan-doi-voi-suc-khoe.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 720452,
  'url': 'https://www.pharmacity.vn/lam-kem-trai-cay-cho-be-tai-nha-loi-ich-va-cach-lam.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 720454,
  'url': 'https://www.pharmacity.vn/nhiem-trung-vet-mo-sau-sinh-nguy-co-va-dieu-tri.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 720458,
  'url': 'https://www.pharmacity.vn/sinh-thuong-khong-co-bao-hiem-het-bao-nhieu-tien.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 720459,
  'url': 'https://www.pharmacity.vn/sua-bi-do-bua-phu-bo-duong-cho-be-yeu.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 720462,
  'url': 'https://www.pharmacity.vn/tac-ong-phong-tinh-la-gi-nguyen-nhan-va-trieu-chung.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 720463,
  'url': 'https://www.pharmacity.vn/tang-sinh-mon-sau-khi-sinh-lieu-co-nen-khau-ngay-lap-tuc.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 720467,
  'url': 'https://www.pharmacity.vn/thieu-hut-collagen.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 720469,
  'url': 'https://www.pharmacity.vn/thoat-vi-ron-thai-nhi-tinh-trang-co-quan-loi-ra-khoi-bung-be.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 720472,
  'url': 'https://www.pharmacity.vn/thuoc-gian-mach-vasodilators.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 720473,
  'url': 'https://www.pharmacity.vn/tieu-duong-thai-ky-co-nguy-hiem-khong.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 720474,
  'url': 'https://www.pharmacity.vn/tim-hieu-nguyen-nhan-dan-den-vo-sinh.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 720475,
  'url': 'https://www.pharmacity.vn/top-5-cach-tri-gau-va-rung-toc-tai-nha-hieu-qua.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 720477,
  'url': 'https://www.pharmacity.vn/trung-ga-va-trung-vit-loai-nao-tot-hon-cho-suc-khoe-ban.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 720481,
  'url': 'https://www.pharmacity.vn/ung-thu-co-tu-cung-nguyen-nhan-trieu-chung-va-cac-giai-doan.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 720485,
  'url': 'https://www.pharmacity.vn/vien-bo-sung-canxi-chung-co-anh-huong-den-thuoc-dieu-tri-tang-huyet-ap-khong.htm',
  'domain': 'www.pharmacity.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 720531,
  'url': 'https://www.qdnd.vn/y-te/benh-vien/ban-giao-nguyen-trang-benh-vien-quan-y-4-tu-quan-khu-7-ve-quan-doan-34-849930',
  'domain': 'www.qdnd.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 720639,
  'url': 'https://www.qdnd.vn/y-te/benh-vien/benh-vien-quan-y-105-khai-truong-khu-kham-benh-chuyen-doi-so-theo-yeu-cau-758857',
  'domain': 'www.qdnd.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 720710,
  'url': 'https://www.qdnd.vn/y-te/benh-vien/benh-vien-quan-y-268-du-dieu-kien-trien-khai-benh-an-dien-tu-843413',
  'domain': 'www.qdnd.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 721149,
  'url': 'https://www.qdnd.vn/y-te/cac-van-de/chu-dong-theo-doi-dien-bien-benh-do-virus-marburg-719377',
  'domain': 'www.qdnd.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 721212,
  'url': 'https://www.qdnd.vn/y-te/cac-van-de/diet-muoi-phuong-phap-hieu-qua-de-phong-benh-chikungunya-842587',
  'domain': 'www.qdnd.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 721325,
  'url': 'https://www.qdnd.vn/y-te/cac-van-de/khong-chu-quan-voi-benh-dau-mua-khi-745750',
  'domain': 'www.qdnd.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 721844,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/3-ngay-lien-tiep-f0-ca-nuoc-lien-tiep-tang-manh-701831',
  'domain': 'www.qdnd.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 721867,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/6-khuyen-cao-cua-bo-y-te-ve-phong-chong-benh-lien-cau-lon-o-nguoi-837393',
  'domain': 'www.qdnd.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 721873,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/8-nguyen-nhan-dan-den-gan-9-400-vien-chuc-y-te-xin-thoi-viec-bo-viec-700496',
  'domain': 'www.qdnd.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 721909,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/bao-dam-an-toan-thuc-pham-phong-chong-ngo-doc-do-bao-lut-849747',
  'domain': 'www.qdnd.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 721947,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/benh-nhan-ghep-tim-dau-tien-tai-benh-vien-trung-uong-quan-doi-108-duoc-ra-vien-767872',
  'domain': 'www.qdnd.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 722232,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/bo-y-te-de-nghi-cac-co-so-nhap-khau-khan-truong-tim-nguon-thuoc-dieu-tri-sot-xuat-huyet-704985',
  'domain': 'www.qdnd.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 722381,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/ca-nuoc-da-tiem-duoc-244-757-059-lieu-vac-xin-phong-covid-19-701357',
  'domain': 'www.qdnd.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 722989,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/hang-chuc-ngan-cong-nhan-duoc-tu-van-cham-soc-suc-khoe-mien-phi-701533',
  'domain': 'www.qdnd.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 723156,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/kham-suc-khoe-tang-qua-ho-tro-nguoi-dan-va-nhan-vien-y-te-bi-anh-huong-boi-bao-lu-876799',
  'domain': 'www.qdnd.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 723159,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/kham-tu-van-suc-khoe-mien-phi-tri-an-nguoi-co-cong-700021',
  'domain': 'www.qdnd.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 723431,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/nhieu-benh-nhan-nguy-kich-sau-vai-ngay-sot-xuat-huyet-846246',
  'domain': 'www.qdnd.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 723556,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/phau-thuat-thay-khop-hang-mien-phi-cho-cac-benh-nhan-ngheo-768151',
  'domain': 'www.qdnd.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 723765,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/thanh-lap-tram-y-te-da-chien-o-vung-lu-nghe-an-839101',
  'domain': 'www.qdnd.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 723778,
  'url': 'https://www.qdnd.vn/y-te/tin-tuc/the-gioi-sap-co-vaccine-dieu-tri-ung-thu-748531',
  'domain': 'www.qdnd.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 724291,
  'url': 'https://www.sggp.org.vn/ben-tre-267-ca-mac-thuy-dau-tai-mot-cong-ty-may-mac-post776570.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 724389,
  'url': 'https://www.sggp.org.vn/benh-vien-ung-buou-tphcm-tiep-nhan-2-may-sinh-thiet-vu-chan-khong-post811569.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 724434,
  'url': 'https://www.sggp.org.vn/bo-y-te-canh-bao-dich-benh-co-nguy-co-bung-phat-trong-mua-thu-dong-post818380.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 724445,
  'url': 'https://www.sggp.org.vn/bo-y-te-dong-y-tiem-vaccine-soi-cho-tre-tu-6-den-duoi-9-thang-tuoi-post767166.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 724450,
  'url': 'https://www.sggp.org.vn/bo-y-te-khuyen-cao-khan-bien-phap-giam-sat-phong-chong-benh-chikungunya-post807721.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 724472,
  'url': 'https://www.sggp.org.vn/bo-y-te-yeu-cau-khan-cap-cung-co-cac-doi-phan-ung-nhanh-ung-pho-virus-nipah-post835677.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 724586,
  'url': 'https://www.sggp.org.vn/chinh-phu-yeu-cau-tang-cuong-trien-khai-cac-giai-phap-kiem-soat-xu-ly-thuoc-gia-post798788.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 724756,
  'url': 'https://www.sggp.org.vn/day-manh-mo-hinh-vien-truong-dau-tu-dao-tao-nhan-luc-y-te-chat-luong-cao-post782865.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 724833,
  'url': 'https://www.sggp.org.vn/dong-nai-day-lui-benh-dai-post814881.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 724950,
  'url': 'https://www.sggp.org.vn/hanh-trinh-cuu-7-nguoi-benh-tu-mo-tang-cua-nguoi-dan-ong-chet-nao-post787004.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 725165,
  'url': 'https://www.sggp.org.vn/lien-tiep-xay-ra-ngo-doc-thuc-pham-bo-y-te-yeu-cau-lam-ro-post838896.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 725195,
  'url': 'https://www.sggp.org.vn/mo-hinh-cham-soc-suc-khoe-nguoi-cao-tuoi-toan-dien-post842767.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 725501,
  'url': 'https://www.sggp.org.vn/san-phu-nguy-kich-sau-khi-an-oc-mo-lay-thai-khan-cap-cuu-me-con-post842146.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 725539,
  'url': 'https://www.sggp.org.vn/su-song-noi-dai-tu-nghia-cu-hien-tang-cuu-nguoi-post770191.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 725564,
  'url': 'https://www.sggp.org.vn/tang-cuong-giam-sat-bep-an-tap-the-post805499.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 725587,
  'url': 'https://www.sggp.org.vn/thanh-lap-khoa-bao-ve-suc-khoe-can-bo-thanh-pho-thuoc-benh-vien-nguyen-trai-post774754.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 725688,
  'url': 'https://www.sggp.org.vn/to-chuc-ngoai-cong-lap-du-kien-duoc-cap-phep-giam-dinh-adn-post807376.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 725743,
  'url': 'https://www.sggp.org.vn/tphcm-hon-327000-nguoi-cao-tuoi-duoc-tham-kham-suc-khoe-post772433.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 725805,
  'url': 'https://www.sggp.org.vn/trach-nhiem-cua-bo-y-te-trong-vu-gan-600-loai-sua-gia-ra-thi-truong-post790729.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 725853,
  'url': 'https://www.sggp.org.vn/tuong-niem-100-nam-ngay-mat-cu-phan-chau-trinh-post844494.html',
  'domain': 'www.sggp.org.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 729716,
  'url': 'https://www.vietnamplus.vn/asean-thuc-day-sang-kien-nham-ngan-chan-dai-dich-trong-tuong-lai-post852451.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 729769,
  'url': 'https://www.vietnamplus.vn/ba-ria-vung-tau-25000-nguoi-tiem-mui-1-vaccine-chua-duoc-tiem-mui-2-post746781.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 729875,
  'url': 'https://www.vietnamplus.vn/benh-nhan-covid-19-chua-tiem-phong-co-nguy-co-tu-vong-cao-gap-5-lan-post846950.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 729937,
  'url': 'https://www.vietnamplus.vn/bien-doi-khi-hau-lam-gia-tang-dich-ta-gan-100-nguoi-o-syria-tu-vong-post836359.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 729962,
  'url': 'https://www.vietnamplus.vn/binh-duong-thanh-lap-162-tram-y-te-luu-dong-ho-tro-f0-tai-nha-post755437.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 730046,
  'url': 'https://www.vietnamplus.vn/buoc-dot-pha-trong-nghien-cuu-phuong-phap-dieu-tri-benh-sot-ret-post916797.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 730490,
  'url': 'https://www.vietnamplus.vn/dam-bao-nguon-nuoc-sach-ve-sinh-moi-truong-trong-mua-bao-lu-post536835.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 730578,
  'url': 'https://www.vietnamplus.vn/dien-vien-manh-truong-mong-chuong-trinh-sua-hoc-duong-duoc-nhan-rong-post580392.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 730837,
  'url': 'https://www.vietnamplus.vn/han-quoc-lai-ghi-nhan-truong-hop-meo-nghi-nhiem-cum-gia-cam-post886112.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 730873,
  'url': 'https://www.vietnamplus.vn/hang-chuc-nguoi-nghi-ngo-bi-phoi-nhiem-hiv-sau-vu-tai-nan-giao-thong-post454119.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 730944,
  'url': 'https://www.vietnamplus.vn/hon-300-chu-ky-keu-goi-gilead-mo-rong-san-xuat-thuoc-lenacapavir-dieu-tri-hiv-post956424.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 731145,
  'url': 'https://www.vietnamplus.vn/lam-sao-de-thuc-day-moi-sang-ma-khong-can-ca-loat-chuong-bao-thuc-post970232.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 731190,
  'url': 'https://www.vietnamplus.vn/lieu-phap-moi-dieu-tri-hieu-qua-benh-viem-loet-dai-trang-vua-va-nang-post850914.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 731497,
  'url': 'https://www.vietnamplus.vn/nga-va-romania-ghi-nhan-nhung-ca-dau-tien-nhiem-bien-the-kraken-post840946.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 731656,
  'url': 'https://www.vietnamplus.vn/nha-vo-dich-an-khoe-the-gioi-giai-nghe-vi-khong-con-cam-thay-doi-post954897.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 731908,
  'url': 'https://www.vietnamplus.vn/olympic-2024-han-quoc-khuyen-cao-nguoi-dan-ve-nguy-co-dich-benh-post965219.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 732147,
  'url': 'https://www.vietnamplus.vn/senegal-nghi-ngo-mot-so-nguoi-hanh-huong-hajj-tu-vong-do-covid-19-post960907.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 732340,
  'url': 'https://www.vietnamplus.vn/thanh-cong-ca-ghep-te-bao-goc-tu-mau-day-ron-cong-dong-dau-tien-post315612.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 732397,
  'url': 'https://www.vietnamplus.vn/them-nhieu-nuoc-thu-hoi-san-pham-sua-cong-thuc-cua-nestle-post1087187.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 733042,
  'url': 'https://www.vietnamplus.vn/who-so-ca-tu-vong-vi-covid-19-giam-95-ke-tu-dau-nam-2023-post859507.vnp',
  'domain': 'www.vietnamplus.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 733750,
  'url': 'https://www.vinmec.com/vie/bai-viet/bac-si-chan-doan-benh-parkinson-nhu-the-nao',
  'domain': 'www.vinmec.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 735773,
  'url': 'https://www.vinmec.com/vie/bai-viet/cac-cach-chua-tran-dich-khop-goi-tai-nha-hieu-qua',
  'domain': 'www.vinmec.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 736000,
  'url': 'https://www.vinmec.com/vie/bai-viet/cac-dinh-huong-dieu-tri-ung-thu-niem-mac-tu-cung-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 736381,
  'url': 'https://www.vinmec.com/vie/bai-viet/cac-phuong-phap-vo-cam-trong-phau-thuat-long-nguc-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 736929,
  'url': 'https://www.vinmec.com/vie/bai-viet/cach-chua-chai-mat-ca-chan-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 737159,
  'url': 'https://www.vinmec.com/vie/bai-viet/cach-giam-ngua-vung-kin-nhanh-nhat-va-an-toan-tai-nha',
  'domain': 'www.vinmec.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 737317,
  'url': 'https://www.vinmec.com/vie/bai-viet/cach-nao-chong-tao-bon-cho-tre-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 739775,
  'url': 'https://www.vinmec.com/vie/bai-viet/co-nen-su-dung-sua-cong-thuc-mo-qua-ngay-quy-dinh-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 739999,
  'url': 'https://www.vinmec.com/vie/bai-viet/cong-dung-chua-benh-cua-la-va-qua-na-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 744189,
  'url': 'https://www.vinmec.com/vie/bai-viet/hoi-chung-nao-gan-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 745528,
  'url': 'https://www.vinmec.com/vie/bai-viet/lam-gi-khi-be-1-thang-tuoi-hay-khoc-dem-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 745606,
  'url': 'https://www.vinmec.com/vie/bai-viet/lam-gi-voi-con-kho-khe-cua-tre-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 750462,
  'url': 'https://www.vinmec.com/vie/bai-viet/quy-trinh-cham-soc-da-toan-khi-mang-thai-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 751734,
  'url': 'https://www.vinmec.com/vie/bai-viet/su-cham-phat-trien-o-tre-em-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 751823,
  'url': 'https://www.vinmec.com/vie/bai-viet/su-khac-biet-giua-phoi-cua-nguoi-hut-thuoc-la-va-phoi-cua-nguoi-binh-thuong-tren-x-quang-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 755539,
  'url': 'https://www.vinmec.com/vie/bai-viet/tre-bi-di-ung-sua-bo-nen-kieng-thuc-pham-gi-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 756107,
  'url': 'https://www.vinmec.com/vie/bai-viet/tre-sinh-non-7-thang-co-kho-nuoi-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 756116,
  'url': 'https://www.vinmec.com/vie/bai-viet/tre-sinh-non-khi-nao-can-kham-mat-phong-benh-vong-mac-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 757070,
  'url': 'https://www.vinmec.com/vie/bai-viet/ung-thu-da-day-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 757707,
  'url': 'https://www.vinmec.com/vie/bai-viet/vang-da-do-tang-bilirubin-gian-tiep-o-tre-so-sinh-vi',
  'domain': 'www.vinmec.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 2000345,
  'url': 'http://www.wujue.com/bkwz/jiaoliu/201107/44557.html',
  'domain': 'www.wujue.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2000457,
  'url': 'http://www.wujue.com/bkwz/jiaoliu/201402/54904.html',
  'domain': 'www.wujue.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2004603,
  'url': 'http://www.wujue.com/gjwx/gjys/bencao/201012/37961.html',
  'domain': 'www.wujue.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2006744,
  'url': 'http://www.wujue.com/gjwx/lilun/yankexue/201103/42384.html',
  'domain': 'www.wujue.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2006819,
  'url': 'http://www.wujue.com/gjwx/lilun/zhongyierke/201009/37588.html',
  'domain': 'www.wujue.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2007799,
  'url': 'http://www.wujue.com/gjwx/mingci/201104/42817.html',
  'domain': 'www.wujue.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2012764,
  'url': 'http://www.wujue.com/pifu/xunmazhen/42499.html',
  'domain': 'www.wujue.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2012772,
  'url': 'http://www.wujue.com/pifu/xunmazhen/42547.html',
  'domain': 'www.wujue.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2012888,
  'url': 'http://www.wujue.com/pifu/ypb/37482.html',
  'domain': 'www.wujue.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2015561,
  'url': 'http://www.wujue.com/yxbk/baby/200704/23486.html',
  'domain': 'www.wujue.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2016277,
  'url': 'http://www.wujue.com/yxbk/baby/201112/47626.html',
  'domain': 'www.wujue.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2021403,
  'url': 'http://www.wujue.com/yxbk/yunyu/201209/51160.html',
  'domain': 'www.wujue.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2021472,
  'url': 'http://www.wujue.com/zhongyi/dbxz/1543.html',
  'domain': 'www.wujue.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2023410,
  'url': 'http://www.wujue.com/zonghe/zatan/201012/38247.html',
  'domain': 'www.wujue.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2024055,
  'url': 'http://www.wujue.com/zyts/dbxz/44309.html',
  'domain': 'www.wujue.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2025594,
  'url': 'http://www.wujue.com/zyzy/jcll/cysx/erx/201210/51619.html',
  'domain': 'www.wujue.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2030103,
  'url': 'http://www.wujue.com/zyzy/ysbj/nvxing/201107/44233.html',
  'domain': 'www.wujue.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2036124,
  'url': 'http://www.wujue.com/zyzy/zycd/h/200703/8044.html',
  'domain': 'www.wujue.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2038559,
  'url': 'http://www.wujue.com/zyzy/zycd/k/200703/9996.html',
  'domain': 'www.wujue.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2051678,
  'url': 'http://www.wujue.com/zyzy/zylc/bzlz/201105/43672.html',
  'domain': 'www.wujue.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 3985161,
  'url': 'https://www.youlai.cn/yyk/article/318049.html',
  'domain': 'www.youlai.cn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 3990123,
  'url': 'https://www.youlai.cn/yyk/article/341079.html',
  'domain': 'www.youlai.cn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 3990727,
  'url': 'https://www.youlai.cn/yyk/article/344012.html',
  'domain': 'www.youlai.cn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 3991531,
  'url': 'https://www.youlai.cn/yyk/article/349121.html',
  'domain': 'www.youlai.cn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 3991674,
  'url': 'https://www.youlai.cn/yyk/article/349691.html',
  'domain': 'www.youlai.cn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 3993830,
  'url': 'https://www.youlai.cn/yyk/article/359314.html',
  'domain': 'www.youlai.cn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 3996862,
  'url': 'https://www.youlai.cn/yyk/article/371465.html',
  'domain': 'www.youlai.cn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 3998987,
  'url': 'https://www.youlai.cn/yyk/article/408248.html',
  'domain': 'www.youlai.cn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 4005461,
  'url': 'https://www.youlai.cn/yyk/article/549139.html',
  'domain': 'www.youlai.cn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 4009846,
  'url': 'https://www.youlai.cn/yyk/article/582616.html',
  'domain': 'www.youlai.cn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 4011319,
  'url': 'https://www.youlai.cn/yyk/article/591900.html',
  'domain': 'www.youlai.cn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 4013405,
  'url': 'https://www.youlai.cn/yyk/article/608121.html',
  'domain': 'www.youlai.cn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 4014813,
  'url': 'https://www.youlai.cn/yyk/article/618050.html',
  'domain': 'www.youlai.cn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 4015329,
  'url': 'https://www.youlai.cn/yyk/article/621057.html',
  'domain': 'www.youlai.cn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 4018000,
  'url': 'https://www.youlai.cn/yyk/article/636373.html',
  'domain': 'www.youlai.cn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 4019569,
  'url': 'https://www.youlai.cn/yyk/article/647756.html',
  'domain': 'www.youlai.cn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 4019858,
  'url': 'https://www.youlai.cn/yyk/article/649367.html',
  'domain': 'www.youlai.cn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 4019891,
  'url': 'https://www.youlai.cn/yyk/article/649711.html',
  'domain': 'www.youlai.cn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 4021303,
  'url': 'https://www.youlai.cn/yyk/article/660805.html',
  'domain': 'www.youlai.cn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 4025046,
  'url': 'https://www.youlai.cn/yyk/article/680990.html',
  'domain': 'www.youlai.cn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 2052429,
  'url': 'http://www.zydcd.com/YiAn/25223sfke.html',
  'domain': 'www.zydcd.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 2052686,
  'url': 'http://www.zydcd.com/wenzhang/72484ryhz.html',
  'domain': 'www.zydcd.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 2052758,
  'url': 'http://www.zydcd.com/wenzhang/72739hgia.html',
  'domain': 'www.zydcd.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 2052836,
  'url': 'http://www.zydcd.com/wenzhang/72833cjci.html',
  'domain': 'www.zydcd.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 2052843,
  'url': 'http://www.zydcd.com/wenzhang/72840tmnl.html',
  'domain': 'www.zydcd.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 2052971,
  'url': 'http://www.zydcd.com/wenzhang/73330krml.html',
  'domain': 'www.zydcd.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 2053034,
  'url': 'http://www.zydcd.com/wenzhang/73635syho.html',
  'domain': 'www.zydcd.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 2053202,
  'url': 'http://www.zydcd.com/wenzhang/74137qvxc.html',
  'domain': 'www.zydcd.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 2053240,
  'url': 'http://www.zydcd.com/wenzhang/74200gbjm.html',
  'domain': 'www.zydcd.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 2053781,
  'url': 'http://www.zydcd.com/wenzhang/75280tmip.html',
  'domain': 'www.zydcd.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 2053794,
  'url': 'http://www.zydcd.com/wenzhang/75528wkxz.html',
  'domain': 'www.zydcd.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 2054051,
  'url': 'http://www.zydcd.com/wenzhang/76318tidd.html',
  'domain': 'www.zydcd.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 2054098,
  'url': 'http://www.zydcd.com/wenzhang/76365mcpu.html',
  'domain': 'www.zydcd.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 2054102,
  'url': 'http://www.zydcd.com/wenzhang/76369ripw.html',
  'domain': 'www.zydcd.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 2054219,
  'url': 'http://www.zydcd.com/wenzhang/76518txbp.html',
  'domain': 'www.zydcd.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 2054407,
  'url': 'http://www.zydcd.com/wenzhang/76796ybjx.html',
  'domain': 'www.zydcd.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 2054507,
  'url': 'http://www.zydcd.com/wenzhang/77001gudq.html',
  'domain': 'www.zydcd.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 2054850,
  'url': 'http://www.zydcd.com/wenzhang/77881hnum.html',
  'domain': 'www.zydcd.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 2095574,
  'url': 'http://www.zydcd.com/zhongyi/zy52942qy/',
  'domain': 'www.zydcd.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 2103294,
  'url': 'http://www.zydcd.com/zhongyi/zy60581xz/',
  'domain': 'www.zydcd.com',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 759341,
  'url': 'https://youmed.vn/tin-tuc/a-t-desloratadin/',
  'domain': 'youmed.vn',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 760447,
  'url': 'https://youmed.vn/tin-tuc/cach-tinh-chi-so-duong-huyet/',
  'domain': 'youmed.vn',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 760867,
  'url': 'https://youmed.vn/tin-tuc/chu-sa-loai-khoang-vat-lam-thuoc-an-than/',
  'domain': 'youmed.vn',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 761598,
  'url': 'https://youmed.vn/tin-tuc/do-mo-hoi-tay-cach-khac-phuc-hieu-qua/',
  'domain': 'youmed.vn',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 761636,
  'url': 'https://youmed.vn/tin-tuc/du-du/',
  'domain': 'youmed.vn',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 761658,
  'url': 'https://youmed.vn/tin-tuc/dung-dich-uong-tonicalcium/',
  'domain': 'youmed.vn',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 762061,
  'url': 'https://youmed.vn/tin-tuc/hoi-chung-loet-truc-trang-don-doc-dau-hieu-nguyen-nhan-chan-doan-va-dieu-tri/',
  'domain': 'youmed.vn',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 762222,
  'url': 'https://youmed.vn/tin-tuc/keto-chay/',
  'domain': 'youmed.vn',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 763265,
  'url': 'https://youmed.vn/tin-tuc/nhung-thuc-pham-trang-da-tot-nhat-co-the-ban-chua-biet/',
  'domain': 'youmed.vn',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 763913,
  'url': 'https://youmed.vn/tin-tuc/sung-nuou-rang-khon/',
  'domain': 'youmed.vn',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 764359,
  'url': 'https://youmed.vn/tin-tuc/thuoc-arcoxia-etoricoxib/',
  'domain': 'youmed.vn',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 764450,
  'url': 'https://youmed.vn/tin-tuc/thuoc-boi-tacrolimus-ointment-cong-dung-cach-dung-va-nhung-luu-y/',
  'domain': 'youmed.vn',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 764762,
  'url': 'https://youmed.vn/tin-tuc/thuoc-nho-mat-emas-glycerin-giam-kho-va-kich-ung-mat/',
  'domain': 'youmed.vn',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 764945,
  'url': 'https://youmed.vn/tin-tuc/thuoc-tributel-trimebutin-trong-dieu-tri-day-hoi-va-cac-benh-tren-duong-tieu-hoa/',
  'domain': 'youmed.vn',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 765062,
  'url': 'https://youmed.vn/tin-tuc/tieu-duong-va-nhung-bien-chung-nguy-hiem/',
  'domain': 'youmed.vn',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 765504,
  'url': 'https://youmed.vn/tin-tuc/ung-thu-gan-giai-doan-dau/',
  'domain': 'youmed.vn',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 765664,
  'url': 'https://youmed.vn/tin-tuc/van-tue-khong-chi-la-cay-canh/',
  'domain': 'youmed.vn',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 766234,
  'url': 'https://youmed.vn/tin-tuc/xet-nghiem-chuc-nang-than/',
  'domain': 'youmed.vn',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 766347,
  'url': 'https://youmed.vn/tin-tuc/xet-nghiem-ure-mau/',
  'domain': 'youmed.vn',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 766380,
  'url': 'https://youmed.vn/tin-tuc/xuat-huyet-nao/',
  'domain': 'youmed.vn',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 4032467,
  'url': 'https://zhongyibaodian.net/a/102591.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 4037203,
  'url': 'https://zhongyibaodian.net/a/108735.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 4042532,
  'url': 'https://zhongyibaodian.net/a/114852.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 4044765,
  'url': 'https://zhongyibaodian.net/a/11732.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 4062420,
  'url': 'https://zhongyibaodian.net/a/13883.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 4070611,
  'url': 'https://zhongyibaodian.net/a/148893.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 4072388,
  'url': 'https://zhongyibaodian.net/a/150820.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 4073700,
  'url': 'https://zhongyibaodian.net/a/15206.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 4082112,
  'url': 'https://zhongyibaodian.net/a/161338.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 4084450,
  'url': 'https://zhongyibaodian.net/a/164041.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 4096716,
  'url': 'https://zhongyibaodian.net/a/18349.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 4104126,
  'url': 'https://zhongyibaodian.net/a/26820.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 4106430,
  'url': 'https://zhongyibaodian.net/a/30017.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 4117903,
  'url': 'https://zhongyibaodian.net/a/45004.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 4126526,
  'url': 'https://zhongyibaodian.net/a/57197.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 4133867,
  'url': 'https://zhongyibaodian.net/a/66554.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 4161584,
  'url': 'https://zhongyibaodian.net/a/a-at/32144.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 4163785,
  'url': 'https://zhongyibaodian.net/a/a-bn/50625.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 4170047,
  'url': 'https://zhongyibaodian.net/a/a-gr/134141.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 4170157,
  'url': 'https://zhongyibaodian.net/a/a-gr/144841.html',
  'domain': 'zhongyibaodian.net',
  'sample_index': 19,
  'sample_role': 'additional'},
 {'id': 4194178,
  'url': 'https://zysjonline.com/articles/misc/30946/',
  'domain': 'zysjonline.com',
  'sample_index': 0,
  'sample_role': 'additional'},
 {'id': 4198112,
  'url': 'https://zysjonline.com/articles/misc/34915/',
  'domain': 'zysjonline.com',
  'sample_index': 1,
  'sample_role': 'additional'},
 {'id': 4210027,
  'url': 'https://zysjonline.com/articles/misc/47082/',
  'domain': 'zysjonline.com',
  'sample_index': 2,
  'sample_role': 'additional'},
 {'id': 4226919,
  'url': 'https://zysjonline.com/articles/misc/72528/',
  'domain': 'zysjonline.com',
  'sample_index': 3,
  'sample_role': 'additional'},
 {'id': 4231007,
  'url': 'https://zysjonline.com/articles/misc/80382/',
  'domain': 'zysjonline.com',
  'sample_index': 4,
  'sample_role': 'additional'},
 {'id': 4231024,
  'url': 'https://zysjonline.com/articles/misc/80411/',
  'domain': 'zysjonline.com',
  'sample_index': 5,
  'sample_role': 'additional'},
 {'id': 4245797,
  'url': 'https://zysjonline.com/books/binglixue/169588/',
  'domain': 'zysjonline.com',
  'sample_index': 6,
  'sample_role': 'additional'},
 {'id': 4254533,
  'url': 'https://zysjonline.com/books/fengshijinnangmilu/87492/',
  'domain': 'zysjonline.com',
  'sample_index': 7,
  'sample_role': 'additional'},
 {'id': 4261981,
  'url': 'https://zysjonline.com/books/hewuqisunshang/171903/',
  'domain': 'zysjonline.com',
  'sample_index': 8,
  'sample_role': 'additional'},
 {'id': 4269725,
  'url': 'https://zysjonline.com/books/jinzhenmichuan/135265/',
  'domain': 'zysjonline.com',
  'sample_index': 9,
  'sample_role': 'additional'},
 {'id': 4270989,
  'url': 'https://zysjonline.com/books/jizhenyixue/181705/',
  'domain': 'zysjonline.com',
  'sample_index': 10,
  'sample_role': 'additional'},
 {'id': 4273522,
  'url': 'https://zysjonline.com/books/liliaoxue/178754/',
  'domain': 'zysjonline.com',
  'sample_index': 11,
  'sample_role': 'additional'},
 {'id': 4284950,
  'url': 'https://zysjonline.com/books/nvkebaojian/192628/',
  'domain': 'zysjonline.com',
  'sample_index': 12,
  'sample_role': 'additional'},
 {'id': 4291135,
  'url': 'https://zysjonline.com/books/qixiaojianbianliangfang/136421/',
  'domain': 'zysjonline.com',
  'sample_index': 13,
  'sample_role': 'additional'},
 {'id': 4307548,
  'url': 'https://zysjonline.com/books/shoushibaoyuan/102697/',
  'domain': 'zysjonline.com',
  'sample_index': 14,
  'sample_role': 'additional'},
 {'id': 4316590,
  'url': 'https://zysjonline.com/books/waitaimiyao/9678/',
  'domain': 'zysjonline.com',
  'sample_index': 15,
  'sample_role': 'additional'},
 {'id': 4319614,
  'url': 'https://zysjonline.com/books/xiaoerweishengzongweilunfang/115048/',
  'domain': 'zysjonline.com',
  'sample_index': 16,
  'sample_role': 'additional'},
 {'id': 4341036,
  'url': 'https://zysjonline.com/books/yizongjinjian/84574/',
  'domain': 'zysjonline.com',
  'sample_index': 17,
  'sample_role': 'additional'},
 {'id': 4379471,
  'url': 'https://zysjonline.com/books/zhongyijibingyuce/15205/',
  'domain': 'zysjonline.com',
  'sample_index': 18,
  'sample_role': 'additional'},
 {'id': 4380018,
  'url': 'https://zysjonline.com/books/zhongyizhiyao5500/14799/',
  'domain': 'zysjonline.com',
  'sample_index': 19,
  'sample_role': 'additional'}]

SOURCE_PROFILE = {'rows': 1220598,
 'id_min': 583,
 'id_max': 7466161,
 'hosts': 94,
 'host_counts': {'article.iiyi.com': 729,
                 'ask.39.net': 32772,
                 'ask.familydoctor.com.cn': 1,
                 'baby.39.net': 3368,
                 'bachmai.gov.vn': 2783,
                 'baidianfeng.familydoctor.com.cn': 60,
                 'baoangiang.com.vn': 10738,
                 'baocantho.com.vn': 12016,
                 'baochinhphu.vn': 2302,
                 'baodanang.vn': 14500,
                 'baogialai.com.vn': 3290,
                 'baohaiphong.vn': 16286,
                 'baolangson.vn': 1981,
                 'baonghean.vn': 15941,
                 'baophutho.vn': 1288,
                 'baoquangninh.vn': 2912,
                 'baoquangtri.vn': 5483,
                 'baotayninh.vn': 1198,
                 'baothanhhoa.vn': 782,
                 'baovinhlong.com.vn': 5113,
                 'benhviennhitrunguong.gov.vn': 1200,
                 'benhvienvietduc.org': 425,
                 'bingli.iiyi.com': 5,
                 'byby.39.net': 37,
                 'cancer.39.net': 736,
                 'care.39.net': 1282,
                 'dantri.com.vn': 2063,
                 'ek.39.net': 448,
                 'fitness.39.net': 185,
                 'fk.39.net': 928,
                 'food.39.net': 594,
                 'gan.39.net': 294,
                 'gc.39.net': 59,
                 'giadinhonline.vn': 16729,
                 'gk.39.net': 90,
                 'hanoimoi.vn': 1283,
                 'health.people.com.cn': 50,
                 'heart.39.net': 1055,
                 'hellobacsi.com': 11747,
                 'jb39.com': 527,
                 'jbk.39.net': 2033,
                 'khoahocphothong.vn': 8143,
                 'laodong.vn': 31282,
                 'man.39.net': 382,
                 'medlatec.vn': 24762,
                 'mega.vietnamplus.vn': 2,
                 'nhandan.vn': 5225,
                 'nhathuoclongchau.com.vn': 79598,
                 'nk.39.net': 762,
                 'pf.39.net': 85,
                 'phunusuckhoe.giadinhonline.vn': 27577,
                 'phuyen.baodaklak.vn': 6252,
                 'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                 'pubmed.ncbi.nlm.nih.gov': 240000,
                 'qihuangzhishu.com': 29,
                 'qy.familydoctor.com.cn': 2,
                 'shen.39.net': 325,
                 'sj.39.net': 197,
                 'special.vietnamplus.vn': 1,
                 'suckhoecongdongonline.vn': 154503,
                 'suckhoedoisong.vn': 85823,
                 'suckhoeviet.org.vn': 5627,
                 'tamanhhospital.vn': 4261,
                 'test.pmphai.com': 2989,
                 'thanhnien.vn': 72111,
                 'thaythuocvietnam.vn': 2927,
                 'tiemchunglongchau.com.vn': 13281,
                 'tienphong.vn': 3021,
                 'tnb.39.net': 717,
                 'tuoitre.vn': 3893,
                 'vietnamnet.vn': 15543,
                 'vnexpress.net': 2055,
                 'vov.vn': 9784,
                 'vov2.vov.vn': 6899,
                 'wei.39.net': 225,
                 'woman.39.net': 687,
                 'www.120ask.com': 98160,
                 'www.a-hospital.com': 4701,
                 'www.baidu.com': 11,
                 'www.cnkang.com': 40046,
                 'www.familydoctor.cn': 2,
                 'www.familydoctor.com.cn': 26819,
                 'www.msdmanuals.cn': 905,
                 'www.pharmacity.vn': 69,
                 'www.qdnd.vn': 3700,
                 'www.sggp.org.vn': 1853,
                 'www.vietnamplus.vn': 3695,
                 'www.vinmec.com': 26129,
                 'www.wujue.com': 708,
                 'www.youlai.cn': 4139,
                 'www.zydcd.com': 204,
                 'youmed.vn': 7189,
                 'zhongyibaodian.net': 6061,
                 'zysjonline.com': 6591},
 'unique_host_counts': {'article.iiyi.com': 729,
                        'ask.39.net': 32772,
                        'ask.familydoctor.com.cn': 1,
                        'baby.39.net': 3368,
                        'bachmai.gov.vn': 2783,
                        'baidianfeng.familydoctor.com.cn': 60,
                        'baoangiang.com.vn': 10738,
                        'baocantho.com.vn': 12016,
                        'baochinhphu.vn': 2302,
                        'baodanang.vn': 14500,
                        'baogialai.com.vn': 3290,
                        'baohaiphong.vn': 16286,
                        'baolangson.vn': 1981,
                        'baonghean.vn': 15941,
                        'baophutho.vn': 1288,
                        'baoquangninh.vn': 2912,
                        'baoquangtri.vn': 5483,
                        'baotayninh.vn': 1198,
                        'baothanhhoa.vn': 782,
                        'baovinhlong.com.vn': 5113,
                        'benhviennhitrunguong.gov.vn': 1200,
                        'benhvienvietduc.org': 425,
                        'bingli.iiyi.com': 5,
                        'byby.39.net': 37,
                        'cancer.39.net': 736,
                        'care.39.net': 1282,
                        'dantri.com.vn': 2063,
                        'ek.39.net': 448,
                        'fitness.39.net': 185,
                        'fk.39.net': 928,
                        'food.39.net': 594,
                        'gan.39.net': 294,
                        'gc.39.net': 59,
                        'giadinhonline.vn': 16729,
                        'gk.39.net': 90,
                        'hanoimoi.vn': 1283,
                        'health.people.com.cn': 50,
                        'heart.39.net': 1055,
                        'hellobacsi.com': 11747,
                        'jb39.com': 527,
                        'jbk.39.net': 2033,
                        'khoahocphothong.vn': 8143,
                        'laodong.vn': 31282,
                        'man.39.net': 382,
                        'medlatec.vn': 24762,
                        'mega.vietnamplus.vn': 2,
                        'nhandan.vn': 5225,
                        'nhathuoclongchau.com.vn': 79598,
                        'nk.39.net': 762,
                        'pf.39.net': 85,
                        'phunusuckhoe.giadinhonline.vn': 27577,
                        'phuyen.baodaklak.vn': 6252,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                        'pubmed.ncbi.nlm.nih.gov': 240000,
                        'qihuangzhishu.com': 29,
                        'qy.familydoctor.com.cn': 2,
                        'shen.39.net': 325,
                        'sj.39.net': 197,
                        'special.vietnamplus.vn': 1,
                        'suckhoecongdongonline.vn': 154503,
                        'suckhoedoisong.vn': 85823,
                        'suckhoeviet.org.vn': 5627,
                        'tamanhhospital.vn': 4261,
                        'test.pmphai.com': 2989,
                        'thanhnien.vn': 72111,
                        'thaythuocvietnam.vn': 2927,
                        'tiemchunglongchau.com.vn': 13281,
                        'tienphong.vn': 3021,
                        'tnb.39.net': 717,
                        'tuoitre.vn': 3893,
                        'vietnamnet.vn': 15543,
                        'vnexpress.net': 2055,
                        'vov.vn': 9784,
                        'vov2.vov.vn': 6899,
                        'wei.39.net': 225,
                        'woman.39.net': 687,
                        'www.120ask.com': 98160,
                        'www.a-hospital.com': 4701,
                        'www.baidu.com': 11,
                        'www.cnkang.com': 40046,
                        'www.familydoctor.cn': 2,
                        'www.familydoctor.com.cn': 26819,
                        'www.msdmanuals.cn': 905,
                        'www.pharmacity.vn': 69,
                        'www.qdnd.vn': 3700,
                        'www.sggp.org.vn': 1853,
                        'www.vietnamplus.vn': 3695,
                        'www.vinmec.com': 26129,
                        'www.wujue.com': 708,
                        'www.youlai.cn': 4139,
                        'www.zydcd.com': 204,
                        'youmed.vn': 7189,
                        'zhongyibaodian.net': 6061,
                        'zysjonline.com': 6591},
 'available_host_counts': {'article.iiyi.com': 729,
                           'ask.39.net': 32772,
                           'ask.familydoctor.com.cn': 1,
                           'baby.39.net': 3368,
                           'bachmai.gov.vn': 2783,
                           'baidianfeng.familydoctor.com.cn': 60,
                           'baoangiang.com.vn': 10738,
                           'baocantho.com.vn': 12016,
                           'baochinhphu.vn': 2302,
                           'baodanang.vn': 14500,
                           'baogialai.com.vn': 3290,
                           'baohaiphong.vn': 16286,
                           'baolangson.vn': 1981,
                           'baonghean.vn': 15941,
                           'baophutho.vn': 1288,
                           'baoquangninh.vn': 2912,
                           'baoquangtri.vn': 5483,
                           'baotayninh.vn': 1198,
                           'baothanhhoa.vn': 782,
                           'baovinhlong.com.vn': 5113,
                           'benhviennhitrunguong.gov.vn': 1200,
                           'benhvienvietduc.org': 425,
                           'bingli.iiyi.com': 5,
                           'byby.39.net': 37,
                           'cancer.39.net': 736,
                           'care.39.net': 1282,
                           'dantri.com.vn': 2063,
                           'ek.39.net': 448,
                           'fitness.39.net': 185,
                           'fk.39.net': 928,
                           'food.39.net': 594,
                           'gan.39.net': 294,
                           'gc.39.net': 59,
                           'giadinhonline.vn': 16729,
                           'gk.39.net': 90,
                           'hanoimoi.vn': 1283,
                           'health.people.com.cn': 50,
                           'heart.39.net': 1055,
                           'hellobacsi.com': 11747,
                           'jb39.com': 527,
                           'jbk.39.net': 2033,
                           'khoahocphothong.vn': 8143,
                           'laodong.vn': 31282,
                           'man.39.net': 382,
                           'medlatec.vn': 24762,
                           'mega.vietnamplus.vn': 2,
                           'nhandan.vn': 5225,
                           'nhathuoclongchau.com.vn': 79578,
                           'nk.39.net': 762,
                           'pf.39.net': 85,
                           'phunusuckhoe.giadinhonline.vn': 27577,
                           'phuyen.baodaklak.vn': 6252,
                           'pmc-ecm-healthblog.beta.pharmacity.io': 5333,
                           'pubmed.ncbi.nlm.nih.gov': 239970,
                           'qihuangzhishu.com': 29,
                           'qy.familydoctor.com.cn': 2,
                           'shen.39.net': 325,
                           'sj.39.net': 197,
                           'special.vietnamplus.vn': 1,
                           'suckhoecongdongonline.vn': 154503,
                           'suckhoedoisong.vn': 85823,
                           'suckhoeviet.org.vn': 5627,
                           'tamanhhospital.vn': 4261,
                           'test.pmphai.com': 2989,
                           'thanhnien.vn': 72111,
                           'thaythuocvietnam.vn': 2927,
                           'tiemchunglongchau.com.vn': 13281,
                           'tienphong.vn': 3021,
                           'tnb.39.net': 717,
                           'tuoitre.vn': 3893,
                           'vietnamnet.vn': 15543,
                           'vnexpress.net': 2055,
                           'vov.vn': 9764,
                           'vov2.vov.vn': 6899,
                           'wei.39.net': 225,
                           'woman.39.net': 687,
                           'www.120ask.com': 98160,
                           'www.a-hospital.com': 4701,
                           'www.baidu.com': 11,
                           'www.cnkang.com': 40046,
                           'www.familydoctor.cn': 2,
                           'www.familydoctor.com.cn': 26819,
                           'www.msdmanuals.cn': 905,
                           'www.pharmacity.vn': 69,
                           'www.qdnd.vn': 3700,
                           'www.sggp.org.vn': 1853,
                           'www.vietnamplus.vn': 3695,
                           'www.vinmec.com': 26129,
                           'www.wujue.com': 708,
                           'www.youlai.cn': 4139,
                           'www.zydcd.com': 204,
                           'youmed.vn': 7189,
                           'zhongyibaodian.net': 6061,
                           'zysjonline.com': 6591},
 'sample_host_counts': {'article.iiyi.com': 20,
                        'ask.39.net': 20,
                        'ask.familydoctor.com.cn': 1,
                        'baby.39.net': 20,
                        'bachmai.gov.vn': 20,
                        'baidianfeng.familydoctor.com.cn': 20,
                        'baoangiang.com.vn': 20,
                        'baocantho.com.vn': 20,
                        'baochinhphu.vn': 20,
                        'baodanang.vn': 20,
                        'baogialai.com.vn': 20,
                        'baohaiphong.vn': 20,
                        'baolangson.vn': 20,
                        'baonghean.vn': 20,
                        'baophutho.vn': 20,
                        'baoquangninh.vn': 20,
                        'baoquangtri.vn': 20,
                        'baotayninh.vn': 20,
                        'baothanhhoa.vn': 20,
                        'baovinhlong.com.vn': 20,
                        'benhviennhitrunguong.gov.vn': 20,
                        'benhvienvietduc.org': 20,
                        'bingli.iiyi.com': 5,
                        'byby.39.net': 20,
                        'cancer.39.net': 20,
                        'care.39.net': 20,
                        'dantri.com.vn': 20,
                        'ek.39.net': 20,
                        'fitness.39.net': 20,
                        'fk.39.net': 20,
                        'food.39.net': 20,
                        'gan.39.net': 20,
                        'gc.39.net': 20,
                        'giadinhonline.vn': 20,
                        'gk.39.net': 20,
                        'hanoimoi.vn': 20,
                        'health.people.com.cn': 20,
                        'heart.39.net': 20,
                        'hellobacsi.com': 20,
                        'jb39.com': 20,
                        'jbk.39.net': 20,
                        'khoahocphothong.vn': 20,
                        'laodong.vn': 20,
                        'man.39.net': 20,
                        'medlatec.vn': 20,
                        'mega.vietnamplus.vn': 2,
                        'nhandan.vn': 20,
                        'nhathuoclongchau.com.vn': 20,
                        'nk.39.net': 20,
                        'pf.39.net': 20,
                        'phunusuckhoe.giadinhonline.vn': 20,
                        'phuyen.baodaklak.vn': 20,
                        'pmc-ecm-healthblog.beta.pharmacity.io': 20,
                        'pubmed.ncbi.nlm.nih.gov': 20,
                        'qihuangzhishu.com': 20,
                        'qy.familydoctor.com.cn': 2,
                        'shen.39.net': 20,
                        'sj.39.net': 20,
                        'special.vietnamplus.vn': 1,
                        'suckhoecongdongonline.vn': 20,
                        'suckhoedoisong.vn': 20,
                        'suckhoeviet.org.vn': 20,
                        'tamanhhospital.vn': 20,
                        'test.pmphai.com': 20,
                        'thanhnien.vn': 20,
                        'thaythuocvietnam.vn': 20,
                        'tiemchunglongchau.com.vn': 20,
                        'tienphong.vn': 20,
                        'tnb.39.net': 20,
                        'tuoitre.vn': 20,
                        'vietnamnet.vn': 20,
                        'vnexpress.net': 20,
                        'vov.vn': 20,
                        'vov2.vov.vn': 20,
                        'wei.39.net': 20,
                        'woman.39.net': 20,
                        'www.120ask.com': 20,
                        'www.a-hospital.com': 20,
                        'www.baidu.com': 11,
                        'www.cnkang.com': 20,
                        'www.familydoctor.cn': 2,
                        'www.familydoctor.com.cn': 20,
                        'www.msdmanuals.cn': 20,
                        'www.pharmacity.vn': 20,
                        'www.qdnd.vn': 20,
                        'www.sggp.org.vn': 20,
                        'www.vietnamplus.vn': 20,
                        'www.vinmec.com': 20,
                        'www.wujue.com': 20,
                        'www.youlai.cn': 20,
                        'www.zydcd.com': 20,
                        'youmed.vn': 20,
                        'zhongyibaodian.net': 20,
                        'zysjonline.com': 20}}

BASELINE_DOMAINS = {'article.iiyi.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'ask.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'ask.familydoctor.com.cn': {'error_type': 'not_previously_tested',
                             'shortfall': 19},
 'baby.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'bachmai.gov.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baidianfeng.familydoctor.com.cn': {'error_type': 'not_previously_tested',
                                     'shortfall': 0},
 'baoangiang.com.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baocantho.com.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baochinhphu.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baodanang.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baogialai.com.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baohaiphong.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baolangson.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baonghean.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baophutho.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baoquangninh.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baoquangtri.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baotayninh.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baothanhhoa.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'baovinhlong.com.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'benhviennhitrunguong.gov.vn': {'error_type': 'not_previously_tested',
                                 'shortfall': 0},
 'benhvienvietduc.org': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'bingli.iiyi.com': {'error_type': 'not_previously_tested', 'shortfall': 15},
 'byby.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'cancer.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'care.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'dantri.com.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'ek.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'fitness.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'fk.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'food.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'gan.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'gc.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'giadinhonline.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'gk.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'hanoimoi.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'health.people.com.cn': {'error_type': 'not_previously_tested',
                          'shortfall': 0},
 'heart.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'hellobacsi.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'jb39.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'jbk.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'khoahocphothong.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'laodong.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'man.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'medlatec.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'mega.vietnamplus.vn': {'error_type': 'not_previously_tested',
                         'shortfall': 18},
 'nhandan.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'nhathuoclongchau.com.vn': {'error_type': 'not_previously_tested',
                             'shortfall': 0},
 'nk.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'pf.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'phunusuckhoe.giadinhonline.vn': {'error_type': 'not_previously_tested',
                                   'shortfall': 0},
 'phuyen.baodaklak.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'pmc-ecm-healthblog.beta.pharmacity.io': {'error_type': 'not_previously_tested',
                                           'shortfall': 0},
 'pubmed.ncbi.nlm.nih.gov': {'error_type': 'not_previously_tested',
                             'shortfall': 0},
 'qihuangzhishu.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'qy.familydoctor.com.cn': {'error_type': 'not_previously_tested',
                            'shortfall': 18},
 'shen.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'sj.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'special.vietnamplus.vn': {'error_type': 'not_previously_tested',
                            'shortfall': 19},
 'suckhoecongdongonline.vn': {'error_type': 'not_previously_tested',
                              'shortfall': 0},
 'suckhoedoisong.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'suckhoeviet.org.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'tamanhhospital.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'test.pmphai.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'thanhnien.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'thaythuocvietnam.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'tiemchunglongchau.com.vn': {'error_type': 'not_previously_tested',
                              'shortfall': 0},
 'tienphong.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'tnb.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'tuoitre.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'vietnamnet.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'vnexpress.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'vov.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'vov2.vov.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'wei.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'woman.39.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.120ask.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.a-hospital.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.baidu.com': {'error_type': 'not_previously_tested', 'shortfall': 9},
 'www.cnkang.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.familydoctor.cn': {'error_type': 'not_previously_tested',
                         'shortfall': 18},
 'www.familydoctor.com.cn': {'error_type': 'not_previously_tested',
                             'shortfall': 0},
 'www.msdmanuals.cn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.pharmacity.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.qdnd.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.sggp.org.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.vietnamplus.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.vinmec.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.wujue.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.youlai.cn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'www.zydcd.com': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'youmed.vn': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'zhongyibaodian.net': {'error_type': 'not_previously_tested', 'shortfall': 0},
 'zysjonline.com': {'error_type': 'not_previously_tested', 'shortfall': 0}}

EXPECTED_URL_COUNT = 1764
EXPECTED_DOMAIN_COUNT = 94
BASELINE_ARCHIVE = ''
BASELINE_ARCHIVE_SHA256 = ''
SOURCE_SHA256 = '87e44a792d7a99fb0fde5dff7f09f1a38a0a1fdf02ddd1ac93a005736e00b0a7'
SOURCE_FILE = 'links_corpus_vi_en_zh.parquet'
SOURCE_REVISION = '71be0e1239a19dfd7a717e2bc13e3bec21bd0334'
SOURCE_URL = 'https://huggingface.co/datasets/AIGuruTinix/ViBioMIR/blob/71be0e1239a19dfd7a717e2bc13e3bec21bd0334/links_corpus_vi_en_zh.parquet'
CRAWLER_SHA256 = '7f2d93d72b8b251cba34dd8080b41e649d47549b0147fa358958822e99eda6ed'
CRAWLER_PAYLOAD = 'c-qZ<ZIj%_dEoc`70{}Dn4xB|vrAHumK@nMMcXV%ltNL?rIr%}JA>JQc4jaFF#F==esEk)9AA7DUtY?&ODa-)sif>UauQ!I7JcUrROx@P%9P|!_ZRMY>qa-wn4P8NyD0Gz19U(A_Vn}KPp_}9e>M4HlD>BPt~bb{qsc}(o1A#_WRezvcPI7US;X0iH;MM*N#60M>5%>{vM8A)vwbhkyks_vkKs93UteDvW$Dxl!_lHxWN{dJ$#kA(g%{0cX%Q7kI?LDA)Vpkd9%XsVbq48V5)Y_^s&q45%!)Ycc*A%UEhfb<85FAUY_Q0(cvb}5lCE=aoW;@bjWnIyI*td6B8BQv9uCs!d=eM&P}eWw<6@HRse1Be8qK2pIO9^osECVX8mltuHw+*Cel49z*W+oFOoBy`OmsJBU|2pZ(ooe?1K<BzGFQ*zC?7*N)KC5gUIf#)h@fJmzs#fI5C)p77jNh3Og&GdV$6M+!;>+rH}HczXVGk!PW3O{oqUX4*T2%iL0srxxM+Ei40N;kiGD0-u3_c%3uBOrSuy}*p&?Mu7TE;HMKhzi2k+ialUWBJ^7$kwtfE=Emljk$Qx)#wt1l&!_zilkacO=MWm$RzOARwnc^Tm1FN&Bx4yqpkKA?C(AKxh_2j1mHInpc|C6fp+O$Jj;hN?^ZaS@It!?m@q-n#qd?bq*kJ<mUU!C!mp20p!(el3|yqRp#8*K7S2Y?bsV_wL;Dwz@(0ng=hQzjDnxe*Q|^yD<k0{jGTKm896bdighk%g=kQSH5!ZwO2ddBsqw^FUNy}wC&v-L;LaO)@1;I|L46KjiM}(D*oEt*T4Asz3|4DZ$Tqv`~KS68tfu(Q4Ct`oiz{s%VHSLOe1wLOOIL#uK}3{?I6k1QJMj6wc1jBnC!!}v~W>vD1yd8K907pKF<Y$aeOQlRDGDlIK;9o#8UxrloVq^RDKzx^LW<svpv6!i!$Ouj<yJxG%)ITKDX_=+Kq~Inhe6dlOoQkm5yhO0bjsem0i$p5j&8`Ob>=h*5V&|?_Rb58UQpeFT(VIesYx}p3VUf)x=SSvuGL<$N-<b=e&_0oa${8AnGJTfZ1xF1;upkmtcaUECK4lJ*K{vpR#y9i3YLsl}E_MktWFa(!IB3c#gqixnQ`M&hr+OgR#tVB-c?sNRr-{qDc<pL$W?AdfV+Fp5abz`HNz-@q*u0O-|BiShi^612upJ@|Ll(IPMV1Y>Pm`6!fY<vEoE}3EMMo9)s@(NIP1_STfg0+A!_@{h99pE_=wW7;e+3#wp+p5FmRzEk&T{Q-FnNMFFG_7&l3e;!I=P3^&ar`C8n<u?ir31l%ylVo;Xz?|kW|-{CR>QZ2H7KZDM9Xb9F@_0EKz#X-K<Yi0iK{ldGy=ys!S|2Zt^3G}u(C|&yg?DTTyi~;J#cy^D2yvUMyt4-5)jMIm30-g-&K)bEV$oS{bOV}zcVDM`F(nX<m7`_eQH+Ll&c`f|7>G{BWlM(FN09O>oN*gR!<cj(oCBQqr8ci1QEg;@m%YS`Q0BQH438K~sk_IpG6rgJ`!UZxLn$*EKP2or6&Eg|(V*?ruWb*QKkqu(2fBZ@a!3g*CJHZ>wDUkcL%<nLS`6$Q-V~_(vW-5pqx%W)Z+x3$n&L?1v-^X5n7zFeoC{E^a4g@w#K<og4qg914O5(||g|*vMc<C(MI-Y~*6a!0QXkIb|iEHTLvVe>RjiY$An9g!F<Ybuhpm<A0dwNVARkTNa@3;9MHBzV;%odXgNK_!4;fUsgfS!RAgP<@Q6fG#z4nXLdp8(pxBus#w<GW=;crD3uktc@>z^wtWCABcOU)KGmmayw08@BlMD9L~pH++27d+YA2UbHC2DG<pMFUh@pF&ZU<1n3)~^*l>~nLzW`yhGTK$P&E~4D!U=i{U%`h>O7(DK@tTRrVrygS!D14k&LH(hDF?<?PX#8-vDbxD-%{3I|-Y)rA#KVX`I(a;&nv%s6690&@eR*bH_)K<MDYwiPzPE}kMd*vG#!E{b{n`JK(p-QU@yjb9Qn^S|2kkNn#+8o8Qsn6Vx26ly<{oypXz|3HQW*&j(+Hc@0?6Pmi*-reqNN@&*Lb=l>Leeb!Rw^bcvzL>VmMGOwHbFlf_-gS_by50I9%;I&}0FyT|kXP>|f?tCS6IhmkW|rZno}UO62@k6%oP5^1_0A%iY#>trN;F9ar0B3*lf}3#vSe>T?+^#DT+UxdK#GtXF<!m1038V?7WJF}#ms~FB!_L8!Pb`Ew>w^!TRj>l*bf*NRkJ2ckIF_qSvmNBs#3!=daI}MdUf51qO97U`p<@`mnVa*-V==^xJ&;C;f>>=Vp1UBAEvO%I3=Y9*us4xggcIBby~1&ceblDG5`UrqqZgJWvwM!vt#euE$d@tn_YaD;9#2f$Q(3UV(@BGR8gzcB$+12XsubVE;Ev21cxg}73!;opMz))g7~m?iu8X+PZg219goP6&n?hbd^od51eO8^Mv%DxhXUjj5di+T56s6HeY;*~B-I*wta`mm<@x~sBK>A;;;r5DWXH8KW)3*>C^hP-*SKGblr*0m4>oLX5>5Apk(cdwnQ%~sUPM`elsiQg0VMyZC^Eu%=ua{X={q3E9Z<gjlB+;ESKqbhM75Bk(Q9P3Eowo%Sw_;{1fd}r0gp?85^ZcG!wrUp`&@nkq*i=9m@M+-Fm7w%VdUv{kpp2%WkY#9VPT|6z5rGWTM{^AON67BdVYr?!obzb>po9byKil?vYF~?onE?>nM+H8e5Tm&S-DoKgUi>;*eu)>&29swb1~GHNgB%lS4b>9H6Wc4(MJ_h<hG3s;z6ex{8?adUBwqpd2Bmgw?fXDP@mZqg2fyKnU<oCFglr=zA+cvp|%pzSJiY`O%uVLfsoif!Bu0J%%w`TR23#H2wCLfN>Jxi_ua>-f#n0rtxRAba85&>G294KT3!%4;lYwM<#Ti_pQB^@bJ>O#F@ZE-V_m39Y#E+$oiwwhw<2V~H-0bxu@Hg+N@$lfrIU`DqDmQBCI)uaG+nk*WKGln_7)&d7!^p@(6EE(5A&7B_{P6+`^$@LaS!aY$D7K=i#rQUougzQOAOL%zR2+%aVUI6hF#wnA+HyF0x3qeFl_2PogBu%^g4P$#rm>i*my)qGJ)YVfE<l2!BVLW;yXzS@ej|BQMlrn(<IpJprV}sO$ZfrmZ+_wo1rFR^evrZzd_P~Us~#*S4<fM3j~p?{dbPlwH1unS%%am0>C^}C_kyNVYZkl+Fw#0(6bMdbdg({t1jO*>}K!^>G5vgl77%&s%d~t(9#LfZ`Hsi_5-j5HVZtD)=!}-6U#Fi>$*#60T>P)zElckpe-(9n~qW=&~j;80x0b&bS}mnK&q^JgCf29$SSw`O21ang#5c?GPwQnCMu27Bq!Y}@EiJoCOoiJMn@qUrO;vt%yYZj^{)4<5~!FVnijO(x+r>Gb}2Og$)0yL8r?*{R%b5-%SVsA3c)Z&V>h}9T2O<zFB{F;PN_=lqzHoeo7tRTdEYH;HO(B30Cz;=+_7{#fF6S#DT@Z&dPg@+o)|DEpw+|gm+wc(4CnxJ^ZLB;Tnozs@aGjlq5{SMDq1f)nne2{s%}sTgb1oxF9kFP78eG>n}3C+;u;sUDqBhd`D#D|nV%r-!A0%})bmEnR3JpXvWK)>A(=(Z!E%?S13(|*D?Q>Wb-;BtWl#s)_Gtf7E9dGL_C)Cy=4LL-x|%@3R^x}7!Y*dQq?`luW%$hR`G?TlIT*<b(~NxD#0Z*%1n8l(jdsm5cr^lUGnO@6TN_NGJoj#*dzJUft>ZyFM}vS6MzorSz`&9s3_%r|j7S~^Vu8NT0=$boFbP{CHkb>k01CkLP<~v@QMU{<JcfXzkxQ06`VyM8;;gX`FdFQ@Bs%GB1t22<UjqLF;Oga$cM0wDpgQECev9m>QzsTu4EFU5CN)GK69DdQT}6=ck<|qGg08J*IAFZaPQty#5OBWtV%GpNf?=ZZzbGaIqdH?W9D1TL$+POd;-a{OGU{@~HlWJDp{9OB&QSR(x)_Tym5-*XfdsGRg5OqBQNk8iXCEy4m773Ut4tSQu-ZeD_&m*%0;D5lKsIT%*&2(Eq+C~kA^tnM)p7EU@;~2=9JocDA6A8D;!_FCe7icN65UKl<=a&u8R%I8GG3lp!WpQh9ghqRqO(>O0qt7T++i;^Y{gU^tX5>~5p!|<a|;$7DOMU4ZCwl_3cx=xLV6%2=^OV5q?_(aX5~{D#+gPmhDI`y!nh_(lT`v#Yc*hkYf;^>ti`gv7_*NSDOd4&NxO~xMMM&7>D7G|A}Cswo`6yKAeXjhAosuqv3p_`;8(cI&N{~boMQtMLXw?O*;<Ry#OfElK_{J}_Sj|!xF*y-i}%q~eL`jt1Fy(ZU5>RvWt1zC>w+d#h^@9o7+jqeqxl@f3}AuOy3w3y4<JL58dMI36f~;hSFpY;6G#a%m2wP6@TRoWKBo@E4MamQOeuls)PD=q&5axTAfCa7ed`8v;Kl$MHz~@nikstNI_ZEML&gLEu{j@(uD#Ri1}}ClZC;|MtNvNL8o^L+n`h+<6mByB6>*#n^*kw;RLgeeaEK9Gm`m5RIgMO`{fe<L5RY{c)cMI-<hM(&B$YN@+wuxzrL6L54&uX6T0;gR0EV37C+s#MvE9`-il9+FBl*F50T}v~E0^0B0S%*p%1G9RxEm3w(cH5XTXW(Ssd8^21&R+buuTd15bK67*wg3H+H8Vgc!;h*kW%O13opb?qbM{IV4}?wF@<^_Mrb9oQQE4^WycyWRBgck>nqgjsYPK~;e~Fy!5ye#!GdHS%3}+=vwNk#+wEiIPPjiw_o7J?kv1oCG0US-tl;>aDkd0PR`yhwBKu&fK?Zd)$}3Tkm+=xu$)P;QFNUzY!1fdODpm~I3>J=;pO9PhRD>0~K9LZIIp{vKVsgR~wkJ+CyE0NYl#Uho)s#*XV}Neqr0SuP*zC!}T0t~;X94(P(2{r}i(-9OLq{8Aw;a2-L6Z_pQ>c+bC<qY5a}ga@We&&<u}%U7(fuy3gvV?O)W&fHOcv<DQ3QMyylaMq7<*=H2O!gSlUN02O8RQ@C^BB)Jdw@9(fKNNnp3Flc(~iBjhA}MW?~gwz1Uprov0}ofCvPkce9Eug1V4T;&|S|4pu%^=SjP9kgZla?@)4itCWI_dcytxLI<owZi|d#1{*9e7<WoRxYo2%51kdrYNhJ9MYzvs<{c)VhB*O4eB44qz6hgb2RFTKcCcw7xKiC=m(Y@mk}cYV!9+OJmaIUbV2U`Jiv0!s_ea|FldBLiaNwv3a8KG`yqF<TDZr#ZAyX@C8E456#UepzaVio}_EL-KtM&ZVtCyc|H?Iyi@*GFj5Uh4%SyU{k_)E)Nr_Jl00#`(j6guTb5KaowFvU2Ufj<kOoq46jkv1Hd8vW&+scamDvCzLOS7~oF(vVtJgPyc6I<7iCH+P#CR~f99O!zVixUBcTwvFhC^sN!CVU)_+BqDG}8I?7Da)A(I7dxeB4O?e1p!oaXCb*q$L&+KIbuTAmIF;H079%KB*e1`ZyElqj3sW*pHuMjoNw;wY#4v^u<(2IhJKojq<t5m`)WsO7(gMhCe;0MZjT<A_ojA?iB}n3A7fVK^ev0ZXI(9Ssm~4iIdcC~0bGx&&OV0BP(!~IuVUnlwmX-<V^ODXR189{a=y><Q<|U7>1EGPLIHI5J^ZSk}h#Qv;0cID4m%f8vLr}09Z3C-fP0h70*6c7ea7#>Fl7g!n8l7DNR>#%LB^#ieZf7S*AkDy%4Lj3T`@vws<^yHt&653OW(JGkSJbS~xj`Q>h<cMN1;ugUuUP^S>7k7xktT^fUHZ9315G(5)WpDEPXTcp<nNzyXU=v`HP(eQFoK==DxACB-rd=HzVy6+D*(dSDZ2O6=h~pX^Meiw(!+%})NIwOuKI#5UwL#}{d`3lp(Kg`z8FH;=`WE6G+Ydcf!`!wl%|_`3Z?{D<rp^s^qEyBWkCdrIYS}Vv?nkmC`ynl1y$N6IfH_%wmNRx@pIl*UD3Q4zf$w8j3+dJo`083S=7(Elqlm24<Kl(=bKm%E)ylqQKQsW_lN@5J6LmdZ@01_{nB>7*3^VOvDRSc>AdBaW3a-r4C{l8pO7B{)bZq0EI0^)nV9>)-`qjZFNQcE$F}VgAcMg9pTLf*<s)#)a)}6hz5>G$sPhsjngmT~pJ1QLCXv=koW|e@ej=RaenOzeoPw|BR8Z}qvxX+rs_1RgS5g)=?`9}_@jiuCo~whM%Lkjp9^NIUu|to3tGj*0yF~wV&gD_({-=}@hw0m}m}nulu|XMtY4$uvHd7*5v%G-|ptiFSztB^YW;*J7meh2Uzic3PN3qFouQ*j&wiR%_(}NxFko_Ux4MC(;4T<{$Db<0Br-koNyCf|Ycd$`Nr&S!cuPF9Lq{KknqVUu{TX9&3RT56oGb1kAOT_*o<&PBNzX`9vSi7m1A(M}i?O|Mwf2z3CSs0zl<~$3tQ`nAY!T#6$#`di9Rmxe4F%1}BK#?Unu7smCrii7zZu=bkF{2@f{bv-Hf)RP(06K0G8AVjoZgSMW;oZ(^$jZD2+k!eE4_xhb`yGJ^z(R6<_sHFf1;A5|X3j8dFW17{+I*AvSD6No+U6L4kb}BBgBXN9QSAi~I)_3`1P)`xmr_kN+{(7paWXwkv#gye9jb)ctX&k31&r~Ef^}rId$rYHMP0kx1qtPHcN_m)#y`J-e_r6IDvQFrsy62mMWJMma?H3S;sR^8gwtQ5{ktO7nnjJ~$oB?)-j?l9nO8!Suzrr}NBoV4PJsw!QDV+yG8>}HjFW=6ZWJw69)Tn}N5>2O#>Milqsz-z#{farhR|1io`KFTUVjE+stBtZN-f(fN**v|D%>gB*+Z>g#8^_BTs%?jp$i;Pz)MxM2I(~IhEoM7l0zpdXU|(&=vr2f&%3wEGn2^2^G3R-CjcwZ(DA{6Ullt?_X;@8wz}<|U*8aR9b2T{57@w08tpBO>pS8A$x{Y(=3i{<0L(<}lOjuh<=(wFJPoaVk^N~1{?@?2Vh`3NLh-AWz4m;!`~PydB{Hu*UN*6qgJaFIuG=)|H%`^&!n#m{a<(eBS6VTARN-%Szzcln7*p1k5b`GV*&Q`QKw4Uk7SpQk9ayT~Wa#blQK(f3xgppZv=&JqNa4*_8R+?p7@1+K5oWc!?G+_rB;96|vhvEibhO=#t^T<btb+4(am!6jmaQha?aQ@Ce|g;+-avZlJD|RwoW_LZjH$tBv33pwEe)3vs>{(|cGocH^m<M!&dv%ut>RGE^3XD;x)idPdoO}yBqKZ+zwKSR<h4}aHo%eu<|H}T3CRvHfe3n)gYB#BCa1Fi`xRZ!Pvv1Yu^Sf>?4z(gu`KRQu@G;n;j#>t8(zw+Zs77SWRJ3UmFfo$#-S_V>zZEB=)|rn3L2Yw`U%t`TQTW1DkWmN#pau}g+T;%s&JeX;yA5FK56@=bWJRPMuZ7TBwVbFFfqfP8Y7;>P>t;MbS^P9XK)rCe$3(m)Omx^swjyiMx@f2Y)6mS0edymxw5>6g5Zvsl*ZR;6E`f)?RLzk+ktr95eZ1!Mwz!ch;ue-yJ!oy*x<Kdn7a(9Uxv|?o&y>9np<Y9t+3Z~0d{d>F3u5J*5)LOdBbUx3Sm<d&_LJbI=0EJB2hG`cD&RhKz|iV)f~iW%(lZ1WCotPDT(SJ??EW<GB#l6FJD9;cAywadTNH<GM;gI!!RZZ!^J})ttKlLxS1sXONm15wa*5@3||l>5-1^*WQ%u2r`*C7!l>WlG&V+*G8Hr0Q;%y^L_a6^OT-pU$;tSk1oD`r&}!tYXl;XN7LqxzWuydiWdv!~tCF0;VMw>KNG&@dd%7vC8`4X&G1_Zg;{-bGQK_S>>-g5HWDbX_ASu8HRiz^7sJ`HS6(1>;mUSZ|rct@ZlBAAaA<c?-@{A})JgsI1npg&=^6i)tKU5A|hNZ{lg~|drT@-34Di*v%Zp9s)f|=-2<!Uy|JVqJ<E6CbLUOZ!U<{<O6=woMh6slYJD{j~}O0vkS%^+B2*!)Iu?FK^@uEwsSDSg)_Yc}83EA!)8)m7H$LtWxLy}0*mnWIc~byW{pqr>Z|oL%DTQXVfx%l6t5OSh>+_Up&fs?i`T=-ErH|H{7Z(5fz1wJHN6tIAZf7S&890Y4!I{vcHJ*w*JDc#)EW9uL-<Jbxm!lL{prXbOmGFhN(%Z)~N)n5QoO*{0|x{Bwo1XEp5WtBm?t@6{A!WcGIHa2XIrAqG$>N*GihL3&v5jL~^dc3LlH!75_f)9at+_%9%eL6G6|7mIqu5Gb<0gNme;LkhUlfni!Jq@=m5hl(cH{xvRrU8y8WvCL!Za1E7_uEe)KGq3Ki?#XqyWPw}llWR80iw#IcoQ&A+Tl#9-XEj~tGxsm)W8Y8bR-$<>lPe*Su=qof!cT4kUCm5LcVrk2PFzYMTxr!z3nctl;cTv0Q;F)JE#T`ku<2CNcus(2&s2&_$p8EbAhkMVL!Ks_**!Xnv4`?AWU7rh_Zg}L5Jn6(g=8>^Hz2XtGy>8wju#mv9Hwm2f=O_itQ4pS8l8^w=@`>6IV2QF=uFcV{oSU&m+9{n`g@iBK97G3@!=1Ycb_0)GM3msE5*M<utXMMza5$)-?QH2c#7E+r#S8$PN^N@`B@O0CS;}q1ij|Hj>!dMocCdZZA3-|q@5_F%eXn2=2~;QKWsk$sGRf9b@r}2*AeXt`%{(LwM1hW*9FCBcjW^}+F8(C6eknfV96*0r5K-Psqy)cb{(4X{N-IJ!OJSacBgKEF>NpNt95Z<)lMjLnd*taLBRAZtECII*X;QE1QRsR2#8&umz4_CNCmwO@U*<+EQ-}`S<*eX1(7Ap2(om{-UH8XH_qPq)3sKPNNq(k!K;yAR-&#Mh)#A3cFi$ZGQ;^|PaR>TXx|nbiHOTxeHVSD7_+8mjbN{zbMgcsd>u%wijo3C$XWV<@KelDggJL`=j%-^0B{>)Q^E0^LR~gDFYSII8TQr{<%cMdV!dDH9m4|;sxBog*{Y?7MvKIn9TTN$z}Yy}QlOhEz$Y7T&^W3ssTiqq8WNYf@5v-9HUO>Ru@0h?qNelIP@-O7#zP<*khb&A{-bwKybr(o;WuXc_w!3T@W0mm;ptXq`)t$r_sUtj_4#M+58GBQU+F3VKDVQ$YEaGp4TNJ(VcZ027ei+J&kZ*}H{JZ)Z$lDVLJK=5_&HJOX6rV($$Esyed;{qZaf7s(xYSU*_6&O5E5ZNI^YmT35r&^B)~KRCL!@N*=X>Y$f`m%N%=oN(TV(lOW|)573mNazqa?%h|b3j=>!txppy!;w!=6dz;=vgnB=H_h<QaqUgOXdp%k;)U$nR`$tCoG&+KB7Xx6;jrcHzQ8(<@Doej?Y5r&Y#B;LS;XZtXP8?w77Ktr0UQ(ht4YO50`_h|{tdKJSrT@wMV69~Q4ctdQCuk2#WApT&Q+!=JIweGEVN;0*P&oR067YH^O0O2eBC9v?^uL#uO<~e9=Eqcnvh#p{LyI+8_>!WI$<qTt5g%))|bgS7(OVa@Avbp_Lj%<+6DL)HM?!DW0yjO2O`t}{~H{ZH($G<@EB4Q;^3!)`WyKOn+ZraO3^EAvL9~u%}=!24H4gm9WqVSX|re#Ve&=eX0X;(o3-hS=C<TFxSCHYSWlxl~MUaHMbEX6cGD!x+gb1Av`oB!d4Cu>J`3S*B$50cCjFjw5<qIlY;gi$%b_r$Tcw!2Y&fMJfvxsQ&H0^^ObU&%;8x0E!nF33QbgQ~RN_iL6}UDsZgWgwpphxZ@>{sC64Y?^46A2Cb(%-WgdST%GsCY(u+uF9QU9Lk&yq{vFq&_M8bdu2FO0>`wgi<8uDQEnM|zp<{Mdd4`fYe)eU&ETQmw4&#aipa_mq=5zd%9e#R0_Bur!FEh(j?(lqq1jbkfFt*2mmxr-uAckU3(kAXGsWG)Syk)BZ-_o^N&JlOfF#M=!0-jhFnFtk??kxi!$VTSBrlihUI=<`rc!6Am#!{KJsFESH$#G-F}x);DI`_3h6$@_R46btu5Y_}fY^0T2bL1RUH@6x;ow;a^hU!G2ljS=<cM`LGngHK8QCN=#v6moz#x)lbODj$9`w&bm*ogH-lFeT4ZVaNPmd5K2kw!U()5zT(4}F&wh^Si$OrAKGU)abKKX)bKXeJ)O7Uh@F=9cb-hI&SpOQ97+OxXUpd@B)(M6S-_OP`uQs2@ke%0O%9NpFH>S}Pj^((a(HBXNs12{V>MT~=LtUhrX+YA&#R8aU;^`&Zj+(i>lAj|oZSN<4?`KSCg-je6Y#!ws(FWjn3$e|T50E+=Vm2Plx{h-l`_tbiV${()8drZZfrKVu-$<P=c-a|qX>)QwOh5&3lx^HO?bJiJbrfiaemjB6r`?n|G{Fjfv{kOmT`k(xtpZ!-{PkHs{fAGl<-^Ew$wkyuED#@-@!rOJDGQOH9<}xhRv7pL=@t@AGi;EiM*vZN?8(W=O>E~u{nM~orJ7R3gRavfjo((98aWpBc_xQWvNMBt`(Gr-kH2|Te`+i#(WZh5z_NXbI!EII@-lD5~-gxcy%NwNvaci~@yKwA9dpS)<M_ZcTSnRz91!mD<vd@{iGIZvkx41M)$vt3bn}N$;)TJt$t=+AS7a^|_{JGTL{Jh_Ba5$|{a3-x{++YS##$}ZSx<v_=%EElJ02;STX-|^xS*cJACOHBwiQsm>fUtV&r})p=dVli^O3gQxis1)9cL$It3(SHwaT7ghEm|Wk+tw&8zb%yJ5zVJ`Oj{4iA0&L)V~Af$n;MmSMlML(2V>2=W%|x*<|@*}op5;vTC+~CcIx8L=@PBLmNlN5nT|Qh#)dW>m{ZOjaZb4!U^(YPHkOi+C4;WcpxOva1j+pkRj6iXRh7>(E2@GP-Bpz}zmy)U56$3)Y2sdV_~5I%`7+VkaQt`>0x1F9ScDjMR^C<xHLQerg-+r!@?U=a_6{OPa%KXBnEV32@A@~Sqp;sqpZk_|{Z#V->BVwIjlfrxGFy#OWKWtilqoXM=Q=NeYjVKc!wkQkhS%rIuiO>+Ti#e#Iz>BqhqiD>zr+>Up#@WQt@0sw1K3BqL$UQ0$W2U6jo-+hucyRaZf!NqjmRWXNsB=BafTh=Xj$pwv*tcQy^h(Goxt4moSWPV@Q_a18IsSQ|G76_{?djBXq5z#EQJ}s>Idia9i~(8_K)n4WqHZhXi9D??jU9iC%3+jtx#AEJlImu9Hjx=StSzVE4r$ZHa1?j!VwxwO7Sb*fHNHRUGh}9OR!V=_5+e5go?A2dj9^|kMnjz%t2)z@f6U;fm)Ukfx%!f0fbzfgmaO8l$~JCsrh+mG%hNyHF=-UKg!`4*DA5gTOE{?g17*?JKNS_Nhzm}OM2&3*jZbe!x8MtbQ4Z!o$R&IQe9Z~21!3;&xJFflL!NOXlKQ6X`;UK7yQ<sO|hRG)A?gADG<8gpNaS7FK%Dn+UC^<Ji2+3!8NnswO^AE2LvS-QpQn6f|R@k?^X}~w|js6zTf|RYxn-qM*q3CU)?`Y^R~q}r238-X!#Va@WgICfBA(geSH9yt_vjx_&y+C?6=MK)m^2~kO0@ROBt>*(%m60t(h4HKdSm<0NUTfiNm(`_YUx%;|+W;R@Z0`%51&Rz1(G<j|c*a+I*o7@QOHzV}O|COmYF1XZnoBcbd~b*1)Q1QFSRoAcKf$%XY-!dvEc5rY}<s$>j8EnjS3Xt~^e)zUmWKiE;BPYmdZ6g4tj%31*XNFiSvWOb?A10<!p%`$FBN4FhQ&rnn_A0@>Csuo_*r$Jb@H%K2zl!k%3jtf+QO!WAE?b&b1h+fpwR?JRK?*VkFd77g{Sx#h~(GupF%wp|2SfZMS(#s$}b;%enQ&@}WY20En+S3~vz_k6j9fi62%w7pkn_0Iq3zMdgcXO4|2FCOOibZeNp7XuEyt2>*_qeXm?h;iGn1URq@j0Q*hQL<xm?<sajaix;k04$3wwOt&kEQDW{i*v@LI%GHvi6ys=A*Jb0@}}BK4I$8aP;OQm2@sz3;_PWR9Aq+@!3D;pzKvmv=w_tTiF-LlhE!E>&izgnVoVenQ_L47(Jmc+W&9JbkGiNV<h$SD<jrJ)C-5k|*G)0bcj0xsm2L0K2}`+`8yuah)03wCFC0&i0TU@*37x3au9WEWeKa(U^2WZ|iM}b)B9+Z_#?^yBPrW7-^u-Y$h)BWV>;l9)CU<S~Ox94w+FjVG<*`k5N6Rkw(oyPTmpT!2HdSGOArTZEj|x$du*E4jiG+0s7+Gd^HB8ZHWxZ%)z;qxo=YkE{Ht$}|=3Uyiok2McTQ~kEcbYa^SM?xKcrk=`BVx2|44mhq;_qM!F5hi9Qum$!M5u)?O+AV}$SHp=^qe$HnG-ANplsB~c6ybo+B>BYwx@8HweW3!8Aq#WvOS!A>;QmNhF0vK@s<0&45~|y?XfqciX7qyaUp1bg^sfkxTj6P3e9W-uzJhpUt<|vrrpnqq=?kQRaAtD>w2yoivn4<z>xvVdTm}NMxG08(sh*}6EE`^oas%u=dD)9EUm9wkQ$bqlZ(q8WFZd6IM-7~UE@Vo@sVE%?6E7CCv7lTOE_yS(zdA%CNQ8jWM<84(ROrjV@F#m;l0XD)A1?-6CDXyQ`e{EWyCg;4DGyN1@Uc4%?~=ae}8fLO1JA{LX`pjffp~1MqmV?I9`=PNEzNJJ<(Lc-7)-!RC|O1b%vB54bK5Wn7<rN28&0*5ifKFcz{P5=i^B3?k(ZhD`K7=97C=mx2Q(0YFOEP^i8p<EM?2wmg7gzZMdlufo;K=6e<_vHP$n-^y-H3gL;YZr~^dWx}vtfNg>(Z81JI+;D*VQYoNj>xk@!nap=ijM)7GA5NsBxofE4-D_j<+9fH$uj29fI!FGu;x+W~f*Tx~X#`-7a=O}e;fA_<0Jo=jtzx(J1kACvedmsJy!{2-KdmsMsqi;X@(~sW$=>5KmQgs79Of0}Ag)obObO{6kGHzJ8TOeP2GKK3iiJ0QFZg*+MbVZ}*JcE}dMGyKsHQJWKx+wwI%&+3bluAzXrp>{u@@@krolgQ3d7*0^F1P^~{G}OXY{MJpK(baBmMC!=mYOnQU6WXAxCsO<B2jnEE0-2Hgl3D%*@D`n8B^k(Icpr%$*P6?oI3@TNFX%@+uidg&x{39^|qWv4P&w@c%?eW<>caIdl+sgI|z;-!j&(042{mUp@5<b*OZkd6FLJUqV?Gje33S4QcpbyS(jYm*Rd;sDGuTW_Nz`XlRCnV?sm7D0Ej-AFLJzpx`a@wq_zf0EK;a-S5xb;kn8RhO|HjIdWyYi?2FP{ag+0r6PxV;)dM++Lb^rSgmXd%NYc|iI+;l4JkXMvPt$`~ZnN$1Rn2%Gig<wG;A%`MKg{4!rX~UpvNic1|LY$<`QaZu`RUJ}eD~W={_*?2_}RP2)}DOxUw`rN&p-b5pMCoNAAa)3zyHaf{u90ZqlcgT>0dwj%fJ8RgP%Y7=662*$xomBm%sn?{lCF)AAj#ZfBe0_{`5QVKlzidQ&hwM`tBz`{EH{=|LoIm{h%&M`oH+;kDq+!&!7Co-#_`s?-A(#{+%b^`qxkX`Fo%K<R?#l|C>+#@u&Fq!#~Co0^V%=_=6wK;k>}(hu`q_9{s}%?kifrD>}<`bNuK>@FIWo9xTS*;}5={dQ<pjEHbVm4_Pf1tzG#_=7aXm<)&rZOTn3t(Sp?RNHN{(UU21V69l|xg9~B~fF!jLnfRs6mh9SG1_&{+Ne2p>=6$3DO@P2p5YF)UW|$5#8}+Om;TBaJ!8Vnz_w#9(&=RI{9eB!eYNP#F=Xa(?<f$7R3buqqH_i$GlKm(Z&glaQ(3T!X-q4C>)?SGdzy+CC&zl)T1-7N`DsYLKshW8$m{f*ndx(eJL8N!%nCZQEEg679oQ{gycr`j4Fu!#?pMXaqdoaz@4LYYsPX#|-f9C!!w3Un=T&EW=J-7}gm6t^H5+_E}m{U5Yh;$cs{9+EfF;W=HY0l<&GC8Bm^!(pxeSQbhOg@0@jt@}Hfa_y9@8W~E(ZUyb(-b++1FSM5kC6;|_yA6hAE1pzT>}}<(6S5fF{L?@<P(@wIL{21p~FCb`QHEd<zM~HFaGU2Pk!`0Ag=QJ<G=jD$A9?0pM2}b4+_k~;2n$~{REl9<A;9&MC|bg?<Eh^t4IGZ1%B}OgTFoTsBGJ>&$Kl@QE!y&F4M*VZGblDgySS6#zK+kcY-~qBc0;~Tuj_BAReywc61mqgCk%vDGz@6gFpK8{jY!W?$;j-qB%qyMfTz~t{#s7{vWDLKw{Zfl7PM!xYP!<L6P(_$%!a>UnM4vEJ_a8xD~1T&YGQ1f_2EZ`MP8~Jffm*?&u5@70q{tA)|#BwyXBm&MH=KC8V-+x}(MYGH)#+pfHmiQyjwM!6v8KQ5>q$g2hx!K*)~78=#UM@2EuFBuGdCBgv%OZmS3Z&Wr;-A4s(dnY8(GiHzJRva<Th`_oI8sEjX(*6bT7s`pChIrR}8P)K95{?gp|mQPHwe@K@C&UOPm5Hm*o`fkkSOgFpttZU<~?-D{MiOVHpxvB{M>^|mx(tt486J#Yf4s@7l$aMyCg1TxN^2(*>`X2k9YSl<mtsabg<ouO7&=U*lNj=4qWLs8t6+7jLBl&XOVNX@e#34>tM$7E!vLjAZok}im&t=1Em>@LtG<-5`LFQ$ecU8&=){#<>y)!!DusH@6KoJ2bP<gB!v*n%Aw5U@wsBKL^qz^38Rx{|;>}96eN+v?loLbtWTFw}IY?2tGOeQMXv~<>Z>vU^#W?Cmyh{k|SX+#l2D_2(!vK`g)pq@b(8oW63B~NyHtqvJ)P8R1zPe{Vbi5qn`c?N9cW>aa|s42?JHd;R=5&*GY{WD6$wr&#wecvy>GKn1c0`_yEX_It6<m*ey6jS(_GIhu&A5(91wGc$pqCZw|JTC!S*PPd{I4=vHf4Mo|KuDwtI~V9H1%Z_(BfL$m$9E-}bEUpXk~-CW46P#SqRO!vN~PfDTeLr`n!EB%XU$(Wo;RteN7NFSBXHuO>b@jtOXuq7Al|jTQ<-nNgIO+QZkP9p%Nu0sr)$-F%8e`C?dyl?m$YwP4Z5ACOV=-V;bRSY<v#TnyCwwtigW~UrqaVOdlPPjQ7-({qMK4^{Ul==J?s1DJ>;t9M9Gp3QAjQ5;%uL8I85<gsIT6-`{wP}?;w9=Q-o&r<pPHxJ%?@bv_q4a$*XL?9%ygOXQgjG-mDx(<ptAg?z`i45i675rAvx;TNlGSfXq)o=bDA(IP;S{kYCDo$OryWp{G0_Tt@wL!31?yV&_ihJL1I!>vI|6Ck<doosOn!m4Xbmb`k-aA8}F^J61v>_}XV@W@x@_E|#izxV#Z{ggJ8a_!z?1a*41jcQD22<4#nIzii_XCxKJs+=$wH*SrOtJD9~snB>Z1hJ{%PfECIyWz0zc`3q)vsz>crgXXd<qq?BB@)5Zjj5rp$?uSo#(`Aw{bxz6>@TMczs)`O$QmEykU{{0)i=}S{TY%;%za8|#jIIM+hOWIX=Y2V&ShF&zsM>lpDp$5ghZI9mE>Bsdp27(Wb6dE6x?wU)@;4H4^hf&Dbf2)b%;hLGNdWUIo&5-WcSqO|pl>#5YnzCyko3s7`lMkF&g~N30wl&>vom#vsw}Hg4q*-iu{aO+UHS)Z!vvHVgQpCFRHTMxU__a~VhgsgBO_#9ql-N4?wAD`g(P)T$FudlI4|EQhu}>z%4R6OxE|MZ5c9n;;`JMlVd2(sJXpXMYhjz1ZV!`*Q77aF$1y`mLsMSRy;q3k_#w!_riMv>lrn~CHLKc~xf5itb^=dG*AgFh1U6;HpDm^_r7w+;_OhD*P8M+EFs^D|NQB-K00Bp*RW2=)soW(bR^o0SZ~EP;WDKxp8Y?hbR&;KT5K+#vC6L&-pyXJ9RD;O6Isx46JB|`sea4+=$mr?-spw55a8ujibg>+N-EBHzNc&7zXUB<-f)PJ0FQf#@C<{i&q0EhUQ!lZ5oI4lwZB6NfT13x>i|IUvL~oR+5*2nJPFgMS=}?6xXvU3{S`T00m|bZNuoj`iH#+cZ$9%5@x@N>_M70ZnQ<k5GsqNsrt^|;0(g4IS6c&`TaRw?AM9G76NzvXCMT)oYF7h#-XsF^eZF%o=ral9=rhAGQ;4eBeAhdKPG|+@1MAtn;1s$qffjf%C_?QZJxR}tn8MA>4;DS2Tz?9jq5)AlQqt;KeeAe$X$83V}4CF%8M#vQs+yRxBPtoDEhg$E#gk*eP@wfm*2?WcGR7()DDARtcG?>9@&DY<g{M?xF=@1mV91V-0CDA(+&N&Z3g4Z(8?Y9tV3M>SeI-Y5moOpDRABt&Y#)fC`)zTVNkAjkNNZbLWA5L>kw*mMiry_}y&f56Vb9;QM286|->AuCFBzgs@>LBA6`Bo|xBKb~&A{-B@UNkMAX?6{sA+Pf|K4^6<E0rm|lPd=j_*~wu#05#{<upGWWX1U@skt0Y07YX;pD=A}ihNwYzxqme=Zml2d<7YxD>t>I^=-`R%OcTTkO@GQ6rt#bn3)DFmV1<kmSSH3L#}~wj2Hu7Bp-Qf5}^R_DazpWOuTgwO6FoTm=&wsqKd^}IS@5@9y;>w)z@#n65jfcklD}pedmk+ZfBx4yptrJ442<5-GFu%LA{IN(_43(_~g8_;GCs->yAo(h$Zm2RMmRLu{4VCXcA?pMr92FRZ=+x<TEoa0hPiTaTOO|q?9l#v5r_#DUVskJ^sv;$^Tz=ucO@7U(rHD7p71p`>GhH=86>>Ro8igww75-Ft4F~Rw}3^e3C*8?UcW)te)Ow=g{q!H{o{ZWr}H5RZ4eUslvr=Lyi;-@ex;w^0e4GE8zlrVju+vsn%SezJ5ywF<QTwX4zs+>MR(GWS-8}YlC2Mo=g$!(dX`FD&n-}Ws^>sZn>4F6;l{7ZZ%CQBAM2v`IA1i0=Sp4wy3No#tPK*FDop!zh=aBH-j==9hM%&_388>X9OZME?s7V>1WD>l>|@Mvm}wsRGsYEV!*rADb<n5vUD;ZlO*EYGrOl1&*&IUqv-(dAtou5j-|HXhXjq2{fh!=xj{1nI7P+}G5E;a>YqCt=#V}Z7OEEM5&&*f;NyU+gCY&1U7Ha0jtg-iDyAqKfM3u#H6d=DvP129Y1p^(1)z|uyso2OIH~~ZfV~D{)M8!wB8c{kg=h}=dZx+|$Qp9ty^|1Q<Zar@aZHedF`X|a?8-w*%hM~I2b`8rkNx%-$1clhM!e_EsLm}gs`GPccR^3KZbaCJ)z)sch76+2uP)$s^ho@2D0nDc|Dc@U6E+cG@PSvVYNCc^`4SDx*rj}$ym=QJ)36j`8O_fCSD`Sg!Zs8^A|J{IXZG3zr4s8O#mWAdNf-9%c-NfHK3o$5Z%Ql#w1eMzTKlB{qL2-6z$$^7mR@nh5}|o$jVZ=?xwWCAMzl!ZK_=R!b&?!C#`;neJ|R@wwbIZqzE`moR(Xlx=Q6cWPO`9UA0Nx2X^Yo6_AYyCIN+M-h0T}4>{ysWi7^zBse_TQpYt(F3ZC099s9EiNGW<QP_CwS`AR5?fE<vj1`7+)s6vqN*NA>K+N4}<Sb<6P?lLSpdZ|+yW!o;)bs_+TgKBoiyiKjR5b}C8g$s1iR#@2q8c|@^nLDBeMQ!`g0tZkuJrLea>!1u{YQs==WH5Xzrv>dYbwa%Z#Jee)6V90rlV~R4WR$1^91`@dB~(I55=1dp13G{4D$fbl*`<pU2XQiK1utF&qDDt^1UByft^SHFfbs)46zW~4D9W|cA|)aLx(m@{h0enyr&}W^)RpH3s>;(@LlB3;RaQj<IhZfhPleLojRE{{F-R~Vkvj#4AY#a>0mq3TF!PwzE=W*K$K|{^*qc0Y%IE6H3EuHT?Q_(lWS>?Ep$VqeyR7kx*s*3^a7!O-k8J}QnZC&s4x8I|tYze;syVzY7Y>U6=L#^@?Cbz6QHlGXLWof=Q4PkSqF~qYAo@5>@cu1b)OCFfj$O+)aH|dzm;(v#U@v<@_fMdniNO_0t1IIMYbIgq<{VS0=JqPKGWe!^MIh+XSw)Qd>T$e?;-+9K$ir?}q~W&u0_NBm*%0gxjdBX^Yp~gd;A-28ahr;deJB7Kf8tDj0!l&Lo|$5eC15Vrk-=0B`1#LzuRQ+X=a|eH&P%X$NA7a^YVyS-eeL#LI@<_lr1cH&=J@f$@614tH8Atc1NHnGY?(*zQZVx07V@3Y`0ScDc=XRWlaC+%6?{-mJ}q5<6=J<I1=tc@>JE~GSw%}1eSlhIkA4KFJdPJeizN%e0*@d56P*+Jiy4M6K(RTW9#Q?*^xxfsn~y*EyP0<kh%mtjYqKLl(XbmcfS-871+=mE_~Cy7BT2omF#d*--h2G;e?k)v|G@4f5D+wbzzM_e)8x^#YoHB`t-lj&f6m>^o$;f8nc}SdRBnh+vmne9#&Y=hgCBxgnmm5^J};y<K%BV!vbX>E;dci$us%WkzLC>n^61_25+hjo@kc+7z&;CCRANl}&%FbH{SE8|zJ2`g?*KO-YCZ5?;pp`_6#9MwW9DMy5&?C_cMRtZ0R7Q>ROiuq^KynN1~<kqLI#;+I1u#1g^HjD^nld@kp;eBCIUd20Ync&R}o|ifh~yCqRy^&>S;{<2i|YRdz9h&0j}hq0Z?4F2i|Mc&I9i&<-WioCf3658*1hr;4XXMZES3K^mk|ZKYppCno@*ujubbz&iA+ez&kxI3H6!BzfJ8$y(@7^y(#qr{mL-Ps8^-l18>$1DpMvEq-1i>!W_V)({%=q-b;kI_`mzRzbjnZa2RTIw?*Q9hn$78qPN}Vs(#0rVb-JeZUW`uy;XRVkAD34;rD0zt2J9Wqwa22S%m7*+dkZD@#^h+za8Ga^~URW?*&uhOFAtCA9t?V^%_JRJf1T`A=tK)!v&>2#K%!6MkQ)qEaa-ZE1b`5dap}z<X4nnw`y%W!mVy7YP1HqY(6eKjZ?aV*f`}PsCt=eckRh~T6gV{#YoAQRqr@?aBX+4c6G4d)bw_M+w^7@(`CIB8z-TE>}}#uFW{=kt}(kRMif!(O?ee9U-yk3b~&DT`(^Em<u{$0OGOqJJT=f!ho=R`2+s~{zRS+8oWbg(mJCD^M(H>|WhZueair_3L<yHgKP!g`1I(NKCO8+<0~iTrBBgv=rI(&^;(}*hE`~F^)M@ZD8Rvk;iz1l>m?Is(mt#YS9nl0R&#^UiN`U0v;4)J1J=ezYsC06OSAus+l(XXB_1n;j)$KVVb(;uEm5!+2dcll!lB<GUv3$k^P&V>frH*G`fUXSabhkEIro^zmQ~HXulF>3ec)5{-q7j8wcH*g!YD(hC`AtMEGDjeevv#mQN%va*C0YJjIuqMFrc+|8@IoFdTjJ$72e}f+g=3~qaUGm-xWr&WbD$l9@PR8w%5s+&##Z(`Jy6w$(gRk#^??pER6J);+V+#Jxjk-lipU)5azC0QC{$c%(qC-1dQF77l~kO0%g4i|1KxWMvVhKw1h0cWa70E`H$0JchFF^b>!eiO2HMdE(4EyUvi{7DLe+ERIFCg&7jqwxa;4r`a(2n#sQq`J!l-GO7f)&A82!_vI6ALGu9Qa&SCTI|MAnQ6=N>4lhDY~Gt+;;D&E|O2YvOukrsrjcxF{rznXi3xYgs6sio&DY2|!3w430Ax&?TBu??$#y@-3%3ZQ(U1$bUiJ>xJZm4VC9*58ZcEDu{+~Qf;xvzjCYu*+Vh&Ct#u9(PzaCYBMJn(;n62YLKh}#t>!dr1PA<Mbe#?zp=sO(bor@dYD`@+N4YRLJ`6N0GvQ68e!n!C3lDF%|YTgJg$v7wG%vMteVAhe2tZ2y0f}oNm(oFRT8}W60{$y+Blxfd;V=qmYU<~Hpt1D7p3^@F9nv)9kAe-qrlQoJi4ur?{02{n8n~p+yDX2SCH=F8&9Q2f@P$H0dT7Zv0Xk(zaeB0GKvl{o=vWnMNLq*1cB(GOivDh^|L`wFrXPl$b9!%EUp|caIbIUQt^o}sRKc1ky$HfL<)UI%+=cE#)&6!a(QJojPkO)%<nMJ7(xgkZIsdQIc><<zYTS~uVI8&n};ZXP+U3(3ZDBr7YaxR8*+-!4duo2p@?T=1CJYCyen`ZWdn3XW7eC|){Hf5#$Be+PlLSm78%ykwwT47Lri5~ue4<@+ClDsXhRvD5E6Dl{~+wVrIoSDN#b&UZc%a<lz}Qm)MlgtT}J_9=P(fKwh-({%<{K1wy7%U<HhQ5@imVc1DOf-30^UDsHhkjC2+(#Vid-r&VG|f6NZ#4C=`(f<7e|?i26FRZ#;hpNLp9s)>KB;R(tLL0$-i{mj'

PROBE_KIND = 'all_hosts_ip_access_probe_v1'
SAMPLING_METHOD = 'fresh_per_host_reservoir_unique_urls_excluding_previous_samples'
PRIORITY_SHA256 = '1021c7ffbd10629fb3137d0102c8bb43106928d785bca3293125f9a199b55604'
PRIORITY_PAYLOAD = 'c-qZ<%aYqjlHi?Rfl=EUKo!YkrmA|>l&CSaq~4OXT9QdwRlSy#0|QAgiAoY+0iaS@IeFgm9{09pV{@Cs9`-bS*h8=Ff7G9>yN7!q0tkRiNxjyxvPuGor-z4!zvJG$d+#^VdhKm@vvo2bZljCXi{|sVD7<;L*+zL%Wa-44<=Isc=iVxg7I9v9N$S0ef8NDKSq!{vTP9f=t-bg0I!=?_W_0h~z1}j<Hqb^o-{pCnmZRma+~qM;-E6bG^iEefwDmU2)-OKB^Ie(c11~B<>~tNM@j}-x<B#P!nMu8Lw_fwx9>1EG*W0*|r;A^bt^EE81~0!iQMr=eS)rQW!aJkww?>gXmS4qcS0-!yw3}`73<0FyT<f1@veDH_nqZP%sIGST8hRQ5M2lGZgLgk=NjktEi+Gtw7c%ShI-Vm?HMpK-JD4-T2+Bpfb-769Wv}=Bi+Ar|zIp9UJ%5`gS)P>FLm1sG9)5V}_r87c^f?rOuQxpp{`pT~VYj6}@q7f|(W>062hny53m;);M<4OcS0C$dH|s|~PsgL*4elS^r(X~Kf!W~jRg_-r0CTYAUsl6suLr-ZxJsPbHNM+k<bXoMmuUg3p2tH$BdmK2rEYt@9?a28<MJxYFT*&`vpl>^(nT=OqpLN-ApSV-PpG{`Tt>+nW>}PYfUigE>?+QKK7CD=UQ#4!QAX)J#?J$f+Zg5C?Fin+gulh;{rguhus3cS|K%|tZR+_&u~sATKc(SkS(?Ino&t7cyVBA8TonjQRpOI6c?a0m-UH%dqxmSk4rnhk7?)Y@(R1LZ(I$r7nyq)(x|;gH^Dm&_Vv#55GV_*DC0^iL^~)<WkE(o#{k~%ei?pc05Jt7hV6P_y0|1&ch6AJV=j*J1rujPz#k)$%6-)xV+QPhJqed%Wp<bw3s`@sXgr?&1&~{YePg$Y%)v*@F(I<mV?Tq;f2zH!a#354afJySen`Mh@Zd=GS?EFjF5fkqNE%4<4e!x_@%qS`2O%e2OsUnqyYS`A<<am7YwRhjcKj+rQnrGQ%0{sKA+Lly2TL*bOD&i=guYziRGyic=CgnQ*m(_9q$JyCGKRLhu<j3N^YM}5oyQ1`Fu@|S&3{VKZJU;sI82&r~yrLfaR*!kS&Y}hM_=jj+#IgthfFq#kdyl<`$G<xP%wBjif05018$5edZCYV!R1XmUVcB5juQNsOMyijhFyf*0dApA0Mq?(f6Zu{P#T2@Y6jgsM^XrM#b(zO8Oa=>$uqse7I55;ytb!i^&j3J3=skaghem*PM<4M73;d%ata5aOWIgDgrCAvuN)~XcPA0`XPqt;B=2`nvUa#Z+IqcRE_Ph=3PU|p7DDyxN(8niZU0h+g7knSBcQFysftpoJ!+eXUUS21zu0BrlU2N_GLGKYzp1_|khnu8KE}~Kq9$(XYZwjE%JHy8{vyjy4los?c;pLN`5CbD`lOdwCU<j4p#yM=C4XlqxM;Dy6-iJ6(mV8cj)#bjTvV=|)#rb)c<Z%tlEMW9`yXU|d#KkHsfaQRy!7?uAD?9~v0MZaBKq?dkxUT0wN~0V|@hWEeY?96`Y*X)Q1+Tp=lCk^)&gR?G)3+YC4(BMqmiRYp5lVeXvK<l<?sg=@Ft;TTpNis09o}^KcY7W5fpY<tCD32Ff@%gRY6(M81Z(2qDO19ME`<l)Wna-QbvO~&uMq*}abDuzXjiWAZ<)h#hH?7AXMzT+|B(g=j90Y5fx@V;AA2-htT9Set2uaVq;G*}MCS{GWf8A6z>sjP1#CQ91?x7T@{@pQwA(IVQGy!8oMo8h8+iBc-}v9b3J;$mf&g<>Vau)Ef-Doaz>jtr29J~xXxU{5RJw=)oIO^Anx0JsLR#7Q8qOu89=ODU4{|tFG^1eD;5G+FMcFV{QBsD`ChQ|1xE0r^H%|Nkpn#lov{)x;3~~qTOP~?lO#z+4+Q>-`$QUlyi-2mK`M|3C=L6bYVF5cSE!Wpmq*{I2!CYyT6{S@}%{0xfpk+MmN1F_2ah4`?UUVEaeA`4HgA-Ui8I$)_#ZmwP&o-l7njjqt#shEkO<zxIlmhJ!qml?EPGR7U``!?{)XXsT@s&3^_MS}j^i`P(!aQmMUI_36-UH~=IUYUFfPO5ptMq6V%`da%lE{9%D1b(7q9jFP!hDZ6+bu4HO)|)Nb!=PR(2@aF&C*3dUF9I|ECPHz@E)k~6V!@z9B=7;-#jJg)DUt>1AxV!XZgah`bC5iCP=Nj(xjmSkD0Imj~({Vz@p@OCdC$cbr7?%vY6702A=9JTqOC_SHBCtBFpbr5dhI6>6!;ZSP$zNA=jBi7b##&GB4<qM(PbsY=Q6slElDWU22j%i!YM24T<!cPPyDyJQ)TSj!S5tzyT<J4#z2=&QvgMSI&`O3lBRzJI4o@8_b)Y9B_eS7dpntvplEkhnsAH9|&GRn_=Q}I{B70Sb8yl666HxCsY~<;&DTgX?<6>yb3oyc7ihGHcnCKZf_nS7Q(WFG?azNNQ6q=!F@1sWTgcPmINl=#h*jI??9R@J`wRGM4|}$AK3keH3o|22K#g!UBif8gREwKZa`tu0AhnHbu=vqe#Xm4n^dN+%55`PE+DrGa{We&%pwYoJ;T4m5w>*R>`&MN>tuyPU-b{i@FE7*77*{|w(o8;N(vi12wb7G8pOvHQRQz1Y^39UbD#j-`aU#C>!f`Y4mbq9h_s<JfGo{Bi=qM;R}t_yKuu;bj6U{=g8B}4e^3Z~L#R6ultyMrO$y%eFM5YX%>94cYh4oA;^!mK6TyB{CtY(0Oj_Egm9-FXo`O2ag~FXUiu~%X;y;xXY~m)<pbRB+4T9q_i?#g6N$BtLk2cBF+%TWpBA-T6eJTk~Oyx)G77H3m2*#loqKg=KZ{90u40Nto%?;dYQJ{#k1TDN!IMPx~MH6>6`DQ#mA9z{;0ved8D5j{aRjRr$L@iSo9ssLK3pm}U7<AxP0?O%i3j#480&BL#L!rrpG&cB9@LIDbHIg7;OX&C}gl?m8fY*_d>#?MvOyV{UR9JS+On0K@Bu~yNbXv%mkP{Kw__#?C@$=`h+HRoimv>t$^PIV^($!T1BnW5?gX<q!7Lv53fTDjEwkwnr9MW%mVwf0^Q~j&o26VNJNX1d(yE_3T<ha49xA5?HJdUiL<_}26&UneYyRQ^p{Ibz}r4(pjF|@UtlR;?oEGl9&2c#giBp>4E$edcTZY#lAmE|^6C%{w=0)2*Z6H`<ubZS1ksSm7<R-r2G16iNaa)^Ws4`|U0;&0Tc#H1-F*bk#+sUGW^D|pT$IJO;}3V<p^wH|=&Eot}Q#gRU9``(eaymupPX#&f78{P<f7s&;bQVPV|(Zw(K8m&vosLVFNUeB(}xCp2pR82$U(*1xeX(Keq0Za>ObKq@|98SmmJu4%POy$Q2Y<t_RNIsHa0WXVHbn?w_r>=pwv|T4<YQ<?yz8*gucwe9Vj%08^Oz*M&{Mh^Lc-(8-b9S5E6mQbDd;b<x@Bl*eSv!9Q2BFJdJ9TqyEk^th5r1mwWuNhuIOQ$jqh10}-4J-ssjyug8|}03rL*BWInV&I-;0LESXKkS9*jXIcrZS}|2@F}{UiSGq0*^aka#7;z_GTZp#gxvGO^{i;z+rXv#Npfeg^~2vUtUT-}p$0EjGW*;8FU7XJbj>gZkkhKb7b!b$bUPr&hirx$iPj)!~es&q;01K7eObr%5|aVE)GYBdCH%RYD4EI;0f$kBgE3hayH3QobvFi&C^C#pn)FFaw=hp*<!z2;j-1dC8Pf%9*<a`2Z1!6@I8zXp6M5@0v?@k7o-^bg!E=slX5ROfsuZ^(&JxX(55QAXWL^!JPS0q1X+hR%n=G62ok%^f0x;zUv%pV@#-~a3xEh)?YUSqI%#zs%$h>zle*jp}<<$y@TEGlb5-Jrb4r*{gHNm-bHH|AinMo81<(<_lB%r&vkAu*U?2N1Rziy*d=Y!jx(%jj5_oWtb1E0z^w29TJ<&!(b1kQPY6nk<N&D)h{t2(A$#VbtV@$Yuc;83^bX?4MMn`Ttx2f0CccUPq_*$5w%%cL^LIcZx3s}DI;uG?{hOwodpq%N;Ki+P&D?GoPR*oPIn}W>5p~)so|5Vpg;UmLax+)39pVsOqHKOoYpxxf<p_)46Rfv%?&=k5-HR9pwj)jIa<>kJU9FfQ(qX8!0TYUB2TEX6vG3EIqo@t`S^NT=^|hDplnV&kUO~Hb^6?{b2#>VEa~01ow{W_`5daHy?XgN&tI*X@XavGzBP>Sjyn?W%m#T>IUhF7RcZyBnIfWFHivtk-O$0U?m_9)dY}82+E@5b<9ZPM(%3L0Ro{wk9?TBkclmaFu1=_>l{8|D>ho`d!gVooF_v|LhFUgMs$9LrU0=>{4nVQc6ZPMj6sR~u!A+$~`LM|{s(MGKuygxfD6nX#@SuuWI`3`Y8hmpXqmKs{A9&QD$YS<&1u5I;;K9CwwP#Nvx3X@e8i-wR_6IP%T7c9sBFs%uIGG&nf`z`|6wKi~o!0EAgggOa8&NRNF1yR&9%dbZU@#gJ9Ba(@(VS+)R7N>#(un6mH7WnrWZw^Bjb|omf#*xQp@qp#JOn_ZPNTNyS>|C2))uI{8S9Q&zXocOfNn6Q9ug2UFKi^zbT39t8;bP6k>b@v7mQx$EnmB>|Ss1~`q~gG!?Y>YQ^#(DxPiSHdU|`#R?u;ZapC5hq?v=Mm3UEE0uS}e%kBPZ*8iT<BAZam+HTgOR+r;FEIclpLQvx7SdE8gL_Lo}~l+=W>RuoYSE-c+aJ2k3mpi9*T5>9Z2iBK&B<~kup2QLJ)q|g^Mv9Q?+*!QwHuuaSbAyP{Ejtr(8R6{^b&det@IgrCR+Sr}5wLyIvp_64$w=IzA$ed&4toEc~h%kK!Oc56m!vvy|hW+z9fnpUEtD_?MC2qsXn%HKdPD5TME1TtX`1#oL8}F@hO9EaU?4zQJ5z8Jpm}ZZ>3`D;C3MZ<0f^`hUweS`|l-J~h!0T>^dE__=j!4=}7$~2SNnIK>bnsMSK*ZW1%`ZtlP1XyfnpD&+ui3p)4I2#c+Sx4{>D?wrD!|Jk8>&T>1TNthsE4t3Bh}xx87<j3%F*-)ZKxyb&JFO+meI8AYav~#yLutpbJi^^I#FMvPn{IhXn^^zMh|4*d#Hh}vgBUxLLF)_LrNj-v{8$oh7C*^lxnI*N~*SyC8`!tsL5x-x0}A~RYI#}udAJ<^)A(Ob1RPM{|$D-z0axYzzcB4@pkXj%y#T&`(s@?cSvG&27NDcWy@lQ`<yaPh~UKysr?CP%K<#~*Weo*gW+X`{u>T5!Y9Bs+Xm!kho;Pa?{90U_!Q-fc~WV;vtn^<9<Jw@nXkOCLJG~v!zG-7=-=iyrhB+D=l*^1u2P(Lh|1ZiDNps<)YTkZ<aWsF(vtVut%L?<srt)%b%Z-|;aU@Otqsfa%N^}St()M^iik?GY3DUWawL~Gk^A(rB&bGV*+`j8@C`FCXr{4OpjDruq|kxt&C-%6u}>*U00r4=0geK!279v7bUY>H$q^$w>Y&)H3X&Z-6D&vtFfWzGHO<oDGAiL<XT<Z+4H?8gR1X$f(;=!faAml%B#lY|T^Tya0|9ABC4i6}GX;LnwnU2tNE$T^-)hgP1~GmIP68mb7b112=#)wfF^6VJCo(p(EF9gqBWn_lS%QR_#yCaQY?Qq99gT~#25W_7=;pX&SbESdk&Gs;U;_V{T5y1_uE`4*(u%1{XXkeSvM(BLqlccsY*jsBR0!Acb)-i#2nl)Z>yQeY7)<_Hy&gI5c%=P(n`ale#Wm`_g{YNM)Ndg3V$|XWbjd&`*Vmwm-)&%nAYZWMn#7z4<`Mu9HZx=y;tG3l&x9c;M9VM)hwt@rz+R@X(V*aBx5iDn&Mx4@1yVM(1zGgXLlvz4-j_AT!2&S!2aILeCrDL=wO-Cc%8T47HW*(3MKHJ>9PUVu79Wfu({t70v~yZ^lR7>Y`AU4jQ0FIDBsj@&Iq2QkuB?za#$Ctpq(N=Of|Q4sRIAro44fk_(f?0^VMk!!yo5s40xRserUf>Zk0ue2QggA}Yzwe(Dk!01(gd!rK_XxDi!R%huVF3K4B&G_qkK_M-K{aepo;RYHsRfAq4w)Ax0<MbJgg~2Q(W!Pm;Np8qyo;CFe>IrqMX>#X20GQE5j^OEu)QBHkhl$WT$(zSF6+Qz@tbJ9XuJ8g}ixK+WH!eU^V@Ny-&WHddJQ00?#w`<GiI<9sFkv@@=Zqz#eo7u@qpt-_x8rrNx*8&FwiA`c0p1gh^9KF-byW2zT37WE@71dc6|<Ax^E#O{0Qe=EQH%h>$GI52T7cq)p|)tm_eTKafqQof!^l3Qu#hEsN_IPz+&Sm<8|!q$L3AcHm%Zm|-+k$C*1dCwsaDRZyLOSc3*5F==Nn6owt%V52RwnY+*%VM^^?{GS?!CY)}>Yt~N1X^oh->=AwJ7)Oz^z19z+E>hDR=aVMAJYWPWt{Zu)y_Qq3n4(iAx-xw;V88R@k#;F7rVp&Bre23@d2lj@ieG;->PyVi^!OpZf*`aIY)MmO{RZxUr(RpUND-=ZAcd~oH1EQkQWiY96oMA5j)V(to(_dX!&+<RFd#nfL!H0-fP5WkN66?GFxV<kbsN}zP{mEBz$#w7&iWOjN@PK;9fOz@Z6`Pw9Hyz`9Z<}?R(WD_5>R*|29`7xHGahLY6ACb5o+Dz=WGLN0vtRd!&m*#7nzAMfuQD#ZU)JL^q>b_y?2I>^Q`A99DG%lwRK9={E?hve?((v=VYpQDTDt89bd8_QJ6#*_MtN-(u6GS7n*>In0oAuySjZ&g&a`BPSf}zszHu7jwMV|dI7>v{#wUVsr1+iF=`G*%C!q1*c$dJBa%?VN}cux^(Nqm5+oiR-=*n?{6H;MU~g$`AFMw8rr!YURdl$ALx5A#DBU9D9T4Sm6<vg`5FzS{gOJ=J!q9PU`CPiIy1c2H0Zu{VnH5;)=s*lv6HiCx`DVl-vOa%{m_^@)DRg7S(P0XtbeUYVb*1QKn=6H*JIO6FQC<f^nNTOJVss}u;8h~SiYhb=(7-1QL21Gr*yL}oAtMVpj9eEZ#ssJaX);QTROS!zsJ%Tz_br)=6vTM?;yNR41C}|XGW8$O1d?FE)voRJmfP^C4UbphP*0C`)qcm=26|weOF%g)k$c7+Tr&<<pPIUn4yTNRT`O!?2JxDZC#7+}qg#c0ZJueDt}d^NrmLfPep%|_E^2MSR<8yBI-79$b4e_rR9RD7DHfO!(8vmK^}fN?s1aO%1K*ZXQB*0t0<u0i)}C82<Y{-70t>dSX$qtiOJ-GgfEv>oHh%7c8sv@HTI!q^C?)FX^Cd^Lhd+TI6SZY5?{B;}cpT2@#NwHOnIG`w0w#?*X0AOU?`>vTcyWLb!r}v9cNF=$O4!F6J-jJXgkgfKG-PATNLE3UhSp@|VRC47MuFUcmX44KO6Tz9G7jt)wrae%{b(`UhFe*Eo|^e?&QX7|)h#B%kJ<{fs9ac$TxpB6P*ugL?$Qt?0OgX3EHsM)>c}%v{(%i<;7A=vtd)pG{FkisnDNF4rRNlcKwt602y*#>pL0;*>_B0LN;X43HMSJmq<r3<&k3ukPOzSt&M@G<U5+I51wAN$2pBvx`y~G!!sA*(0)FEO04PZzrskkvq=E**aCoFPEqKkq$v_*ZkkD}~i<`LC=qaY3;3yEu=dl4kSjorgB+fPfIog4hyDhCMx>xGmYqw1Mi#AsW&=^jwV49;yvaU5@2lhMTeKq>Bly6z-&e}MO#-VE4M;l(Rjid3PX7o9H&Z+>gpn^3zMt*Wb9^WdjW`|CI6R5HLYzgqa!|916GdWa{udJ8v6wd(e@W22AW~ii6U`U1G0qUV0YX=9`+uiI>Nau-tT3=(zK#bsF?*wi65!(&VN=r;LYdbgjgc`g+F35{YV#L~izrtFJegpd^G1|;5WOGmyDxs!6O6Rj=l&&|UG+B)<vJbwShisbv&>>hIm$*`B_q4c6--fBh#+tQA8gwoLhylEu0~Od&;T?%E_DQzaSKJ)sJ}JR0t6VA7i>?KfT((1nLkEfjGyw6f=#UM}5e;NG)|xY{I@SrnJEt@R;KUP^+O2zBll^r<jXlytvh=!;>>oOiY|`!5`S&HKd{H;>FDtKfP$gO8CsQdhIV>*Io%P9)u1SKL*!`uxeY>M<%}h_2VF_8fdINSsDKo-&-#OKuqpBVhHSQc{Jz3O2xuEWlN~zj(^2_k&cIe*5hTiRXpTg~+Xw^GrYijY1bH%@k2O(s@R4Hz#Ri#G7P=dn~K6U1txnmIFAqFa5_PeBya>rg81e^)CBm7b}swD)q1bhtGBL}d%K;}6R<)AyR!x=I{dh{eEWjMJ#q82nQq9tH6q-q9q_WS)k1X_*~5&11rp<PFjgPZzyAPXck*RW=4PNIoZn6<EPd(hR7Ip&e2uJ~(eKLNES6=zC}r!pF{Wn6?BvSXyfh_aZ+$h~UcJ77$$Z1%PbL(59q4t4HzHT>faS-HmfuTS%2xi>bb=}cx+xQjm-57>@Qx>M_Sa|vvVS^7Ko3RG@)#^68E^G-bDY)I6-2c%aijjJQ5+*}-8odV>gj_I`0ccE3yAEmunRlwmmsFL|Zme7!Qti40I@SfdlE`$y_M!F!4hRn_eS}0~26@+0p?71`Kui&Y2b%i9RRRViR61S$T!b-%6qxmXi-4JkKp}6BxzTx%S?>1N}pnAW5_vVN9FW!ZxFW#Sqe|q!$g(-mIP*8TmQ8*ZVmG2XMl2NM46*~&T5Ha#O<ttQkyk$tD0_kAmf#wUT2bDI~8LA~=c@C9f;~=SF&;DcXvyQbrgHe*3r10eS={Go)F5-_qNzP+)@Onvbfr+HJFqh(PbkRN7Lfku*X^1blUM?-Wrl(kd(VZHWg!W7Wf(?KHbjgL0Z3lklm~9|2j=<-c%IN_z?2c|9fjUPCmD34w3Z;`pZH&rE%U$^e>vGhYCTC$)&B8YfsK%v}flb<VrCbE+lQ$a5Zbg9g&wTy_NND`-%r*bV_?mxV3L<x+`G!!68LBbmvg$$Mhk9GfA&Q7$7TUyM@LX`X?rW7LZ6pu@O;`+S9u>XINwL*ofWa!N&T|3oWM7c8Y7?zcMoaqYi^ed`QnbD=7YfS-DF_?&bu)pqR7t?JSNmt<bF5Vr)cA#;8hETwK~Py<SjdcShq~n1@c8^p_WF5WL@R|*+RR;J^ni~8wR3DAABXb{juTyy-8ixjH4nO4(wX1$jYxL#8P^e71tc}-AhK-`&Jho(-3b-|u_u~1?@tjp)YJ0080vbOq1GWO<zX}f;7(@iD82NBk*YD5+6QvdM(oq*1Ln8Gyo*;+j@iJQ;bgU$R5ip)p%Psk&7HTxL~&4>X2HZy^S}Oky7Csg>tFx!UrX<D_1FIe+R62=|M73v-v9Y;|M!3X`j>y5d+D!#`R@sofiJVa{*MGQM*aF9|GOO7_ps;)7zAexC&|+Ttx*v9mBX$Vs-rU5gLq6YL4Sp2nmAN0F_h#Ho4%oCnrK=(qGwQ;11D&_r6BWvvY$r@-0F~@LP2i=Bwq3-84Y+k#FH~uNfyI_evb@i?P3=riI=*##YR{AwhZ<dsa<K|P*)J2dC&zKG;%Dh?s)L7vmJ#$t#**QM3=BA3;EmOOVo`We=~08u&ON@t6VB;idv<bq4qu+?5j(Y2A-LuzMjI5g0e(1Mt4xi$YW$`)t8>BwzN6VQTHmIG13&fEmj)s5~=<}JbO;xet@C$4#=?Ft^1Qc{jMbNJWp{8b0jFQ>9<D2vXSz}tz44n#iiOlCX@s7p;3j%j7h6pC7`{FK#&cMg+{3c6Jj*nGFs*~>MQ5eVRex%<Bl?wJJmIgI-R%zJ{6p*(Qq2{yXIkGMj-fXVr(BSj(}_If+pqKJ6HE46<JhQH-OnMkqnepe%(fLD^3Z?^JOcu;#;k=OG?m#5$h~$OoOz}7u#^YS%4oGvuS_8qNfnvthl5z#ShVobAG@d$-+8T)Z9!4;+$^fa4_GUC|8|h)meLP@aDZ*1`%$IR)v$FfHo3D4B*Pj<?x}uC+u49<xdT}{(=#{_M(5GRInJMu4k-;ejUEv5^o;eFx|QxZPSZtwmdVe*~gn}@L1hkf=wz=4+R_2$?z0rm$euL;CzINL42<<i~#qz(RSaxcj{sHL>=JC0L?-1@YOwzk30%o0T+ISE>8UWgjn`w+;(>YLOny?Jgl!rHG`Xtx3d2=uY-{G6thbiZt+(~)|#7JXI7fM88X#^+e$XHkNu>Ssa<+*GY959rB!SdM~t+X_Zs#kXVfq&s{|6Z!tH$0nkBYYP^2?gAD@*3QS;z`KN%2>1pVzf3su0sQz`i=y7lU6Be}Y+U9Z(C0JREwZPHF7>|$@yFDqwuz4kW;XAXnR+7i6PhEscU>oc7UynQ2H91~N64+Y&bAa@IMQV$P(CqTwfI@u#|)XU*nu8B2lG*+Cr>L1hnk`76%*)C5PD$raN9r7&VLsjgcylYskYyTSEI)XfdzPm*wiI2#4AjtiH4*qG<a(CvxkRu*SS0FRx0#&$l^N_GjV{l4Zq(N0LZM{O>qJf9a=%srCl83Hpt33-{&-5k8FmBvfRzWE%KLtZgpDmYi8L1!@MQ>G&@aq9($`P?14UWAkNtjA416R`K3YQ!hbYmD=ya(j1gxNRX;e;Oe%$Rt`w>0PG(pM3uV}}i!reLm#%Z}uVngJ#vFO?<fV7zNKGenVkUp0nVvNWNu?owrMJ_~cmdd%vknhCiLJ+&fABSe<h)<>`$;x6qEOwJu@wC|t337>!W=9#*#K}*2gQbqoy)9x0m>(1zKX*YxL<lXKymRgMm<JO-2@!L1A|8sc2G|WoRzWwfL`0U%K|D}D<g4{+~+xF9Ht0DhGxpz&ONo#}bJA1XAU6m>nJ%{itw0vEwqvIt8CBl96Y=-roQ6*rW>4H7DVLgXTBS0LHdx4wDXA#~W{R))oAN5sw9(uPGW;Wo4-zAI)^)2Mf=#FBdNO7RI#HZA+&cZ-<HO?3lF3m5|6oQF!&u>jrvDuzw9b4mgZ59K^Q*qUJ8Dpw4<LU7okY!BoMFnCBLQ<{wo929ct*(XEaFx5{iQ2&HsJt8=kH(`S8;xOx0l$M+!*TiBujzi2>WvxlF12TQ0<Pl2lV&6Q>7>z?`JvuqS`-hUA!!C5fx-{?+cy8&`UubX)oS5lw!x7cODT9h)=hl8oUwM#9ISpjZnXAi6qnKUH?|u6s9#rY|9KIiWzH_A-W@mFT_zvF<N{?M=(6;xn(m+R#?Ty#oaiF->ZH*c*xG@Wwu-7(CkEivi#ovP7#m?$#1}?2daPI4f*UQkUnP0gz#jY4u!87qbx4a7hQ>?Nt3pVFHcX0fQkAAedAef#K5mCw-5li@xTfg&t5*RxQA^;d<R-pSt=Rw%NCdD(8@%7Rn3`oZ3CCP}CQXp<DyBoJtGK1cloAa(dz2C00XPQ>Hl?eCP?BUlf_tKY<@Aw?Vw5OeQPB86Rea}-9`IQXa4Oa2Hv3M|B6gejlTFt3oF92Sjw9Wzw*%=3WCUb>!IcxdS`gvX-RL~Yem#OIh8<$#g@i<r%{1O6i-0V&V@J}4-6Cs12VeTmOW=sW!`H4mAnm#H(dL?dmE=3mD32=L0s_L}VvQk)*pDmKW;J0f0nizywX1;Orfk?~=|)}nLRJC*1-~75&tLrh<<r;UAKtxrefr|{b9C)ZF?=CcQG5XKSI@$yuU<`kIO2P@MjIt;a02FoFo5KqB&`-Y?9Js#EiNn&nZ=0^F*q#XiG1Z5trQUN1W7E6l5bzWeL?R*3NzooKYjk@yVJvOWJLB^^na*>g%z3lqE;gbj}$|2^3G%b@MZz3Qv=?E`W27O^;1!De0Vc^w1|<2z_{=}ZqPe}JU}UG{m5?)mtx?ckQ)i%gJKc_Up7t?(0$xkke~s*#@J&;L7t@%{_NZXTwc*PRAwK(8LS<EB%81((s>afGQw*e6Sx<E*f}|`p`?k9XGHRRrE@~UG5q;w)K)?Qlp4`s_N_@aYDbEbuu3^@#X5S<xbnFhDw?=8x5F6`Kn$@bFK&Mz?Vq=9i@RP&+PaH6cgJ0OX>6wUP4Zyel1dmD8ZtMM4+vNlqntD)&*c5&$KrnQ<NL4r(7J}HjW2CEf~(>c3ai9J6p^8Oww^{5E*cywPanvo0-7(sJ)N-Nmv9IC+<q%zGYr`sOVYKwUT1VZ3szRBhPaB`%E{*#TcOBMze=gg<ZbP+b!;|%q}@h-mcwD14f*X5WkSdrnNohB`_k(9pwbN0r6GrmbxxjA>Y*V$38$nkWjycTtu*~|R&?cx?4VwFenak75L!DpmX@w8t6EbAB>Sy@sPbNU6WGdoPIP#f*1rD*WlSBh`&mudqN?Q4!Y~1?*tcV5@0$_$9NEez>kJHKka0>9QdNEkQ%sT7o`mWU{YBH!Lwq{4zelGph-!O;cOxY8Sn9r4k7{J>X)>)^94Rz>qrNa|bGZ>X(S~x!2e1<{9S$;+aKG}E-(3_^5hDr@7x9N-vDyrG<e=GICyzIvet{~p-354`Tnx9#c3&Ok{V>Fg^3&s!hvUKT#^cJLty;zk$J<)MvO~4xxb5mU38QzQp^G8H4V^zY@%J|k6s!Q!Kg&K=L$eU7*@Su@%Du7d8Ddqk0aLljKtaIm3-lQ9V|bhURzdWsnMd1bmXHD0zsP|112zvKjz$-9Y_%K}fDQrwE}?D}?|<gMe-S==`s~{m;oGOD-)jFAM0+-v;}#uhEhdYC)kP4_7EfRP=>>2Rih2Q)=5r*gLhGquG$rkYp0>!zXp1_tz>M6)`?hf$TjYtu%PWZWBXWQiHc6mp@HEffy?A>1B7FTO{Nd&6=Wl+vZJ@$bqk>XwiodNIgk!e_Jeeu~9mwKL`LDx+>eD@<8s0DBjjW79H3#ASq}E`)gxua(Al4<Tj9)^c1n*EVF;)NHbIICb*{Qm^9VOXLnt^^5Dp4M|Mso@|y2isL_~shl1T1F9G9C?>)gc6nv|~SD$x*dSL1@1er9;zOgePH;`SzPWdGH|Zn`tuCX<2hUh5fuvE>>mzgu5?@rj_R3D_=^{m6N7fa)1<)=m@8dYIBSL(AHK|GgNj~!Fe5n=D=d|RdFA{`q(rF>Gbg<Cur^4GI&g8$0yX#P|y4Ek$U|Yy|bS1<cK4Bd_=E&m&1;_3>GPTtvUZ-woX@}8%05DqRh02K%;IR$-H;bZ!DzVaaCel=J3eTwr*v(J<x08)5p<sq`1aGX{J%9e%<TsfgX=tv$)GZ8(XkrsE#^ceB1}2ih|iA)MQ`?x#wbTl^|I=<N&1<TL?qyIn&!{<aYYDGAS{0%(_`)Go8#Q@yNEr>GT-iI1<#JxYEBi^S$Dg7_hH^Pgz6IMz*G)YT;5fEknb8mOk<nv!JfZD3BjVxtzeJX&DF0bsGL8@r+IpTTB0<6Tw<n1~kL}Bs!##IJW)P03-Qd(a0;Lqf6_ek$qiIYg(c1jCF?nxHDs&+~-)&MW+t9FLFlIa->c7q%yWuK^Bf$Pc>w1@>F}NsJJoS{C^=XcVO8Z)Q}$fccTxj3F2cFS9fs!ZUY&trQeg!@PKFr`L_bB&gk9@helQ|4MdMg%+(VdSEPq7TWfQeG+a~<3lM3=s5_`ZErRI`FtGzK9AFT|;mzfwcFisOYVakuhz6Hc<TLpQq3?*g@t0yk(LJ@*nB{brG-x7<N&;xNCep3Bq@^y_alFNxTLC99IRO??$JlgTs$N^OV#;9A090{nQzUHssk6PmiTmjt)Mj_T|Lsm$9UXNWi5(qf+>2xeR(GcIXs|5qY<x|s^#1_$RE`J'


import base64
import hashlib
import importlib.util
import zlib

def sha256(body):
    return hashlib.sha256(body).hexdigest()

crawler_bytes = zlib.decompress(base64.b85decode(CRAWLER_PAYLOAD))
if sha256(crawler_bytes) != CRAWLER_SHA256:
    raise ValueError('Snapshot crawler bị hỏng.')
if (len(SAMPLES) != EXPECTED_URL_COUNT or len({r['domain'] for r in SAMPLES}) != EXPECTED_DOMAIN_COUNT
        or len({r['url'] for r in SAMPLES}) != len(SAMPLES) or len({r['id'] for r in SAMPLES}) != len(SAMPLES)):
    raise ValueError('Danh sách mẫu thiếu hoặc trùng ID/URL.')
# Runtime filtering is limited to the freshly embedded URLs; no source download is needed.
if SAMPLE_SEED != EMBEDDED_SAMPLE_SEED:
    raise ValueError('Mẫu đã nhúng theo seed cố định; build lại notebook để đổi SAMPLE_SEED.')
ALL_SAMPLES = SAMPLES
TARGET_DOMAINS = tuple(sorted(SOURCE_PROFILE['host_counts'])) if TARGET_DOMAINS is None else tuple(TARGET_DOMAINS)
if (not TARGET_DOMAINS or len(set(TARGET_DOMAINS)) != len(TARGET_DOMAINS)
        or set(TARGET_DOMAINS) - set(SOURCE_PROFILE['host_counts'])):
    raise ValueError('TARGET_DOMAINS phải thuộc danh sách hostname trong corpus.')
if type(SAMPLES_PER_DOMAIN) is not int or not 1 <= SAMPLES_PER_DOMAIN <= 20:
    raise ValueError('Notebook nhúng tối đa 20 URL/hostname; SAMPLES_PER_DOMAIN phải từ 1 đến 20.')
SAMPLES = [r for r in ALL_SAMPLES if r['domain'] in TARGET_DOMAINS and r['sample_index'] < SAMPLES_PER_DOMAIN]
NETWORK_CHECK_DOMAINS = TARGET_DOMAINS
SOURCE_PROFILE = json.loads(json.dumps(SOURCE_PROFILE))
SOURCE_PROFILE['sample_host_counts'] = {domain: sum(r['domain'] == domain for r in SAMPLES)
                                      for domain in SOURCE_PROFILE['host_counts']}
EXPECTED_DOMAIN_COUNT = len(TARGET_DOMAINS)
EXPECTED_URL_COUNT = len(SAMPLES)
BASELINE_DOMAINS = {domain: BASELINE_DOMAINS[domain] for domain in TARGET_DOMAINS}
if any(SOURCE_PROFILE['sample_host_counts'][domain] != min(
        SAMPLES_PER_DOMAIN, SOURCE_PROFILE['available_host_counts'][domain]) for domain in TARGET_DOMAINS):
    raise ValueError('Số mẫu không khớp số URL duy nhất có sẵn.')
sample_bytes = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in SAMPLES).encode('utf-8')
manifest_path = RUN_DIR / 'manifest.json'
sample_path = RUN_DIR / 'sample.jsonl'
crawler_path = RUN_DIR / 'crawler_snapshot.py'
if manifest_path.exists():
    saved = json.loads(manifest_path.read_text(encoding='utf-8'))
    if (saved.get('probe_kind') != PROBE_KIND
            or saved.get('sample_sha256') != sha256(sample_bytes)
            or saved.get('source_sha256') != SOURCE_SHA256):
        raise ValueError('Run cũ không khớp mẫu của notebook này.')
    if not sample_path.exists() or sample_path.read_bytes() != sample_bytes:
        raise ValueError('Danh sách mẫu của run cũ đã thay đổi.')
    if not crawler_path.exists() or sha256(crawler_path.read_bytes()) != CRAWLER_SHA256:
        raise ValueError('Snapshot khác run cũ; hãy tạo run mới.')
    if json.loads((RUN_DIR / 'source_profile.json').read_text(encoding='utf-8')) != SOURCE_PROFILE:
        raise ValueError('Source profile của run cũ đã thay đổi.')
else:
    sample_path.write_bytes(sample_bytes)
    crawler_path.write_bytes(crawler_bytes)
    (RUN_DIR / 'source_profile.json').write_text(
        json.dumps(SOURCE_PROFILE, ensure_ascii=False, indent=2), encoding='utf-8')
    manifest_path.write_text(json.dumps({
        'created_at': datetime.now(timezone.utc).isoformat(),
        'probe_kind': PROBE_KIND, 'dataset': 'AIGuruTinix/ViBioMIR',
        'source': SOURCE_URL, 'source_file': SOURCE_FILE, 'source_revision': SOURCE_REVISION,
        'source_sha256': SOURCE_SHA256, 'source_rows': SOURCE_PROFILE['rows'],
        'sampling': SAMPLING_METHOD, 'seed': SAMPLE_SEED, 'selected': len(SAMPLES),
        'target_domains': list(TARGET_DOMAINS), 'samples_per_domain': SAMPLES_PER_DOMAIN,
        'selected_domains': EXPECTED_DOMAIN_COUNT,
        'baseline_archive': BASELINE_ARCHIVE, 'baseline_archive_sha256': BASELINE_ARCHIVE_SHA256,
        'sample_sha256': sha256(sample_bytes),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

spec = importlib.util.spec_from_file_location('domain_probe_crawler', crawler_path)
crawler = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = crawler
spec.loader.exec_module(crawler)
priority_path = RUN_DIR / 'priority_probe_snapshot.py'
priority_bytes = zlib.decompress(base64.b85decode(PRIORITY_PAYLOAD))
if sha256(priority_bytes) != PRIORITY_SHA256:
    raise ValueError('Priority probe snapshot bị hỏng.')
if priority_path.exists() and sha256(priority_path.read_bytes()) != PRIORITY_SHA256:
    raise ValueError('Priority probe khác run cũ; tạo run mới.')
priority_path.write_bytes(priority_bytes)
priority_spec = importlib.util.spec_from_file_location('domain_priority_probe', priority_path)
priority_probe = importlib.util.module_from_spec(priority_spec)
sys.modules[priority_spec.name] = priority_probe
priority_spec.loader.exec_module(priority_probe)
if not isinstance(USER_AGENT, str) or not USER_AGENT.strip() or any(c in USER_AGENT for c in ('\r', '\n')):
    raise ValueError('USER_AGENT phải là chuỗi header hợp lệ.')
crawler.UA = USER_AGENT
crawler.ROBOT_AGENT = USER_AGENT.split('/', 1)[0]
(RUN_DIR / 'baseline_domains.json').write_text(json.dumps(BASELINE_DOMAINS, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Đã nạp {len(SAMPLES)} URL mẫu và crawler {crawler.VERSION}.')


ip_path = RUN_DIR / 'ip_diagnostics_snapshot.py'
ip_bytes = zlib.decompress(base64.b85decode(IP_DIAGNOSTICS_PAYLOAD))
if sha256(ip_bytes) != IP_DIAGNOSTICS_SHA256:
    raise ValueError('IP diagnostics snapshot bị hỏng.')
if ip_path.exists() and sha256(ip_path.read_bytes()) != IP_DIAGNOSTICS_SHA256:
    raise ValueError('IP diagnostics khác run cũ; tạo run mới.')
ip_path.write_bytes(ip_bytes)
ip_spec = importlib.util.spec_from_file_location('domain_ip_diagnostics', ip_path)
IP_DIAGNOSTICS = importlib.util.module_from_spec(ip_spec)
sys.modules[ip_spec.name] = IP_DIAGNOSTICS
ip_spec.loader.exec_module(IP_DIAGNOSTICS)


## 4. Chuẩn bị bảng kết quả và phân loại lỗi

Output riêng: `vao_duoc.csv`, `khong_vao_duoc.csv`, `chua_chay.csv`,
`loi_theo_loai.csv`, `can_xem_noi_dung.csv`; bảng tổng là `ket_qua.csv`.
Các CSV dùng UTF-8 BOM để mở được tiếng Việt trong Excel.
`error_detail` giữ nguyên lỗi kỹ thuật; `reason` diễn giải tiếng Việt.
Mở `ket_qua.html` để lọc theo domain hoặc trạng thái.
`uu_tien_fix_domain.csv/html` xếp hostname còn lỗi theo số URL dataset giảm dần.
`tong_hop_domain.csv` gồm cả hostname đã phục hồi; `WEB_ERRORS.md` là báo cáo lỗi chi tiết.


In [ ]:
import csv
import html
import shutil
from collections import Counter
from IPython.display import HTML, FileLink, display

STATUS_REASONS = {
    'robots_unknown': 'Không xác định được chính sách robots.txt; crawler dừng trước khi tải trang.',
    'robots_denied': 'robots.txt trả HTTP 401/403; crawler dừng theo chính sách robots.',
    'robots_disallowed': 'robots.txt không cho phép crawler tải URL này.',
    'http_error': 'Trang đích trả HTTP ngoài khoảng 200–299.',
    'network_error': 'Không tải được trang do lỗi kết nối sau retry.',
    'unsafe_address': 'Địa chỉ đích không phải IP public được crawler cho phép.',
    'invalid_url': 'URL không hợp lệ hoặc dùng cổng ngoài 80/443.',
    'oversized': 'Phản hồi vượt giới hạn kích thước của crawler.',
    'response_deadline': 'Tải phản hồi vượt giới hạn thời gian tổng.',
    'rate_limited_deferred': 'Server yêu cầu chờ Retry-After quá lâu; crawler hoãn URL.',
    'redirect_limit': 'Vượt số lần redirect cho phép.',
    'unsupported_content_type': 'Đã tải phản hồi nhưng định dạng không được hỗ trợ.',
    'unexpected_error': 'Lỗi ngoài dự kiến; xem error_detail.',
}

def classify_error(status, detail, http_status):
    lower = detail.lower()
    if status == 'http_error':
        return f'page_http_{http_status}'
    if status == 'robots_disallowed':
        return 'robots_disallowed'
    if status.startswith('robots_') and 'robots http ' in lower:
        return 'robots_http_' + lower.split('robots http ', 1)[1].split()[0]
    if 'html instead of robots.txt' in lower:
        return 'robots_html_instead_of_txt'
    prefix = 'robots_' if status.startswith('robots_') else ''
    if 'waiting for domain lock' in lower:
        return prefix + 'domain_lock_budget'
    if 'request budget exhausted' in lower or 'retry budget exhausted' in lower:
        return prefix + 'budget_exhausted'
    if 'winerror 10013' in lower or 'network is unreachable' in lower:
        return prefix + 'environment_network_blocked'
    if any(s in lower for s in ('timed out', 'timeout')):
        return prefix + 'timeout'
    if any(s in lower for s in ('sslerror', 'certificate verify failed', 'ssl:')):
        return prefix + 'ssl_error'
    if any(s in lower for s in ('nameresolutionerror', 'getaddrinfo failed', 'name or service not known', 'failed to resolve')):
        return prefix + 'dns_error'
    if any(s in lower for s in ('connection refused', 'failed to establish a new connection', 'connection reset')):
        return prefix + 'connection_error'
    return status

def result_rows():
    rows = []
    for sample in sorted(SAMPLES, key=lambda s: (-SOURCE_PROFILE['host_counts'][s['domain']], s['domain'], s['sample_index'])):
        record_path = RUN_DIR / 'records' / f"{sample['id']}.json"
        record = json.loads(record_path.read_text(encoding='utf-8')) if record_path.exists() else {}
        if record and (record.get('id'), record.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Checkpoint không khớp ID/URL nguồn.')
        status = record.get('status', 'pending')
        page_http = record.get('http_status')
        robots = record.get('robots', {})
        # Robots failure may happen before fetch() attaches its evidence to the record.
        if status.startswith('robots_') and not robots:
            from urllib.parse import urlsplit
            target = record.get('requests', [{}])[-1].get('url', sample['url']) if record.get('requests') else sample['url']
            if record.get('redirects'):
                target = record['redirects'][-1]['to']
            p = urlsplit(target)
            robot_file = RUN_DIR / 'robots' / (sha256(f'{p.scheme}://{p.netloc}'.encode()) + '.json')
            if robot_file.exists():
                robots = json.loads(robot_file.read_text(encoding='utf-8'))
        detail = record.get('error', '')
        browser_file = RUN_DIR / 'browser_records' / f"{sample['id']}.json"
        browser = json.loads(browser_file.read_text(encoding='utf-8')) if browser_file.exists() else {}
        if browser and (browser.get('id'), browser.get('url')) != (sample['id'], sample['url']):
            raise ValueError('Browser checkpoint không khớp ID/URL.')
        requests_access = ('CHUA_CHAY' if status == 'pending' else
                  'VAO_DUOC' if page_http is not None and 200 <= page_http < 300 else 'KHONG_VAO_DUOC')
        if DIRECT_PUBLIC_PROBE:
            access, effective_engine, effective = priority_probe.access_outcome(record, browser)
            if record.get('challenge_detected'):
                requests_access = 'KHONG_VAO_DUOC'
        else:
            access, effective_engine, effective = requests_access, 'requests', record
        flags = record.get('quality_flags', [])
        if status == 'skipped_selenium_first':
            requests_access = 'CHUA_CHAY'
            reason, error_type, stage = 'Bỏ Requests; dùng Selenium trực tiếp theo cấu hình.', '', 'browser_primary'
        elif status == 'pending':
            reason, error_type, stage = 'Chưa chạy URL này.', '', 'pending'
        elif status == 'extracted':
            stage = 'done'
            error_type = ('qc_' + record['quality_status'] if record.get('quality_status') in ('review', 'rejected') else '')
            reason = ('Tải được và nội dung đạt QC tự động.' if record.get('quality_status') == 'accepted'
                      else 'Tải được, nhưng nội dung cần xem lại hoặc bị QC loại: ' + ', '.join(flags))
        else:
            stage = ('robots' if status.startswith('robots_') else 'extraction' if access == 'VAO_DUOC' else 'fetch')
            error_type = record.get('error_kind') or classify_error(status, detail, page_http)
            reason = ('Trang trả HTML xác minh/JavaScript thay vì nội dung bài viết.'
                      if record.get('challenge_detected') else STATUS_REASONS.get(status, status))
        requests_reason = reason
        if effective_engine == 'selenium':
            stage = 'done_browser'
            reason = ('Selenium tải trực tiếp. QC: ' if effective.get('browser_route') == 'selenium_first'
                      else 'Selenium cải thiện tải/trích xuất so với Requests. QC: ') + effective.get('quality_status', 'unknown')
            if access == 'KHONG_VAO_DUOC':
                stage, error_type = 'browser', effective.get('error_kind') or effective.get('status', 'browser_error')
                detail, reason = effective.get('error', ''), 'Selenium không lấy được bài viết.'
        rows.append({
            'domain': sample['domain'], 'dataset_url_count': SOURCE_PROFILE['host_counts'][sample['domain']],
            'sample_id': sample['id'], 'sample_url': sample['url'], 'access_result': access,
            'requests_access_result': requests_access, 'effective_engine': effective_engine,
            'browser_route': effective.get('browser_route', ''),
            'effective_http_status': effective.get('http_status'),
            'effective_quality_status': effective.get('quality_status', ''),
            'effective_char_count': effective.get('char_count'),
            'effective_quality_flags': '; '.join(effective.get('quality_flags', [])),
            'http_status': page_http, 'robots_http_status': robots.get('status'),
            'robots_attempt_count': robots.get('attempt_count'),
            'robots_cache_hit': robots.get('cache_hit'),
            'robots_fetch_id': robots.get('fetch_id', ''),
            'robots_error_kind': robots.get('error_kind', ''),
            'robots_elapsed_seconds': robots.get('elapsed_seconds'),
            'robots_lock_wait_seconds': robots.get('lock_wait_seconds'),
            'crawl_status': status, 'failure_stage': stage, 'error_type': error_type,
            'reason': reason, 'requests_reason': requests_reason,
            'error_detail': detail, 'quality_status': record.get('quality_status', ''),
            'quality_flags': '; '.join(flags), 'char_count': record.get('char_count'),
            'title': record.get('title', ''), 'final_url': record.get('final_url', ''),
            'elapsed_seconds': record.get('elapsed_seconds'),
            'reused_from': record.get('reused_from', ''),
            'request_attempt_count': len(record.get('requests', [])),
            'retryable': record.get('retryable'), 'retry_deferred': record.get('retry_deferred', ''),
            'page_attempted': bool(record.get('requests')),
            'browser_status': browser.get('status', ''),
            'browser_http_status': browser.get('http_status'),
            'browser_quality_status': browser.get('quality_status', ''),
            'browser_char_count': browser.get('char_count'),
            'browser_final_url': browser.get('final_url', ''),
            'browser_error': browser.get('error', ''),
            'browser_attempt_count': browser.get('attempt_count'),
            'sample_role': sample['sample_role'], 'sample_index': sample['sample_index'],
        })
    return rows

def domain_rows(rows):
    result = []
    for domain in sorted(BASELINE_DOMAINS, key=lambda h: (-SOURCE_PROFILE['host_counts'][h], h)):
        selected = [r for r in rows if r['domain'] == domain]
        failed = [r for r in selected if r['access_result'] == 'KHONG_VAO_DUOC']
        content = [r for r in selected if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted']
        pending = sum(r['access_result'] == 'CHUA_CHAY' for r in selected)
        errors = Counter(r['error_type'] for r in failed)
        errors.update('qc_' + (r['effective_quality_status'] or 'unknown') for r in content)
        result.append({
            'domain': domain, 'dataset_url_count': SOURCE_PROFILE['host_counts'][domain],
            'baseline_error_type': BASELINE_DOMAINS[domain]['error_type'],
            'selected_urls': len(selected), 'additional_urls': sum(r['sample_role'] == 'additional' for r in selected),
            'missing_additional_urls': BASELINE_DOMAINS[domain]['shortfall'],
            'completed_urls': len(selected) - pending, 'pending_urls': pending,
            'page_not_attempted': sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in selected),
            'accessible_urls': sum(r['access_result'] == 'VAO_DUOC' for r in selected),
            'access_failures': len(failed), 'content_issues': len(content),
            'total_problem_urls': len(failed) + len(content),
            'additional_problem_urls': sum(r['sample_role'] == 'additional' for r in failed + content),
            'error_types': '; '.join(f'{kind}: {n}' for kind, n in sorted(errors.items())),
            'test_status': 'CHUA_CHAY' if pending == len(selected) else 'CHUA_HOAN_TAT' if pending else 'CON_LOI' if failed or content else 'DA_PHUC_HOI',
        })
    return result

def write_csv(name, rows, fields):
    temporary = RUN_DIR / (name + '.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(RUN_DIR / name)

def save_outputs(show=True):
    rows = result_rows()
    fields = list(rows[0])
    counts = Counter(r['access_result'] for r in rows)
    summary = {'selected': len(rows), 'vao_duoc': counts['VAO_DUOC'],
               'khong_vao_duoc': counts['KHONG_VAO_DUOC'], 'chua_chay': counts['CHUA_CHAY'],
               'crawl_status_counts': dict(Counter(r['crawl_status'] for r in rows)),
               'quality_status_counts': dict(Counter(r['quality_status'] for r in rows if r['quality_status'])),
               'effective_quality_status_counts': dict(Counter(r['effective_quality_status'] for r in rows if r['effective_quality_status'])),
               'created_at': datetime.now(timezone.utc).isoformat()}
    domains = domain_rows(rows)
    priority = [dict(priority_rank=i, **r) for i, r in enumerate((r for r in domains if r['total_problem_urls']), 1)]
    summary.update(hostnames=len(domains), domains_with_errors=len(priority),
                   priority_order='dataset_url_count descending', user_agent=USER_AGENT,
                   page_not_attempted=sum(not r['page_attempted'] and r['access_result'] != 'CHUA_CHAY' for r in rows),
                   additional_urls=sum(r['sample_role'] == 'additional' for r in rows))
    summary.update(request_mode='direct_public_page_probe' if DIRECT_PUBLIC_PROBE else 'robots_enforced',
                   requests_http_2xx=sum(r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' for r in rows),
                   browser_access_recovered=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] != 'VAO_DUOC' and r['browser_route'] != 'selenium_first' for r in rows),
                   browser_content_improved=sum(r['access_result'] == 'VAO_DUOC' and r['effective_engine'] == 'selenium' and r['requests_access_result'] == 'VAO_DUOC' for r in rows),
                   browser_primary=sum(r['browser_route'] == 'selenium_first' for r in rows),
                   browser_tested=sum(bool(r['browser_status']) for r in rows),
                   browser_http_2xx=sum(r['browser_http_status'] is not None and 200 <= r['browser_http_status'] < 300 for r in rows))
    write_csv('tong_hop_domain.csv', domains, list(domains[0]))
    write_csv('uu_tien_fix_domain.csv', priority, ['priority_rank', *domains[0]])
    (RUN_DIR / 'uu_tien_fix_domain.json').write_text(json.dumps(priority, ensure_ascii=False, indent=2), encoding='utf-8')
    write_csv('ket_qua.csv', rows, fields)
    write_csv('requests_vs_selenium.csv', rows, fields)
    for value, filename in [('VAO_DUOC', 'vao_duoc.csv'), ('KHONG_VAO_DUOC', 'khong_vao_duoc.csv'), ('CHUA_CHAY', 'chua_chay.csv')]:
        write_csv(filename, [r for r in rows if r['access_result'] == value], fields)
    write_csv('can_xem_noi_dung.csv', [r for r in rows if r['access_result'] == 'VAO_DUOC' and r['effective_quality_status'] != 'accepted'], fields)
    failures = Counter(r['error_type'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC')
    errors = [{'error_type': kind, 'domain_count': len({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}),
               'url_count': n,
               'domains': '; '.join(sorted({r['domain'] for r in rows if r['access_result'] == 'KHONG_VAO_DUOC' and r['error_type'] == kind}))}
              for kind, n in failures.most_common()]
    write_csv('loi_theo_loai.csv', errors, ['error_type', 'domain_count', 'url_count', 'domains'])
    (RUN_DIR / 'ket_qua.json').write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
    (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    md = ['# Các website còn lỗi sau lượt test bổ sung', '',
          f"Thời điểm UTC: {summary['created_at']}. Run: `{RUN_DIR.name}`.", '',
          f"Đã chạy {len(rows)-counts['CHUA_CHAY']}/{len(rows)} URL thuộc {len(domains)} hostname. "
          f"Tải được sau Requests/Selenium: {counts['VAO_DUOC']}; chưa tải được: {counts['KHONG_VAO_DUOC']}; chưa chạy: {counts['CHUA_CHAY']}.", '',
          f"Requests tải được: {summary['requests_http_2xx']}; chọn kết quả Selenium: {summary['browser_recovered']} "
          f"(phục hồi truy cập: {summary['browser_access_recovered']}; cải thiện nội dung: {summary['browser_content_improved']}). "
          'Các cột lỗi Requests vẫn giữ nguyên dù Selenium phục hồi thành công.', '',
          f"{summary['page_not_attempted']} URL đã xử lý nhưng Requests chưa gửi tới bài viết (gồm PubMed dùng Selenium trực tiếp). "
          'Các trường hợp này được phân biệt với bài viết thực sự trả HTTP lỗi.', '',
          f'HTTP User-Agent: `{USER_AGENT}`.', '',
          f"Bộ mẫu có {len(rows)} URL thuộc {len(domains)} hostname, gồm {summary['additional_urls']} URL bổ sung. "
          'Gồm lỗi truy cập và lỗi nội dung/QC. Số URL dataset là số dòng của hostname, chưa loại URL trùng.', '',
          'Ưu tiên theo số URL dataset giảm dần; không suy ra toàn bộ URL của domain đều bị lỗi. '
          'Dừng ở robots là chưa request bài viết, không đồng nghĩa website không mở được trong trình duyệt.', '',
          '| Ưu tiên | Hostname | Số URL dataset | Mẫu đã chạy | Lỗi truy cập | Lỗi nội dung | Loại lỗi |',
          '|---:|---|---:|---:|---:|---:|---|']
    for item in priority:
        md.append(f"| {item['priority_rank']} | {item['domain']} | {item['dataset_url_count']:,} | {item['completed_urls']}/{item['selected_urls']} | {item['access_failures']} | {item['content_issues']} | {item['error_types']} |")
    md += ['', '## Domain đã phục hồi trong các mẫu', '']
    md += [f"- {r['domain']}: {r['accessible_urls']}/{r['selected_urls']} URL tải được và đạt QC." for r in domains if r['test_status'] == 'DA_PHUC_HOI'] or ['Chưa xác định domain phục hồi hoàn toàn.']
    md += ['', '## Domain thiếu URL bổ sung', '']
    md += [f"- {r['domain']}: lấy được {r['selected_urls']}/{SAMPLES_PER_DOMAIN} URL mới; dataset có {r['dataset_url_count']} dòng." for r in domains if r['missing_additional_urls']]
    md += ['', '## Chi tiết lỗi theo domain', '']
    for item in priority:
        md += [f"### {item['priority_rank']}. {item['domain']}", '',
               f"{item['dataset_url_count']:,} URL nguồn; {item['page_not_attempted']} mẫu đã xử lý nhưng chưa request bài viết. "
               f"Trạng thái trước lượt thử: `{item['baseline_error_type']}`.", '']
        for row in rows:
            if row['domain'] != item['domain'] or row['access_result'] == 'CHUA_CHAY' or (row['access_result'] == 'VAO_DUOC' and row['effective_quality_status'] == 'accepted'):
                continue
            detail = (row['error_detail'] or row['quality_flags']).replace('\n', ' ').replace('`', "'")
            md += [f"- ID {row['sample_id']} ({row['sample_role']}): [URL](<{row['sample_url']}>) — `{row['error_type']}`. "
                   f"HTTP trang: {row['http_status']}; HTTP robots: {row['robots_http_status']}. {row['reason']} Lỗi gốc: `{detail}`."]
        md.append('')
    (RUN_DIR / 'WEB_ERRORS.md').write_text('\n'.join(md), encoding='utf-8')

    visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'sample_role', 'access_result',
                      'http_status', 'robots_http_status', 'robots_attempt_count', 'robots_cache_hit', 'error_type', 'reason', 'error_detail',
                      'quality_status', 'quality_flags', 'char_count']
    labels = ['Domain', 'Số URL nguồn', 'ID mẫu', 'URL mẫu', 'Mẫu cũ/mới', 'Vào được?', 'HTTP trang',
              'HTTP robots', 'Lần thử robots', 'Cache hit robots', 'Loại lỗi', 'Lý do', 'Lỗi gốc', 'QC', 'Cờ QC', 'Số ký tự']
    if DIRECT_PUBLIC_PROBE:
        visible_fields = ['domain', 'dataset_url_count', 'sample_id', 'sample_url', 'http_status',
                          'error_type', 'error_detail', 'quality_status', 'char_count',
                          'browser_status', 'browser_http_status', 'browser_quality_status', 'browser_char_count', 'browser_error',
                          'access_result', 'effective_engine']
        labels = ['Domain', 'Số URL nguồn', 'ID', 'URL', 'HTTP Requests', 'Lỗi Requests', 'Chi tiết',
                  'QC Requests', 'Ký tự Requests', 'Selenium', 'HTTP Selenium', 'QC Selenium', 'Ký tự Selenium', 'Lỗi Selenium',
                  'Kết quả cuối', 'Engine tải được']
    def esc(value):
        return html.escape(str(value if value is not None else ''), quote=True)
    body = []
    for row in rows:
        cells = []
        for field in visible_fields:
            value = row[field]
            text = (f'<a href="{esc(value)}" target="_blank" rel="noopener noreferrer">{esc(value)}</a>'
                    if field == 'sample_url' else esc(value))
            cells.append('<td>' + text + '</td>')
        body.append(f'<tr data-access="{row["access_result"]}">' + ''.join(cells) + '</tr>')
    style = """<style>body{font:14px/1.5 system-ui,sans-serif;margin:20px;color:#172b4d}
    .table-wrap{overflow:auto;max-height:70vh;border:1px solid #ccd4df}table{border-collapse:collapse;width:100%}
    th,td{padding:9px;border-bottom:1px solid #dce2e9;text-align:left;vertical-align:top}th{background:#e8eef6;position:sticky;top:0}
    td{min-width:95px;max-width:330px;overflow-wrap:anywhere}td:nth-child(4),td:nth-child(9),td:nth-child(10){min-width:260px}
    input,select{padding:8px;margin:0 10px 14px 0}a{color:#1654a0}</style>"""
    table = '<div class="table-wrap"><table><thead><tr>' + ''.join('<th>' + esc(label) + '</th>' for label in labels)
    table += '</tr></thead><tbody>' + ''.join(body) + '</tbody></table></div>'
    heading = f'<h1>Kết quả thử crawl {len(rows)} URL / {len(domains)} hostname</h1><p>Vào được: {counts["VAO_DUOC"]} · Không vào được: {counts["KHONG_VAO_DUOC"]} · Chưa chạy: {counts["CHUA_CHAY"]}</p>'
    heading += '<p>Vào được tính cả Requests và Selenium phục hồi; trang challenge được loại. Các cột Requests vẫn giữ lỗi gốc, QC là kết quả riêng. Kết quả chỉ áp dụng cho các URL mẫu.</p>'
    controls = '<input id="search" placeholder="Tìm domain, URL hoặc lỗi" aria-label="Tìm kiếm"><select id="access" aria-label="Kết quả"><option value="">Tất cả</option><option>VAO_DUOC</option><option>KHONG_VAO_DUOC</option><option>CHUA_CHAY</option></select><span id="count"></span>'
    priority_fields = ['priority_rank', 'domain', 'dataset_url_count', 'selected_urls', 'completed_urls', 'pending_urls', 'access_failures', 'content_issues', 'error_types']
    priority_labels = ['Ưu tiên', 'Domain', 'Số URL dataset', 'Mẫu', 'Đã chạy', 'Chưa chạy', 'Lỗi truy cập', 'Lỗi nội dung', 'Loại lỗi']
    priority_heading = '<h2>Domain còn lỗi — số URL dataset giảm dần</h2><p>Gồm lỗi truy cập và lỗi nội dung/QC. Số URL nguồn không phải số URL đã xác minh lỗi.</p>'
    priority_table = '<div class="table-wrap"><table id="priority"><thead><tr>' + ''.join('<th>'+esc(label)+'</th>' for label in priority_labels)
    priority_table += '</tr></thead><tbody>' + ''.join('<tr>' + ''.join('<td>'+esc(r[f])+'</td>' for f in priority_fields) + '</tr>' for r in priority) + '</tbody></table></div>'
    (RUN_DIR / 'uu_tien_fix_domain.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><title>Ưu tiên domain lỗi</title>'+style+priority_heading+priority_table+'</html>', encoding='utf-8')
    javascript = """<script>const rs=[...document.querySelectorAll('tbody tr')],s=document.getElementById('search'),a=document.getElementById('access');
    function filter(){let n=0;for(const r of rs){const ok=r.textContent.toLowerCase().includes(s.value.toLowerCase())&&(!a.value||r.dataset.access===a.value);r.hidden=!ok;n+=ok?1:0}document.getElementById('count').textContent=n+' hostname'}
    s.addEventListener('input',filter);a.addEventListener('change',filter);filter();</script>"""
    javascript = javascript.replace("document.querySelectorAll('tbody tr')", "document.querySelectorAll('tr[data-access]')").replace("n+' hostname'", "n+' URL'")
    (RUN_DIR / 'ket_qua.html').write_text('<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Kết quả crawl domain</title>' + style + heading + priority_heading + priority_table + '<h2>Từng URL mẫu</h2>' + controls + table + javascript + '</html>', encoding='utf-8')
    if IP_CHECK_ENABLED:
        summary['ip_diagnostics'] = IP_DIAGNOSTICS.build_report(crawler, RUN_DIR, SAMPLES, SOURCE_PROFILE)
        (RUN_DIR / 'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    # Archive outside the run directory to avoid including the archive itself.
    zip_path = Path(shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name))
    if show:
        print(json.dumps(summary, ensure_ascii=False, indent=2))
        display(HTML(style + priority_heading + priority_table))
        display(HTML(style + heading + table))
        print('File kết quả:', RUN_DIR)
        print('ZIP:', zip_path)
    return rows, summary, zip_path

# Create reports before networking, so partial runs have explicit CHUA_CHAY rows.
rows, summary, zip_path = save_outputs(show=False)
print('Đã chuẩn bị báo cáo và phân loại lỗi.')


## 5. Chẩn đoán mạng, crawl và đối chiếu Selenium

IP check đo Requests/Chrome qua ipify; ghi riêng lỗi dịch vụ đo IP, khác biệt IP và proxy.
Mỗi hostname được tra DNS và thử homepage một lần, có timeout/ngân sách hữu hạn,
không retry liên tục khi gặp 403/429. Lỗi kiểm tra IP không ngăn crawler thử trang mẫu.

PubMed đi Selenium trực tiếp; các hostname khác dùng engine/cấu hình chung với notebook corpus.
Hostname xử lý tuần tự, giới hạn tốc độ theo registered domain. Selenium fallback cho lỗi
HTTP/mạng/timeout/challenge và QC review/rejected. `off` tắt browser.
Ngắt cell giữ các checkpoint đã hoàn tất; chạy lại ô crawl để tiếp tục phần còn thiếu.

`ip_block_report.csv/html`, `IP_BLOCK_REPORT.md` và `ip_url_evidence.csv` được cập nhật
khi xuất báo cáo, kể cả ở finally sau gián đoạn. Chạy ô cuối để tạo lại báo cáo từ raw/checkpoint.
HTTP 2xx không đồng nghĩa nội dung đạt QC; 20 URL không đại diện cho toàn bộ hostname.


In [ ]:
import argparse
import os
import socket
import time

try:
    if IP_CHECK_ENABLED:
        network = IP_DIAGNOSTICS.collect_network(
            crawler, priority_probe, RUN_DIR, NETWORK_CHECK_DOMAINS, user_agent=USER_AGENT,
            browser_user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY,
            browser_ip=IP_BROWSER_EGRESS_CHECK and BROWSER_TEST_MODE != 'off', homepage=IP_HOMEPAGE_CHECK,
            workers=WORKERS, timeout=IP_CHECK_TIMEOUT, budget=IP_CHECK_BUDGET, force=IP_RECHECK_ON_RESUME)
        print('IP Requests:', network.get('requests_ips', []), '; IP Selenium:', network.get('selenium_ips', []))
        rows, summary, zip_path = save_outputs(show=False)
    if DIRECT_PUBLIC_PROBE:
        if BROWSER_TEST_MODE != 'off':
            priority_probe.browser_preflight(crawler, RUN_DIR, user_agent=BROWSER_USER_AGENT,
                                             chrome_binary=CHROME_BINARY, load_timeout=BROWSER_LOAD_TIMEOUT)
        if RECOVERY_RUN_DIR:
            recovery = priority_probe.seed_successful_requests(crawler, RUN_DIR, SAMPLES, RECOVERY_RUN_DIR)
            print('Recovery:', len(recovery['reused_ids']), 'URL reuse raw;', len(recovery['retry_ids']), 'URL thử lại.')
        primary_enabled = PUBMED_SELENIUM_FIRST and BROWSER_TEST_MODE != 'off'
        primary, primary_new = priority_probe.prepare_pubmed_selenium(
            crawler, RUN_DIR, SAMPLES, enabled=primary_enabled, max_new=MAX_NEW)
        if primary:
            priority_probe.crawl_browser(
                crawler, RUN_DIR, primary, user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY,
                delay=DELAY, load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS, log_every=LOG_EVERY)
            if any(not (RUN_DIR / 'browser_records' / f"{s['id']}.json").exists() for s in primary):
                raise RuntimeError('PubMed Selenium chưa hoàn tất; xem browser_setup_error.json và resume.')
        request_samples = [s for s in SAMPLES if not primary_enabled
                           or crawler.host(s['url']) != 'pubmed.ncbi.nlm.nih.gov']
        request_limit = None if MAX_NEW is None else MAX_NEW - primary_new
        if request_limit != 0:
            priority_probe.crawl_requests(
                crawler, RUN_DIR, request_samples, workers=WORKERS, delay=DELAY,
                attempts=PAGE_ATTEMPTS, user_agent=USER_AGENT,
                timeout=(PAGE_CONNECT_TIMEOUT, PAGE_READ_TIMEOUT), budget=PAGE_REQUEST_BUDGET, max_new=request_limit, log_every=LOG_EVERY)
        records = [crawler.load_json(RUN_DIR / 'records' / f"{s['id']}.json") for s in request_samples
                   if (RUN_DIR / 'records' / f"{s['id']}.json").exists()]
        browser_samples = priority_probe.select_browser_samples(request_samples, records, BROWSER_TEST_MODE)
        if browser_samples:
            priority_probe.crawl_browser(crawler, RUN_DIR, browser_samples,
                                        user_agent=BROWSER_USER_AGENT, chrome_binary=CHROME_BINARY, delay=DELAY,
                                        load_timeout=BROWSER_LOAD_TIMEOUT, attempts=BROWSER_ATTEMPTS, log_every=LOG_EVERY)
        crawler.report(argparse.Namespace(out=str(RUN_DIR)))
    else:
        crawler.locked_crawl(argparse.Namespace(
            out=str(RUN_DIR), workers=WORKERS, delay=DELAY, attempts=ATTEMPTS, max_new=MAX_NEW,
            log_every=LOG_EVERY,
            robots_attempts=ROBOTS_ATTEMPTS, robots_connect_timeout=ROBOTS_CONNECT_TIMEOUT,
            robots_read_timeout=ROBOTS_READ_TIMEOUT, robots_retry_budget=ROBOTS_RETRY_BUDGET,
            robots_failure_ttl=ROBOTS_FAILURE_TTL,
        ))
finally:
    rows, summary, zip_path = save_outputs()


## 6. Xem bằng chứng và tải output

Mở **`ip_block_report.html`** để xem IP, kết luận từng hostname, số 403/429 và homepage đối chứng.
`ip_url_evidence.csv` ghi từng ID/URL/engine, phân loại, đoạn thông báo và đường dẫn raw/screenshot.
`network_check.json` ghi thời điểm, IP Requests/Chrome, DNS và việc có biến proxy (không ghi giá trị proxy).
`network_controls/` giữ raw/redirect/header của homepage và các lượt đo IP.

Không suy từ 403 thành chặn IP. Nếu trang ghi rõ IP/network bị chặn, đối chiếu thông báo với IP
đã đo; IP có thể khác theo proxy hoặc đích. Các lỗi parser, thiếu abstract, DNS, SSL, 404 và lỗi Chrome
được phân biệt với bằng chứng chặn truy cập.

ZIP chứa mẫu, snapshot, kết quả, raw và screenshot, gồm cả output dở dang.
Kaggle: tải ZIP trong Output rồi gửi lại để đối chiếu; có thể chạy cùng notebook local làm đối chứng.


In [ ]:
# Có thể chạy lại ô này để xuất báo cáo từ checkpoint mà không crawl thêm.
rows, summary, zip_path = save_outputs(show=False)
failed_rows = [r for r in rows if r['access_result'] == 'KHONG_VAO_DUOC']
print(f'Không vào được: {len(failed_rows)}/{len(rows)} URL, thuộc {len({r["domain"] for r in failed_rows})} hostname')
for row in failed_rows:
    print(f"{row['domain']} | {row['error_type']} | {row['error_detail']}")
print('\nOutput:', RUN_DIR)
for name in ['IP_BLOCK_REPORT.md', 'ip_block_report.html', 'ip_block_report.csv', 'ip_url_evidence.csv', 'network_check.json', 'WEB_ERRORS.md', 'uu_tien_fix_domain.html', 'uu_tien_fix_domain.csv', 'tong_hop_domain.csv',
             'requests_vs_selenium.csv',
             'ket_qua.html', 'ket_qua.csv', 'vao_duoc.csv', 'khong_vao_duoc.csv',
             'loi_theo_loai.csv', 'can_xem_noi_dung.csv', 'chua_chay.csv']:
    if (RUN_DIR / name).exists():
        display(FileLink(str(RUN_DIR / name)))
display(FileLink(str(zip_path)))

if 'google.colab' in sys.modules:
    import ipywidgets as widgets
    from google.colab import files
    button = widgets.Button(description='Tải ZIP kết quả')
    button.on_click(lambda _: files.download(str(zip_path)))
    display(button)
